In [ ]:
import marimo as mo

mo.md(
    r"""
    # ECA-3WD: certified three-way authentication of AI-generated images

    Complete code for every number, table and figure in the paper and its supplement: data,
    training, certification, calibration, attacks, analyses and paper figures. No LLMs or agents
    run anywhere in this pipeline. Labels: `y = 1` fake, `y = 0` real. Scores are **realness**:
    larger means more authentic.

    | part | cells | what | needs |
    |---|---|---|---|
    | **I. Pipeline** | S0–S12 | theory self-tests, CommunityForensics / OpenFake / RAID data, training of BASE, PGD-AT, CLIP probe, Cohen and ECA-3WD (ADRT), baselines and gates, attacks, certification, CERT-CAL, evidence floor, sequential certification, report | GPU; run through the RUNNER cell |
    | **II. Corrected calibration** | F1–F7 | inventory, gap filling, framing threshold re-calibrated on S_lo of reals, (alpha, beta) grid, corrected S9–S11, overall comparison | CPU, then GPU for F5 |
    | **III. Analyses and new experiments** | A1–A11, B1–B6, C1–C9, C1b, C6b–C6d | feasibility frontier, guarantee validity over re-splits, certificate audit, bootstrap intervals, three-way cost, OpenFake, source mass, defended ROC, stress test, larger N, extra seeds, ADRT ablations | CPU; GPU cells are resumable |
    | **IV. Paper** | FIG, R1 | paper figures; frozen selection / final-calibration procedure, Table 2 ablations on the official split, flag/defer baseline, finite-N bound | CPU |

    **How to run.** Optional environment variables (set them as notebook secrets, never in the
    code): `HF_TOKEN` and `HF_RESULTS_REPO` mirror results and weights to a private Hugging Face
    repo so that a session that ends can resume; `ALLOW_TRAIN=1` unlocks training in S5;
    `RUN_STAGES=all` (or a comma-separated list such as `S6,S8`) runs Part I without the UI;
    `SMOKE=1` and `V3_SMOKE=1` use tiny sizes for testing. Every stage skips work whose outputs
    already exist locally or in the results repo, so nothing that has finished is run again.

    **Hardware used.** One RTX 6000 Ada; peak RAM stays below 30 GB.
    """
)

In [ ]:
# ---- 1. SETUP: environment, imports, budgets, helpers --------------------------------
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import gc
import io
import json
import math
import time
import ctypes
import shutil
import hashlib
import contextlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

try:
    import psutil
except ImportError:
    psutil = None
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

import torch
import torch.nn as nn
import torch.nn.functional as F

RAM_CAP_GB = float(os.environ.get("RAM_CAP_GB", "30"))
RAM_SOFT_GB = RAM_CAP_GB - 4.0
VRAM_CAP_GB = float(os.environ.get("VRAM_CAP_GB", "70"))
START_TIME = time.time()

def rss_gb(include_children=True):
    """Resident memory of this process and its workers, in GB."""
    if psutil is not None:
        _p = psutil.Process()
        _tot = _p.memory_info().rss
        if include_children:
            for _c in _p.children(recursive=True):
                try:
                    _tot += _c.memory_info().rss
                except psutil.Error:
                    pass
        return _tot / 2**30
    with open("/proc/self/status") as _f:
        for _line in _f:
            if _line.startswith("VmRSS:"):
                return int(_line.split()[1]) / 2**20
    return float("nan")

def ram_guard(extra_gb, what="allocation"):
    """Raise a readable error before an allocation that would cross the soft RAM limit."""
    _used = rss_gb()
    if _used + extra_gb > RAM_SOFT_GB:
        raise MemoryError(
            f"{what}: need ~{extra_gb:.1f} GB on top of {_used:.1f} GB; soft limit "
            f"{RAM_SOFT_GB:.0f} GB (hard cap {RAM_CAP_GB:.0f} GB). Chunk or memory-map it."
        )

def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass
    return rss_gb()

def gpu_setup():
    if not torch.cuda.is_available():
        return {"device": "cpu", "torch": torch.__version__}
    _free, _total = torch.cuda.mem_get_info()
    _total_gb = _total / 2**30
    _cap = min(VRAM_CAP_GB, _total_gb * 0.95)
    torch.cuda.set_per_process_memory_fraction(min(1.0, _cap / _total_gb))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    return {
        "device": "cuda",
        "name": torch.cuda.get_device_name(0),
        "total_gb": round(_total_gb, 1),
        "cap_gb": round(_cap, 1),
        "torch": torch.__version__,
    }

def gpu_free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

def vram_report(reset_peak=False):
    if not torch.cuda.is_available():
        return {}
    _out = {
        "allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
        "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    }
    if reset_peak:
        torch.cuda.reset_peak_memory_stats()
    return _out

def atomic_write(path, writer):
    """writer(tmp) writes the file; it is renamed into place only when complete."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    _base, _ext = os.path.splitext(path)
    _tmp = f"{_base}.tmp{_ext}"
    writer(_tmp)
    os.replace(_tmp, path)

def cfg_hash(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]

def amp_ctx(device):
    """bf16 autocast on GPU; plain fp32 on CPU."""
    if device == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()

In [ ]:
# ---- 2. HARDWARE REPORT ----------------------------------------------------------------
HW = gpu_setup()
DEVICE = HW["device"]
_du = shutil.disk_usage(".")
mo.md(
    f"**Hardware:** `{HW}`  \n**RAM in use:** {rss_gb():.2f} GB · "
    f"**disk free:** {_du.free / 2**30:.0f} GB · **CPUs:** {os.cpu_count()}"
)

In [ ]:
# ---- 3. CONFIG: every size, count and path in one place -----------------------------
SMOKE = os.environ.get("SMOKE", "0") == "1"
_work = os.environ.get("ECA_WORK", "eca_work_smoke" if SMOKE else "eca_work")
CFG = {
    "seed": 0,
    "smoke": SMOKE,
    # Training lock: S5 (the multi-hour training) only runs when ALLOW_TRAIN=1 is set,
    # or in SMOKE mode (tiny). Without it, S5 is skipped and later stages use whatever
    # weights already exist locally or in the HF results repo.
    "allow_train": SMOKE or os.environ.get("ALLOW_TRAIN", "0") == "1",
    "work": _work,
    "deadline_h": 11.0,
    "hf": {
        "results_repo": os.environ.get("HF_RESULTS_REPO", ""),
        "sync": os.environ.get("HF_SYNC", "1") == "1",
        "sync_cache": os.environ.get("HF_SYNC_CACHE", "0") == "1",
        "private": True,
    },
    "cf": {
        "repo": "OwensLab/CommunityForensics-Small",
        # None = index every shard (~2-5 min, metadata columns only).
        "index_shards": [0, 117] if SMOKE else None,
        "n_fake_shards": 1 if SMOKE else 16,
        "n_manual_shards": 0 if SMOKE else 2,
        "real_shards": {"COCO": 1} if SMOKE else {"FFHQ": 3, "COCO": 8, "LandscapesHQ": 4},
        "max_rows_per_shard": 40 if SMOKE else None,
        "split_frac": [0.6, 0.2, 0.2],
    },
    "of": {
        "repo": "ComplexDataLab/OpenFake",
        "shards": [0] if SMOKE else [0, 1, 2],
        "per_model": 4 if SMOKE else 200,
        "n_real": 12 if SMOKE else 2400,
        "max_row_groups": 1 if SMOKE else 60,
    },
    "raid": {
        "repo": "aimagelab/RAID",
        "n_per_folder": 3 if SMOKE else 400,
        "budgets": ["original", "epsilon8", "epsilon16", "epsilon32"],
        "folders": ["real", "gen_0", "gen_1", "gen_2", "gen_3"],
    },
    "img": {"store": 256, "crop": 224, "max_short": 512},
    "model": {
        "backbone": "convnext_atto.d2_in1k" if SMOKE else "convnext_tiny.fb_in22k",
        "drop_path": 0.1,
        "clip": "openai/clip-vit-base-patch32" if SMOKE else "openai/clip-vit-large-patch14",
    },
    "train": {
        "epochs": 1 if SMOKE else 6,
        "batch": 8 if SMOKE else 64,
        "lr": 1e-4,
        "wd": 0.05,
        "warmup_frac": 0.05,
        "k_noise": 2 if SMOKE else 4,
        "workers": 0 if SMOKE else min(6, max(1, (os.cpu_count() or 2) - 2)),
        "max_steps": 3 if SMOKE else None,
        "pgd_at_eps": 2 / 255,
        "pgd_at_steps": 2 if SMOKE else 3,
        "probe_epochs": 5 if SMOKE else 100,
    },
    "eca": {
        # ECA-3WD at three noise levels; sigma=0.06 is the safety net if heavier noise
        # erases forensic evidence (RQ3). Cohen (symmetric, no ADRT) is the ablation.
        "sigmas": [0.25] if SMOKE else [0.06, 0.12, 0.25],
        "cohen_sigmas": [0.25] if SMOKE else [0.12],
        "mu": 0.5,
        "lam_af": 4.0,
        "lam_fr": 1.0,
        "mF": 1.5,   # margins and temperature in units of sigma
        "mR": 0.5,
        "T": 0.1,
        "gamma": 1e-3,
    },
    "cert": {
        "N": 16 if SMOKE else 256,
        "alpha_mc": 1e-3,
        "img_batch": 4 if SMOKE else 8,
        "chunk": 32 if SMOKE else 1024,
        "eps_cert": [0.0, 0.06, 0.12, 0.25, 0.5],
        # None => certified budget eps_c = sigma for each model (the eps/sigma = 1 design
        # point of Proposition 6); set a number to force one budget for all models.
        "main_eps": None,
    },
    "cal": {
        "beta": 0.2 if SMOKE else 0.01,      # framing rate on reals
        "alpha": 0.5 if SMOKE else 0.01,     # expected accepted-fake rate over new generators (CRC)
        "alpha_q": 0.5 if SMOKE else 0.10,   # per-generator level for the quantile guarantee
        "eta": 0.05,
        "delta": 0.05,
        "min_per_gen": 1 if SMOKE else 20,
        "gate_keep": 0.95,
        "maha_fit_n": 64 if SMOKE else 6000,
    },
    "attack": {
        "n": 4 if SMOKE else 500,
        "batch": 4 if SMOKE else 50,
        "steps": 2 if SMOKE else 50,
        "eot_steps": 2 if SMOKE else 20,
        "eot_m": 2 if SMOKE else 16,
        "mc_eot": 2 if SMOKE else 4,
        "mc_eval": 2 if SMOKE else 8,
        "linf": [1 / 255, 2 / 255, 4 / 255, 8 / 255],
        "l2": [0.06, 0.12, 0.25, 0.5, 1.0, 2.0],
    },
    "floor": {"n": 4 if SMOKE else 512, "vae": "stabilityai/sd-vae-ft-mse"},
    "seq": {"n": 4 if SMOKE else 200, "block": 8 if SMOKE else 32, "n_max": 32 if SMOKE else 512, "n_saliency": 2 if SMOKE else 6},
}
DATA_TAG = cfg_hash([CFG["seed"], CFG["cf"], CFG["of"], CFG["raid"], CFG["img"]])
MODEL_TAG = cfg_hash([DATA_TAG, CFG["model"], CFG["train"], CFG["eca"]])
RES_TAG = cfg_hash([MODEL_TAG, CFG["cert"], CFG["cal"], CFG["attack"], CFG["floor"], CFG["seq"]])
# Folder names hash the config, and the config includes a machine-dependent value
# (data-loader workers = CPU count), so a resumed session on another machine computes a
# different hash. Existing folders are therefore reused: locally under the work folder,
# otherwise in the HF results repo. If several exist, the one with the most files wins
# (ties: most recent); if none exists, the hashed names are used.
def _hub_dirs():
    _repo, _tok = CFG["hf"]["results_repo"], os.environ.get("HF_TOKEN") or None
    if not (_repo and _tok):
        return {}
    try:
        from huggingface_hub import HfApi as _HfApi

        _cnt = {}
        for _f in _HfApi(token=_tok).list_repo_files(_repo, repo_type="model"):
            _top = _f.split("/", 1)[0]
            _cnt[_top] = _cnt.get(_top, 0) + 1
        return _cnt
    except Exception:
        return {}

_hub = _hub_dirs()

def _find(prefix, fallback):
    _loc = []
    if os.path.isdir(_work):
        for _d in os.listdir(_work):
            _full = os.path.join(_work, _d)
            if _d.startswith(prefix + "_") and os.path.isdir(_full):
                _n = sum(len(_fs) for _, _, _fs in os.walk(_full))
                _loc.append((_n, os.path.getmtime(_full), _d))
    _loc = [x for x in _loc if x[0] > 0]
    if _loc:
        return max(_loc)[2], "local"
    _h = [(n, d) for d, n in _hub.items() if d.startswith(prefix + "_")]
    if _h:
        return max(_h)[1], "hub"
    return fallback, "not found"

_found = {k: _find(k, f"{k}_{t}") for k, t in [("data", DATA_TAG), ("models", MODEL_TAG), ("results", RES_TAG)]}
PATHS = {
    "data": os.path.join(_work, _found["data"][0]),
    "models": os.path.join(_work, _found["models"][0]),
    "res": os.path.join(_work, _found["results"][0]),
    "dl": os.path.join(_work, "_downloads"),
}
for _p in PATHS.values():
    os.makedirs(_p, exist_ok=True)
{"SMOKE": SMOKE, **{k: f"{v[0]} ({v[1]})" for k, v in _found.items()}}

In [ ]:
# ---- 4. HUGGING FACE SYNC + LOGGING ---------------------------------------------------
# Results and checkpoints are mirrored to a private model repo so that a session that
# ends (molab keeps only file-browser files) can resume. Token comes from secrets only.
from huggingface_hub import HfApi, hf_hub_download, snapshot_download, HfFileSystem

HF_TOKEN = os.environ.get("HF_TOKEN") or None
_repo = CFG["hf"]["results_repo"]
HF_ON = bool(HF_TOKEN and _repo and CFG["hf"]["sync"])
HF_API = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
if HF_ON:
    try:
        HF_API.create_repo(_repo, repo_type="model", private=CFG["hf"]["private"], exist_ok=True)
    except Exception as _e:  # never print the token
        print(f"[hf] could not create/access repo {_repo}: {type(_e).__name__}; sync disabled")
        HF_ON = False

def log(msg):
    _line = (
        f"[{(time.time() - START_TIME) / 60:7.1f} min | RAM {rss_gb():5.1f} GB | "
        f"VRAM {vram_report().get('peak_gb', 0):5.1f} GB] {msg}"
    )
    print(_line, flush=True)
    with open(os.path.join(CFG["work"], "progress.log"), "a") as _f:
        _f.write(_line + "\n")

def _rel(path):
    return os.path.relpath(path, CFG["work"]).replace(os.sep, "/")

def hf_push(path, message=None):
    """Upload one file or a folder (non-recursive patterns) to the results repo."""
    if not HF_ON or not os.path.exists(path):
        return False
    try:
        if os.path.isdir(path):
            HF_API.upload_folder(
                folder_path=path, path_in_repo=_rel(path), repo_id=_repo, repo_type="model",
                allow_patterns=["*.json", "*.parquet", "*.npz", "*.png", "*.csv", "*.md", "*.pt"],
                commit_message=message or f"sync {_rel(path)}",
            )
        else:
            HF_API.upload_file(
                path_or_fileobj=path, path_in_repo=_rel(path), repo_id=_repo, repo_type="model",
                commit_message=message or f"sync {_rel(path)}",
            )
        return True
    except Exception as _e:
        log(f"[hf] push failed for {_rel(path)}: {type(_e).__name__}")
        return False

def ensure(path):
    """True if `path` exists locally, pulling it from the results repo when possible."""
    if os.path.exists(path):
        return True
    if not HF_ON:
        return False
    try:
        hf_hub_download(_repo, _rel(path), repo_type="model", local_dir=CFG["work"], token=HF_TOKEN)
        return os.path.exists(path)
    except Exception:
        return False

def time_left_h():
    return CFG["deadline_h"] - (time.time() - START_TIME) / 3600

{"hf_sync": HF_ON, "repo": _repo or "(unset)", "paths": PATHS}

In [ ]:
# ---- 5. MEMORY PLAN (arithmetic before allocation) ------------------------------------
_S = CFG["img"]["store"]
_per = _S * _S * 3
_shards = CFG["cf"]["n_fake_shards"] + CFG["cf"]["n_manual_shards"] + sum(CFG["cf"]["real_shards"].values())
_n_cf = _shards * (CFG["cf"]["max_rows_per_shard"] or 2992)
_cache_gb = _n_cf * _per / 2**30
_shard_gb = 4.2  # largest CommunityForensics-Small shard (FFHQ)
_b = CFG["train"]["batch"] * CFG["train"]["k_noise"]
mo.md(
    f"""
    | quantity | estimate | budget |
    |---|---|---|
    | crop cache on disk ({_n_cf:,} × {_S}² × 3 uint8, memory-mapped) | {_cache_gb:.1f} GB disk | not in RAM |
    | one shard while cropping (Arrow, memory-mapped) + decode buffers | ≤ {_shard_gb + 1.5:.1f} GB RAM | RAM 30 GB |
    | training loader ({CFG['train']['workers']} fork workers sharing memmaps) | ≈ {0.4 * CFG['train']['workers'] + 2:.1f} GB RAM | RAM 30 GB |
    | ConvNeXt-T step, {_b} noisy 224² images, bf16 | ≈ 18–25 GB VRAM | VRAM 70 GB |
    | EOT attack, {CFG['attack']['batch']}×{CFG['attack']['eot_m']} images with grad | ≈ 20–35 GB VRAM | VRAM 70 GB |
    | certification chunk {CFG['cert']['chunk']} images, no grad | ≈ 6 GB VRAM | VRAM 70 GB |

    Data dir: `{PATHS['data']}` · models: `{PATHS['models']}` · results: `{PATHS['res']}`
    """
)

In [ ]:
# PIL lives in its own tiny cell so the library cell above can name it as a dependency.
from PIL import Image, ImageFilter

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
# ---- 6. LIBRARY: images, chunk store, splits ------------------------------------------
def stable_u(key):
    """Deterministic number in [0, 1) from a string (used for splits and crops)."""
    return int(hashlib.sha1(key.encode()).hexdigest()[:12], 16) / 16**12

def split_of(key, frac):
    _u = stable_u(key)
    if _u < frac[0]:
        return "train"
    if _u < frac[0] + frac[1]:
        return "cal"
    return "test"

def decode_rgb(b):
    _im = Image.open(io.BytesIO(b))
    _im.load()
    return _im.convert("RGB")

def normalise_scale(im, max_short, store):
    """Same rule for both classes: short side > max_short is downscaled to max_short;
    short side < store is upscaled to store (rare). Kills the scale shortcut."""
    _w, _h = im.size
    _s = min(_w, _h)
    if _s > max_short:
        _r = max_short / _s
        im = im.resize((max(store, round(_w * _r)), max(store, round(_h * _r))), Image.BICUBIC)
    _w, _h = im.size
    _s = min(_w, _h)
    if _s < store:
        _r = store / _s
        im = im.resize((math.ceil(_w * _r), math.ceil(_h * _r)), Image.BICUBIC)
    return im

def crop_store(im, store, mode, key):
    _w, _h = im.size
    if mode == "center":
        _x0, _y0 = (_w - store) // 2, (_h - store) // 2
    else:
        _u, _v = stable_u(key + "x"), stable_u(key + "y")
        _x0, _y0 = int(_u * (_w - store + 1)), int(_v * (_h - store + 1))
    return np.asarray(im.crop((_x0, _y0, _x0 + store, _y0 + store)), dtype=np.uint8)

def prep_image(b, cfg_img, mode, key):
    _im = normalise_scale(decode_rgb(b), cfg_img["max_short"], cfg_img["store"])
    return crop_store(_im, cfg_img["store"], mode, key)

def save_chunk(path_npy, arrs, meta, store):
    _a = np.stack(arrs) if len(arrs) else np.zeros((0, store, store, 3), np.uint8)
    _base, _ = os.path.splitext(path_npy)
    _tmp = f"{_base}.tmp.npy"
    np.save(_tmp, _a)
    meta.reset_index(drop=True).to_parquet(_base + ".tmp.parquet")
    os.replace(_base + ".tmp.parquet", _base + ".parquet")
    os.replace(_tmp, path_npy)

def chunk_done(path_npy):
    return os.path.exists(path_npy) and os.path.exists(os.path.splitext(path_npy)[0] + ".parquet")

class Store:
    """Images in many .npy chunks (memory-mapped, opened lazily so forked workers share
    pages) plus one concatenated metadata table. Index = row of `meta`."""

    def __init__(self, files):
        self.files = sorted(files)
        self._arrs = None
        _metas = []
        for _ci, _f in enumerate(self.files):
            _m = pd.read_parquet(os.path.splitext(_f)[0] + ".parquet")
            _m["_c"] = _ci
            _m["_l"] = np.arange(len(_m))
            _metas.append(_m)
        self.meta = pd.concat(_metas, ignore_index=True) if _metas else pd.DataFrame()
        self._c = self.meta["_c"].to_numpy() if len(self.meta) else np.zeros(0, int)
        self._l = self.meta["_l"].to_numpy() if len(self.meta) else np.zeros(0, int)

    def __len__(self):
        return len(self.meta)

    def _open(self):
        if self._arrs is None:
            self._arrs = [np.load(_f, mmap_mode="r") for _f in self.files]

    def get(self, i):
        self._open()
        return np.asarray(self._arrs[self._c[i]][self._l[i]])

    def batch(self, idx):
        return np.stack([self.get(int(_i)) for _i in idx])

    def __getstate__(self):
        _d = dict(self.__dict__)
        _d["_arrs"] = None
        return _d

def open_store(dirpath, prefix):
    if not os.path.isdir(dirpath):
        return Store([])
    _fs = [os.path.join(dirpath, _f) for _f in os.listdir(dirpath)
           if _f.startswith(prefix) and _f.endswith(".npy") and ".tmp" not in _f]
    _fs = [_f for _f in _fs if chunk_done(_f)]
    return Store(_fs)

def center224(a, store=256, crop=224):
    _o = (store - crop) // 2
    return a[..., _o:_o + crop, _o:_o + crop, :]

def to_tensor01(batch_u8, device):
    """uint8 NHWC numpy -> float NCHW in [0,1] on device."""
    import torch as _t

    return _t.from_numpy(np.ascontiguousarray(batch_u8)).to(device).permute(0, 3, 1, 2).float().div_(255.0)

def save_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w") as _f:
        json.dump(obj, _f, indent=1, default=float)
    os.replace(path + ".tmp", path)

def load_json(path):
    with open(path) as _f:
        return json.load(_f)

In [ ]:
# ---- 7. LIBRARY: models, augmentation, datasets ----------------------------------------
import timm
from timm.layers import DropPath

class Detector(nn.Module):
    """Real-vs-synthetic scorer. Input x in [0,1] (NCHW). Output: realness logit."""

    def __init__(self, backbone, drop_path=0.1, pretrained=True):
        super().__init__()
        self.net = timm.create_model(backbone, pretrained=pretrained, num_classes=1, drop_path_rate=drop_path)
        _cfg = self.net.pretrained_cfg
        self.register_buffer("mean", torch.tensor(_cfg.get("mean", (0.485, 0.456, 0.406))).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(_cfg.get("std", (0.229, 0.224, 0.225))).view(1, 3, 1, 1))

    def forward_both(self, x):
        _f = self.net.forward_features((x - self.mean) / self.std)
        _pre = self.net.forward_head(_f, pre_logits=True)
        return self.net.get_classifier()(_pre).squeeze(1), _pre

    def forward(self, x):
        return self.forward_both(x)[0]

def set_mc(model, on):
    """Stochastic depth active at eval time (MC-dropout gate)."""
    for _m in model.modules():
        if isinstance(_m, DropPath):
            _m.train(on)

class ClipProbe(nn.Module):
    """UnivFD-style baseline: frozen CLIP image embedding + linear head."""

    def __init__(self, name):
        super().__init__()
        from transformers import CLIPVisionModelWithProjection

        self.enc = CLIPVisionModelWithProjection.from_pretrained(name)
        self.enc.requires_grad_(False)
        self.enc.eval()
        _d = self.enc.config.projection_dim
        self.head = nn.Linear(_d, 1)
        self.register_buffer("mean", torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1))

    def features(self, x):
        _e = self.enc(pixel_values=(x - self.mean) / self.std).image_embeds
        return F.normalize(_e.float(), dim=-1)

    def forward_both(self, x):
        _f = self.features(x)
        return self.head(_f).squeeze(1), _f

    def forward(self, x):
        return self.forward_both(x)[0]

def launder_train(a, rng, crop):
    """Training augmentation on a 256-px uint8 crop, identical for both classes:
    mild rescale, random crop, flip, blur, JPEG (q 65-100, p=0.8)."""
    _im = Image.fromarray(a)
    if rng.random() < 0.3:
        _s = rng.uniform(0.9, 1.4)
        _n = int(round(a.shape[0] * _s))
        _im = _im.resize((_n, _n), Image.BICUBIC)
    _W = _im.size[0]
    _x0, _y0 = int(rng.integers(0, _W - crop + 1)), int(rng.integers(0, _W - crop + 1))
    _im = _im.crop((_x0, _y0, _x0 + crop, _y0 + crop))
    if rng.random() < 0.5:
        _im = _im.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.2:
        _im = _im.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.1, 1.5))))
    if rng.random() < 0.8:
        _buf = io.BytesIO()
        _im.save(_buf, "JPEG", quality=int(rng.integers(65, 101)))
        _im = Image.open(io.BytesIO(_buf.getvalue())).convert("RGB")
    return np.array(_im, dtype=np.uint8)

def launder_eval(a, kind, crop=224):
    """Deterministic test-time laundering, then centre crop."""
    _im = Image.fromarray(a)
    if kind == "jpeg75":
        _buf = io.BytesIO()
        _im.save(_buf, "JPEG", quality=75)
        _im = Image.open(io.BytesIO(_buf.getvalue())).convert("RGB")
    elif kind == "resize075":
        _n = a.shape[0]
        _im = _im.resize((int(_n * 0.75),) * 2, Image.BICUBIC).resize((_n, _n), Image.BICUBIC)
    elif kind == "blur1":
        _im = _im.filter(ImageFilter.GaussianBlur(1.0))
    return center224(np.asarray(_im, dtype=np.uint8), a.shape[0], crop)

class TrainDS(torch.utils.data.Dataset):
    def __init__(self, store, idx, crop):
        self.store, self.idx, self.crop = store, np.asarray(idx), crop
        self.y = store.meta["label"].to_numpy()[self.idx].astype(np.float32)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, j):
        _rng = np.random.default_rng([torch.initial_seed() % 2**32, int(j)])
        _a = launder_train(self.store.get(int(self.idx[j])), _rng, self.crop)
        return torch.from_numpy(np.ascontiguousarray(_a)), torch.tensor(self.y[j])

def make_loader(store, idx, batch, workers, seed, epoch):
    _ds = TrainDS(store, idx, CFG["img"]["crop"])
    _y = _ds.y
    _w = np.where(_y > 0.5, 0.5 / max(1, _y.sum()), 0.5 / max(1, (1 - _y).sum()))
    _g = torch.Generator().manual_seed(seed * 1000 + epoch)
    _sampler = torch.utils.data.WeightedRandomSampler(torch.as_tensor(_w, dtype=torch.double), len(_ds), replacement=True, generator=_g)
    _kw = {}
    if workers > 0:
        _kw = {"multiprocessing_context": "fork", "persistent_workers": False, "prefetch_factor": 4}
    return torch.utils.data.DataLoader(_ds, batch_size=batch, sampler=_sampler, num_workers=workers,
                                       drop_last=True, pin_memory=torch.cuda.is_available(), **_kw)

def load_detector(path, device):
    _sd = torch.load(path, map_location=device, weights_only=True)
    _m = Detector(CFG["model"]["backbone"], CFG["model"]["drop_path"], pretrained=False).to(device)
    _m.load_state_dict(_sd)
    _m.eval()
    return _m

def load_clip_probe(path, device):
    _m = ClipProbe(CFG["model"]["clip"]).to(device)
    _m.head.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    _m.eval()
    return _m

In [ ]:
# ---- 8. LIBRARY: smoothing, certification, calibration, metrics ------------------------
from scipy.stats import beta as _betad, norm as _norm, rankdata as _rankdata

def cp_two_sided(k, n, alpha):
    """Clopper-Pearson interval for a binomial proportion, vectorised."""
    k = np.asarray(k, float)
    n = np.asarray(n, float) * np.ones_like(k)
    _lo = np.where(k > 0, _betad.ppf(alpha / 2, np.maximum(k, 1e-9), n - k + 1), 0.0)
    _hi = np.where(k < n, _betad.ppf(1 - alpha / 2, k + 1, np.maximum(n - k, 1e-9)), 1.0)
    return _lo, _hi

def cp_upper_one_sided(c, n, eta):
    c = np.asarray(c, float)
    return np.where(c < n, _betad.ppf(1 - eta, c + 1, np.maximum(n - c, 1e-9)), 1.0)

def scores_from_counts(k, N, sigma, alpha):
    """S = sigma * Phi^-1(p): 1-Lipschitz in l2 (Salman et al. 2019). Returns
    lower / upper confidence scores and a point estimate."""
    _lo, _hi = cp_two_sided(k, N, alpha)
    _c = lambda p: np.clip(p, 1e-12, 1 - 1e-12)
    return (sigma * _norm.ppf(_c(_lo)), sigma * _norm.ppf(_c(_hi)),
            sigma * _norm.ppf((np.asarray(k, float) + 0.5) / (N + 1)))

@torch.no_grad()
def smooth_votes(model, x, sigma, N, chunk, device):
    """Number of Gaussian-noise draws (out of N) on which the base model says 'real'."""
    _b = x.shape[0]
    _m = max(1, min(N, chunk // max(1, _b)))
    _votes = torch.zeros(_b, device=x.device)
    _done = 0
    while _done < N:
        _mm = min(_m, N - _done)
        _xr = x.repeat_interleave(_mm, 0)
        _xr = _xr + sigma * torch.randn_like(_xr)
        with amp_ctx(device):
            _lg = model(_xr).float()
        _votes += (_lg > 0).view(_b, _mm).sum(1)
        _done += _mm
    return _votes.cpu().numpy()

def conformal_flag_threshold(s_hi_real, beta):
    """Flag iff S_hi < tau_f (STRICT). With tau_f the floor((n+1)beta)-th smallest
    calibration score, exchangeability gives P(new real flagged) <= beta even when
    scores tie, which they do: S_hi takes at most N+1 distinct values."""
    _s = np.sort(np.asarray(s_hi_real))
    _k = int(np.floor((len(_s) + 1) * beta))
    return -np.inf if _k < 1 else float(_s[_k - 1])

def crc_accept_threshold(per_gen_scores, alpha, extra_mask=None):
    """Conformal risk control over generators: smallest t with
    (K/(K+1)) * mean_k AF_k(t) + 1/(K+1) <= alpha, AF_k(t) = mean(score >= t).
    Guarantees E[AF of a new exchangeable generator] <= alpha."""
    _K = len(per_gen_scores)
    if _K == 0 or (_K + 1) * alpha < 1:
        return np.inf, {"K": _K, "feasible": False}
    _allv = np.unique(np.concatenate([np.asarray(_s) for _s in per_gen_scores]))
    _cands = np.concatenate([_allv, [np.inf]])
    _srt = [np.sort(np.asarray(_s)) for _s in per_gen_scores]

    def _risk(t):
        _af = [1.0 - np.searchsorted(_s, t, side="left") / max(1, len(_s)) for _s in _srt]
        return _K / (_K + 1) * float(np.mean(_af)) + 1.0 / (_K + 1)

    _lo, _hi = 0, len(_cands) - 1  # risk is non-increasing in t: binary search
    while _lo < _hi:
        _mid = (_lo + _hi) // 2
        if _risk(_cands[_mid]) <= alpha:
            _hi = _mid
        else:
            _lo = _mid + 1
    return float(_cands[_lo]), {"K": _K, "feasible": True, "risk_bound": _risk(_cands[_lo])}

def quantile_accept_threshold(per_gen_scores, alpha_q, eta, delta):
    """Theorem 5: per-generator critical thresholds from a one-sided Clopper-Pearson
    bound, then the ceil((K+1)(1-delta))-th smallest. P(AF_new <= alpha_q) >= 1-delta-eta."""
    _th = []
    for _s in per_gen_scores:
        _s = np.sort(np.asarray(_s))[::-1]
        _n = len(_s)
        _ok = [c for c in range(_n + 1) if cp_upper_one_sided(c, _n, eta) <= alpha_q]
        if not _ok:
            _th.append(np.inf)
            continue
        _cmax = max(_ok)
        _th.append(np.inf if _cmax >= _n else float(np.nextafter(_s[_cmax], np.inf)))
    _th = np.sort(np.asarray(_th))
    _K = len(_th)
    _r = int(np.ceil((_K + 1) * (1 - delta)))
    return (float(_th[_r - 1]) if 1 <= _r <= _K else np.inf), _th

def decide(s_lo, s_hi, tau_a, tau_f, eps):
    """2 = Accept-authentic, 1 = Defer, 0 = Flag-synthetic (certified, Theorem 3)."""
    _A = np.asarray(s_lo) >= tau_a + eps
    _Fl = (np.asarray(s_hi) < tau_f - eps) & ~_A
    return np.where(_A, 2, np.where(_Fl, 0, 1))

def three_way_metrics(dec, y):
    dec, y = np.asarray(dec), np.asarray(y)
    _r, _f = dec[y == 0], dec[y == 1]
    _acc = (dec == 2)
    return {
        "AF": float(np.mean(_f == 2)) if len(_f) else float("nan"),
        "FR": float(np.mean(_r == 0)) if len(_r) else float("nan"),
        "cov_real": float(np.mean(_r != 1)) if len(_r) else float("nan"),
        "cov_fake": float(np.mean(_f != 1)) if len(_f) else float("nan"),
        "accept_real_rate": float(np.mean(_r == 2)) if len(_r) else float("nan"),
        "accept_precision": float(np.mean(y[_acc] == 0)) if _acc.any() else float("nan"),
        "n_real": int(len(_r)), "n_fake": int(len(_f)),
    }

def auroc_real(score, y):
    """AUROC with 'real' as the positive class and realness as the score."""
    score, y = np.asarray(score, float), np.asarray(y)
    _pos = y == 0
    _n1, _n0 = _pos.sum(), (~_pos).sum()
    if _n1 == 0 or _n0 == 0:
        return float("nan")
    _r = _rankdata(score)
    return float((_r[_pos].sum() - _n1 * (_n1 + 1) / 2) / (_n1 * _n0))

def tpr_at_fpr(score, y, fpr=0.01):
    """Detection view: share of fakes caught when only `fpr` of reals are flagged."""
    score, y = np.asarray(score, float), np.asarray(y)
    if (y == 0).sum() == 0 or (y == 1).sum() == 0:
        return float("nan")
    _t = np.quantile(score[y == 0], fpr)
    return float(np.mean(score[y == 1] < _t))

In [ ]:
# ---- 9. LIBRARY: attacks ---------------------------------------------------------------
def pgd(objective, x0, eps, norm, steps, rand_init=True):
    """Generic PGD. objective(x) -> (loss_to_minimise [B], success_mask [B]).
    Keeps the first successful iterate per sample (any step counts)."""
    x0 = x0.detach()
    _alpha = 2.5 * eps / max(1, steps)
    if rand_init:
        if norm == "linf":
            _d = torch.empty_like(x0).uniform_(-eps, eps)
        else:
            _d = torch.randn_like(x0)
            _d = _d / _d.flatten(1).norm(dim=1).view(-1, 1, 1, 1).clamp_min(1e-12) * eps * 0.5
    else:
        _d = torch.zeros_like(x0)
    _d = ((x0 + _d).clamp(0, 1) - x0).detach()
    _best = x0.clone()
    _succ = torch.zeros(x0.shape[0], dtype=torch.bool, device=x0.device)
    for _t in range(steps + 1):
        _d.requires_grad_(True)
        _x = (x0 + _d).clamp(0, 1)
        _loss, _ok = objective(_x)
        _new = _ok.detach() & ~_succ
        _best[_new] = _x.detach()[_new]
        _succ |= _ok.detach()
        if _t == steps:
            break
        (_g,) = torch.autograd.grad(_loss.sum(), _d)
        with torch.no_grad():
            if norm == "linf":
                _d = (_d - _alpha * _g.sign()).clamp(-eps, eps)
            else:
                _gn = _g.flatten(1).norm(dim=1).view(-1, 1, 1, 1).clamp_min(1e-12)
                _d = _d - _alpha * _g / _gn
                _n = _d.flatten(1).norm(dim=1).view(-1, 1, 1, 1).clamp_min(1e-12)
                _d = _d * torch.clamp(eps / _n, max=1.0)
            _d = ((x0 + _d).clamp(0, 1) - x0).detach()
    _last = (x0 + _d).clamp(0, 1).detach()
    _out = torch.where(_succ.view(-1, 1, 1, 1), _best, _last)
    return _out, _succ

def pgd_linf_train(model, x, target_real, eps, steps, device):
    """PGD-AT inner maximisation (cross-entropy on the realness logit)."""
    _obj = lambda xx: (
        -torch.nn.functional.binary_cross_entropy_with_logits(model(xx).float(), target_real, reduction="none"),
        torch.zeros(xx.shape[0], dtype=torch.bool, device=xx.device),
    )
    with amp_ctx(device):
        _xa, _ = pgd(_obj, x, eps, "linf", steps)
    return _xa.detach()

def eot_smoothed_attack(model, x0, sigma, eps, norm, steps, m, device, goal="real"):
    """SmoothAdv-style attack on the soft smoothed classifier: maximise
    log mean_j sigmoid(+-h(x + delta + eta_j)) with m noise draws per step."""
    _sgn = 1.0 if goal == "real" else -1.0

    def _obj(x):
        _xr = x.repeat_interleave(m, 0)
        _xr = _xr + sigma * torch.randn_like(_xr)
        with amp_ctx(device):
            _lg = model(_xr).float().view(x.shape[0], m)
        _p = torch.sigmoid(_sgn * _lg).mean(1).clamp_min(1e-12)
        return -torch.log(_p), torch.zeros(x.shape[0], dtype=torch.bool, device=x.device)

    _xa, _ = pgd(_obj, x0, eps, norm, steps)
    return _xa.detach()

## Part I: pipeline

Each stage S0–S12 is a function; the RUNNER cell below executes the selected stages in order. S5 trains only when `ALLOW_TRAIN=1`.

In [ ]:
# ---- S0. THEORY SELF-TESTS (CPU, ~1-2 min): numerically check every proved claim ------
# Each block constructs adversarial-to-the-theory instances and reports the worst case.
# A violation here means a bug in the mathematics or the code, BEFORE any GPU time.
def stage_S0_theory():
    from scipy.optimize import minimize
    from scipy.stats import norm, beta as betad

    _t0 = time.time()
    _rng = np.random.default_rng(CFG["seed"])
    _out = {}

    # (A1) Prop. 2: min-norm joint attack  <=  r_max * sqrt(k / lambda_min(G)),
    #      and <= r_max * sqrt(k) when all pairwise cosines are >= 0 (LP-duality proof).
    def _minnorm(U, r):
        G = U.T @ U
        f = lambda lam: lam @ G @ lam - 2 * r @ lam
        g = lambda lam: 2 * G @ lam - 2 * r
        res = minimize(f, np.ones(len(r)) * 0.1, jac=g, bounds=[(0, None)] * len(r), method="L-BFGS-B",
                       options={"ftol": 1e-14, "gtol": 1e-12, "maxiter": 5000})
        u = U @ res.x
        return np.linalg.norm(u), float((U.T @ u - r).min())

    _worst_gen, _worst_nonneg, _worst_feas = 0.0, 0.0, 0.0
    for _trial in range(300):
        _k = int(_rng.integers(2, 9))
        _d = 300
        _common = _rng.normal(size=_d)
        _a = _rng.uniform(-0.5, 2.0)
        _U = np.stack([_a * _common + _rng.normal(size=_d) * _rng.uniform(0.2, 2) for _ in range(_k)], 1)
        _U /= np.linalg.norm(_U, axis=0, keepdims=True)
        _r = _rng.uniform(0.1, 1.0, size=_k)
        _G = _U.T @ _U
        _lmin = np.linalg.eigvalsh(_G).min()
        _cost, _slack = _minnorm(_U, _r)
        _worst_feas = min(_worst_feas, _slack)
        _worst_gen = max(_worst_gen, _cost / (_r.max() * np.sqrt(_k / max(_lmin, 1e-12))))
        if (_G - np.eye(_k)).min() >= 0:
            _worst_nonneg = max(_worst_nonneg, _cost / (_r.max() * np.sqrt(_k)))
    _out["A1_prop2"] = {"max cost / (r_max*sqrt(k/lambda_min)) [must be <= 1]": _worst_gen,
                        "max cost / (r_max*sqrt(k)) when cosines >= 0 [must be <= 1]": _worst_nonneg,
                        "min constraint slack of solver [~0]": _worst_feas}

    # (A2) Theorem 3 ingredient: S = sigma * Phi^-1(p) is 1-Lipschitz (Salman et al. Lemma 2).
    #      Random non-smooth base classifiers, huge Monte-Carlo, finite differences.
    _ratios = []
    _sig = 0.25
    for _trial in range(12):
        _dd = 8
        _W1 = _rng.normal(size=(_dd, 32)) * 3
        _b1 = _rng.normal(size=32)
        _w2 = _rng.normal(size=32)
        _h = lambda X: (np.tanh(X @ _W1 + _b1) @ _w2 > 0)
        _x = _rng.normal(size=_dd) * 0.3
        _dir = _rng.normal(size=_dd)
        _dir /= np.linalg.norm(_dir)
        _eps = 0.15
        _M = 400_000
        _Z = _rng.normal(size=(_M, _dd)) * _sig  # common random numbers for both points
        _p0 = _h(_x + _Z).mean()
        _p1 = _h(_x + _eps * _dir + _Z).mean()
        _c = lambda p: np.clip(p, 1e-6, 1 - 1e-6)
        _dS = abs(_sig * (norm.ppf(_c(_p1)) - norm.ppf(_c(_p0))))
        _ratios.append(_dS / _eps)
    _out["A2_lipschitz"] = {"max |dS|/||dx|| over random nets [must be <= 1 up to MC noise]": float(max(_ratios)),
                            "median": float(np.median(_ratios))}

    # (A3) Conformal framing control WITH TIES (S_hi takes at most N+1 values).
    _N, _alpha_mc, _beta = 64, 1e-3, 0.05
    _viol_strict, _viol_leq = [], []
    for _trial in range(400):
        _pr = _rng.beta(8, 1.5, size=601)  # reals: mostly vote 'real'
        _kk = _rng.binomial(_N, _pr)
        _hi = np.where(_kk < _N, betad.ppf(1 - _alpha_mc / 2, _kk + 1, np.maximum(_N - _kk, 1e-9)), 1.0)
        _cal, _new = _hi[:600], _hi[600]
        _s = np.sort(_cal)
        _tf = _s[int(np.floor(601 * _beta)) - 1]
        _viol_strict.append(_new < _tf)
        _viol_leq.append(_new <= _tf)
    _out["A3_conformal_ties"] = {"P(flag real), strict '<' [must be <= beta]": float(np.mean(_viol_strict)),
                                 "P(flag real), '<=' (not guaranteed under ties)": float(np.mean(_viol_leq)),
                                 "beta": _beta}

    # (A4) Theorem 5' (conformal risk control over generators): E[AF_new] <= alpha.
    #      Generators differ in difficulty (heavy heterogeneity) and sample size.
    def _crc(per, alpha):
        K = len(per)
        vals = np.unique(np.concatenate(per))
        cands = np.concatenate([vals, [np.inf]])
        srt = [np.sort(v) for v in per]
        risk = lambda t: K / (K + 1) * np.mean([1 - np.searchsorted(v, t, "left") / len(v) for v in srt]) + 1 / (K + 1)
        lo, hi = 0, len(cands) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if risk(cands[mid]) <= alpha:
                hi = mid
            else:
                lo = mid + 1
        return cands[lo]

    _afs, _alpha = [], 0.02
    for _trial in range(300):
        _K = 120
        _mus = _rng.normal(-2.0, 1.2, size=_K + 1)  # per-generator mean fake score
        _ns = _rng.integers(15, 60, size=_K + 1)
        _per = [_rng.normal(m, 1.0, size=n) for m, n in zip(_mus, _ns)]
        _tau = _crc(_per[:_K], _alpha)
        _afs.append(float(np.mean(_per[_K] >= _tau)))
    _out["A4_crc_generators"] = {"E[AF of new generator] [must be <= alpha]": float(np.mean(_afs)),
                                 "alpha": _alpha, "std_err": float(np.std(_afs) / np.sqrt(len(_afs)))}

    # (A5) Theorem 4 + sampling cap: the largest certifiable radii for the configured N.
    _caps = {}
    for _n in [128, 256, 512, 1024]:
        _z = float(norm.ppf((CFG["cert"]["alpha_mc"] / 2) ** (1 / _n)))
        _caps[f"N={_n}"] = {"z_N": round(_z, 3),
                            **{f"sigma={s}: two-way {s * _z:.3f} | three-way {2 * s * _z:.3f}": None
                               for s in sorted(set(CFG["eca"]["sigmas"] + CFG["eca"]["cohen_sigmas"]))}}
    _out["A5_radius_caps"] = _caps
    _out["seconds"] = time.time() - _t0
    _p = os.path.join(PATHS["res"], "S0_theory.json")
    save_json(_p, _out)
    hf_push(_p)
    _ok = (_out["A1_prop2"]["max cost / (r_max*sqrt(k/lambda_min)) [must be <= 1]"] <= 1 + 1e-6
           and _out["A1_prop2"]["max cost / (r_max*sqrt(k)) when cosines >= 0 [must be <= 1]"] <= 1 + 1e-6
           and _out["A2_lipschitz"]["max |dS|/||dx|| over random nets [must be <= 1 up to MC noise]"] <= 1.1
           and _out["A3_conformal_ties"]["P(flag real), strict '<' [must be <= beta]"] <= _beta + 0.025
           and _out["A4_crc_generators"]["E[AF of new generator] [must be <= alpha]"] <= _alpha + 3 * _out["A4_crc_generators"]["std_err"])
    log(f"S0: theory self-tests {'PASS' if _ok else 'FAIL'} in {_out['seconds']:.0f}s -> {_p}")
    return {"pass": bool(_ok), **{k: v for k, v in _out.items() if k.startswith("A") and k != "A5_radius_caps"}}

In [ ]:
# ---- S1. INDEX CommunityForensics-Small and choose shards ------------------------------
def stage_S1_index():
    import pyarrow.parquet as pq

    _cfg = CFG["cf"]
    _sel_path = os.path.join(PATHS["data"], "cf_selection.parquet")
    if ensure(_sel_path):
        _sel = pd.read_parquet(_sel_path)
        log(f"S1: selection exists ({len(_sel)} rows)")
        return {"rows": len(_sel)}
    _fs = HfFileSystem(token=HF_TOKEN)
    _files = _fs.glob(f"datasets/{_cfg['repo']}/data/*.parquet")
    _num = lambda f: int(os.path.basename(f).split("_")[-1].split(".")[0])
    _files = sorted(_files, key=_num)
    if _cfg["index_shards"] is not None:
        _files = [f for f in _files if _num(f) in set(_cfg["index_shards"])]
    _cols = ["model_name", "label", "subset", "nsfw_flag", "format"]

    def _one(f):
        _t = pq.ParquetFile(f, filesystem=_fs).read(columns=_cols).to_pandas()
        _t["shard"] = _num(f)
        _t["row"] = np.arange(len(_t))
        return _t

    with ThreadPoolExecutor(8) as _ex:
        _idx = pd.concat(list(_ex.map(_one, _files)), ignore_index=True)
    log(f"S1: indexed {len(_files)} shards, {len(_idx):,} rows, {_idx[_idx.label == 1].model_name.nunique():,} generators")
    _idx.to_parquet(os.path.join(PATHS["data"], "cf_index.parquet"))

    # Fake shards: greedy for generator diversity (Theorem 5 needs many generators).
    _fake = _idx[_idx.label == 1]
    _chosen = []
    for _subset, _n in [("Systematic", _cfg["n_fake_shards"]), ("Manual", _cfg["n_manual_shards"])]:
        _sh = {s: set(g.model_name) for s, g in _fake[_fake.subset == _subset].groupby("shard")}
        _seen = set()
        for _ in range(min(_n, len(_sh))):
            _best = max((s for s in _sh if s not in _chosen), key=lambda s: (len(_sh[s] - _seen), -s))
            _chosen.append(_best)
            _seen |= _sh[_best]
    # Real shards: per-source quotas, evenly spaced over that source's shards.
    _real = _idx[_idx.label == 0]
    _dom = _real.groupby("shard").model_name.agg(lambda s: s.value_counts().index[0])
    for _src, _n in _cfg["real_shards"].items():
        _cand = sorted(_dom[_dom == _src].index.tolist())
        if _cand and _n > 0:
            _pick = np.unique(np.linspace(0, len(_cand) - 1, min(_n, len(_cand))).round().astype(int))
            _chosen += [_cand[i] for i in _pick]
    _sel = _idx[_idx.shard.isin(_chosen) & (_idx.nsfw_flag != True)].copy()  # noqa: E712
    if _cfg["max_rows_per_shard"]:
        _keep = []
        for _shd, _g in _sel.groupby("shard"):
            _pos = np.unique(np.linspace(0, len(_g) - 1, min(len(_g), _cfg["max_rows_per_shard"])).round().astype(int))
            _keep.append(_g.iloc[_pos])
        _sel = pd.concat(_keep)
    _fr = _cfg["split_frac"]
    _sel["split"] = [
        split_of(f"gen:{m}", _fr) if l == 1 else split_of(f"img:{s}:{r}", _fr)
        for m, l, s, r in zip(_sel.model_name, _sel.label, _sel.shard, _sel.row)
    ]
    _sel["dataset"] = "cf"
    _sel.reset_index(drop=True).to_parquet(_sel_path)
    hf_push(_sel_path)
    _summary = _sel.groupby(["label", "split"]).agg(n=("row", "size"), gens=("model_name", "nunique")).reset_index()
    log(f"S1: chose shards {sorted(_chosen)}\n{_summary}")
    return {"rows": len(_sel), "shards": sorted(_chosen)}

In [ ]:
# ---- S2. CROP CACHE for CommunityForensics (one shard at a time, resumable) -----------
def stage_S2_cache():
    import pyarrow.parquet as pq

    _cfg = CFG["cf"]
    _sel = pd.read_parquet(os.path.join(PATHS["data"], "cf_selection.parquet"))
    _out = os.path.join(PATHS["data"], "cf")
    os.makedirs(_out, exist_ok=True)
    _n_done = 0
    for _shard, _g in _sel.groupby("shard"):
        _npy = os.path.join(_out, f"cf_{_shard:03d}.npy")
        if chunk_done(_npy) or (CFG["hf"]["sync_cache"] and ensure(_npy) and ensure(_npy[:-4] + ".parquet")):
            _n_done += 1
            continue
        if time_left_h() < 0.3:
            log("S2: session deadline near; stopping (re-run to resume)")
            break
        _local = hf_hub_download(_cfg["repo"], f"data/HFCF_small_{_shard}.parquet", repo_type="dataset",
                                 local_dir=PATHS["dl"], token=HF_TOKEN)
        _want = dict(zip(_g.row.tolist(), _g.split.tolist()))
        _pf = pq.ParquetFile(_local, memory_map=True)
        _arrs, _meta, _r0 = [], [], 0
        with ThreadPoolExecutor(8) as _ex:
            for _b in _pf.iter_batches(batch_size=128, columns=["image_data", "model_name", "label"]):
                _rows = [(_r0 + i) for i in range(_b.num_rows) if (_r0 + i) in _want]
                _data = _b.column("image_data")
                _mn = _b.column("model_name").to_pylist()
                _lb = _b.column("label").to_pylist()

                def _job(r, _data=_data, _r0=_r0):
                    _mode = "random" if _want[r] == "train" else "center"
                    try:
                        return prep_image(_data[r - _r0].as_py(), CFG["img"], _mode, f"cf:{_shard}:{r}")
                    except Exception:
                        return None

                for r, a in zip(_rows, _ex.map(_job, _rows)):
                    if a is None:
                        continue
                    _arrs.append(a)
                    _meta.append({"shard": _shard, "row": r, "model_name": _mn[r - _r0],
                                  "label": int(_lb[r - _r0]), "split": _want[r], "dataset": "cf"})
                _r0 += _b.num_rows
        del _pf
        save_chunk(_npy, _arrs, pd.DataFrame(_meta), CFG["img"]["store"])
        os.remove(_local)
        if CFG["hf"]["sync_cache"]:
            hf_push(_npy)
            hf_push(_npy[:-4] + ".parquet")
        _n_done += 1
        log(f"S2: shard {_shard} -> {len(_arrs)} crops ({_n_done}/{_sel.shard.nunique()})  RAM {trim_ram():.1f} GB")
    return {"chunks": _n_done, "of": int(_sel.shard.nunique())}

In [ ]:
# ---- S3. OpenFake test subset (2025-26 generators), row-group level fetch -------------
def stage_S3_openfake():
    import pyarrow.parquet as pq

    _cfg = CFG["of"]
    _out = os.path.join(PATHS["data"], "of")
    os.makedirs(_out, exist_ok=True)
    _fs = HfFileSystem(token=HF_TOKEN)
    _have = open_store(_out, "of_").meta
    _cnt = _have.groupby("model").size().to_dict() if len(_have) else {}
    _n_real = int((_have.label == 0).sum()) if len(_have) else 0
    _fetched = len([p for p in os.listdir(_out) if p.startswith("of_") and p.endswith(".npy") and ".tmp" not in p])
    for _shard in _cfg["shards"]:
        _f = f"datasets/{_cfg['repo']}/core/test-{_shard:05d}-of-00013.parquet"
        _pf = pq.ParquetFile(_f, filesystem=_fs)
        _md = _pf.metadata
        _sizes = [_md.row_group(i).num_rows for i in range(_md.num_row_groups)]
        _m = _pf.read(columns=["label", "model", "release_date", "type"]).to_pandas()
        _m["rg"] = np.repeat(np.arange(len(_sizes)), _sizes)
        _m["rr"] = np.concatenate([np.arange(s) for s in _sizes])
        for _ in range(_md.num_row_groups):
            if _fetched >= _cfg["max_row_groups"] or time_left_h() < 0.3:
                break

            def _useful(rg):
                _g = _m[_m.rg == rg]
                _u = 0
                for _mod, _gg in _g.groupby("model"):
                    if (_gg.label == "real").all():
                        _u += min(len(_gg), max(0, _cfg["n_real"] - _n_real))
                    else:
                        _u += min(len(_gg), max(0, _cfg["per_model"] - _cnt.get(_mod, 0)))
                return _u

            _done_rgs = {int(os.path.basename(p).split("_")[2]) for p in os.listdir(_out)
                         if p.startswith(f"of_{_shard:02d}_") and p.endswith(".npy")}
            _cands = [rg for rg in range(_md.num_row_groups) if rg not in _done_rgs]
            if not _cands:
                break
            _rg = max(_cands, key=_useful)
            if _useful(_rg) == 0:
                break
            _npy = os.path.join(_out, f"of_{_shard:02d}_{_rg:03d}_.npy")
            _t = _pf.read_row_group(_rg, columns=["image", "label", "model", "release_date", "type"]).to_pandas()
            _arrs, _meta = [], []
            for _i, _row in _t.iterrows():
                _real = _row.label == "real"
                if _real and _n_real >= _cfg["n_real"]:
                    continue
                if (not _real) and _cnt.get(_row.model, 0) >= _cfg["per_model"]:
                    continue
                try:
                    _a = prep_image(_row.image["bytes"], CFG["img"], "center", f"of:{_shard}:{_rg}:{_i}")
                except Exception:
                    continue
                _arrs.append(_a)
                _meta.append({"model": _row.model, "model_name": _row.model, "label": 0 if _real else 1,
                              "release_date": _row.release_date, "type": _row.type, "split": "test", "dataset": "openfake"})
                if _real:
                    _n_real += 1
                else:
                    _cnt[_row.model] = _cnt.get(_row.model, 0) + 1
            save_chunk(_npy, _arrs, pd.DataFrame(_meta), CFG["img"]["store"])
            _fetched += 1
            log(f"S3: shard {_shard} rg {_rg}: +{len(_arrs)} (reals {_n_real}, fake models {len(_cnt)})")
    _st = open_store(_out, "of_")
    return {"images": len(_st), "models": int(_st.meta.model.nunique()) if len(_st) else 0}

In [ ]:
# ---- S4. RAID transfer-attack set (paired ids across budgets) -------------------------
def stage_S4_raid():
    _cfg = CFG["raid"]
    _out = os.path.join(PATHS["data"], "raid")
    os.makedirs(_out, exist_ok=True)
    if all(chunk_done(os.path.join(_out, f"raid_{b}.npy")) for b in _cfg["budgets"]):
        log("S4: RAID cache exists")
        return {"done": True}
    _fs = HfFileSystem(token=HF_TOKEN)
    _ids = sorted(os.path.basename(p).split("_gen_0")[0]
                  for p in _fs.ls(f"datasets/{_cfg['repo']}/original/gen_0", detail=False))[: _cfg["n_per_folder"]]

    def _name(b, f, i):
        return f"{b}/{f}/{i}_{f}.jpg" if b == "original" else f"{b}/{f}/{i}_{f}_adv.png"

    _files = [_name(b, f, i) for b in _cfg["budgets"] for f in _cfg["folders"] for i in _ids]
    _tmp = os.path.join(PATHS["dl"], "raid")
    snapshot_download(_cfg["repo"], repo_type="dataset", allow_patterns=_files, local_dir=_tmp,
                      max_workers=16, token=HF_TOKEN)
    _S = CFG["img"]["store"]
    for _b in _cfg["budgets"]:
        _arrs, _meta = [], []
        for _f in _cfg["folders"]:
            for _i in _ids:
                _p = os.path.join(_tmp, _name(_b, _f, _i))
                if not os.path.exists(_p):
                    continue
                _im = Image.open(_p).convert("RGB")
                if min(_im.size) != _S:  # originals of reals are 324 px; RAID attacked 256-px copies
                    _r = _S / min(_im.size)
                    _im = _im.resize((round(_im.size[0] * _r), round(_im.size[1] * _r)), Image.BICUBIC)
                _w, _h = _im.size
                _x0, _y0 = (_w - _S) // 2, (_h - _S) // 2
                _arrs.append(np.asarray(_im.crop((_x0, _y0, _x0 + _S, _y0 + _S)), dtype=np.uint8))
                _meta.append({"id": _i, "folder": _f, "budget": _b, "model_name": _f,
                              "label": 0 if _f == "real" else 1, "split": "test", "dataset": "raid"})
        save_chunk(os.path.join(_out, f"raid_{_b}.npy"), _arrs, pd.DataFrame(_meta), _S)
        log(f"S4: RAID {_b}: {len(_arrs)} images")
    shutil.rmtree(_tmp, ignore_errors=True)
    return {"ids": len(_ids)}

In [ ]:
# ---- S5. TRAINING: BASE, PGD-AT, Cohen, ECA-3WD (ADRT); CLIP probe --------------------
def train_detector(name, mode, sigma=None):
    """mode in {'base','pgdat','cohen','eca'}. Resumable per epoch; synced to the Hub."""
    _dir = os.path.join(PATHS["models"], name)
    os.makedirs(_dir, exist_ok=True)
    _final = os.path.join(_dir, "final.pt")
    if ensure(_final):
        log(f"S5[{name}]: final weights exist")
        return _final
    if not CFG["allow_train"]:
        log(f"S5[{name}]: no weights and training is locked (set ALLOW_TRAIN=1); skipped")
        return None
    _tc, _ec = CFG["train"], CFG["eca"]
    _store = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _tr = np.where(_store.meta.split.to_numpy() == "train")[0]
    torch.manual_seed(CFG["seed"])
    _model = Detector(CFG["model"]["backbone"], CFG["model"]["drop_path"], pretrained=True).to(DEVICE)
    _opt = torch.optim.AdamW(_model.parameters(), lr=_tc["lr"], weight_decay=_tc["wd"])
    _steps_ep = len(_tr) // _tc["batch"]
    if _tc["max_steps"]:
        _steps_ep = min(_steps_ep, _tc["max_steps"])
    _total = max(1, _steps_ep * _tc["epochs"])
    _warm = max(1, int(_total * _tc["warmup_frac"]))
    _sched = torch.optim.lr_scheduler.LambdaLR(
        _opt, lambda s: min(1.0, (s + 1) / _warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / _total))))
    _ep0, _hist = 0, []
    _last = os.path.join(_dir, "last.pt")
    if ensure(_last):
        _ck = torch.load(_last, map_location=DEVICE, weights_only=False)
        _model.load_state_dict(_ck["model"])
        _opt.load_state_dict(_ck["opt"])
        _sched.load_state_dict(_ck["sched"])
        _ep0, _hist = _ck["epoch"] + 1, _ck["hist"]
        log(f"S5[{name}]: resumed at epoch {_ep0}")
    _k = _tc["k_noise"]
    for _ep in range(_ep0, _tc["epochs"]):
        _model.train()
        _dl = make_loader(_store, _tr, _tc["batch"], _tc["workers"], CFG["seed"], _ep)
        _t0, _agg = time.time(), {"loss": 0.0, "n": 0}
        for _it, (_xb, _yb) in enumerate(_dl):
            if _it >= _steps_ep:
                break
            _x = _xb.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            _t = 1.0 - _yb.to(DEVICE).float()  # target = realness
            if mode == "pgdat":
                _h = _x.shape[0] // 2
                _model.eval()
                _xa = pgd_linf_train(_model, _x[:_h], _t[:_h], _tc["pgd_at_eps"], _tc["pgd_at_steps"], DEVICE)
                _model.train()
                _x = torch.cat([_xa, _x[_h:]])
            if mode in ("eca", "cohen"):
                _xr = _x.repeat_interleave(_k, 0)
                _xr = _xr + sigma * torch.randn_like(_xr)
                with amp_ctx(DEVICE):
                    _lg = _model(_xr).float()
                _loss = F.binary_cross_entropy_with_logits(_lg, _t.repeat_interleave(_k))
                if mode == "eca":
                    _q = torch.sigmoid(_lg).view(-1, _k).mean(1).clamp(_ec["gamma"], 1 - _ec["gamma"])
                    _S = sigma * torch.special.ndtri(_q)
                    _T = _ec["T"] * sigma
                    _isF, _isR = _t < 0.5, _t > 0.5
                    _laf = (_T * F.softplus((_S + _ec["mF"] * sigma) / _T))[_isF].mean() if _isF.any() else _S.sum() * 0
                    _lfr = (_T * F.softplus((_ec["mR"] * sigma - _S) / _T))[_isR].mean() if _isR.any() else _S.sum() * 0
                    _loss = _loss + _ec["mu"] * (_ec["lam_af"] * _laf + _ec["lam_fr"] * _lfr) / sigma
            else:
                with amp_ctx(DEVICE):
                    _lg = _model(_x).float()
                _loss = F.binary_cross_entropy_with_logits(_lg, _t)
            _opt.zero_grad(set_to_none=True)
            _loss.backward()
            torch.nn.utils.clip_grad_norm_(_model.parameters(), 5.0)
            _opt.step()
            _sched.step()
            _agg["loss"] += float(_loss.item())
            _agg["n"] += 1
            if _it % 100 == 0:
                log(f"S5[{name}] ep {_ep} it {_it}/{_steps_ep} loss {_agg['loss'] / _agg['n']:.4f} {vram_report()}")
            if time_left_h() < 0.25:
                log(f"S5[{name}]: deadline near, saving mid-epoch is skipped; re-run to redo epoch {_ep}")
                return None
        _hist.append({"epoch": _ep, "loss": _agg["loss"] / max(1, _agg["n"]), "sec": time.time() - _t0})
        atomic_write(_last, lambda p: torch.save({"model": _model.state_dict(), "opt": _opt.state_dict(),
                                                  "sched": _sched.state_dict(), "epoch": _ep, "hist": _hist}, p))
        hf_push(_last, f"{name} epoch {_ep}")
        log(f"S5[{name}]: epoch {_ep} done {_hist[-1]}")
    _model.eval()
    atomic_write(_final, lambda p: torch.save(_model.state_dict(), p))
    save_json(os.path.join(_dir, "train_log.json"), {"mode": mode, "sigma": sigma, "hist": _hist})
    hf_push(_final, f"{name} final")
    hf_push(os.path.join(_dir, "train_log.json"))
    del _model, _opt
    gpu_free()
    return _final

def train_clip_probe():
    _dir = os.path.join(PATHS["models"], "clip")
    os.makedirs(_dir, exist_ok=True)
    _final = os.path.join(_dir, "head.pt")
    if ensure(_final):
        return _final
    if not CFG["allow_train"]:
        log("S5[clip]: no head and training is locked (set ALLOW_TRAIN=1); skipped")
        return None
    _store = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _tr = np.where(_store.meta.split.to_numpy() == "train")[0]
    _m = ClipProbe(CFG["model"]["clip"]).to(DEVICE)
    _feats = []
    with torch.no_grad():
        for _s in range(0, len(_tr), 128):
            _x = to_tensor01(center224(_store.batch(_tr[_s:_s + 128])), DEVICE)
            with amp_ctx(DEVICE):
                _feats.append(_m.features(_x).float().cpu())
    _Fx = torch.cat(_feats).to(DEVICE)
    _y = torch.as_tensor(1.0 - _store.meta.label.to_numpy()[_tr], dtype=torch.float32, device=DEVICE)
    _w = torch.where(_y > 0.5, 0.5 / _y.sum().clamp_min(1), 0.5 / (1 - _y).sum().clamp_min(1)) * len(_y)
    _opt = torch.optim.AdamW(_m.head.parameters(), lr=1e-3, weight_decay=1e-4)
    for _e in range(CFG["train"]["probe_epochs"]):
        for _s in torch.randperm(len(_y), device=DEVICE).split(1024):
            _l = (F.binary_cross_entropy_with_logits(_m.head(_Fx[_s]).squeeze(1), _y[_s], reduction="none") * _w[_s]).mean()
            _opt.zero_grad()
            _l.backward()
            _opt.step()
    atomic_write(_final, lambda p: torch.save(_m.head.state_dict(), p))
    hf_push(_final, "clip probe head")
    log(f"S5[clip]: probe trained on {len(_y)} features, last loss {float(_l):.4f}")
    del _m, _Fx
    gpu_free()
    return _final

def stage_S5_train():
    _out = {"base": train_detector("base", "base"), "pgdat": train_detector("pgdat", "pgdat")}
    for _s in CFG["eca"]["sigmas"]:
        _out[f"eca_{_s}"] = train_detector(f"eca_{_s}", "eca", sigma=_s)
    for _s in CFG["eca"]["cohen_sigmas"]:
        _out[f"cohen_{_s}"] = train_detector(f"cohen_{_s}", "cohen", sigma=_s)
    _out["clip"] = train_clip_probe()
    return _out

In [ ]:
# ---- S6. BASELINES: scores, gates, conformal calibration, first-order budget (Prop. 1)
def baseline_models():
    _m = []
    for _n in ["base", "pgdat"]:
        _p = os.path.join(PATHS["models"], _n, "final.pt")
        if os.path.exists(_p):
            _m.append((_n, "cnn", _p))
    _p = os.path.join(PATHS["models"], "clip", "head.pt")
    if os.path.exists(_p):
        _m.append(("clip", "clip", _p))
    return _m

def load_any(kind, path):
    return load_detector(path, DEVICE) if kind == "cnn" else load_clip_probe(path, DEVICE)

@torch.no_grad()
def base_outputs(model, kind, x, mc_passes):
    """Realness logit s, penultimate features, and MC (stochastic depth) variance."""
    with amp_ctx(DEVICE):
        _s, _f = model.forward_both(x)
    _var = torch.zeros_like(_s, dtype=torch.float32)
    if kind == "cnn" and mc_passes > 1:
        set_mc(model, True)
        with amp_ctx(DEVICE):
            _ss = torch.stack([model.forward_both(x)[0].float() for _ in range(mc_passes)])
        set_mc(model, False)
        _var = _ss.var(0)
    return _s.float(), _f.float(), _var

def run_store(model, kind, store, idx, mc_passes, launder=None, bs=64):
    _S, _Fs, _V = [], [], []
    for _i in range(0, len(idx), bs):
        _b = store.batch(idx[_i:_i + bs])
        _b = np.stack([launder_eval(a, launder) for a in _b]) if launder else center224(_b)
        _s, _f, _v = base_outputs(model, kind, to_tensor01(_b, DEVICE), mc_passes)
        _S.append(_s.cpu())
        _Fs.append(_f.cpu())
        _V.append(_v.cpu())
    if not _S:
        return np.zeros(0), np.zeros((0, 1)), np.zeros(0)
    return torch.cat(_S).numpy(), torch.cat(_Fs).numpy(), torch.cat(_V).numpy()

def fit_maha(feats, y):
    """Class-conditional means, shared covariance (Lee et al. 2018), shrinkage."""
    _mu_r = feats[y == 0].mean(0)
    _mu_f = feats[y == 1].mean(0) if (y == 1).any() else _mu_r
    _c = np.concatenate([feats[y == 0] - _mu_r, feats[y == 1] - _mu_f])
    _cov = _c.T @ _c / max(1, len(_c) - 1)
    _cov = _cov + 1e-3 * np.trace(_cov) / len(_cov) * np.eye(len(_cov))
    return {"mu_r": _mu_r.astype(np.float32), "P": np.linalg.inv(_cov).astype(np.float32)}

def maha_gate(feats, mp):
    """Gate value c = -Mahalanobis distance to the REAL class (larger = more typical)."""
    _d = feats - mp["mu_r"]
    return -np.einsum("ij,jk,ik->i", _d, mp["P"], _d)

VARIANTS = {"cnn": ["conf", "maha", "mc", "all3"], "clip": ["conf", "maha"]}

def gate_mask(variant, s, cm, cv, thr):
    _ok = np.ones_like(s, dtype=bool)
    if variant in ("maha", "all3"):
        _ok &= cm >= thr["maha"]
    if variant in ("mc", "all3"):
        _ok &= -cv >= thr["mc"]
    return _ok

def baseline_decide(variant, s, cm, cv, thr):
    _acc = (s >= thr[f"tau_a_{variant}"]) & gate_mask(variant, s, cm, cv, thr)
    _flag = (s < thr["tau_f"]) & ~_acc
    return np.where(_acc, 2, np.where(_flag, 0, 1))

def stage_S6_baselines():
    if not baseline_models():
        log("S6: no baseline weights (S5 not trained yet); skipped")
        return {"skipped": "no weights"}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _of = open_store(os.path.join(PATHS["data"], "of"), "of_")
    _rd = open_store(os.path.join(PATHS["data"], "raid"), "raid_")
    _meta = _cf.meta
    _y = _meta.label.to_numpy()
    _sp = _meta.split.to_numpy()
    _rng = np.random.default_rng(CFG["seed"])
    _cal = np.where(_sp == "cal")[0]
    _test = np.where(_sp == "test")[0]
    _trn = np.where(_sp == "train")[0]
    _fit = _rng.choice(_trn, min(len(_trn), CFG["cal"]["maha_fit_n"]), replace=False)
    _mc = CFG["attack"]["mc_eval"]
    _all = {}
    for _name, _kind, _path in baseline_models():
        _jp = os.path.join(PATHS["res"], f"S6_{_name}.json")
        if all(ensure(os.path.join(PATHS["res"], f"S6_{_name}{_x}")) for _x in [".json", "_maha.npz", "_scores.parquet"]):
            _all[_name] = load_json(_jp)
            log(f"S6[{_name}]: results exist, skipping")
            continue
        _model = load_any(_kind, _path)
        _sf, _ff, _ = run_store(_model, _kind, _cf, _fit, 1)
        _mp = fit_maha(_ff, _y[_fit])
        _sc, _fc, _vc = run_store(_model, _kind, _cf, _cal, _mc)
        _cmc = maha_gate(_fc, _mp)
        _ycal = _y[_cal]
        _keep = CFG["cal"]["gate_keep"]
        _thr = {
            "maha": float(np.quantile(_cmc[_ycal == 0], 1 - _keep)),
            "mc": float(np.quantile(-_vc[_ycal == 0], 1 - _keep)),
            "tau_f": conformal_flag_threshold(_sc[_ycal == 0], CFG["cal"]["beta"]),
        }
        _gens = _meta.model_name.to_numpy()[_cal]
        for _v in VARIANTS[_kind]:
            _gm = gate_mask(_v, _sc, _cmc, _vc, _thr)
            # Gated acceptance is a superlevel set of s restricted to the gate: give gate
            # failures score -inf, then CRC over calibration generators.
            _eff = np.where(_gm, _sc, -np.inf)
            _per = [_eff[(_ycal == 1) & (_gens == g)] for g in np.unique(_gens[_ycal == 1])
                    if ((_ycal == 1) & (_gens == g)).sum() >= CFG["cal"]["min_per_gen"]]
            _t, _info = crc_accept_threshold(_per, CFG["cal"]["alpha"])
            _thr[f"tau_a_{_v}"] = max(_t, _thr["tau_f"])
            _thr[f"crc_{_v}"] = _info
        np.savez(os.path.join(PATHS["res"], f"S6_{_name}_maha.npz"), **_mp)
        _res = {"thresholds": _thr, "eval": {}}
        _sets = {"cf_test": (_cf, _test, None), "cf_test_jpeg75": (_cf, _test, "jpeg75"),
                 "cf_test_resize075": (_cf, _test, "resize075"), "cf_test_blur1": (_cf, _test, "blur1"),
                 "openfake": (_of, np.arange(len(_of)), None)}
        for _b in CFG["raid"]["budgets"]:
            _sets[f"raid_{_b}"] = (_rd, np.where(_rd.meta.budget.to_numpy() == _b)[0] if len(_rd) else np.zeros(0, int), None)
        _rows = []
        for _sn, (_st, _ix, _ld) in _sets.items():
            if len(_ix) == 0:
                continue
            _s, _f, _v = run_store(_model, _kind, _st, _ix, _mc, launder=_ld)
            _cm = maha_gate(_f, _mp)
            _yy = _st.meta.label.to_numpy()[_ix]
            _ent = {"auroc": auroc_real(_s, _yy), "tpr@1%fpr": tpr_at_fpr(_s, _yy)}
            for _v2 in VARIANTS[_kind]:
                _ent[_v2] = three_way_metrics(baseline_decide(_v2, _s, _cm, _v, _thr), _yy)
            _res["eval"][_sn] = _ent
            _rows.append(pd.DataFrame({"set": _sn, "s": _s, "maha": _cm, "mcvar": _v, "y": _yy,
                                       "model_name": _st.meta.model_name.to_numpy()[_ix]}))
        pd.concat(_rows).to_parquet(os.path.join(PATHS["res"], f"S6_{_name}_scores.parquet"))
        # Proposition 1: first-order l_inf evasion budget eps1 = (tau_a - s) / ||grad s||_1
        _fk = _test[_y[_test] == 1][: CFG["attack"]["n"]]
        _e1 = []
        for _i in range(0, len(_fk), 16):
            _x = to_tensor01(center224(_cf.batch(_fk[_i:_i + 16])), DEVICE).requires_grad_(True)
            with amp_ctx(DEVICE):
                _s = _model(_x).float()
            (_g,) = torch.autograd.grad(_s.sum(), _x)
            _gap = (_thr["tau_a_conf"] - _s.detach()).clamp_min(0)
            _e1.append((_gap / _g.abs().flatten(1).sum(1).clamp_min(1e-12)).cpu().numpy())
        _e1 = np.concatenate(_e1) if _e1 else np.zeros(0)
        _res["first_order_eps1"] = {"median_x255": float(np.median(_e1) * 255) if len(_e1) else None,
                                    "q90_x255": float(np.quantile(_e1, 0.9) * 255) if len(_e1) else None}
        np.save(os.path.join(PATHS["res"], f"S6_{_name}_eps1.npy"), _e1)
        save_json(os.path.join(PATHS["res"], f"S6_{_name}.json"), _res)
        log(f"S6[{_name}]: cf_test {_res['eval'].get('cf_test', {}).get('auroc')}, eps1 median {_res['first_order_eps1']['median_x255']}/255")
        _all[_name] = _res
        del _model
        gpu_free()
    hf_push(PATHS["res"], "S6 baselines")
    return {k: v["eval"].get("cf_test", {}).get("auroc") for k, v in _all.items()}

In [ ]:
# ---- S7. ATTACKS ON BASELINES: gate-blind vs joint (Prop. 1); sqrt(k) check (Prop. 2)
def _constraints(model, kind, x, thr, mp_t, variant, mc_m):
    """Constraint values c_i(x) that acceptance requires (c_i >= thr_i)."""
    _s, _f = model.forward_both(x)
    _cs = [(_s.float(), thr[f"tau_a_{variant}"], thr["_scale_s"])]
    if variant in ("maha", "all3"):
        _d = _f.float() - mp_t["mu_r"]
        _cs.append((-(_d @ mp_t["P"] * _d).sum(1), thr["maha"], thr["_scale_maha"]))
    if variant in ("mc", "all3") and kind == "cnn":
        set_mc(model, True)
        _ss = torch.stack([model.forward_both(x)[0].float() for _ in range(mc_m)])
        set_mc(model, False)
        _cs.append((-_ss.var(0), thr["mc"], thr["_scale_mc"]))
    return _cs

def stage_S7_attacks():
    if not baseline_models():
        log("S7: no baseline weights; skipped")
        return {"skipped": "no weights"}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _fk = _test[_y[_test] == 1][: CFG["attack"]["n"]]
    _ac = CFG["attack"]
    _out = {}
    for _name, _kind, _path in baseline_models():
        if ensure(os.path.join(PATHS["res"], f"S7_{_name}.json")):
            _out[_name] = load_json(os.path.join(PATHS["res"], f"S7_{_name}.json"))["joint"]
            log(f"S7[{_name}]: results exist, skipping")
            continue
        _r6 = load_json(os.path.join(PATHS["res"], f"S6_{_name}.json"))
        _thr = dict(_r6["thresholds"])
        _mpz = np.load(os.path.join(PATHS["res"], f"S6_{_name}_maha.npz"))
        _mp_t = {k: torch.as_tensor(_mpz[k], device=DEVICE) for k in ["mu_r", "P"]}
        _mp = {k: _mpz[k] for k in ["mu_r", "P"]}
        import pandas as _pd_s7

        _sc = _pd_s7.read_parquet(os.path.join(PATHS["res"], f"S6_{_name}_scores.parquet"))
        _ref = _sc[(_sc.set == "cf_test") & (_sc.y == 0)]
        _thr["_scale_s"] = float(_ref.s.std() + 1e-6)
        _thr["_scale_maha"] = float(_ref.maha.std() + 1e-6)
        _thr["_scale_mc"] = float(_ref.mcvar.std() + 1e-6)
        _model = load_any(_kind, _path)
        for _p in _model.parameters():
            _p.requires_grad_(False)
        _res = {"blind": {}, "joint": {}, "prop2": {}}
        _grid = [("linf", e) for e in _ac["linf"]] + [("l2", e) for e in _ac["l2"]]
        _minjoint = {v: np.full(len(_fk), np.inf) for v in VARIANTS[_kind]}

        def _eval_dec(xa):
            _s, _f, _v = base_outputs(_model, _kind, xa, _ac["mc_eval"])
            _cm = -np.einsum("ij,jk,ik->i", _f.cpu().numpy() - _mp["mu_r"], _mp["P"], _f.cpu().numpy() - _mp["mu_r"])
            return {v: baseline_decide(v, _s.cpu().numpy(), _cm, _v.cpu().numpy(), _thr) for v in VARIANTS[_kind]}

        for _nrm, _eps in _grid:
            _key = f"{_nrm}_{_eps * 255:.0f}/255" if _nrm == "linf" else f"{_nrm}_{_eps}"
            _accB = {v: [] for v in VARIANTS[_kind]}
            _accJ = {v: [] for v in VARIANTS[_kind]}
            for _i in range(0, len(_fk), _ac["batch"]):
                _x0 = to_tensor01(center224(_cf.batch(_fk[_i:_i + _ac["batch"]])), DEVICE)
                # Gate-blind: maximise the realness logit only (what prior selective papers test).
                _objB = lambda x: (-model_s(x), torch.zeros(x.shape[0], dtype=torch.bool, device=x.device))

                def model_s(x):
                    with amp_ctx(DEVICE):
                        return _model(x).float()

                _xb, _ = pgd(_objB, _x0, _eps, _nrm, _ac["steps"])
                for _v, _d in _eval_dec(_xb).items():
                    _accB[_v].append(_d == 2)
                # Joint: every acceptance constraint, normalised hinge deficits.
                for _v in VARIANTS[_kind]:
                    def _objJ(x, _v=_v):
                        with amp_ctx(DEVICE):
                            _cs = _constraints(_model, _kind, x, _thr, _mp_t, _v, _ac["mc_eot"])
                        _loss = sum(torch.relu(t + 0.05 * sc - c) / sc for c, t, sc in _cs)
                        _ok = torch.stack([c >= t for c, t, sc in _cs]).all(0)
                        return _loss - 0.01 * _cs[0][0] / _cs[0][2], _ok

                    _xj, _ = pgd(_objJ, _x0, _eps, _nrm, _ac["steps"])
                    _acc = _eval_dec(_xj)[_v] == 2
                    _accJ[_v].append(_acc)
                    if _nrm == "l2":
                        _sl = slice(_i, _i + len(_acc))
                        _minjoint[_v][_sl] = np.where(_acc, np.minimum(_minjoint[_v][_sl], _eps), _minjoint[_v][_sl])
            _res["blind"][_key] = {v: float(np.concatenate(a).mean()) for v, a in _accB.items()}
            _res["joint"][_key] = {v: float(np.concatenate(a).mean()) for v, a in _accJ.items()}
            log(f"S7[{_name}] {_key}: blind {_res['blind'][_key]} | joint {_res['joint'][_key]}")
        # Proposition 2: first-order deficits and gradient cosines at clean fakes (l2).
        if _kind == "cnn":
            _rows = []
            for _i in range(0, len(_fk), _ac["batch"]):
                _x = to_tensor01(center224(_cf.batch(_fk[_i:_i + _ac["batch"]])), DEVICE).requires_grad_(True)
                with amp_ctx(DEVICE):
                    _cs = _constraints(_model, _kind, _x, _thr, _mp_t, "all3", _ac["mc_eval"])
                _gs = [torch.autograd.grad(c.sum(), _x, retain_graph=True)[0].flatten(1).float() for c, _, _ in _cs]
                _def = [torch.relu(t - c.detach()) for c, t, _ in _cs]
                _r = torch.stack([d / g.norm(dim=1).clamp_min(1e-12) for d, g in zip(_def, _gs)], 1)
                _u = [g / g.norm(dim=1, keepdim=True).clamp_min(1e-12) for g in _gs]
                _cos = torch.stack([(_u[a] * _u[b]).sum(1) for a in range(3) for b in range(a + 1, 3)], 1)
                _rows.append(torch.cat([_r, _cos], 1).detach().cpu().numpy())
            _rows = np.concatenate(_rows)
            _r, _cos = _rows[:, :3], _rows[:, 3:]
            _active = (_r > 0).sum(1)
            _rho = np.clip(_cos.mean(1), 0, None)
            _kk = np.maximum(_active, 1)
            _bound = _r.max(1) * np.sqrt(_kk / (1 + (_kk - 1) * _rho))
            _emp = _minjoint["all3"]
            _ok = np.isfinite(_emp) & (_r.max(1) > 0)
            _res["prop2"] = {
                "median_cos": [float(np.median(_cos[:, j])) for j in range(3)],
                "median_r_max": float(np.median(_r.max(1))),
                "median_bound": float(np.median(_bound)),
                "median_emp_over_rmax": float(np.median(_emp[_ok] / _r.max(1)[_ok])) if _ok.any() else None,
                "median_sqrt_factor": float(np.median(np.sqrt(_kk / (1 + (_kk - 1) * _rho)))),
                "share_emp_le_bound": float(np.mean(_emp[_ok] <= _bound[_ok] * 1.2)) if _ok.any() else None,
                "note": "empirical eps is quantised to the l2 grid; first-order bound is local",
            }
        save_json(os.path.join(PATHS["res"], f"S7_{_name}.json"), _res)
        _out[_name] = _res["joint"]
        del _model
        gpu_free()
    hf_push(PATHS["res"], "S7 attacks")
    return _out

In [ ]:
# ---- S8. ECA-3WD: certification, CERT-CAL, evaluation, generator-level guarantee ------
def smoothed_models():
    _out = []
    _cands = [("eca", s) for s in CFG["eca"]["sigmas"]] + [("cohen", s) for s in CFG["eca"]["cohen_sigmas"]]
    for _v, _s in _cands:
        _p = os.path.join(PATHS["models"], f"{_v}_{_s}", "final.pt")
        if ensure(_p):
            _out.append((f"{_v}_{_s}", _s, _p))
    return _out

def certify_store(model, store, idx, sigma, launder=None):
    _c = CFG["cert"]
    _k = []
    for _i in range(0, len(idx), _c["img_batch"]):
        _b = store.batch(idx[_i:_i + _c["img_batch"]])
        _b = np.stack([launder_eval(a, launder) for a in _b]) if launder else center224(_b)
        _k.append(smooth_votes(model, to_tensor01(_b, DEVICE), sigma, _c["N"], _c["chunk"], DEVICE))
    return np.concatenate(_k) if _k else np.zeros(0)

def cert_calibrate(s_hi_cal, y_cal, gens_cal):
    _cc = CFG["cal"]
    _tf = conformal_flag_threshold(s_hi_cal[y_cal == 0], _cc["beta"])
    _per, _names = [], []
    for _g in np.unique(gens_cal[y_cal == 1]):
        _m = (y_cal == 1) & (gens_cal == _g)
        if _m.sum() >= _cc["min_per_gen"]:
            _per.append(s_hi_cal[_m])
            _names.append(_g)
    _ta, _info = crc_accept_threshold(_per, _cc["alpha"])  # fakes calibrated on S_hi (critique #4)
    _tq, _thetas = quantile_accept_threshold(_per, _cc["alpha_q"], _cc["eta"], _cc["delta"])
    return {"tau_f": _tf, "tau_a": max(_ta, _tf), "tau_q": max(_tq, _tf), "crc": _info, "K": len(_per)}

def stage_S8_certify():
    if not smoothed_models():
        log("S8: no smoothed-model weights (S5 not trained yet); skipped")
        return {"skipped": "no weights"}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _of = open_store(os.path.join(PATHS["data"], "of"), "of_")
    _rd = open_store(os.path.join(PATHS["data"], "raid"), "raid_")
    _y = _cf.meta.label.to_numpy()
    _sp = _cf.meta.split.to_numpy()
    _gn = _cf.meta.model_name.to_numpy()
    _cal, _test = np.where(_sp == "cal")[0], np.where(_sp == "test")[0]
    _N, _a = CFG["cert"]["N"], CFG["cert"]["alpha_mc"]
    _summary = {}
    for _name, _sigma, _path in smoothed_models():
        if ensure(os.path.join(PATHS["res"], f"S8_{_name}.json")) and ensure(os.path.join(PATHS["res"], f"S8_{_name}_counts.parquet")):
            _summary[_name] = load_json(os.path.join(PATHS["res"], f"S8_{_name}.json"))["eval"].get("cf_test", {}).get("auroc")
            log(f"S8[{_name}]: results exist, skipping")
            continue
        _model = load_detector(_path, DEVICE)
        _kc = certify_store(_model, _cf, _cal, _sigma)
        _lo, _hi, _ = scores_from_counts(_kc, _N, _sigma, _a)
        _th = cert_calibrate(_hi, _y[_cal], _gn[_cal])
        from scipy.stats import norm as _nm

        _zN = float(_nm.ppf((_a / 2) ** (1.0 / _N)))
        _res = {"sigma": _sigma, "thresholds": _th, "eval": {},
                "theory": {
                    "z_N": _zN,
                    "max_two_way_radius (sigma*z_N)": _sigma * _zN,
                    "max_three_way_radius (2*sigma*z_N)": 2 * _sigma * _zN,
                    "acceptance_margin_needed_for_real (tau_a + eps_c), main eps": _th["tau_a"] + (CFG["cert"]["main_eps"] or _sigma),
                    "linf_budgets_as_l2_at_224": {f"{e * 255:.0f}/255": e * (3 * 224 * 224) ** 0.5 for e in CFG["attack"]["linf"]},
                    "guarantee": f"E[AF_eps] <= {CFG['cal']['alpha']} + alpha_mc over exchangeable CF generators, any l2 attack <= eps_c, for ANY base model",
                }}
        _sets = {"cf_test": (_cf, _test, None), "cf_test_jpeg75": (_cf, _test, "jpeg75"),
                 "cf_test_resize075": (_cf, _test, "resize075"), "cf_test_blur1": (_cf, _test, "blur1"),
                 "openfake": (_of, np.arange(len(_of)), None)}
        for _b in CFG["raid"]["budgets"]:
            _sets[f"raid_{_b}"] = (_rd, np.where(_rd.meta.budget.to_numpy() == _b)[0] if len(_rd) else np.zeros(0, int), None)
        _rows = [pd.DataFrame({"set": "cf_cal", "k": _kc, "y": _y[_cal], "model_name": _gn[_cal]})]
        for _sn, (_st, _ix, _ld) in _sets.items():
            if len(_ix) == 0:
                continue
            _k = certify_store(_model, _st, _ix, _sigma, _ld)
            _l, _h, _p = scores_from_counts(_k, _N, _sigma, _a)
            _yy = _st.meta.label.to_numpy()[_ix]
            _ent = {"auroc": auroc_real(_p, _yy), "tpr@1%fpr": tpr_at_fpr(_p, _yy)}
            for _e in CFG["cert"]["eps_cert"]:
                _ent[f"eps{_e}"] = three_way_metrics(decide(_l, _h, _th["tau_a"], _th["tau_f"], _e), _yy)
            _res["eval"][_sn] = _ent
            _df = pd.DataFrame({"set": _sn, "k": _k, "y": _yy, "model_name": _st.meta.model_name.to_numpy()[_ix]})
            if "release_date" in _st.meta:
                _df["release_date"] = _st.meta.release_date.to_numpy()[_ix]
            _rows.append(_df)
        # Theorem 5 check on HELD-OUT generators (test split never touched calibration).
        _tdf = _rows[1] if len(_rows) > 1 else None
        if _tdf is not None:
            _l, _h, _ = scores_from_counts(_tdf.k.to_numpy(), _N, _sigma, _a)
            _tdf = _tdf.assign(acc_crc=_h >= _th["tau_a"], acc_q=_h >= _th["tau_q"])
            _pg = _tdf[_tdf.y == 1].groupby("model_name").agg(n=("k", "size"), af_crc=("acc_crc", "mean"), af_q=("acc_q", "mean"))
            _pg = _pg[_pg.n >= CFG["cal"]["min_per_gen"]]
            _res["generator_guarantee"] = {
                "n_test_generators": int(len(_pg)),
                "mean_AF_crc (target <= alpha)": float(_pg.af_crc.mean()) if len(_pg) else None,
                "alpha": CFG["cal"]["alpha"],
                "share_AF_q<=alpha_q (target >= 1-delta-eta)": float((_pg.af_q <= CFG["cal"]["alpha_q"]).mean()) if len(_pg) else None,
                "target_share": 1 - CFG["cal"]["delta"] - CFG["cal"]["eta"],
            }
        pd.concat(_rows).to_parquet(os.path.join(PATHS["res"], f"S8_{_name}_counts.parquet"))
        save_json(os.path.join(PATHS["res"], f"S8_{_name}.json"), _res)
        log(f"S8[{_name}]: tau_a {_th['tau_a']:.3f} tau_f {_th['tau_f']:.3f} K={_th['K']} "
            f"cf_test {_res['eval'].get('cf_test', {}).get('eps' + str(CFG['cert']['main_eps'] or _sigma))}")
        _summary[_name] = _res["eval"].get("cf_test", {}).get("auroc")
        del _model
        gpu_free()
    hf_push(PATHS["res"], "S8 certification")
    return _summary

In [ ]:
# ---- S9. ADAPTIVE EOT ATTACKS on smoothed models, then re-certification (Thm 3) -------
def stage_S9_eot():
    if not smoothed_models():
        log("S9: no smoothed-model weights (S5 not trained yet); skipped")
        return {"skipped": "no weights"}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _ac, _c = CFG["attack"], CFG["cert"]
    _fk = _test[_y[_test] == 1][: _ac["n"]]
    _rl = _test[_y[_test] == 0][: _ac["n"]]
    _out = {}
    for _name, _sigma, _path in smoothed_models():
        if ensure(os.path.join(PATHS["res"], f"S9_{_name}.json")):
            _out[_name] = load_json(os.path.join(PATHS["res"], f"S9_{_name}.json"))["evasion"]
            log(f"S9[{_name}]: results exist, skipping")
            continue
        if not ensure(os.path.join(PATHS["res"], f"S8_{_name}.json")):
            log(f"S9[{_name}]: no S8 certification yet; skipped")
            continue
        _th = load_json(os.path.join(PATHS["res"], f"S8_{_name}.json"))["thresholds"]
        _model = load_detector(_path, DEVICE)
        for _p in _model.parameters():
            _p.requires_grad_(False)
        _res = {"evasion": {}, "framing": {}}
        _grid = [("linf", e) for e in _ac["linf"]] + [("l2", e) for e in _ac["l2"]]
        for _nrm, _eps in _grid:
            _key = f"{_nrm}_{_eps * 255:.0f}/255" if _nrm == "linf" else f"{_nrm}_{_eps}"
            for _goal, _ix, _tag in [("real", _fk, "evasion"), ("fake", _rl, "framing")]:
                _ks = []
                for _i in range(0, len(_ix), _ac["batch"]):
                    _x0 = to_tensor01(center224(_cf.batch(_ix[_i:_i + _ac["batch"]])), DEVICE)
                    _xa = eot_smoothed_attack(_model, _x0, _sigma, _eps, _nrm, _ac["eot_steps"], _ac["eot_m"], DEVICE, goal=_goal)
                    _ks.append(smooth_votes(_model, _xa, _sigma, _c["N"], _c["chunk"], DEVICE))
                _l, _h, _ = scores_from_counts(np.concatenate(_ks), _c["N"], _sigma, _c["alpha_mc"])
                _ent = {}
                for _e in _c["eps_cert"]:
                    _d = decide(_l, _h, _th["tau_a"], _th["tau_f"], _e)
                    _ent[f"eps_c{_e}"] = float(np.mean(_d == (2 if _tag == "evasion" else 0)))
                _res[_tag][_key] = _ent
            log(f"S9[{_name}] {_key}: AF {_res['evasion'][_key]} | FR {_res['framing'][_key]}")
        save_json(os.path.join(PATHS["res"], f"S9_{_name}.json"), _res)
        _out[_name] = _res["evasion"]
        del _model
        gpu_free()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    hf_push(PATHS["res"], "S9 EOT attacks")
    return _out

In [ ]:
# ---- S10. EVIDENCE-ENERGY FLOOR with SD-VAE reconstruction pairs (Theorem 4) ----------
def stage_S10_floor():
    from diffusers import AutoencoderKL

    if ensure(os.path.join(PATHS["res"], "S10_floor.json")):
        log("S10: results exist, skipping")
        return {"a_l2_median": load_json(os.path.join(PATHS["res"], "S10_floor.json"))["a_l2"]["median"]}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    if len(_cf.meta) == 0 or "label" not in _cf.meta.columns:
        log("S10: no CommunityForensics crops (run S1-S2 first); skipped")
        return {"skipped": "no data"}
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _rl = _test[_y[_test] == 0][: CFG["floor"]["n"]]
    _vae = AutoencoderKL.from_pretrained(CFG["floor"]["vae"]).to(DEVICE).eval()
    _xr, _xf = [], []
    with torch.no_grad():
        for _i in range(0, len(_rl), 16):
            _x = to_tensor01(_cf.batch(_rl[_i:_i + 16]), DEVICE)  # full 256 px
            _z = _vae.encode(_x * 2 - 1).latent_dist.mode()
            _rec = ((_vae.decode(_z).sample + 1) / 2).clamp(0, 1)
            # quantise like a saved PNG, then centre-crop to the detector's 224 px input
            _rec = (_rec * 255).round() / 255
            _xr.append(_x[:, :, 16:240, 16:240].cpu())
            _xf.append(_rec[:, :, 16:240, 16:240].cpu())
    del _vae
    gpu_free()
    _xr, _xf = torch.cat(_xr), torch.cat(_xf)
    _anorm = (_xf - _xr).flatten(1).norm(dim=1).numpy()
    _eg = np.array([0.0, 0.06, 0.12, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0])
    _res = {"a_l2": {"median": float(np.median(_anorm)), "q05": float(np.quantile(_anorm, 0.05)),
                     "q95": float(np.quantile(_anorm, 0.95))},
            "eps_grid": _eg.tolist(),
            "floor_three_way": [float(np.mean(_anorm > e)) for e in _eg],
            "cap_two_way": [float(np.mean(_anorm > 2 * e)) for e in _eg],
            "models": {}}
    _N, _a = CFG["cert"]["N"], CFG["cert"]["alpha_mc"]
    for _name, _sigma, _path in smoothed_models():
        if not ensure(os.path.join(PATHS["res"], f"S8_{_name}.json")):
            log(f"S10[{_name}]: no S8 certification yet; skipped")
            continue
        _th = load_json(os.path.join(PATHS["res"], f"S8_{_name}.json"))["thresholds"]
        _m = load_detector(_path, DEVICE)
        _kr = np.concatenate([smooth_votes(_m, _xr[i:i + 8].to(DEVICE), _sigma, _N, CFG["cert"]["chunk"], DEVICE) for i in range(0, len(_xr), 8)])
        _kf = np.concatenate([smooth_votes(_m, _xf[i:i + 8].to(DEVICE), _sigma, _N, CFG["cert"]["chunk"], DEVICE) for i in range(0, len(_xf), 8)])
        _lr, _hr, _ = scores_from_counts(_kr, _N, _sigma, _a)
        _lf, _hf, _ = scores_from_counts(_kf, _N, _sigma, _a)
        # Robustly authenticated pair at budget e: real certified-accepted AND fake certified never-accepted.
        _R = [float(np.mean((_lr >= _th["tau_a"] + e) & (_hf < _th["tau_a"]))) for e in _eg]
        from scipy.special import ndtri as _ndtri_s10

        _zN = float(_ndtri_s10((_a / 2) ** (1.0 / _N)))
        _res["models"][_name] = {"robust_pairs": _R,
                                 "upper_bound_min(floor, sampling cap)": [float(min(f, 1.0 if e < 2 * _sigma * _zN else 0.0)) for f, e in zip(_res["floor_three_way"], _eg)],
                                 "AF_vae_clean": float(np.mean(_lf >= _th["tau_a"] + (CFG["cert"]["main_eps"] or _sigma)))}
        log(f"S10[{_name}]: robust pairs {_R[:5]} vs floor {_res['floor_three_way'][:5]}")
        del _m
        gpu_free()
    save_json(os.path.join(PATHS["res"], "S10_floor.json"), _res)
    hf_push(PATHS["res"], "S10 floor")
    return {"a_l2_median": _res["a_l2"]["median"]}

In [ ]:
# ---- S11. SEQUENTIAL CERTIFICATION, EXPLANATION CERTIFICATES, STEIN SALIENCY ----------
from scipy.stats import norm as _nrm_s11

def seq_certify(model, x, sigma, th, eps, block, n_max, alpha):
    """Vectorised sequential test: stop each image once its decision is determined."""
    _B = n_max // block
    _a = alpha / _B
    _b = x.shape[0]
    _k = np.zeros(_b)
    _n = np.zeros(_b)
    _dec = np.full(_b, -1)
    for _ in range(_B):
        _act = np.where(_dec < 0)[0]
        if len(_act) == 0:
            break
        _k[_act] += smooth_votes(model, x[_act], sigma, block, CFG["cert"]["chunk"], DEVICE)
        _n[_act] += block
        _lo, _hi = cp_two_sided(_k[_act], _n[_act], _a)
        _sl = sigma * _nrm_s11.ppf(np.clip(_lo, 1e-12, 1 - 1e-12))
        _sh = sigma * _nrm_s11.ppf(np.clip(_hi, 1e-12, 1 - 1e-12))
        _A = _sl >= th["tau_a"] + eps
        _Fl = _sh < th["tau_f"] - eps
        _D = (_sh < th["tau_a"] + eps) & (_sl > th["tau_f"] - eps)
        _dec[_act] = np.where(_A, 2, np.where(_Fl, 0, np.where(_D, 1, -1)))
    _dec[_dec < 0] = 1
    return _dec, _n

@torch.no_grad()
def stein_saliency(model, x1, sigma, N, block):
    """grad p(x) = E[eta * 1{h(x+eta)=real}] / sigma^2 (Stein), with a control variate;
    grad S = sigma * grad p / phi(Phi^-1(p)). Uses only the certification samples."""
    _se = torch.zeros_like(x1[0])
    _sv = torch.zeros_like(x1[0])
    _k = 0.0
    for _ in range(N // block):
        _eta = sigma * torch.randn((block,) + tuple(x1.shape[1:]), device=x1.device)
        with amp_ctx(DEVICE):
            _v = (model(x1 + _eta).float() > 0).float()
        _sv += (_eta * _v.view(-1, 1, 1, 1)).sum(0)
        _se += _eta.sum(0)
        _k += float(_v.sum())
    _n = (N // block) * block
    _p = min(max(_k / _n, 1e-4), 1 - 1e-4)
    _gp = (_sv - _p * _se) / (_n * sigma**2)
    return (sigma * _gp / _nrm_s11.pdf(_nrm_s11.ppf(_p))).cpu().numpy(), _p

def explain(s_lo, s_hi, th, eps, alpha, delta):
    if s_lo >= th["tau_a"] + eps:
        return (f"ACCEPT-AUTHENTIC: lower authenticity score {s_lo:.3f} >= acceptance line {th['tau_a']:.3f} "
                f"+ attack budget {eps}; certified l2 margin {s_lo - th['tau_a']:.3f}. Calibrated so a new "
                f"generator's expected accepted-fake rate is <= {alpha:.0%}.")
    if s_hi < th["tau_f"] - eps:
        return (f"FLAG-SYNTHETIC: upper authenticity score {s_hi:.3f} <= framing line {th['tau_f']:.3f} "
                f"- budget {eps}; no l2 perturbation <= {eps} could make a real image look this synthetic "
                f"more often than the calibrated framing rate.")
    return (f"DEFER: scores [{s_lo:.3f}, {s_hi:.3f}] lie inside the defended band "
            f"({th['tau_f'] - eps:.3f}, {th['tau_a'] + eps:.3f}); the evidence cannot be defended at budget {eps}.")

def stage_S11_seq():
    if not smoothed_models():
        log("S11: no smoothed-model weights (S5 not trained yet); skipped")
        return {"skipped": "no weights"}
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    if ensure(os.path.join(PATHS["res"], "S11_seq.json")):
        log("S11: results exist, skipping")
        return {k: v["mean_samples"] for k, v in load_json(os.path.join(PATHS["res"], "S11_seq.json")).items()}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _rng = np.random.default_rng(CFG["seed"])
    _ix = _rng.choice(_test, min(len(_test), CFG["seq"]["n"]), replace=False)
    _res = {}
    for _name, _sigma, _path in smoothed_models():
        if not ensure(os.path.join(PATHS["res"], f"S8_{_name}.json")):
            log(f"S11[{_name}]: no S8 certification yet; skipped")
            continue
        _th = load_json(os.path.join(PATHS["res"], f"S8_{_name}.json"))["thresholds"]
        _eps = CFG["cert"]["main_eps"] or _sigma
        _m = load_detector(_path, DEVICE)
        _x = to_tensor01(center224(_cf.batch(_ix)), DEVICE)
        _t0 = time.time()
        _ds, _ns = [], []
        for _i in range(0, len(_ix), 32):
            _d, _n = seq_certify(_m, _x[_i:_i + 32], _sigma, _th, _eps, CFG["seq"]["block"], CFG["seq"]["n_max"], CFG["cert"]["alpha_mc"])
            _ds.append(_d)
            _ns.append(_n)
        _dt = (time.time() - _t0) / max(1, len(_ix))
        _ds, _ns = np.concatenate(_ds), np.concatenate(_ns)
        _kf = np.concatenate([smooth_votes(_m, _x[i:i + 8], _sigma, CFG["cert"]["N"], CFG["cert"]["chunk"], DEVICE) for i in range(0, len(_ix), 8)])
        _l, _h, _ = scores_from_counts(_kf, CFG["cert"]["N"], _sigma, CFG["cert"]["alpha_mc"])
        _full = decide(_l, _h, _th["tau_a"], _th["tau_f"], _eps)
        _ex = [explain(float(_l[i]), float(_h[i]), _th, _eps, CFG["cal"]["alpha"], CFG["cal"]["delta"]) for i in range(min(6, len(_ix)))]
        _res[_name] = {"mean_samples": float(_ns.mean()), "ms_per_image": 1000 * _dt,
                       "agreement_with_fixed_N": float(np.mean(_ds == _full)), "examples": _ex,
                       "labels_of_examples": _y[_ix[:6]].tolist()}
        # Stein saliency for a few test fakes
        _fk = _test[_y[_test] == 1][: CFG["seq"]["n_saliency"]]
        _fig, _axs = plt.subplots(2, len(_fk), figsize=(3 * len(_fk), 6), squeeze=False)
        for _j, _i in enumerate(_fk):
            _img = center224(_cf.get(int(_i)))
            _g, _p = stein_saliency(_m, to_tensor01(_img[None], DEVICE), _sigma, CFG["seq"]["n_max"], CFG["seq"]["block"])
            _sal = np.abs(_g).sum(0)
            _axs[0, _j].imshow(_img)
            _axs[0, _j].set_title(f"fake, p_real={_p:.2f}", fontsize=8)
            _axs[1, _j].imshow(_sal, cmap="magma")
            _axs[1, _j].set_title("|grad S| (Stein)", fontsize=8)
            for _a in _axs[:, _j]:
                _a.axis("off")
        _fig.tight_layout()
        _fig.savefig(os.path.join(PATHS["res"], f"S11_{_name}_saliency.png"), dpi=110)
        plt.close(_fig)
        log(f"S11[{_name}]: mean samples {_res[_name]['mean_samples']:.0f}, {_res[_name]['ms_per_image']:.1f} ms/img")
        del _m, _x
        gpu_free()
    save_json(os.path.join(PATHS["res"], "S11_seq.json"), _res)
    hf_push(PATHS["res"], "S11 sequential + explanations")
    return {k: v["mean_samples"] for k, v in _res.items()}

In [ ]:
# ---- S12. REPORT: coverage (what is done / left out), tables, headline figure ---------
# Read-only with respect to training: it never trains, attacks or certifies anything.
def pull_results_from_hub():
    """Fetch results + small artefacts already in the HF repo (fresh session)."""
    _repo = CFG["hf"]["results_repo"]
    if not (HF_TOKEN and _repo):
        return "hub pull skipped (HF_TOKEN / HF_RESULTS_REPO unset)"
    _pat = [os.path.relpath(PATHS["res"], CFG["work"]).replace(os.sep, "/") + "/*"]
    try:
        snapshot_download(_repo, repo_type="model", local_dir=CFG["work"], token=HF_TOKEN,
                          allow_patterns=_pat)
        return "hub pull ok"
    except Exception as _e:
        return f"hub pull failed: {type(_e).__name__}"

def _hub_files():
    _repo = CFG["hf"]["results_repo"]
    if not (HF_TOKEN and _repo):
        return set()
    try:
        from huggingface_hub import HfApi as _Api
        return set(_Api(token=HF_TOKEN).list_repo_files(_repo, repo_type="model"))
    except Exception:
        return set()

def coverage_report():
    """One row per expected artefact: present locally / on the Hub / missing, and
    which stage produces it. Uses only file existence, never runs a stage."""
    _hub = _hub_files()

    def _st(path):
        _rel = os.path.relpath(path, CFG["work"]).replace(os.sep, "/")
        return "local" if os.path.exists(path) else ("hub" if _rel in _hub else "MISSING")

    def _chunks(sub, prefix):
        _d = os.path.join(PATHS["data"], sub)
        _n = len([f for f in os.listdir(_d) if f.startswith(prefix) and f.endswith(".npy")
                  and ".tmp" not in f]) if os.path.isdir(_d) else 0
        return f"{_n} chunks" if _n else "MISSING"

    _r, _m = PATHS["res"], PATHS["models"]
    _sm = [f"eca_{s}" for s in CFG["eca"]["sigmas"]] + [f"cohen_{s}" for s in CFG["eca"]["cohen_sigmas"]]
    _bl = ["base", "pgdat", "clip"]
    _rows = [("S0", "theory self-tests", _st(os.path.join(_r, "S0_theory.json"))),
             ("S1", "CommunityForensics index", _st(os.path.join(PATHS["data"], "cf_index.parquet"))),
             ("S2", "CF crop cache", _chunks("cf", "cf_")),
             ("S3", "OpenFake crops", _chunks("of", "of_")),
             ("S4", "RAID crops", _chunks("raid", "raid_"))]
    for _n in _bl:
        _rows.append(("S5", f"weights {_n}", _st(os.path.join(_m, _n, "head.pt" if _n == "clip" else "final.pt"))))
    for _n in _sm:
        _rows.append(("S5", f"weights {_n}", _st(os.path.join(_m, _n, "final.pt"))))
    for _n in _bl:
        _rows.append(("S6", f"baseline eval {_n}", _st(os.path.join(_r, f"S6_{_n}.json"))))
        _rows.append(("S7", f"attacks {_n}", _st(os.path.join(_r, f"S7_{_n}.json"))))
    for _n in _sm:
        _rows.append(("S8", f"certification {_n}", _st(os.path.join(_r, f"S8_{_n}.json"))))
        _rows.append(("S9", f"EOT attack {_n}", _st(os.path.join(_r, f"S9_{_n}.json"))))
    _rows += [("S10", "evidence-energy floor", _st(os.path.join(_r, "S10_floor.json"))),
              ("S11", "sequential / explanations", _st(os.path.join(_r, "S11_seq.json")))]
    _df = pd.DataFrame(_rows, columns=["stage", "artefact", "status"])
    # Which paper claims are currently backed by a result file
    _has = lambda prefix: any(r[2] != "MISSING" for r in _rows if r[0] == prefix)
    _claims = {
        "Thm 0 / S0 self-tests": _has("S0"),
        "Prop 1-2 (gate collapse, sqrt(k) stacking)": _has("S7"),
        "Thm 5 / 5' (CERT-CAL guarantee)": _has("S8"),
        "Thm 3 (no attack amplification, EOT)": _has("S9"),
        "Thm 4 (abstention dividend / floor)": _has("S10"),
        "Sequential + explanation certificates": _has("S11"),
    }
    return _df, _claims

def stage_S12_report():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    log(f"S12: {pull_results_from_hub()}")
    _r = PATHS["res"]
    _rows = []
    for _f in sorted(os.listdir(_r)):
        if _f.startswith("S6_") and _f.endswith(".json"):
            _n = _f[3:-5]
            _j = load_json(os.path.join(_r, _f))
            for _set, _e in _j["eval"].items():
                for _v, _m in _e.items():
                    if isinstance(_m, dict):
                        _rows.append({"method": f"{_n}+{_v}", "set": _set, "auroc": _e["auroc"], **_m})
        if _f.startswith("S8_") and _f.endswith(".json"):
            _n = _f[3:-5]
            _j = load_json(os.path.join(_r, _f))
            _me = CFG["cert"]["main_eps"] or _j["sigma"]
            for _set, _e in _j["eval"].items():
                _m = _e.get(f"eps{_me}")
                if _m:
                    _rows.append({"method": f"{_n} (eps_c={_me})", "set": _set, "auroc": _e["auroc"], **_m})
    _clean = pd.DataFrame(_rows)
    if len(_clean):
        _clean.to_csv(os.path.join(_r, "table_clean.csv"), index=False)
    _cov, _claims = coverage_report()
    _cov.to_csv(os.path.join(_r, "COVERAGE.csv"), index=False)
    _missing = _cov[_cov.status == "MISSING"]
    with open(os.path.join(_r, "REPORT.md"), "w") as _fh:
        _fh.write("# ECA-3WD run report\n\n## Claims backed by a result file\n\n")
        for _k, _v in _claims.items():
            _fh.write(f"- [{'x' if _v else ' '}] {_k}\n")
        _fh.write(f"\n## Left out ({len(_missing)} artefacts)\n\n")
        for _row in _missing.itertuples():
            _fh.write(f"- {_row.stage}: {_row.artefact}\n")
        if len(_clean):
            try:
                _tbl = _clean.round(4).to_markdown(index=False)
            except ImportError:  # tabulate not installed
                _tbl = _clean.round(4).to_string(index=False)
            _fh.write("\n## Clean / transfer table\n\n" + _tbl + "\n")
    # Headline: accepted-fake rate under joint (baselines) / EOT (smoothed) l2 attacks on
    # held-out generators -- the certified regime. l_inf results stay in the tables.
    _fig, _ax = plt.subplots(1, 2, figsize=(11, 4))
    try:
        for _f in sorted(os.listdir(_r)):
            if _f.startswith("S7_") and _f.endswith(".json"):
                _j = load_json(os.path.join(_r, _f))
                for _part, _ls in [("joint", "-"), ("blind", ":")]:
                    _keys = sorted([k for k in _j[_part] if k.startswith("l2_")], key=lambda k: float(k.split("_")[1]))
                    _xs = [float(k.split("_")[1]) for k in _keys]
                    _v = "all3" if "all3" in _j[_part][_keys[0]] else "maha"
                    _ax[0].plot(_xs, [_j[_part][k][_v] for k in _keys], _ls, marker="o", label=f"{_f[3:-5]}+{_v} ({_part})")
            if _f.startswith("S9_") and _f.endswith(".json"):
                _j = load_json(os.path.join(_r, _f))
                _me = CFG["cert"]["main_eps"] or load_json(os.path.join(_r, "S8_" + _f[3:]))["sigma"]
                _keys = sorted([k for k in _j["evasion"] if k.startswith("l2_")], key=lambda k: float(k.split("_")[1]))
                _xs = [float(k.split("_")[1]) for k in _keys]
                _ax[0].plot(_xs, [_j["evasion"][k][f"eps_c{_me}"] for k in _keys], "-", lw=2.5, marker="s", label=f"ECA {_f[3:-5]} (eps_c={_me})")
    except Exception as _e:  # partial / older result files must not kill the report
        log(f"S12: headline panel 1 incomplete ({type(_e).__name__}: {_e})")
    _ax[0].set_xlabel("l2 attack budget")
    _ax[0].set_ylabel("accepted-fake rate")
    _ax[0].set_title("Defended accepted-fake rate (held-out generators)")
    _ax[0].legend(fontsize=7)
    if os.path.exists(os.path.join(_r, "S10_floor.json")):
        _j = load_json(os.path.join(_r, "S10_floor.json"))
        _j["models"] = {k: v for k, v in _j.get("models", {}).items() if "robust_pairs" in v}
        _ax[1].plot(_j["eps_grid"], _j["floor_three_way"], "k-", label="three-way floor P(||a||>eps)")
        _ax[1].plot(_j["eps_grid"], _j["cap_two_way"], "k--", label="two-way cap P(||a||>2 eps)")
        for _n, _m in _j["models"].items():
            _ax[1].plot(_j["eps_grid"], _m["robust_pairs"], marker="o", label=f"ECA {_n}")
        _ax[1].set_xscale("symlog", linthresh=0.05)
        _ax[1].set_xlabel("l2 budget eps")
        _ax[1].set_ylabel("share of robustly authenticated pairs")
        _ax[1].set_title("Evidence-energy floor (SD-VAE pairs)")
        _ax[1].legend(fontsize=7)
    _fig.tight_layout()
    _png = os.path.join(_r, "headline.png")
    _fig.savefig(_png, dpi=130)
    plt.close(_fig)
    hf_push(_r, "S12 report")
    log(f"S12: report written to {_r}")
    return {"table_rows": len(_clean), "figure": _png, "missing": int(len(_missing)),
            "claims_backed": sum(_claims.values()), "report": os.path.join(_r, "REPORT.md")}

In [ ]:
# ---- RUN STATUS: what exists, what is left out. Reads files only; never trains. -------
_cov, _claims = coverage_report()
_miss = _cov[_cov.status == "MISSING"]
mo.vstack([
    mo.md(f"### Run status\n**Training lock:** {'OFF (ALLOW_TRAIN=1 or SMOKE)' if CFG['allow_train'] else 'ON (S5 will not train)'}  \n"
          f"**Artefacts present:** {len(_cov) - len(_miss)} / {len(_cov)}  \n"
          + "  \n".join(f"{'✅' if v else '⬜'} {k}" for k, v in _claims.items())),
    mo.ui.table(_cov, page_size=30, selection=None),
])

In [ ]:
# ---- RUN CONTROLS ----------------------------------------------------------------------
STAGE_NAMES = ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10", "S11", "S12"]
stages_ui = mo.ui.multiselect(options=STAGE_NAMES, value=[], label="Stages to run (in order)")
go_btn = mo.ui.run_button(label="Run selected stages")
mo.hstack([stages_ui, go_btn], justify="start")

In [ ]:
# ---- RUNNER: executes the chosen stages in pipeline order -----------------------------
_env = os.environ.get("RUN_STAGES", "")
_sel = set(stages_ui.value) if go_btn.value else set()
if _env:
    _sel |= set(STAGE_NAMES) if _env == "all" else {s.strip() for s in _env.split(",")}
mo.stop(not _sel, mo.md("Pick stages and press **Run selected stages** (or set `RUN_STAGES`)."))
if "S5" in _sel and not CFG["allow_train"]:
    log("S5 requested but training is locked (ALLOW_TRAIN not set): S5 only fetches existing weights")
_fns = {"S0": stage_S0_theory, "S1": stage_S1_index, "S2": stage_S2_cache, "S3": stage_S3_openfake, "S4": stage_S4_raid,
        "S5": stage_S5_train, "S6": stage_S6_baselines, "S7": stage_S7_attacks, "S8": stage_S8_certify,
        "S9": stage_S9_eot, "S10": stage_S10_floor, "S11": stage_S11_seq, "S12": stage_S12_report}
RUN_LOG = []
for _s in STAGE_NAMES:
    if _s in _sel:
        _t = time.time()
        log(f"=== {_s} start")
        try:
            _o = _fns[_s]()
        except Exception as _e:  # one failing stage must not block the rest (esp. S12)
            _o = f"FAILED: {type(_e).__name__}: {_e}"
            log(f"=== {_s} {_o}")
        RUN_LOG.append({"stage": _s, "minutes": round((time.time() - _t) / 60, 2), "out": str(_o)[:300]})
        log(f"=== {_s} done in {RUN_LOG[-1]['minutes']} min; RAM {trim_ram():.1f} GB")
RUN_LOG

In [ ]:
# ---- RESULTS VIEWER: reads whatever is on disk (independent of the runner) -------------
_r = PATHS["res"]
_parts = []
if os.path.exists(os.path.join(_r, "table_clean.csv")):
    _t = pd.read_csv(os.path.join(_r, "table_clean.csv"))
    _t = _t[_t.set.isin(["cf_test", "openfake", "raid_epsilon16"])][
        ["method", "set", "auroc", "AF", "FR", "cov_real", "cov_fake", "accept_precision"]]
    _parts.append(mo.md("### Clean / transfer results (three-way)"))
    _parts.append(mo.ui.table(_t.round(4), page_size=20))
for _f in sorted(os.listdir(_r)) if os.path.isdir(_r) else []:
    if _f.startswith(("S7_", "S9_")) and _f.endswith(".json"):
        _j = load_json(os.path.join(_r, _f))
        _k = "joint" if _f.startswith("S7_") else "evasion"
        _parts.append(mo.md(f"**{_f}** ({_k})"))
        _parts.append(mo.ui.table(pd.DataFrame(_j[_k]).T.round(3), page_size=10))
if os.path.exists(os.path.join(_r, "headline.png")):
    _parts.append(mo.image(os.path.join(_r, "headline.png")))
mo.vstack(_parts) if _parts else mo.md("_No results yet._")

## Part II: corrected calibration

The flag rule is "flag if S_hi(x') < tau_f - eps". For the framing guarantee, a flagged attacked
real must imply S(x) < tau_f, and S(x) < tau_f implies S_lo(x) < tau_f, so tau_f has to be
calibrated on **S_lo of reals**; the first pass (S8) calibrated it on S_hi. The acceptance side
(tau_a on S_hi of fakes) is unchanged. The first pass's alpha = beta = 1% also turned out
infeasible for the smoothed models, so F4 reports the pre-registered grid alpha, beta in
{1, 2, 5, 10}% from the saved vote counts, and F5 re-runs S9–S11 at the corrected thresholds.

In [ ]:
# ---- F1. PULL THE FIRST-PASS RESULTS FROM THE HUB (resume) -----------------------------------
def fu_rel(path):
    return os.path.relpath(path, CFG["work"]).replace(os.sep, "/")

FU_HUB = bool(HF_TOKEN and CFG["hf"]["results_repo"])

def fu_hub_files():
    if not FU_HUB:
        return set()
    try:
        return set(HF_API.list_repo_files(CFG["hf"]["results_repo"], repo_type="model"))
    except Exception as _e:
        print(f"[hub] listing failed: {type(_e).__name__}")
        return set()

def fu_hub_pull(patterns):
    if not FU_HUB:
        return False
    try:
        snapshot_download(CFG["hf"]["results_repo"], repo_type="model", local_dir=CFG["work"],
                          token=HF_TOKEN, allow_patterns=patterns)
        return True
    except Exception as _e:
        print(f"[hub] pull failed: {type(_e).__name__}")
        return False

def fu_push(message):
    """Results folder only (never weights, never data)."""
    return hf_push(PATHS["res"], message)

_ok = fu_hub_pull([fu_rel(PATHS["res"]) + "/*", fu_rel(PATHS["data"]) + "/cf_selection.parquet"])
{"hub": CFG["hf"]["results_repo"] or "(HF_RESULTS_REPO unset)", "hub_on": FU_HUB, "pulled": _ok,
 "result_files_local": len(os.listdir(PATHS["res"])), "results_folder": os.path.basename(PATHS["res"]),
 "disk_free_gb": round(shutil.disk_usage(CFG["work"]).free / 2**30, 1)}

In [ ]:
# ---- F2. INVENTORY ------------------------------------------------------------------------
FU_BASELINES = ["base", "pgdat", "clip"]
FU_SMOOTHED = ([(f"eca_{s}", s) for s in CFG["eca"]["sigmas"]]
               + [(f"cohen_{s}", s) for s in CFG["eca"]["cohen_sigmas"]])
FU_SIG = dict(FU_SMOOTHED)
FU_SETS = (["cf_test", "cf_test_jpeg75", "cf_test_resize075", "cf_test_blur1", "openfake"]
           + [f"raid_{b}" for b in CFG["raid"]["budgets"]])
FU_SET_SOURCE = {s: ("of" if s == "openfake" else "raid" if s.startswith("raid_") else "cf") for s in FU_SETS}

def fu_weight(name):
    return os.path.join(PATHS["models"], name, "head.pt" if name == "clip" else "final.pt")

def fu_res(fname):
    return os.path.join(PATHS["res"], fname)

def fu_inventory():
    _hub = fu_hub_files()
    _where = lambda p: "local" if os.path.exists(p) else ("hub" if fu_rel(p) in _hub else None)
    _rows = []

    def _add(stage, unit, status, detail="", model=None, sets=None):
        _rows.append({"stage": stage, "unit": unit, "status": status, "detail": detail, "model": model, "sets": sets})

    for _sub, _st in [("cf", "S2"), ("of", "S3"), ("raid", "S4")]:
        _d = os.path.join(PATHS["data"], _sub)
        _loc = len(open_store(_d, _sub + "_").meta) if os.path.isdir(_d) else 0
        _h = sum(1 for f in _hub if f.startswith(fu_rel(_d) + "/") and f.endswith(".npy"))
        _add(_st, f"data:{_sub}", "done" if (_loc or _h) else "missing", f"{_loc} images local, {_h} chunks on Hub")
    _add("S0", "theory self-tests", "done" if _where(fu_res("S0_theory.json")) else "missing")
    _w = {n: _where(fu_weight(n)) for n in FU_BASELINES + list(FU_SIG)}
    for _n, _v in _w.items():
        _add("S5", f"weights:{_n}", "done" if _v else "missing", _v or "not trained (run S5)", _n)

    def _eval_unit(stage, n, fname):
        _p = fu_res(fname)
        if not os.path.exists(_p):
            _add(stage, f"{stage}:{n}", "missing", "", n)
            return
        _have = set(load_json(_p).get("eval", {}).keys())
        _gap = [s for s in FU_SETS if s not in _have]
        _add(stage, f"{stage}:{n}", "incomplete" if _gap else "done",
             ("sets not evaluated: " + ", ".join(_gap)) if _gap else "", n, _gap or None)

    for _n in FU_BASELINES:
        _eval_unit("S6", _n, f"S6_{_n}.json")
        _add("S7", f"S7:{_n}", "done" if os.path.exists(fu_res(f"S7_{_n}.json")) else "missing", "", _n)
    for _n in FU_SIG:
        _eval_unit("S8", _n, f"S8_{_n}.json")
    # first-pass S9-S11 use the uncorrected thresholds: kept if present, never filled
    for _st, _f in [("S9", "S9_{}.json"), ("S10", "S10_floor.json"), ("S11", "S11_seq.json")]:
        _ex = any(os.path.exists(fu_res(_f.format(n))) for n in FU_SIG) if "{}" in _f else os.path.exists(fu_res(_f))
        _add(_st, f"{_st} (original thresholds)", "done" if _ex else "missing", "superseded by the corrected version")
    for _n in FU_SIG:
        _src = fu_res(f"S8_{_n}_counts.parquet")
        _c = fu_res(f"S8c_{_n}.json")
        _st = "done" if os.path.exists(_c) else "missing"
        _add("S8c", f"S8c:{_n}", _st, "re-calibration from saved vote counts" + ("" if os.path.exists(_src) else " (needs S8 counts)"), _n)
        for _cs, _cf in [("S9c", f"S9c_{_n}.json")]:
            _add(_cs, f"{_cs}:{_n}", "done" if os.path.exists(fu_res(_cf)) else "missing", "", _n)
    for _cs, _cf in [("S10c", "S10c_floor.json"), ("S11c", "S11c_seq.json"), ("S11d", "S11d_saliency_agreement.json")]:
        _add(_cs, _cs, "done" if os.path.exists(fu_res(_cf)) else "missing")
    _df = pd.DataFrame(_rows)

    def _action(r):
        if r.status == "done":
            return "keep"
        if r.stage in ("S2", "S3", "S4"):
            return "fetch only if a gap needs it"
        if r.stage == "S5":
            return "not available (Part II never trains)"
        if r.stage in ("S9", "S10", "S11"):
            return "not needed (corrected version below)"
        if isinstance(r.model, str) and not _w.get(r.model):
            return "skip: model has no weights"
        if r.stage == "S8c":
            return "F4 correction (CPU)"
        if r.stage in ("S9c", "S10c", "S11c", "S11d"):
            return "F5 corrected run"
        if r.stage in ("S6", "S8") and r.status == "incomplete":
            return "F3: evaluate missing sets (original thresholds)"
        return f"F3: run original {r.stage} for this unit"

    _df["action"] = [_action(r) for r in _df.itertuples()]
    return _df

FU_INV0 = fu_inventory()

In [ ]:
# ---- F2b. PLAN (what this run will compute) ------------------------------------------------------------------
_f3 = FU_INV0[FU_INV0.action.str.startswith("F3")]
_f5 = FU_INV0[FU_INV0.action.str.startswith("F5")]
_est = {"S0": 0.01, "S6": 0.25, "S7": 1.2, "S8": 1.5}
_h3 = sum(_est.get(s, 0) * (0.4 if "missing sets" in a else 1) for s, a in zip(_f3.stage, _f3.action))
_h5 = 1.7 * (_f5.stage == "S9c").sum() + 0.5 * (_f5.stage == "S10c").sum() + 0.5 * (_f5.stage == "S11c").sum() + 0.2 * (_f5.stage == "S11d").sum()
mo.vstack([
    mo.md(f"### Inventory of the first pass\n"
          f"**done:** {(FU_INV0.status == 'done').sum()} · **incomplete:** {(FU_INV0.status == 'incomplete').sum()} · "
          f"**missing:** {(FU_INV0.status == 'missing').sum()}"),
    mo.ui.table(FU_INV0.drop(columns=["sets"]), page_size=50, selection=None),
    mo.md(f"**This run will compute:** {len(_f3)} gap units (~{_h3:.1f} h GPU) and {len(_f5)} corrected units (~{_h5:.1f} h GPU)."),
])

In [ ]:
# ---- F3. FILL THE GAPS (first-pass evaluation stages only; never trains; runs automatically) --------------------
_go = True
FILL_LOG = {"ran": False, "steps": []}

def fu_step(name, fn, store):
    if time_left_h() < 0.25:
        store["steps"].append({"step": name, "result": "not started: session clock nearly out; re-run"})
        return
    _t = time.time()
    try:
        _msg = str(fn())[:300]
    except Exception as _e:  # one failing unit must not block the others or the comparison
        _msg = f"FAILED: {type(_e).__name__}: {_e}"
    store["steps"].append({"step": name, "minutes": round((time.time() - _t) / 60, 2), "result": _msg})
    log(f"[correction] {name}: {_msg}")
    fu_push(f"correction: {name}")

def fu_ensure_data(sub):
    """Evaluation data: local -> Hub copy -> fetch stage (S1-S4 fetch data only)."""
    _d = os.path.join(PATHS["data"], sub)
    if os.path.isdir(_d) and len(open_store(_d, sub + "_").meta):
        return "local"
    if any(f.startswith(fu_rel(_d) + "/") for f in fu_hub_files()):
        fu_hub_pull([fu_rel(_d) + "/*"])
        if len(open_store(_d, sub + "_").meta):
            return "pulled from Hub"
    if sub == "cf":
        stage_S1_index()
        stage_S2_cache()
    elif sub == "of":
        stage_S3_openfake()
    else:
        stage_S4_raid()
    return "fetched by data stage" if len(open_store(_d, sub + "_").meta) else "STILL EMPTY (source unavailable)"

def _stores():
    return {s: open_store(os.path.join(PATHS["data"], s), s + "_") for s in ("cf", "of", "raid")}

def _set_index(stores, sn):
    _cf = stores["cf"]
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0] if len(_cf.meta) else np.zeros(0, int)
    if sn.startswith("cf_test"):
        return _cf, _test, {"cf_test": None, "cf_test_jpeg75": "jpeg75", "cf_test_resize075": "resize075",
                            "cf_test_blur1": "blur1"}[sn]
    if sn == "openfake":
        return stores["of"], np.arange(len(stores["of"].meta)), None
    _rd = stores["raid"]
    return _rd, (np.where(_rd.meta.budget.to_numpy() == sn[5:])[0] if len(_rd.meta) else np.zeros(0, int)), None

def _keep_copy(files):
    _dst = os.path.join(PATHS["res"], "_superseded", time.strftime("%Y%m%d-%H%M%S"))
    os.makedirs(_dst, exist_ok=True)
    for _f in files:
        if os.path.exists(_f):
            shutil.copy2(_f, _dst)

@torch.no_grad()
def _run_store(model, kind, store, idx, mc, launder=None, bs=64):
    _S, _Fs, _V = [], [], []
    for _i in range(0, len(idx), bs):
        _b = store.batch(idx[_i:_i + bs])
        _b = np.stack([launder_eval(a, launder) for a in _b]) if launder else center224(_b)
        _s, _f, _v = base_outputs(model, kind, to_tensor01(_b, DEVICE), mc)
        _S.append(_s.cpu())
        _Fs.append(_f.cpu())
        _V.append(_v.cpu())
    return torch.cat(_S).numpy(), torch.cat(_Fs).numpy(), torch.cat(_V).numpy()

def _complete_S6(name, gaps):
    """Only the missing sets, with the first-pass S6 thresholds and Mahalanobis fit."""
    _jp, _sp = fu_res(f"S6_{name}.json"), fu_res(f"S6_{name}_scores.parquet")
    _keep_copy([_jp, _sp])
    _j = load_json(_jp)
    _thr = _j["thresholds"]
    _mpz = np.load(fu_res(f"S6_{name}_maha.npz"))
    _mp = {k: _mpz[k] for k in ["mu_r", "P"]}
    _kind = "clip" if name == "clip" else "cnn"
    ensure(fu_weight(name))
    _model = load_any(_kind, fu_weight(name))
    _st_all, _rows, _done = _stores(), [], []
    for _sn in gaps:
        _st, _ix, _ld = _set_index(_st_all, _sn)
        if len(_ix) == 0:
            continue
        _s, _f, _v = _run_store(_model, _kind, _st, _ix, CFG["attack"]["mc_eval"], launder=_ld)
        _cm = maha_gate(_f, _mp)
        _yy = _st.meta.label.to_numpy()[_ix]
        _ent = {"auroc": auroc_real(_s, _yy), "tpr@1%fpr": tpr_at_fpr(_s, _yy)}
        for _v2 in VARIANTS[_kind]:
            _ent[_v2] = three_way_metrics(baseline_decide(_v2, _s, _cm, _v, _thr), _yy)
        _j["eval"][_sn] = _ent
        _rows.append(pd.DataFrame({"set": _sn, "s": _s, "maha": _cm, "mcvar": _v, "y": _yy,
                                   "model_name": _st.meta.model_name.to_numpy()[_ix]}))
        _done.append(_sn)
    if _rows:
        _old = [pd.read_parquet(_sp)] if os.path.exists(_sp) else []
        pd.concat(_old + _rows).to_parquet(_sp)
    _j.setdefault("followup", []).append({"added_sets": _done, "thresholds": "original"})
    save_json(_jp, _j)
    del _model
    gpu_free()
    return {"added": _done, "still_unavailable": [s for s in gaps if s not in _done]}

def _complete_S8(name, sigma, gaps):
    """Certify only the missing sets; vote counts are what the correction (F4) uses."""
    _jp, _cp = fu_res(f"S8_{name}.json"), fu_res(f"S8_{name}_counts.parquet")
    _keep_copy([_jp, _cp])
    _j = load_json(_jp)
    _th = _j["thresholds"]
    _N, _a = CFG["cert"]["N"], CFG["cert"]["alpha_mc"]
    ensure(fu_weight(name))
    _model = load_detector(fu_weight(name), DEVICE)
    _st_all, _rows, _done = _stores(), [], []
    for _sn in gaps:
        _st, _ix, _ld = _set_index(_st_all, _sn)
        if len(_ix) == 0:
            continue
        _k = certify_store(_model, _st, _ix, sigma, _ld)
        _l, _h, _p = scores_from_counts(_k, _N, sigma, _a)
        _yy = _st.meta.label.to_numpy()[_ix]
        _ent = {"auroc": auroc_real(_p, _yy), "tpr@1%fpr": tpr_at_fpr(_p, _yy)}
        for _e in CFG["cert"]["eps_cert"]:
            _ent[f"eps{_e}"] = three_way_metrics(decide(_l, _h, _th["tau_a"], _th["tau_f"], _e), _yy)
        _j["eval"][_sn] = _ent
        _df = pd.DataFrame({"set": _sn, "k": _k, "y": _yy, "model_name": _st.meta.model_name.to_numpy()[_ix]})
        if "release_date" in _st.meta:
            _df["release_date"] = _st.meta.release_date.to_numpy()[_ix]
        _rows.append(_df)
        _done.append(_sn)
    if _rows:
        _old = [pd.read_parquet(_cp)] if os.path.exists(_cp) else []
        pd.concat(_old + _rows).to_parquet(_cp)
    _j.setdefault("followup", []).append({"added_sets": _done, "thresholds": "original"})
    save_json(_jp, _j)
    del _model
    gpu_free()
    return {"added": _done, "still_unavailable": [s for s in gaps if s not in _done]}

if _go:
    FILL_LOG["ran"] = True
    _todo = FU_INV0[FU_INV0.action.str.startswith("F3")]
    for _m in sorted(set(m for m in _todo.model if isinstance(m, str))):
        ensure(fu_weight(_m))
    _need = set()
    if len(_todo[_todo.stage != "S0"]):
        _need.add("cf")
    for _r in _todo.itertuples():
        if _r.stage in ("S6", "S8"):
            _need |= ({FU_SET_SOURCE[s] for s in _r.sets} if _r.sets else set(FU_SET_SOURCE.values()))
    for _src in sorted(_need):
        fu_step(f"data {_src}", lambda _s=_src: fu_ensure_data(_s), FILL_LOG)
    _miss = lambda st: ((_todo.stage == st) & (_todo.status == "missing")).any()
    if (_todo.stage == "S0").any():
        fu_step("S0 theory self-tests", stage_S0_theory, FILL_LOG)
    if _miss("S6"):
        fu_step("S6 missing baseline units", stage_S6_baselines, FILL_LOG)
    for _r in _todo[(_todo.stage == "S6") & (_todo.status == "incomplete")].itertuples():
        fu_step(f"S6 missing sets {_r.model}", lambda _r=_r: _complete_S6(_r.model, _r.sets), FILL_LOG)
    if _miss("S7"):
        fu_step("S7 missing attack units", stage_S7_attacks, FILL_LOG)
    if _miss("S8"):
        fu_step("S8 missing certification units", stage_S8_certify, FILL_LOG)
    for _r in _todo[(_todo.stage == "S8") & (_todo.status == "incomplete")].itertuples():
        fu_step(f"S8 missing sets {_r.model}", lambda _r=_r: _complete_S8(_r.model, FU_SIG[_r.model], _r.sets), FILL_LOG)
    gc.collect()
FILL_LOG

In [ ]:
# ---- F4. CORRECTION: re-calibrate from the saved vote counts (CPU, minutes) ---------------
# Decision rule is unchanged: accept iff S_lo(x') >= tau_a + eps, flag iff S_hi(x') < tau_f - eps.
#  * tau_a: CRC over generators on S_hi of calibration fakes (as before; this was right).
#  * tau_f: conformal quantile of S_lo of calibration reals (CORRECTED; was S_hi).
#    Flagged attacked real => S(x') <= S_hi(x') < tau_f - eps => S(x) < tau_f => S_lo(x) < tau_f
#    (w.p. >= 1 - alpha_mc/2), and P(S_lo(new real) < tau_f) <= beta by exchangeability, ties included.
#  * Feasibility: acceptance is possible at budget eps iff alpha >= alpha_min(eps) =
#    K/(K+1) * mean_g P(S_hi >= sigma z_N - eps) + 1/(K+1); flagging iff beta >= beta_min(eps) =
#    (#{S_lo(real) <= eps - sigma z_N} + 1) / (n+1). Both are reported.
#  * Operating point: pre-registered grid; alpha* = smallest grid alpha whose acceptance rate of
#    selection-half reals at eps = sigma is >= USEFUL, using tau_a fitted on selection-half
#    generators; beta* likewise for flagging selection-half fakes. Thresholds are then fitted on
#    the OTHER half only, so the guarantees hold at the selected levels.
from scipy.stats import norm as _fu_norm

_ = FILL_LOG
ALPHA_GRID = [0.01, 0.02, 0.05, 0.10]
BETA_GRID = [0.01, 0.02, 0.05, 0.10]
USEFUL = 0.05

def fu_eval_counts(df, sigma, tau_a, tau_f, sets=None):
    _N, _am = CFG["cert"]["N"], CFG["cert"]["alpha_mc"]
    _out = {}
    for _sn, _g in df[df.set != "cf_cal"].groupby("set"):
        if sets and _sn not in sets:
            continue
        _l, _h, _p = scores_from_counts(_g.k.to_numpy(), _N, sigma, _am)
        _yy = _g.y.to_numpy()
        _e = {"auroc": auroc_real(_p, _yy), "tpr@1%fpr": tpr_at_fpr(_p, _yy)}
        for _eps in CFG["cert"]["eps_cert"]:
            _e[f"eps{_eps}"] = three_way_metrics(decide(_l, _h, tau_a, tau_f, _eps), _yy)
        _out[_sn] = _e
    return _out

def fu_recalibrate(name, sigma):
    _cp = fu_res(f"S8_{name}_counts.parquet")
    _df = pd.read_parquet(_cp)
    _N, _am = CFG["cert"]["N"], CFG["cert"]["alpha_mc"]
    _zN = float(_fu_norm.ppf((_am / 2) ** (1.0 / _N)))
    _smax, _smin = sigma * _zN, -sigma * _zN
    _cal = _df[_df.set == "cf_cal"]
    _k, _y, _gn = _cal.k.to_numpy(), _cal.y.to_numpy(), _cal.model_name.to_numpy()
    _lo, _hi, _ = scores_from_counts(_k, _N, sigma, _am)
    _gens = [g for g in np.unique(_gn[_y == 1]) if ((_y == 1) & (_gn == g)).sum() >= CFG["cal"]["min_per_gen"]]
    _hi_by = {g: _hi[(_y == 1) & (_gn == g)] for g in _gens}
    _ri = np.where(_y == 0)[0]
    _lo_r = _lo[_ri]
    _K, _n = len(_gens), len(_ri)
    _epsg = CFG["cert"]["eps_cert"]
    # ---- diagnostics (why alpha = beta = 1% may be infeasible)
    _top = sorted(({"generator": g, "n": int(len(v)), "share_all_votes_real": float(np.mean(_k[(_y == 1) & (_gn == g)] == _N)),
                    "share_votes_real_ge_90pct": float(np.mean(_k[(_y == 1) & (_gn == g)] >= 0.9 * _N))}
                   for g, v in _hi_by.items()), key=lambda d: -d["share_votes_real_ge_90pct"])[:15]
    _diag = {
        "K_generators": _K, "n_real": _n, "z_N": _zN, "max_S_lo": _smax, "min_S_hi": _smin,
        "share_cal_reals_all_votes_fake": float(np.mean(_k[_ri] == 0)),
        "share_cal_fakes_all_votes_real": float(np.mean(_k[_y == 1] == _N)),
        "alpha_min_for_any_acceptance": {str(e): float(_K / (_K + 1) * np.mean([np.mean(v >= _smax - e) for v in _hi_by.values()]) + 1 / (_K + 1)) for e in _epsg},
        "beta_min_for_any_flagging": {str(e): float((np.sum(_lo_r <= _smin + e) + 1) / (_n + 1)) for e in _epsg},
        "hardest_generators": _top,
    }
    # ---- full pre-registered grid on the whole calibration set (each point valid on its own)
    _grid = {}
    for _al in ALPHA_GRID:
        _ta, _inf = crc_accept_threshold(list(_hi_by.values()), _al)
        for _be in BETA_GRID:
            _tf = conformal_flag_threshold(_lo_r, _be)
            _grid[f"alpha={_al},beta={_be}"] = {
                "alpha": _al, "beta": _be, "tau_a": max(_ta, _tf), "tau_f": _tf, "crc": _inf,
                "eval": fu_eval_counts(_df, sigma, max(_ta, _tf), _tf, sets=["cf_test", "openfake"])}
    # ---- split-sample selection of the main operating point
    _rng = np.random.default_rng(CFG["seed"] + 2027)
    _gp = list(_rng.permutation(_gens))
    _GA, _GB = _gp[: _K // 2], _gp[_K // 2:]
    _rp = _rng.permutation(_ri)
    _RA, _RB = _rp[: _n // 2], _rp[_n // 2:]
    _fA = np.concatenate([_hi_by[g] for g in _GA]) if _GA else np.zeros(0)
    _sel_a = None
    for _al in ALPHA_GRID:
        _ta, _ = crc_accept_threshold([_hi_by[g] for g in _GA], _al)
        if np.isfinite(_ta) and np.mean(_lo[_RA] >= _ta + sigma) >= USEFUL:
            _sel_a = _al
            break
    _sel_b = None
    for _be in BETA_GRID:
        _tf = conformal_flag_threshold(_lo[_RA], _be)
        if np.isfinite(_tf) and np.mean(_fA < _tf - sigma) >= USEFUL:
            _sel_b = _be
            break
    _a_star = _sel_a if _sel_a is not None else ALPHA_GRID[-1]
    _b_star = _sel_b if _sel_b is not None else BETA_GRID[-1]
    _ta, _inf = crc_accept_threshold([_hi_by[g] for g in _GB], _a_star)
    _tf = conformal_flag_threshold(_lo[_RB], _b_star)
    _th = {"tau_a": max(_ta, _tf), "tau_f": _tf, "alpha": _a_star, "beta": _b_star, "K": len(_GB),
           "n_real_calib": int(len(_RB)), "crc": _inf, "accept_basis": "S_hi of fakes (CRC over generators)",
           "flag_basis": "S_lo of reals (corrected)",
           "selection": {"rule": f"smallest grid level reaching {USEFUL:.0%} usefulness at eps=sigma on the selection half",
                         "alpha_selected": _sel_a, "beta_selected": _sel_b,
                         "acceptance_useful": _sel_a is not None, "flagging_useful": _sel_b is not None}}
    _ev = fu_eval_counts(_df, sigma, _th["tau_a"], _th["tau_f"])
    # held-out generator check (cf_test generators never touched calibration)
    _t = _df[(_df.set == "cf_test") & (_df.y == 1)]
    _hc = {}
    if len(_t):
        _l, _h, _ = scores_from_counts(_t.k.to_numpy(), _N, sigma, _am)
        _pg = (_t.assign(acc=_l >= _th["tau_a"], bound=_h >= _th["tau_a"]).groupby("model_name")
               .agg(n=("k", "size"), acc=("acc", "mean"), bound=("bound", "mean")))
        _pg = _pg[_pg.n >= CFG["cal"]["min_per_gen"]]
        _r = _df[(_df.set == "cf_test") & (_df.y == 0)]
        _lr, _, _ = scores_from_counts(_r.k.to_numpy(), _N, sigma, _am)
        _hc = {"n_test_generators": int(len(_pg)),
               "mean_AF_clean_over_test_generators": float(_pg.acc.mean()) if len(_pg) else None,
               "mean_P(S_hi>=tau_a)_over_test_generators (guarantee event, target <= alpha)": float(_pg.bound.mean()) if len(_pg) else None,
               "alpha": _a_star,
               "P(S_lo(real) < tau_f) on test reals (target <= beta)": float(np.mean(_lr < _th["tau_f"])) if len(_lr) else None,
               "beta": _b_star}
    # Theorem 5 (quantile version, audited at alpha_q): fitted on the calibration half
    _tq, _ = quantile_accept_threshold([_hi_by[g] for g in _GB], CFG["cal"]["alpha_q"], CFG["cal"]["eta"], CFG["cal"]["delta"])
    _th["tau_q"] = float(max(_tq, _th["tau_f"]))

    def _per_gen(setname):
        _t = _df[(_df.set == setname) & (_df.y == 1)]
        if not len(_t):
            return []
        _l, _h, _ = scores_from_counts(_t.k.to_numpy(), _N, sigma, _am)
        _t = _t.assign(acc0=_l >= _th["tau_a"], acc_sig=_l >= _th["tau_a"] + sigma, bound=_h >= _th["tau_a"],
                       acc_q=_h >= _th["tau_q"], vote_real=_t.k.to_numpy() / _N)
        _agg = {"n": ("k", "size"), "AF_clean": ("acc0", "mean"), "AF_certified_eps_sigma": ("acc_sig", "mean"),
                "P_S_hi_ge_tau_a": ("bound", "mean"), "AF_q": ("acc_q", "mean"), "mean_real_vote": ("vote_real", "mean")}
        if "release_date" in _t.columns:
            _agg["release_date"] = ("release_date", "first")
        _g = _t.groupby("model_name").agg(**_agg).reset_index()
        if "release_date" in _g.columns:
            _g["release_date"] = _g["release_date"].astype(str)
        return _g.to_dict("records")

    _pg_cf, _pg_of = _per_gen("cf_test"), _per_gen("openfake")
    _elig = [g for g in _pg_cf if g["n"] >= CFG["cal"]["min_per_gen"]]
    _thm5 = {"alpha_q": CFG["cal"]["alpha_q"], "tau_q": _th["tau_q"],
             "share_test_generators_AF_q<=alpha_q": float(np.mean([g["AF_q"] <= CFG["cal"]["alpha_q"] for g in _elig])) if _elig else None,
             "target_share (1-delta-eta)": 1 - CFG["cal"]["delta"] - CFG["cal"]["eta"],
             "share_test_generators_P(S_hi>=tau_a)<=alpha*": float(np.mean([g["P_S_hi_ge_tau_a"] <= _a_star for g in _elig])) if _elig else None,
             "n_test_generators": len(_elig)}
    # RQ3: vote distributions on held-out generators
    _tt = _df[_df.set == "cf_test"]
    _vf, _vr = _tt[_tt.y == 1].k.to_numpy() / _N, _tt[_tt.y == 0].k.to_numpy() / _N
    _votes = {"fake_real_vote_q99": float(np.quantile(_vf, 0.99)) if len(_vf) else None,
              "share_fakes_le_10pct_real_votes": float(np.mean(_vf <= 0.10)) if len(_vf) else None,
              "real_real_vote_median": float(np.median(_vr)) if len(_vr) else None,
              "share_reals_ge_90pct_real_votes": float(np.mean(_vr >= 0.90)) if len(_vr) else None}
    _as_run = load_json(fu_res(f"S8_{name}.json"))
    _out = {"sigma": sigma, "thresholds": _th, "as_run_thresholds": _as_run.get("thresholds"),
            "diagnostics": _diag, "grid": _grid, "eval": _ev, "heldout_check": _hc,
            "theorem5_quantile": _thm5, "votes": _votes,
            "per_generator": {"cf_test": _pg_cf, "openfake": _pg_of},
            "source_rows": int(len(_df)), "version": 2}
    save_json(fu_res(f"S8c_{name}.json"), _out)
    return _out

RECAL = {}
for _n, _s in FU_SMOOTHED:
    if not os.path.exists(fu_res(f"S8_{_n}_counts.parquet")) or not os.path.exists(fu_res(f"S8_{_n}.json")):
        continue
    _c = fu_res(f"S8c_{_n}.json")
    if os.path.exists(_c) and load_json(_c).get("version") == 2 and load_json(_c).get("source_rows") == len(pd.read_parquet(fu_res(f"S8_{_n}_counts.parquet"), columns=["k"])):
        RECAL[_n] = load_json(_c)
        continue
    RECAL[_n] = fu_recalibrate(_n, _s)
    log(f"[correction] {_n}: tau_a {RECAL[_n]['thresholds']['tau_a']:.3f} tau_f {RECAL[_n]['thresholds']['tau_f']:.3f} "
        f"alpha* {RECAL[_n]['thresholds']['alpha']} beta* {RECAL[_n]['thresholds']['beta']}")
if RECAL:
    fu_push("correction (S8c)")
# Thresholds handed to the corrected S9-S11 (all models).
TH_MAIN = {}
for _n, _r in RECAL.items():
    _t = _r["thresholds"]
    # A model is attacked unless its corrected rule can neither accept nor flag at ANY radius
    # (eps_c = 0 is the loosest). Only then are its attack outcomes identically zero by
    # construction, so attacking it would add no information.
    _can_accept = np.isfinite(_t["tau_a"]) and _t["tau_a"] <= _r["diagnostics"]["max_S_lo"]
    _can_flag = np.isfinite(_t["tau_f"]) and _t["tau_f"] > _r["diagnostics"]["min_S_hi"]
    # Every model is attacked, usable or not: the post-attack vote counts are saved, so
    # any other alpha / beta can be evaluated later without re-attacking.
    TH_MAIN[_n] = dict(_t, can_accept_at_sigma=bool(_can_accept), can_flag_at_sigma=bool(_can_flag))
RECAL_TABLE = pd.DataFrame([{
    "model": _n, "alpha*": _r["thresholds"]["alpha"], "beta*": _r["thresholds"]["beta"],
    "tau_a": _r["thresholds"]["tau_a"], "tau_f": _r["thresholds"]["tau_f"],
    "as-run tau_a": (_r["as_run_thresholds"] or {}).get("tau_a"),
    "alpha_min (eps=sigma)": _r["diagnostics"]["alpha_min_for_any_acceptance"].get(str(_r["sigma"])),
    "beta_min (eps=sigma)": _r["diagnostics"]["beta_min_for_any_flagging"].get(str(_r["sigma"])),
    "accept_real cf_test (eps=sigma)": _r["eval"].get("cf_test", {}).get(f"eps{_r['sigma']}", {}).get("accept_real_rate"),
    "AF cf_test (eps=sigma)": _r["eval"].get("cf_test", {}).get(f"eps{_r['sigma']}", {}).get("AF"),
    "FR cf_test (eps=sigma)": _r["eval"].get("cf_test", {}).get(f"eps{_r['sigma']}", {}).get("FR"),
    "can accept at eps=sigma": TH_MAIN[_n]["can_accept_at_sigma"],
    "can flag at eps=sigma": TH_MAIN[_n]["can_flag_at_sigma"]} for _n, _r in RECAL.items()])
RECAL_TABLE

In [ ]:
# ---- F5a. CORRECTED S9: adaptive EOT attacks at the corrected thresholds -----------------
# Same attack as S9; also saves the post-attack vote counts, so any later
# re-calibration can re-decide offline without re-attacking.
def stage_S9c_eot(th_map):
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _ac, _c = CFG["attack"], CFG["cert"]
    _fk = _test[_y[_test] == 1][: _ac["n"]]
    _rl = _test[_y[_test] == 0][: _ac["n"]]
    _out = {}
    for _name, _sigma, _path in smoothed_models():
        if _name not in th_map:
            continue
        _jp = os.path.join(PATHS["res"], f"S9c_{_name}.json")
        if os.path.exists(_jp):
            _out[_name] = "exists"
            continue
        _th = th_map[_name]
        _model = load_detector(_path, DEVICE)
        for _p in _model.parameters():
            _p.requires_grad_(False)
        _res = {"thresholds": _th, "evasion": {}, "framing": {}, "evasion_laundered": {k: {} for k in ("jpeg75", "resize075", "blur1")}}
        _rows = []
        _grid = [("linf", e) for e in _ac["linf"]] + [("l2", e) for e in _ac["l2"]]
        for _nrm, _eps in _grid:
            if time_left_h() < 0.25:
                raise RuntimeError("session clock nearly out; re-run to continue (finished models are kept)")
            _key = f"{_nrm}_{_eps * 255:.0f}/255" if _nrm == "linf" else f"{_nrm}_{_eps}"
            for _goal, _ix, _tag in [("real", _fk, "evasion"), ("fake", _rl, "framing")]:
                _ks = []
                _kl = {k: [] for k in _res["evasion_laundered"]}
                for _i in range(0, len(_ix), _ac["batch"]):
                    _x0 = to_tensor01(center224(_cf.batch(_ix[_i:_i + _ac["batch"]])), DEVICE)
                    _xa = eot_smoothed_attack(_model, _x0, _sigma, _eps, _nrm, _ac["eot_steps"], _ac["eot_m"], DEVICE, goal=_goal)
                    _ks.append(smooth_votes(_model, _xa, _sigma, _c["N"], _c["chunk"], DEVICE))
                    if _tag == "evasion":  # protocol: laundering applied AFTER the attack
                        _u8 = (_xa.detach().clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy()
                        for _lk in _kl:
                            _xl = to_tensor01(np.stack([launder_eval(a, _lk, crop=a.shape[0]) for a in _u8]), DEVICE)
                            _kl[_lk].append(smooth_votes(_model, _xl, _sigma, _c["N"], _c["chunk"], DEVICE))
                for _lk, _v in _kl.items():
                    if _v:
                        _kv = np.concatenate(_v)
                        _rows.append(pd.DataFrame({"attack": _key, "goal": f"evasion+{_lk}", "k": _kv}))
                        _ll, _hl, _ = scores_from_counts(_kv, _c["N"], _sigma, _c["alpha_mc"])
                        _res["evasion_laundered"][_lk][_key] = {f"eps_c{_e}": float(np.mean(decide(_ll, _hl, _th["tau_a"], _th["tau_f"], _e) == 2))
                                                                for _e in _c["eps_cert"]}
                _kk = np.concatenate(_ks)
                _rows.append(pd.DataFrame({"attack": _key, "goal": _tag, "k": _kk}))
                _l, _h, _ = scores_from_counts(_kk, _c["N"], _sigma, _c["alpha_mc"])
                _res[_tag][_key] = {f"eps_c{_e}": float(np.mean(decide(_l, _h, _th["tau_a"], _th["tau_f"], _e) == (2 if _tag == "evasion" else 0)))
                                    for _e in _c["eps_cert"]}
            log(f"S9c[{_name}] {_key}: AF {_res['evasion'][_key]} | FR {_res['framing'][_key]}")
        pd.concat(_rows).to_parquet(os.path.join(PATHS["res"], f"S9c_{_name}_counts.parquet"))
        save_json(_jp, _res)
        _out[_name] = "done"
        del _model
        gpu_free()
        fu_push(f"S9c {_name}")
    return _out

In [ ]:
# ---- F5c. CORRECTED S10: EVIDENCE-ENERGY FLOOR at corrected thresholds; with SD-VAE reconstruction pairs (Theorem 4) ----------
def stage_S10c_floor(th_map):
    from diffusers import AutoencoderKL

    if ensure(os.path.join(PATHS["res"], "S10c_floor.json")):
        log("S10c: results exist, skipping")
        return {"a_l2_median": load_json(os.path.join(PATHS["res"], "S10c_floor.json"))["a_l2"]["median"]}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _rl = _test[_y[_test] == 0][: CFG["floor"]["n"]]
    _vae = AutoencoderKL.from_pretrained(CFG["floor"]["vae"]).to(DEVICE).eval()
    _xr, _xf = [], []
    with torch.no_grad():
        for _i in range(0, len(_rl), 16):
            _x = to_tensor01(_cf.batch(_rl[_i:_i + 16]), DEVICE)  # full 256 px
            _z = _vae.encode(_x * 2 - 1).latent_dist.mode()
            _rec = ((_vae.decode(_z).sample + 1) / 2).clamp(0, 1)
            # quantise like a saved PNG, then centre-crop to the detector's 224 px input
            _rec = (_rec * 255).round() / 255
            _xr.append(_x[:, :, 16:240, 16:240].cpu())
            _xf.append(_rec[:, :, 16:240, 16:240].cpu())
    del _vae
    gpu_free()
    _xr, _xf = torch.cat(_xr), torch.cat(_xf)
    _anorm = (_xf - _xr).flatten(1).norm(dim=1).numpy()
    _eg = np.array([0.0, 0.06, 0.12, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0])
    _res = {"a_l2": {"median": float(np.median(_anorm)), "q05": float(np.quantile(_anorm, 0.05)),
                     "q95": float(np.quantile(_anorm, 0.95))},
            "eps_grid": _eg.tolist(),
            "floor_three_way": [float(np.mean(_anorm > e)) for e in _eg],
            "cap_two_way": [float(np.mean(_anorm > 2 * e)) for e in _eg],
            "models": {}}
    _N, _a = CFG["cert"]["N"], CFG["cert"]["alpha_mc"]
    for _name, _sigma, _path in smoothed_models():
        if _name not in th_map:
            continue
        _th = th_map[_name]
        _m = load_detector(_path, DEVICE)
        _kr = np.concatenate([smooth_votes(_m, _xr[i:i + 8].to(DEVICE), _sigma, _N, CFG["cert"]["chunk"], DEVICE) for i in range(0, len(_xr), 8)])
        _kf = np.concatenate([smooth_votes(_m, _xf[i:i + 8].to(DEVICE), _sigma, _N, CFG["cert"]["chunk"], DEVICE) for i in range(0, len(_xf), 8)])
        _lr, _hr, _ = scores_from_counts(_kr, _N, _sigma, _a)
        _lf, _hf, _ = scores_from_counts(_kf, _N, _sigma, _a)
        # Robustly authenticated pair at budget e: real certified-accepted AND fake certified never-accepted.
        _R = [float(np.mean((_lr >= _th["tau_a"] + e) & (_hf < _th["tau_a"]))) for e in _eg]
        from scipy.special import ndtri as _ndtri

        _zN = float(_ndtri((_a / 2) ** (1.0 / _N)))
        _res["models"][_name] = {"robust_pairs": _R,
                                 "upper_bound_min(floor, sampling cap)": [float(min(f, 1.0 if e < 2 * _sigma * _zN else 0.0)) for f, e in zip(_res["floor_three_way"], _eg)],
                                 "thresholds": _th,
                                 "AF_vae_clean": float(np.mean(_lf >= _th["tau_a"] + (CFG["cert"]["main_eps"] or _sigma)))}
        log(f"S10c[{_name}]: robust pairs {_R[:5]} vs floor {_res['floor_three_way'][:5]}")
        del _m
        gpu_free()
    save_json(os.path.join(PATHS["res"], "S10c_floor.json"), _res)
    hf_push(PATHS["res"], "S10c floor (corrected)")
    return {"a_l2_median": _res["a_l2"]["median"]}

In [ ]:
# ---- F5d. CORRECTED S11: SEQUENTIAL CERTIFICATION, EXPLANATION CERTIFICATES, STEIN SALIENCY ----------
# seq_certify, explain and stein_saliency are defined in S11; only the thresholds change here.
from scipy.stats import norm as _nrm


def stage_S11c_seq(th_map):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    if ensure(os.path.join(PATHS["res"], "S11c_seq.json")):
        log("S11c: results exist, skipping")
        return {k: v["mean_samples"] for k, v in load_json(os.path.join(PATHS["res"], "S11c_seq.json")).items()}
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _rng = np.random.default_rng(CFG["seed"])
    _ix = _rng.choice(_test, min(len(_test), CFG["seq"]["n"]), replace=False)
    _res = {}
    for _name, _sigma, _path in smoothed_models():
        if _name not in th_map:
            continue
        _th = th_map[_name]
        _eps = CFG["cert"]["main_eps"] or _sigma
        _m = load_detector(_path, DEVICE)
        _x = to_tensor01(center224(_cf.batch(_ix)), DEVICE)
        _t0 = time.time()
        _ds, _ns = [], []
        for _i in range(0, len(_ix), 32):
            _d, _n = seq_certify(_m, _x[_i:_i + 32], _sigma, _th, _eps, CFG["seq"]["block"], CFG["seq"]["n_max"], CFG["cert"]["alpha_mc"])
            _ds.append(_d)
            _ns.append(_n)
        _dt = (time.time() - _t0) / max(1, len(_ix))
        _ds, _ns = np.concatenate(_ds), np.concatenate(_ns)
        _kf = np.concatenate([smooth_votes(_m, _x[i:i + 8], _sigma, CFG["cert"]["N"], CFG["cert"]["chunk"], DEVICE) for i in range(0, len(_ix), 8)])
        _l, _h, _ = scores_from_counts(_kf, CFG["cert"]["N"], _sigma, CFG["cert"]["alpha_mc"])
        _full = decide(_l, _h, _th["tau_a"], _th["tau_f"], _eps)
        _ex = [explain(float(_l[i]), float(_h[i]), _th, _eps, _th.get("alpha", CFG["cal"]["alpha"]), CFG["cal"]["delta"]) for i in range(min(6, len(_ix)))]
        _res[_name] = {"mean_samples": float(_ns.mean()), "ms_per_image": 1000 * _dt,
                       "agreement_with_fixed_N": float(np.mean(_ds == _full)), "examples": _ex,
                       "labels_of_examples": _y[_ix[:6]].tolist()}
        # Stein saliency for a few test fakes
        _fk = _test[_y[_test] == 1][: CFG["seq"]["n_saliency"]]
        _fig, _axs = plt.subplots(2, len(_fk), figsize=(3 * len(_fk), 6), squeeze=False)
        for _j, _i in enumerate(_fk):
            _img = center224(_cf.get(int(_i)))
            _g, _p = stein_saliency(_m, to_tensor01(_img[None], DEVICE), _sigma, CFG["seq"]["n_max"], CFG["seq"]["block"])
            _sal = np.abs(_g).sum(0)
            _axs[0, _j].imshow(_img)
            _axs[0, _j].set_title(f"fake, p_real={_p:.2f}", fontsize=8)
            _axs[1, _j].imshow(_sal, cmap="magma")
            _axs[1, _j].set_title("|grad S| (Stein)", fontsize=8)
            for _a in _axs[:, _j]:
                _a.axis("off")
        _fig.tight_layout()
        _fig.savefig(os.path.join(PATHS["res"], f"S11c_{_name}_saliency.png"), dpi=110)
        plt.close(_fig)
        log(f"S11c[{_name}]: mean samples {_res[_name]['mean_samples']:.0f}, {_res[_name]['ms_per_image']:.1f} ms/img")
        del _m, _x
        gpu_free()
    save_json(os.path.join(PATHS["res"], "S11c_seq.json"), _res)
    hf_push(PATHS["res"], "S11c sequential (corrected)")
    return {k: v["mean_samples"] for k, v in _res.items()}

In [ ]:
# ---- F5e. STEIN SALIENCY vs KNOWN ARTIFACTS on aligned SD-VAE pairs (secondary question) ---
# For a pair (x_real, x_fake = VAE(x_real)) the artifact map a = x_fake - x_real is known exactly,
# so we can measure whether the certification-sample saliency |grad S| points at it.
def stage_S11d_saliency(th_map, n_pairs=32):
    from diffusers import AutoencoderKL
    from scipy.stats import spearmanr as _spr

    _jp = os.path.join(PATHS["res"], "S11d_saliency_agreement.json")
    if os.path.exists(_jp):
        return load_json(_jp)
    _cf = open_store(os.path.join(PATHS["data"], "cf"), "cf_")
    _y = _cf.meta.label.to_numpy()
    _test = np.where(_cf.meta.split.to_numpy() == "test")[0]
    _rl = _test[_y[_test] == 0][:n_pairs]
    _vae = AutoencoderKL.from_pretrained(CFG["floor"]["vae"]).to(DEVICE).eval()
    _xr, _xf = [], []
    with torch.no_grad():
        for _i in range(0, len(_rl), 8):
            _x = to_tensor01(_cf.batch(_rl[_i:_i + 8]), DEVICE)
            _rec = ((_vae.decode(_vae.encode(_x * 2 - 1).latent_dist.mode()).sample + 1) / 2).clamp(0, 1)
            _rec = (_rec * 255).round() / 255
            _xr.append(_x[:, :, 16:240, 16:240].cpu())
            _xf.append(_rec[:, :, 16:240, 16:240].cpu())
    del _vae
    gpu_free()
    _xr, _xf = torch.cat(_xr), torch.cat(_xf)
    _amap = (_xf - _xr).abs().sum(1).numpy()
    _pool = lambda m: m.reshape(28, 8, 28, 8).mean((1, 3))  # 8x8 pooling tames Monte-Carlo noise
    _res = {"n_pairs": int(len(_xf)), "iou_top10pct_if_random": 0.01 / 0.19, "models": {}}
    for _name, _sigma, _path in smoothed_models():
        if _name not in th_map:
            continue
        _m = load_detector(_path, DEVICE)
        _rho, _iou = [], []
        for _j in range(len(_xf)):
            _g, _ = stein_saliency(_m, _xf[_j:_j + 1].to(DEVICE), _sigma, CFG["seq"]["n_max"], CFG["seq"]["block"])
            _sp, _ap = _pool(np.abs(_g).sum(0)), _pool(_amap[_j])
            _rho.append(float(_spr(_sp.ravel(), _ap.ravel()).correlation))
            _ts, _ta = _sp >= np.quantile(_sp, 0.9), _ap >= np.quantile(_ap, 0.9)
            _iou.append(float((_ts & _ta).sum() / max(1, (_ts | _ta).sum())))
        _res["models"][_name] = {"spearman_mean": float(np.nanmean(_rho)), "spearman_median": float(np.nanmedian(_rho)),
                                 "iou_top10pct_mean": float(np.mean(_iou))}
        log(f"S11d[{_name}]: saliency vs artifact spearman {_res['models'][_name]['spearman_mean']:.3f}, "
            f"top-10% IoU {_res['models'][_name]['iou_top10pct_mean']:.3f} (random {_res['iou_top10pct_if_random']:.3f})")
        del _m
        gpu_free()
    save_json(_jp, _res)
    hf_push(PATHS["res"], "S11d saliency agreement")
    return _res

In [ ]:
# ---- F5b. RUN CORRECTED S9-S11 (runs automatically) ------------------------------
_go = True
CORR_LOG = {"ran": False, "steps": [], "models": sorted(TH_MAIN)}
if _go and TH_MAIN:
    CORR_LOG["ran"] = True
    fu_step("data cf", lambda: fu_ensure_data("cf"), CORR_LOG)
    fu_step("S9c EOT attacks", lambda: stage_S9c_eot(TH_MAIN), CORR_LOG)
    fu_step("S10c evidence floor", lambda: stage_S10c_floor(TH_MAIN), CORR_LOG)
    fu_step("S11c sequential + explanations", lambda: stage_S11c_seq(TH_MAIN), CORR_LOG)
    fu_step("S11d saliency vs known artifacts", lambda: stage_S11d_saliency(TH_MAIN), CORR_LOG)
elif _go:
    CORR_LOG["steps"].append({"step": "all", "result": "no model has a usable corrected operating point; nothing to attack"})
CORR_LOG

In [ ]:
# ---- F6. OVERALL COMPARISON (reads result files only) -----------------------------------
from scipy.stats import beta as _bdist

_ = (FILL_LOG, CORR_LOG, RECAL)
CMP_DIR = os.path.join(PATHS["res"], "comparison")
os.makedirs(CMP_DIR, exist_ok=True)
FU_INV1 = fu_inventory()
_lj = lambda f: load_json(fu_res(f)) if os.path.exists(fu_res(f)) else None
_S6 = {n: _lj(f"S6_{n}.json") for n in FU_BASELINES if _lj(f"S6_{n}.json")}
_S7 = {n: _lj(f"S7_{n}.json") for n in FU_BASELINES if _lj(f"S7_{n}.json")}
_S8 = {n: _lj(f"S8_{n}.json") for n in FU_SIG if _lj(f"S8_{n}.json")}
_S8c = {n: _lj(f"S8c_{n}.json") for n in FU_SIG if _lj(f"S8c_{n}.json")}
_S9 = {n: _lj(f"S9c_{n}.json") or _lj(f"S9_{n}.json") for n in FU_SIG if (_lj(f"S9c_{n}.json") or _lj(f"S9_{n}.json"))}
_S9_tag = {n: "corrected" if _lj(f"S9c_{n}.json") else "as-run" for n in _S9}
_S10 = _lj("S10c_floor.json") or _lj("S10_floor.json")
_S10_tag = "corrected" if _lj("S10c_floor.json") else "as-run"
_S11 = _lj("S11c_seq.json") or _lj("S11_seq.json")
_S0 = _lj("S0_theory.json")
_ALPHA_BASE = CFG["cal"]["alpha"]

def _cp_ci(rate, n, a=0.05):
    if not n or rate is None or np.isnan(rate):
        return (np.nan, np.nan)
    _k = round(rate * n)
    return (0.0 if _k == 0 else _bdist.ppf(a / 2, _k, n - _k + 1), 1.0 if _k == n else _bdist.ppf(1 - a / 2, _k + 1, n - _k))

_r = []
for _n, _j in _S6.items():
    for _sn, _e in _j["eval"].items():
        for _v, _m in _e.items():
            if isinstance(_m, dict):
                _r.append({"family": "baseline", "method": f"{_n}+{_v} (alpha={_ALPHA_BASE})", "point": "deployed",
                           "set": _sn, "auroc": _e["auroc"], **_m})
for _src, _tag in [(_S8c, "corrected"), (_S8, "as-run")]:
    for _n, _j in _src.items():
        _t = _j["thresholds"]
        _lab = (f"{_n} [corrected alpha={_t['alpha']}, beta={_t['beta']}]" if _tag == "corrected"
                else f"{_n} [as-run alpha=beta={_ALPHA_BASE}]")
        for _sn, _e in _j["eval"].items():
            for _k, _m in _e.items():
                if _k.startswith("eps") and isinstance(_m, dict):
                    _ec = float(_k[3:])
                    _pt = "eps_c=0 (clean)" if _ec == 0 else ("eps_c=sigma (certified)" if np.isclose(_ec, FU_SIG[_n]) else f"eps_c={_ec}")
                    _r.append({"family": _tag, "method": _lab, "model": _n, "eps_c": _ec, "point": _pt,
                               "set": _sn, "auroc": _e["auroc"], **_m})
MET = pd.DataFrame(_r)
TABLES = {}
_cols = ["method", "point", "auroc", "AF", "AF_ci95", "FR", "accept_real_rate", "coverage", "accept_precision", "n_real", "n_fake"]
if len(MET):
    MET["AF_ci95"] = [f"[{l:.3f}, {h:.3f}]" for l, h in (_cp_ci(a, n) for a, n in zip(MET.AF, MET.n_fake))]
    MET["coverage"] = (MET.cov_real * MET.n_real + MET.cov_fake * MET.n_fake) / (MET.n_real + MET.n_fake)
    _d = MET[MET.point.isin(["deployed", "eps_c=0 (clean)", "eps_c=sigma (certified)"])]
    TABLES["T1_clean_cf_test"] = _d[_d.set == "cf_test"][_cols]
    TABLES["T2_laundering_AF"] = _d[_d.set.str.startswith("cf_test")].pivot_table(index=["method", "point"], columns="set", values="AF").reset_index()
    TABLES["T3_transfer_openfake_raid"] = _d[_d.set.isin(["openfake"] + [s for s in FU_SETS if s.startswith("raid_")])][["set"] + _cols]
    _c = MET[(MET.family == "corrected") & (MET.set == "cf_test")]
    if len(_c):
        TABLES["T5_certification_curve_cf_test"] = _c[["method", "model", "eps_c", "AF", "AF_ci95", "FR", "accept_real_rate", "coverage"]]
_r4 = []
for _n, _j in _S7.items():
    for _part in ("joint", "blind"):
        for _bud, _g in _j.get(_part, {}).items():
            for _v, _af in _g.items():
                _r4.append({"method": f"{_n}+{_v}", "attack": f"{_part} PGD", "budget": _bud, "rate": _af,
                            "main": _part == "joint" and _v in ("conf", "all3", "maha")})
for _n, _j in _S9.items():
    for _tag, _lab in [("evasion", "EOT evasion (fake accepted)"), ("framing", "EOT framing (real flagged)")]:
        for _bud, _g in _j.get(_tag, {}).items():
            for _k, _v in _g.items():
                _r4.append({"method": f"{_n} [{_S9_tag[_n]}] eps_c={_k[5:]}", "attack": _lab, "budget": _bud, "rate": _v,
                            "main": _tag == "evasion" and np.isclose(float(_k[5:]), FU_SIG[_n])})
ATT = pd.DataFrame(_r4)
if len(ATT):
    TABLES["T4_adaptive_attacks"] = ATT.drop(columns=["main"]).pivot_table(index=["method", "attack"], columns="budget", values="rate").reset_index()
_r6 = []
for _s in CFG["eca"]["cohen_sigmas"]:
    _e, _c2 = _S8c.get(f"eca_{_s}"), _S8c.get(f"cohen_{_s}")
    if _e and _c2:
        for _eps in CFG["cert"]["eps_cert"]:
            _me, _mc = _e["eval"].get("cf_test", {}).get(f"eps{_eps}"), _c2["eval"].get("cf_test", {}).get(f"eps{_eps}")
            if _me and _mc:
                _r6.append({"sigma": _s, "eps_c": _eps, "alpha* ECA": _e["thresholds"]["alpha"], "alpha* Cohen": _c2["thresholds"]["alpha"],
                            "accept_real ECA": _me["accept_real_rate"], "accept_real Cohen": _mc["accept_real_rate"],
                            "AF ECA": _me["AF"], "AF Cohen": _mc["AF"], "FR ECA": _me["FR"], "FR Cohen": _mc["FR"]})
if _r6:
    TABLES["T6_ADRT_vs_Cohen"] = pd.DataFrame(_r6)
_r7 = []
for _n, _j in _S8c.items():
    _t, _hc = _j["thresholds"], _j.get("heldout_check", {})
    _r7.append({"method": _n, "alpha*": _t["alpha"], "beta*": _t["beta"], "tau_a": _t["tau_a"], "tau_f": _t["tau_f"],
                "K (calibration half)": _t["K"], "CRC feasible": _t["crc"].get("feasible"), "CRC risk bound": _t["crc"].get("risk_bound"),
                "acceptance useful (selection half)": _t["selection"]["acceptance_useful"],
                "flagging useful (selection half)": _t["selection"]["flagging_useful"],
                "test generators": _hc.get("n_test_generators"),
                "mean P(S_hi>=tau_a) test gens [<= alpha*]": _hc.get("mean_P(S_hi>=tau_a)_over_test_generators (guarantee event, target <= alpha)"),
                "P(S_lo<tau_f) test reals [<= beta*]": _hc.get("P(S_lo(real) < tau_f) on test reals (target <= beta)")})
for _n, _j in _S6.items():
    for _k, _v in _j["thresholds"].items():
        if _k.startswith("crc_"):
            _r7.append({"method": f"{_n}+{_k[4:]}", "alpha*": _ALPHA_BASE, "tau_a": _j["thresholds"].get(f"tau_a_{_k[4:]}"),
                        "tau_f": _j["thresholds"]["tau_f"], "CRC feasible": _v.get("feasible"), "CRC risk bound": _v.get("risk_bound")})
if _r7:
    TABLES["T7_guarantees"] = pd.DataFrame(_r7)
_r8 = []
if _S0:
    for _k, _v in _S0.items():
        if isinstance(_v, dict):
            for _kk, _vv in _v.items():
                if isinstance(_vv, (int, float)) and not isinstance(_vv, bool):
                    _r8.append({"item": f"S0 {_k}", "quantity": _kk, "value": _vv})
for _n, _j in _S6.items():
    for _kk, _vv in (_j.get("first_order_eps1") or {}).items():
        _r8.append({"item": f"Prop 1 first-order evasion budget, {_n}", "quantity": _kk, "value": _vv})
for _n, _j in _S7.items():
    for _kk, _vv in (_j.get("prop2") or {}).items():
        if isinstance(_vv, (int, float)):
            _r8.append({"item": f"Prop 2 gate stacking, {_n}", "quantity": _kk, "value": _vv})
if _S11:
    for _n, _v in _S11.items():
        for _kk in ("mean_samples", "ms_per_image", "agreement_with_fixed_N"):
            if _kk in _v:
                _r8.append({"item": f"Sequential certification, {_n}", "quantity": _kk, "value": _v[_kk]})
if _r8:
    TABLES["T8_theory_and_efficiency"] = pd.DataFrame(_r8)
if _S10:
    _f = pd.DataFrame({"eps": _S10["eps_grid"], "three-way floor": _S10["floor_three_way"], "two-way cap": _S10["cap_two_way"]})
    for _n, _m in _S10.get("models", {}).items():
        if "robust_pairs" in _m:
            _f[f"robust pairs {_n} [{_S10_tag}]"] = _m["robust_pairs"]
    TABLES["T9_evidence_energy_floor"] = _f
_r10, _r11 = [], []
for _n, _j in _S8c.items():
    _dg = _j["diagnostics"]
    for _e in CFG["cert"]["eps_cert"]:
        _r10.append({"model": _n, "eps_c": _e, "alpha_min for any acceptance": _dg["alpha_min_for_any_acceptance"][str(_e)],
                     "beta_min for any flagging": _dg["beta_min_for_any_flagging"][str(_e)],
                     "cal reals with 0 real votes": _dg["share_cal_reals_all_votes_fake"],
                     "cal fakes with all votes real": _dg["share_cal_fakes_all_votes_real"]})
    for _gk, _gv in _j["grid"].items():
        _m = _gv["eval"].get("cf_test", {}).get(f"eps{_j['sigma']}", {})
        _m0 = _gv["eval"].get("cf_test", {}).get("eps0.0", {})
        _r11.append({"model": _n, "alpha": _gv["alpha"], "beta": _gv["beta"], "tau_a": _gv["tau_a"], "tau_f": _gv["tau_f"],
                     "accept_real (eps=0)": _m0.get("accept_real_rate"), "AF (eps=0)": _m0.get("AF"),
                     "accept_real (eps=sigma)": _m.get("accept_real_rate"), "AF (eps=sigma)": _m.get("AF"),
                     "FR (eps=sigma)": _m.get("FR"), "fakes flagged (eps=sigma)": (_m.get("cov_fake", np.nan) - _m.get("AF", np.nan))})
if _r10:
    TABLES["T10_feasibility_diagnostics"] = pd.DataFrame(_r10)
    TABLES["T11_alpha_beta_grid_cf_test"] = pd.DataFrame(_r11)
    TABLES["T12_hardest_generators"] = pd.DataFrame([dict(model=_n, **g) for _n, _j in _S8c.items() for g in _j["diagnostics"]["hardest_generators"]])

# ---- research-question tables (pre-registered hypotheses from the blueprint) --------------
_S11d = _lj("S11d_saliency_agreement.json")
_amc = CFG["cert"]["alpha_mc"]

def _bud_x(b):
    _v = b.split("_", 1)[1]
    return float(_v.split("/")[0]) / float(_v.split("/")[1]) if "/" in _v else float(_v)

# T13 Prop 1: first-order budget vs the l_inf budget at which joint attacks reach 50% AF
_r13 = []
for _n, _j in _S6.items():
    _e1 = (_j.get("first_order_eps1") or {}).get("median_x255")
    _jt = (_S7.get(_n) or {}).get("joint", {})
    _ks = sorted([k for k in _jt if k.startswith("linf_")], key=_bud_x)
    _e50, _first = None, False
    for _a, _b in zip([None] + _ks, _ks):
        if _jt[_b].get("conf", 0) >= 0.5:
            if _a is None:
                _e50, _first = _bud_x(_b) * 255, True
            else:
                _ya, _yb = _jt[_a]["conf"], _jt[_b]["conf"]
                _xa, _xb = _bud_x(_a) * 255, _bud_x(_b) * 255
                _e50 = _xa + (0.5 - _ya) * (_xb - _xa) / max(1e-9, _yb - _ya)
            break
    _r13.append({"model": _n, "Prop 1 median first-order budget (x/255)": _e1,
                 "measured l_inf budget for 50% AF, confidence gate (x/255)": _e50,
                 "already >= 50% at the smallest budget (so the true value is lower)": _first})
if _r13:
    TABLES["T13_prop1_prediction"] = pd.DataFrame(_r13)
# T14 RQ3: vote distributions and clean AUROC drop from smoothing
_base_auc = ((_S6.get("base") or {}).get("eval", {}).get("cf_test", {}) or {}).get("auroc")
_r14 = []
for _n, _j in _S8c.items():
    _v = _j.get("votes", {})
    _auc = _j["eval"].get("cf_test", {}).get("auroc")
    _r14.append({"model": _n, "sigma": _j["sigma"], **_v, "AUROC smoothed": _auc, "AUROC base": _base_auc,
                 "AUROC drop (points)": (100 * (_base_auc - _auc)) if (_auc is not None and _base_auc is not None) else None})
if _r14:
    TABLES["T14_RQ3_votes_and_auroc"] = pd.DataFrame(_r14)
# T15 RQ5: per-generator results; OpenFake release-date trend
_r15, _r15s = [], []
for _n, _j in _S8c.items():
    for _setn in ("cf_test", "openfake"):
        for _g in _j.get("per_generator", {}).get(_setn, []):
            _r15.append({"model": _n, "set": _setn, **_g})
    _of = pd.DataFrame(_j.get("per_generator", {}).get("openfake", []))
    if len(_of) >= 3 and "release_date" in _of.columns:
        from scipy.stats import spearmanr as _spr2

        _d = pd.to_datetime(_of.release_date, errors="coerce")
        _ok = _d.notna()
        if _ok.sum() >= 3:
            _rho = _spr2(_d[_ok].map(pd.Timestamp.toordinal), _of.P_S_hi_ge_tau_a[_ok]).correlation
            _r15s.append({"model": _n, "OpenFake generators": int(_ok.sum()),
                          "Spearman(release date, P(S_hi>=tau_a))": float(_rho),
                          "mean AF clean OpenFake": float(_of.AF_clean.mean()), "alpha*": _j["thresholds"]["alpha"]})
if _r15:
    TABLES["T15_per_generator"] = pd.DataFrame(_r15)
if _r15s:
    TABLES["T15b_openfake_release_date_trend"] = pd.DataFrame(_r15s)
# T16 Theorem 5 quantile audit
_r16 = [dict(model=_n, **_j["theorem5_quantile"]) for _n, _j in _S8c.items() if _j.get("theorem5_quantile")]
if _r16:
    TABLES["T16_theorem5_quantile_audit"] = pd.DataFrame(_r16)
# T17 laundering applied after the attack
_r17 = []
for _n, _j in _S9.items():
    for _lk, _d in (_j.get("evasion_laundered") or {}).items():
        for _bud, _g in _d.items():
            _r17.append({"model": _n, "laundering": _lk, "attack budget": _bud,
                         "AF (eps_c=0)": _g.get("eps_c0.0"), f"AF (eps_c=sigma)": _g.get(f"eps_c{FU_SIG[_n]}"),
                         "AF without laundering (eps_c=sigma)": _j["evasion"].get(_bud, {}).get(f"eps_c{FU_SIG[_n]}")})
if _r17:
    TABLES["T17_attack_then_laundering"] = pd.DataFrame(_r17)
# T18 saliency agreement
if _S11d:
    TABLES["T18_saliency_vs_artifacts"] = pd.DataFrame([dict(model=_n, **_v, iou_if_random=_S11d["iou_top10pct_if_random"])
                                                        for _n, _v in _S11d["models"].items()])

# T0 RQ verdicts against the pre-registered criteria
def _verdict(ok, bad):
    return "supported" if ok else ("falsified" if bad else "not supported (not falsified)")

_r0 = []
try:
    _g = [(n, v, j["joint"]["l2_0.25"][v], j["joint"]["linf_1/255"][v]) for n, j in _S7.items() for v in j["joint"]["l2_0.25"]]
    _r0.append({"RQ": "RQ1 gates vs joint attacker", "evidence": "; ".join(f"{n}+{v}: l2 0.25 {a:.2f}, linf 1/255 {b:.2f}" for n, v, a, b in _g),
                "verdict": _verdict(all(a > 0.5 and b > 0.5 for _, _, a, b in _g), any(a < 0.2 for _, _, a, _ in _g))})
except Exception as _e:
    _r0.append({"RQ": "RQ1 gates vs joint attacker", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
try:
    _q = {n: j["prop2"]["median_emp_over_rmax"] / max(1e-9, j["prop2"]["median_sqrt_factor"])
          for n, j in _S7.items() if "median_emp_over_rmax" in (j.get("prop2") or {})}  # k >= 2 gates only
    _r0.append({"RQ": "RQ2 gate stacking within sqrt(k/lambda_min)", "evidence": "; ".join(f"{n}: median measured/bound {v:.2f}" for n, v in _q.items()),
                "verdict": _verdict(all(v <= 1.2 for v in _q.values()), all(v > 1.2 for v in _q.values()))})
except Exception as _e:
    _r0.append({"RQ": "RQ2 gate stacking", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
try:
    _m12, _m06 = _S8c["eca_0.12"]["votes"], _S8c["eca_0.06"]["votes"]
    _ok = lambda v: v["fake_real_vote_q99"] <= 0.10 and v["real_real_vote_median"] >= 0.90
    _r0.append({"RQ": "RQ3 evidence survives noise", "evidence": f"sigma 0.12: fake q99 {_m12['fake_real_vote_q99']:.2f}, real median {_m12['real_real_vote_median']:.2f}; "
                f"sigma 0.06: fake q99 {_m06['fake_real_vote_q99']:.2f}, real median {_m06['real_real_vote_median']:.2f}",
                "verdict": _verdict(_ok(_m12), not _ok(_m06))})
except Exception as _e:
    _r0.append({"RQ": "RQ3 evidence survives noise", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
try:
    _c = _S8c["eca_0.12"]
    _m = _c["eval"]["cf_test"]["eps0.12"]
    _bl = {f"{n}+{v}": j["joint"]["l2_0.12"][v] for n, j in _S7.items() for v in j["joint"]["l2_0.12"]}
    _eca_att = (_S9.get("eca_0.12") or {}).get("evasion", {}).get("l2_0.12", {}).get("eps_c0.12")
    _raid_e = _c["eval"].get("raid_epsilon16", {}).get("eps0.0", {}).get("AF")
    _raid_b = min([j["eval"].get("raid_epsilon16", {}).get(v, {}).get("AF", 1) for j in _S6.values()
                   for v in j["eval"].get("raid_epsilon16", {}) if isinstance(j["eval"]["raid_epsilon16"][v], dict)] or [None])
    _acc_any = max(_S8c[n]["eval"]["cf_test"][f"eps{FU_SIG[n]}"]["accept_real_rate"] for n in _S8c if n.startswith("eca_"))
    _r0.append({"RQ": "RQ4 defended risk at matched budget (sigma=0.12)",
                "evidence": f"ECA AF {_m['AF']:.3f} vs alpha*+alpha_MC {_c['thresholds']['alpha'] + _amc:.3f}; accept_real {_m['accept_real_rate']:.2f}; "
                            f"ECA AF under EOT l2 0.12: {_eca_att}; baselines joint AF at l2 0.12: min {min(_bl.values()):.2f}; "
                            f"RAID eps16 AF ECA {_raid_e} vs best baseline {_raid_b}",
                "verdict": _verdict(_m["AF"] <= _c["thresholds"]["alpha"] + _amc and _m["accept_real_rate"] >= 0.5 and min(_bl.values()) > 0.5,
                                    _acc_any < 0.2)})
except Exception as _e:
    _r0.append({"RQ": "RQ4 defended risk", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
try:
    _hcs = {n: (j["heldout_check"]["mean_P(S_hi>=tau_a)_over_test_generators (guarantee event, target <= alpha)"], j["thresholds"]["alpha"],
                j["heldout_check"]["n_test_generators"]) for n, j in _S8c.items()}
    _viol = {n: v > a + 2 * np.sqrt(a * (1 - a) / max(1, k)) for n, (v, a, k) in _hcs.items()}
    _tr = TABLES.get("T15b_openfake_release_date_trend")
    _r0.append({"RQ": "RQ5 generator-level guarantee", "evidence": "; ".join(f"{n}: {v:.3f} vs alpha* {a}" for n, (v, a, _) in _hcs.items())
                + ("" if _tr is None else "; OpenFake date trend rho " + ", ".join(f"{r['model']} {r['Spearman(release date, P(S_hi>=tau_a))']:.2f}" for r in _tr.to_dict("records"))),
                "verdict": _verdict(not any(_viol.values()), any(_viol.values()))})
except Exception as _e:
    _r0.append({"RQ": "RQ5 generator-level guarantee", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
try:
    _fl = np.array(_S10["floor_three_way"])
    _bad = any(np.any(np.array(m["robust_pairs"]) > _fl + 1e-9) for m in _S10["models"].values() if "robust_pairs" in m)
    _cap = max(2 * FU_SIG[n] * _S8c[n]["diagnostics"]["z_N"] for n in _S8c)
    _near1 = all(f >= 0.95 for f, e in zip(_fl, _S10["eps_grid"]) if e <= _cap)
    _r0.append({"RQ": "RQ6 information vs estimation", "evidence": f"floor >= robust pairs everywhere: {not _bad}; floor >= 0.95 at certifiable eps: {_near1}",
                "verdict": _verdict(_near1 and not _bad, _bad)})
except Exception as _e:
    _r0.append({"RQ": "RQ6 information vs estimation", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
try:
    _ms = {n: v["mean_samples"] for n, v in _S11.items()}
    _sal = {n: (v["spearman_mean"], v["iou_top10pct_mean"]) for n, v in (_S11d or {}).get("models", {}).items()}
    _r0.append({"RQ": "Secondary: sequential samples < 128; saliency on artifacts",
                "evidence": "; ".join(f"{n}: {v:.0f} samples" for n, v in _ms.items()) + "; "
                            + "; ".join(f"{n}: rho {a:.2f}, IoU {b:.3f}" for n, (a, b) in _sal.items()),
                "verdict": _verdict(all(v < 128 for v in _ms.values()) and all(a > 0 for a, _ in _sal.values()), False)})
except Exception as _e:
    _r0.append({"RQ": "Secondary", "evidence": f"not available ({type(_e).__name__})", "verdict": "n/a"})
TABLES = {"T0_research_question_verdicts": pd.DataFrame(_r0), **TABLES}

FIGS = {}

def _bud_val(b):
    _v = b.split("_", 1)[1]
    if "/" in _v:
        _a, _b = _v.split("/")
        return float(_a) / float(_b)
    return float(_v)

if len(ATT):
    for _norm in ("l2", "linf"):
        _sel = ATT[ATT.main.astype(bool) & ATT.budget.str.startswith(_norm + "_")]
        if not len(_sel):
            continue
        _fig, _ax = plt.subplots(figsize=(7.5, 4.2))
        for (_m, _a), _g in _sel.groupby(["method", "attack"]):
            _g = _g.assign(x=_g.budget.map(_bud_val)).sort_values("x")
            _ax.plot(_g.x * (255 if _norm == "linf" else 1), _g.rate, marker="o",
                     lw=2.5 if _a.startswith("EOT") else 1.2, label=f"{_m} [{_a.split(' ')[0]}]")
        if _norm == "l2":
            for _al in sorted({j["thresholds"]["alpha"] for j in _S8c.values()}):
                _ax.axhline(_al + _amc, color="k", ls="--", lw=0.8, label=f"certified bound alpha*+alpha_MC = {_al + _amc:.3f}")
        _ax.set_ylim(-0.02, 1.02)
        _ax.set_xlabel("l2 budget" if _norm == "l2" else "l_inf budget (x/255)")
        _ax.set_ylabel("accepted-fake rate under attack")
        _ax.set_title(f"Adaptive attacks ({_norm}){'  [empirical only]' if _norm == 'linf' else ''}")
        _ax.legend(fontsize=6, ncol=2)
        _fig.tight_layout()
        FIGS[f"F_attack_{_norm}"] = _fig
if "T11_alpha_beta_grid_cf_test" in TABLES:
    _t = TABLES["T11_alpha_beta_grid_cf_test"]
    _t = _t[_t.beta == _t.beta.min()]
    _fig, _ax = plt.subplots(1, 2, figsize=(11, 4))
    for _n, _g in _t.groupby("model"):
        _g = _g.sort_values("alpha")
        _ax[0].plot(_g.alpha, _g["accept_real (eps=sigma)"], marker="o", label=_n)
        _ax[1].plot(_g.alpha, _g["AF (eps=sigma)"], marker="o", label=_n)
    _ax[1].plot(sorted(_t.alpha.unique()), sorted(_t.alpha.unique()), "k:", lw=1, label="y = alpha")
    _ax[0].set_title("Risk-coverage: certified accepted-real rate at eps=sigma")
    _ax[1].set_title("Accepted-fake rate at eps=sigma vs target alpha")
    for _a in _ax:
        _a.set_xlabel("alpha (pre-registered grid)")
        _a.set_ylim(-0.02, 1.02)
        _a.legend(fontsize=7)
    _fig.tight_layout()
    FIGS["F_risk_coverage"] = _fig
if "T5_certification_curve_cf_test" in TABLES:
    _fig, _ax = plt.subplots(1, 2, figsize=(11, 4))
    for _m, _g in TABLES["T5_certification_curve_cf_test"].groupby("method"):
        _g = _g.sort_values("eps_c")
        _ax[0].plot(_g.eps_c, _g.accept_real_rate, marker="o", label=_m)
        _ax[1].plot(_g.eps_c, _g.AF, marker="o", label=_m)
    _ax[0].set_title("Certified accepted-real rate (corrected)")
    _ax[1].set_title("Accepted-fake rate at certification radius")
    for _a in _ax:
        _a.set_ylim(-0.02, 1.02)
        _a.set_xlabel("certification radius eps_c (l2)")
        _a.legend(fontsize=6)
    _fig.tight_layout()
    FIGS["F_certification"] = _fig
if _S10:
    _fig, _ax = plt.subplots(figsize=(6, 4))
    _ax.plot(_S10["eps_grid"], _S10["floor_three_way"], "k-", label="three-way floor")
    _ax.plot(_S10["eps_grid"], _S10["cap_two_way"], "k--", label="two-way cap")
    for _n, _m in _S10.get("models", {}).items():
        if "robust_pairs" in _m:
            _ax.plot(_S10["eps_grid"], _m["robust_pairs"], marker="o", label=_n)
    _ax.set_xscale("symlog", linthresh=0.05)
    _ax.set_xlabel("l2 budget eps")
    _ax.set_ylabel("share of robustly authenticated pairs")
    _ax.set_title(f"Evidence-energy floor (Thm 4) [{_S10_tag}]")
    _ax.legend(fontsize=7)
    _fig.tight_layout()
    FIGS["F_floor"] = _fig

for _k, _t in TABLES.items():
    _t.to_csv(os.path.join(CMP_DIR, f"{_k}.csv"), index=False)
for _k, _f in FIGS.items():
    _f.savefig(os.path.join(CMP_DIR, f"{_k}.png"), dpi=130)
    plt.close(_f)
FU_INV1.drop(columns=["sets"]).to_csv(os.path.join(CMP_DIR, "inventory.csv"), index=False)

def _md(df):
    try:
        return df.round(4).to_markdown(index=False)
    except ImportError:
        return df.round(4).to_string(index=False)

_left = FU_INV1[(FU_INV1.status != "done") & ~FU_INV1.stage.isin(["S9", "S10", "S11"])]
with open(os.path.join(CMP_DIR, "COMPARISON.md"), "w") as _fh:
    _fh.write("# ECA-3WD overall comparison\n\n"
              "First-pass results plus the gaps filled by F3 (units it completed carry a "
              "`followup` entry in their JSON). Smoothed models are reported twice: **as-run** (alpha = beta = "
              f"{_ALPHA_BASE}, framing threshold on S_hi) and **corrected** (framing threshold on S_lo; operating "
              "point from the pre-registered grid by the split-sample rule). Baselines keep alpha = "
              f"{_ALPHA_BASE}, which is stricter than any corrected alpha*, so the comparison favours the baselines. "
              "l_inf results are empirical only.\n\n")
    _fh.write(f"## Still not available ({len(_left)})\n\n")
    for _row in _left.itertuples():
        _fh.write(f"- {_row.unit}: {_row.status} {('- ' + _row.detail) if _row.detail else ''} -> {_row.action}\n")
    for _lab, _lg in [("Filled in this session", FILL_LOG), ("Corrected runs in this session", CORR_LOG)]:
        if _lg.get("ran"):
            _fh.write(f"\n## {_lab}\n\n")
            for _s in _lg["steps"]:
                _fh.write(f"- {_s['step']}: {_s['result']}\n")
    for _k, _t in TABLES.items():
        _fh.write(f"\n## {_k}\n\n{_md(_t)}\n")
    for _k in FIGS:
        _fh.write(f"\n![{_k}]({_k}.png)\n")
_pushed = fu_push("correction: overall comparison")
{"tables": list(TABLES), "figures": list(FIGS), "still_missing": int(len(_left)), "folder": CMP_DIR, "pushed": _pushed}

In [ ]:
# ---- F7. VIEWER ---------------------------------------------------------------------------
_tabs = {"Correction": mo.ui.table(RECAL_TABLE.round(4), selection=None) if len(RECAL_TABLE) else mo.md("No S8 vote counts yet.")}
_left = FU_INV1[(FU_INV1.status != "done") & ~FU_INV1.stage.isin(["S9", "S10", "S11"])]
_tabs["Still missing"] = mo.ui.table(_left.drop(columns=["sets"]), page_size=30, selection=None) if len(_left) else mo.md("Everything expected is present.")
for _k, _t in TABLES.items():
    _tabs[_k] = mo.ui.table(_t.round(4), page_size=25, selection=None)
for _k in FIGS:
    _tabs[_k] = mo.image(os.path.join(CMP_DIR, f"{_k}.png"))
mo.ui.tabs(_tabs)

## Part III: analyses and new experiments

Part A reads only saved results (CPU). Parts B and C add experiments; every GPU unit skips work whose output already exists, locally or in the results repo.

In [ ]:
# ---- V-CONFIG: settings for the new analyses and experiments --------------------------
V3_SMOKE = os.environ.get("V3_SMOKE", "0") == "1"  # tiny sizes, for testing the notebook only
V3 = {
    "out": os.path.join(CFG["work"], "v3"),
    "N": CFG["cert"]["N"],
    "alpha_mc": CFG["cert"]["alpha_mc"],
    "d": 3 * CFG["img"]["crop"] ** 2,  # input dimension, for l_inf -> l2 conversion
    "min_per_gen": CFG["cal"]["min_per_gen"],
    # Pre-registered grids. Every (alpha, beta) point is a valid guarantee on its own;
    # the main operating point is chosen on calibration data only (A3 selection).
    "alpha_grid": [0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30],
    "beta_grid": [0.01, 0.02, 0.05, 0.10],
    "main_alphas": [0.01, 0.02, 0.05, 0.10],
    "main_beta": 0.05,
    "eps_rel": [0.0, 0.25, 0.5, 1.0, 1.5],  # certified budget in units of sigma
    "select_alpha": 0.05,
    "n_resplits": 20 if V3_SMOKE else 300,
    "n_boot": 50 if V3_SMOKE else 2000,
    "losses": [  # (lambda_AF, lambda_FR, lambda_D); pi_F = 0.5
        (10, 1, 0.05), (10, 1, 0.2), (50, 5, 0.1), (100, 1, 0.05), (100, 10, 0.2)],
    "roc_attack": {"n": 8 if V3_SMOKE else 500, "batch": 4 if V3_SMOKE else 50,
                   "steps": 2 if V3_SMOKE else 50, "restarts": 1 if V3_SMOKE else 2,
                   "l2": [0.06, 0.12, 0.25, 0.5], "linf": [0.5 / 255, 1 / 255, 2 / 255]},
    # 3x the steps and 2x the noise draws of S9c, 2 restarts; batch x m = 512 images with grad
    # (S9c used 50 x 16 = 800), so VRAM stays below what S9c needed.
    "stress": {"n": 4 if V3_SMOKE else 200, "batch": 2 if V3_SMOKE else 16,
               "steps": 2 if V3_SMOKE else 60, "m": 2 if V3_SMOKE else 32,
               "restarts": 1 if V3_SMOKE else 2, "l2_rel": [0.5, 1.0, 1.5]},
    "bigN": {"Ns": [16, 32] if V3_SMOKE else [256, 1024], "per_gen_cap": 2 if V3_SMOKE else 20, "n_real": 8 if V3_SMOKE else 2000},
    "seeds": {"seeds": [1, 2], "models": ["base", "eca"], "per_gen_cap": 2 if V3_SMOKE else 20,
              "n_real": 8 if V3_SMOKE else 3000, "max_steps": 3 if V3_SMOKE else None},
    # Part C. C2: the B1 attack on the adversarially trained baseline, 4x the steps and 2.5x the
    # restarts (does its empirical robustness survive a stronger attacker?). C6: ADRT loss ablation.
    "c2": {"n": 8 if V3_SMOKE else 500, "batch": 4 if V3_SMOKE else 50, "steps": 3 if V3_SMOKE else 200,
           "restarts": 1 if V3_SMOKE else 5, "l2": [0.06, 0.12, 0.25], "linf": []},
    "c6": {"variants": {"afonly": {"lam_fr": 0.0}, "fronly": {"lam_af": 0.0}}},
    "run": {"B1": True, "B3": True, "B4": True, "B5": True, "C2": True, "C6": True},
}
for _sub in ["", "figs", "tables", "gpu", "stamps"]:
    os.makedirs(os.path.join(V3["out"], _sub), exist_ok=True)

def v_rel(path):
    return os.path.relpath(path, CFG["work"]).replace(os.sep, "/")

def v_path(*parts):
    return os.path.join(V3["out"], *parts)

# Slow CPU stages (the re-split analyses) are skipped when their outputs already exist, locally or
# pulled from the Hub by A1, AND were made by the current version of that stage's code on the same
# results folder. Bump a version whenever a stage's code changes, so stale outputs are recomputed.
# Part B units skip by output file (see each B cell); they never re-run once their file exists.
V_STAMPS = {"A4": "1", "A11": "1"}
V_LEGACY_OK = {"A4"}  # outputs made before stamps existed, by code identical to the current version

def v_cached(stage, files):
    if not all(os.path.exists(f) for f in files):
        return False
    _st = v_path("stamps", f"{stage}.json")
    if os.path.exists(_st):
        _j = load_json(_st)
        return _j.get("version") == V_STAMPS[stage] and _j.get("results") == os.path.basename(PATHS["res"])
    return stage in V_LEGACY_OK

def v_stamp(stage):
    _st = v_path("stamps", f"{stage}.json")
    save_json(_st, {"version": V_STAMPS[stage], "results": os.path.basename(PATHS["res"]),
                    "time": time.strftime("%Y-%m-%d %H:%M")})
    v_push(_st)

def v_push(path, message=None):
    """Push a v3 file or folder to the results repo (results only: never weights or data)."""
    if not HF_ON or not os.path.exists(path):
        return False
    try:
        if os.path.isdir(path):
            HF_API.upload_folder(folder_path=path, path_in_repo=v_rel(path), repo_id=CFG["hf"]["results_repo"],
                                 repo_type="model", commit_message=message or f"v3 {v_rel(path)}",
                                 allow_patterns=["*.json", "*.csv", "*.md", "*.png", "*.pdf", "*.parquet", "*.tex"])
        else:
            HF_API.upload_file(path_or_fileobj=path, path_in_repo=v_rel(path), repo_id=CFG["hf"]["results_repo"],
                               repo_type="model", commit_message=message or f"v3 {v_rel(path)}")
        return True
    except Exception as _e:
        log(f"[hf] v3 push failed for {v_rel(path)}: {type(_e).__name__}")
        return False

{"smoke": V3_SMOKE, "out": V3["out"], "N": V3["N"], "alpha_mc": V3["alpha_mc"]}

In [ ]:
# ---- A1. PULL every saved result from the Hub (small files only; no weights, no data) ----
def a1_pull():
    _repo = CFG["hf"]["results_repo"]
    if not (HF_TOKEN and _repo):
        return {"pulled": False, "why": "HF_TOKEN or HF_RESULTS_REPO not set; using local files only"}
    _res = v_rel(PATHS["res"])
    _mod = v_rel(PATHS["models"])
    _dat = v_rel(PATHS["data"])
    _pats = [f"{_res}/*.json", f"{_res}/*.parquet", f"{_res}/*.npz", f"{_res}/*.npy", f"{_res}/*.csv",
             f"{_res}/*.md", f"{_res}/*.png", f"{_res}/comparison/*", f"{_mod}/*/train_log.json",
             f"{_dat}/cf_selection.parquet", "v3/*"]
    try:
        snapshot_download(_repo, repo_type="model", local_dir=CFG["work"], token=HF_TOKEN,
                          allow_patterns=_pats, max_workers=8)
    except Exception as _e:
        return {"pulled": False, "why": f"{type(_e).__name__}: {str(_e)[:200]}"}
    return {"pulled": True}

A1 = a1_pull()
_dat = os.path.relpath(PATHS["data"], CFG["work"])
_skip = ("_downloads", os.path.join(_dat, "cf") + os.sep, os.path.join(_dat, "of") + os.sep,
         os.path.join(_dat, "raid") + os.sep)
A1_FILES = sorted(r for r in (os.path.relpath(os.path.join(_d, _f), CFG["work"])
                              for _d, _, _fs in os.walk(CFG["work"]) for _f in _fs)
                  if not r.startswith(_skip))
mo.md(f"**A1:** `{A1}` · results folder `{os.path.basename(PATHS['res'])}` · "
      f"{len(os.listdir(PATHS['res']))} files in it · {len(A1_FILES)} small files in total")

In [ ]:
# ---- A2. DIGEST: every saved number in one table + a readable report --------------------
def _flatten(obj, prefix=""):
    if isinstance(obj, dict):
        for _k, _v in obj.items():
            yield from _flatten(_v, f"{prefix}.{_k}" if prefix else str(_k))
    elif isinstance(obj, list) and len(obj) <= 64 and all(not isinstance(_v, (dict, list)) for _v in obj):
        yield prefix, json.dumps(obj, default=float)
    elif isinstance(obj, list):
        for _i, _v in enumerate(obj[:500]):
            yield from _flatten(_v, f"{prefix}[{_i}]")
    else:
        yield prefix, obj

def a2_digest():
    _rows = []
    _files = sorted(f for f in os.listdir(PATHS["res"]) if f.endswith(".json"))
    _models = [os.path.join(PATHS["models"], d, "train_log.json") for d in sorted(os.listdir(PATHS["models"]))
               if os.path.exists(os.path.join(PATHS["models"], d, "train_log.json"))] if os.path.isdir(PATHS["models"]) else []
    for _p in [os.path.join(PATHS["res"], f) for f in _files] + _models:
        try:
            _j = load_json(_p)
        except Exception as _e:
            _rows.append({"file": os.path.relpath(_p, CFG["work"]), "key": "(unreadable)", "value": str(_e)[:100]})
            continue
        for _k, _v in _flatten(_j):
            _rows.append({"file": os.path.relpath(_p, CFG["work"]), "key": _k,
                          "value": _v if isinstance(_v, (int, float, str, bool)) or _v is None else str(_v)})
    _df = pd.DataFrame(_rows, columns=["file", "key", "value"])
    _df["value"] = [str(v) for v in _df["value"]]
    _df.to_csv(v_path("tables", "A2_digest_all_numbers.csv"), index=False)
    with open(v_path("A2_REPORT_ALL_RESULTS.md"), "w") as _fh:
        _fh.write(f"# Every saved result ({time.strftime('%Y-%m-%d %H:%M')})\n\nResults folder: `{os.path.basename(PATHS['res'])}`\n\n")
        for _f, _g in _df.groupby("file", sort=True):
            _fh.write(f"## {_f}\n\n")
            for _r in _g.itertuples():
                _fh.write(f"- `{_r.key}` = {str(_r.value)[:300]}\n")
            _fh.write("\n")
        _cmp = os.path.join(PATHS["res"], "comparison")
        if os.path.isdir(_cmp):
            for _f in sorted(os.listdir(_cmp)):
                if _f.endswith(".csv"):
                    _t = pd.read_csv(os.path.join(_cmp, _f))
                    _fh.write(f"## comparison/{_f}\n\n{_t.round(4).to_string(index=False, max_rows=200)}\n\n")
    return _df

A2 = a2_digest()
mo.vstack([mo.md(f"**A2:** {len(A2):,} saved numbers from {A2.file.nunique()} files → "
                 f"`{v_path('A2_REPORT_ALL_RESULTS.md')}`"),
           mo.ui.table(A2.groupby("file").size().rename("numbers").reset_index(), selection=None, page_size=40)])

In [ ]:
# ---- V-LIBRARY: analysis helpers (pure numpy; all inputs are saved vote counts) --------
from scipy.stats import beta as v_beta, norm as v_norm, spearmanr as v_spearman

def v_zN(N, alpha_mc):
    """Largest |S| / sigma a Clopper-Pearson bound can certify with N votes."""
    return float(v_norm.ppf((alpha_mc / 2) ** (1.0 / N)))

def v_scores(k, sigma, N=None):
    _N = N or V3["N"]
    return scores_from_counts(np.asarray(k, float), _N, sigma, V3["alpha_mc"])

def v_crc(per_gen, alpha):
    """Conformal risk control over generators, vectorised. Smallest t with
    K/(K+1) * mean_g P_g(v >= t) + 1/(K+1) <= alpha. Same rule as crc_accept_threshold."""
    _K = len(per_gen)
    if _K == 0:
        return np.inf, False
    _c = (alpha - 1.0 / (_K + 1)) * (_K + 1) / _K  # allowed generator-averaged tail mass
    if _c < 0:
        return np.inf, False
    _v = np.concatenate([np.asarray(p, float) for p in per_gen])
    _w = np.concatenate([np.full(len(p), 1.0 / (len(p) * _K)) for p in per_gen])
    _o = np.argsort(_v, kind="stable")
    _v, _w = _v[_o], _w[_o]
    _u, _first = np.unique(_v, return_index=True)
    _tail = np.cumsum(_w[::-1])[::-1][_first]  # weight of v >= u_j
    _ok = np.where(_tail <= _c + 1e-12)[0]
    return (float(_u[_ok[0]]) if len(_ok) else np.inf), True

def v_risk(per_gen, t):
    _K = len(per_gen)
    if _K == 0:
        return np.nan
    return _K / (_K + 1) * float(np.mean([np.mean(np.asarray(p) >= t) for p in per_gen])) + 1 / (_K + 1)

def v_gen_avg(values, gens, t, min_n):
    """Generator-averaged P(v >= t), over generators with >= min_n images; also the share
    of generators whose own rate exceeds a level, and the count."""
    _d = pd.DataFrame({"v": np.asarray(values) >= t, "g": np.asarray(gens)})
    _r = _d.groupby("g").v.agg(["mean", "size"])
    _r = _r[_r["size"] >= min_n]
    return (float(_r["mean"].mean()) if len(_r) else np.nan), _r

def v_flag_threshold(s_lo_real, beta):
    return conformal_flag_threshold(np.asarray(s_lo_real), beta)

def v_cp(k, n, a=0.05):
    """Two-sided Clopper-Pearson interval for k successes out of n."""
    if n == 0:
        return (np.nan, np.nan)
    _lo = 0.0 if k == 0 else float(v_beta.ppf(a / 2, k, n - k + 1))
    _hi = 1.0 if k == n else float(v_beta.ppf(1 - a / 2, k + 1, n - k))
    return _lo, _hi

def v_smoothed():
    """(name, sigma) for every smoothed model with saved S8 vote counts."""
    _out = []
    for _f in sorted(os.listdir(PATHS["res"])):
        if _f.startswith("S8_") and _f.endswith("_counts.parquet"):
            _n = _f[3:-len("_counts.parquet")]
            _j = os.path.join(PATHS["res"], f"S8_{_n}.json")
            _s = load_json(_j)["sigma"] if os.path.exists(_j) else float(_n.split("_")[-1])
            _out.append((_n, float(_s)))
    return _out

def v_counts(name):
    _p = os.path.join(PATHS["res"], f"S8_{name}_counts.parquet")
    return pd.read_parquet(_p) if os.path.exists(_p) else None

def v_cal_parts(df, sigma, N=None):
    """Calibration arrays: per-generator S_hi / S_lo / point score of fakes, S_lo / S_hi of reals."""
    _c = df[df.set == "cf_cal"]
    _lo, _hi, _pt = v_scores(_c.k.to_numpy(), sigma, N)
    _y, _g = _c.y.to_numpy(), _c.model_name.to_numpy()
    _gens = [g for g in np.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= V3["min_per_gen"]]
    _m = {g: (_y == 1) & (_g == g) for g in _gens}
    return {"gens": _gens, "hi": {g: _hi[_m[g]] for g in _gens}, "lo": {g: _lo[_m[g]] for g in _gens},
            "pt": {g: _pt[_m[g]] for g in _gens}, "lo_r": _lo[_y == 0], "hi_r": _hi[_y == 0], "pt_r": _pt[_y == 0]}

def v_rule(cal, alpha, beta):
    """CERT-CAL (corrected): tau_a by CRC on S_hi of calibration fakes, tau_f by the
    conformal quantile of S_lo of calibration reals; tau_a >= tau_f."""
    _ta, _ok = v_crc([cal["hi"][g] for g in cal["gens"]], alpha)
    _tf = v_flag_threshold(cal["lo_r"], beta)
    return {"tau_a": float(max(_ta, _tf)), "tau_f": float(_tf), "alpha": alpha, "beta": beta,
            "K": len(cal["gens"]), "n_real": int(len(cal["lo_r"])), "feasible": bool(_ok)}

def v_eval_set(df_set, sigma, rule, eps, N=None, pre=None):
    """Three-way metrics on one evaluation set, plus the guarantee events the theorems bound.
    `pre` = (S_lo, S_hi) already computed for df_set (saves recomputing Clopper-Pearson bounds)."""
    _lo, _hi = pre if pre is not None else v_scores(df_set.k.to_numpy(), sigma, N)[:2]
    _y = df_set.y.to_numpy()
    _d = decide(_lo, _hi, rule["tau_a"], rule["tau_f"], eps)
    _m = three_way_metrics(_d, _y)
    _f = _y == 1
    if _f.any():
        _ga, _ = v_gen_avg(_hi[_f], df_set.model_name.to_numpy()[_f], rule["tau_a"], V3["min_per_gen"])
        _m["guarantee_event_gen_avg P(S_hi>=tau_a)"] = _ga
        _m["guarantee_event_pooled P(S_hi>=tau_a)"] = float(np.mean(_hi[_f] >= rule["tau_a"]))
    if (~_f).any():
        _m["framing_event P(S_lo<tau_f)"] = float(np.mean(_lo[~_f] < rule["tau_f"]))
    _m["n_accept_real"] = int(np.sum((_d == 2) & ~_f))
    return _m

def v_select_model(models, eps_rel=1.0):
    """Pick the main smoothed model on CALIBRATION data only: generators and reals are split in
    two halves by hash; thresholds are fitted on half A and real acceptance at eps = sigma is
    measured on half B. Test sets are never looked at."""
    _rows = []
    for _n, _s in models:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        _gA = [g for g in _cal["gens"] if stable_u("sel:" + str(g)) < 0.5]
        _rA = np.array([stable_u(f"selr:{i}") < 0.5 for i in range(len(_cal["lo_r"]))])
        _ta, _ = v_crc([_cal["hi"][g] for g in _gA], V3["select_alpha"])
        _tf = v_flag_threshold(_cal["lo_r"][_rA], V3["main_beta"])
        _ta = max(_ta, _tf)
        _acc = float(np.mean(_cal["lo_r"][~_rA] >= _ta + eps_rel * _s)) if (~_rA).any() else np.nan
        _rows.append({"model": _n, "sigma": _s, "K_half": len(_gA), "tau_a_half": _ta,
                      "accept_real_halfB_eps_sigma": _acc})
    _t = pd.DataFrame(_rows)
    _best = _t.sort_values(["accept_real_halfB_eps_sigma", "sigma"], ascending=[False, False]).iloc[0]["model"] if len(_t) else None
    return _best, _t

def v_parse_attack(key):
    """'l2_0.25' -> ('l2', 0.25, 0.25); 'linf_2/255' -> ('linf', 2/255, l2 radius of that ball)."""
    _nrm, _v = key.split("_", 1)
    if _nrm == "linf":
        _e = float(_v.split("/")[0]) / 255
        return "linf", _e, _e * np.sqrt(V3["d"])
    return "l2", float(_v), float(_v)

def v_savefig(fig, name):
    fig.savefig(v_path("figs", name + ".png"), dpi=140, bbox_inches="tight")
    fig.savefig(v_path("figs", name + ".pdf"), bbox_inches="tight")
    plt.close(fig)
    return v_path("figs", name + ".png")

V_MODELS = v_smoothed()
V_BASELINES = [n for n in ["base", "pgdat", "clip"] if os.path.exists(os.path.join(PATHS["res"], f"S6_{n}.json"))]
V_BEST, V_SELECT = v_select_model(V_MODELS) if V_MODELS else (None, pd.DataFrame())
if len(V_SELECT):
    V_SELECT.to_csv(v_path("tables", "A3_model_selection_calibration_only.csv"), index=False)
mo.vstack([mo.md(f"**Smoothed models with vote counts:** {[m for m, _ in V_MODELS]} · **baselines:** {V_BASELINES} · "
                 f"**main model (chosen on calibration halves only):** `{V_BEST}`"),
           mo.ui.table(V_SELECT.round(4), selection=None) if len(V_SELECT) else mo.md("_No S8 vote counts found._")])

In [ ]:
# ---- A3. FEASIBILITY FRONTIER and certified operating curves (CPU) ---------------------
# Proposition (feasibility). S_lo <= sigma z_N for every image, so some real can be accepted at
# budget eps only if tau_a <= sigma z_N - eps, i.e. only if
#     alpha >= alpha_min(eps) = K/(K+1) * mean_g P_g(S_hi >= sigma z_N - eps) + 1/(K+1).
# Symmetrically, flagging needs beta >= beta_min(eps) = (#{S_lo(real) <= -sigma z_N + eps} + 1)/(n+1).
# alpha_min is a property of the base detector under noise; no calibration trick can go below it.
def a3_frontier(name, sigma):
    _df = v_counts(name)
    _cal = v_cal_parts(_df, sigma)
    _zN = v_zN(V3["N"], V3["alpha_mc"])
    _smax = sigma * _zN
    _per = [_cal["hi"][g] for g in _cal["gens"]]
    _n = len(_cal["lo_r"])
    _front = []
    for _er in V3["eps_rel"]:
        _e = _er * sigma
        _front.append({"model": name, "sigma": sigma, "eps_over_sigma": _er, "eps": _e,
                       "alpha_min": v_risk(_per, _smax - _e),
                       "beta_min": float((np.sum(_cal["lo_r"] <= -_smax + _e) + 1) / (_n + 1)),
                       "share_cal_fakes_all_votes_real": float(np.mean(np.concatenate(_per) >= _smax - 1e-9)) if _per else np.nan,
                       "K": len(_per), "n_real_cal": _n, "z_N": _zN})
    _rows = []
    _sets = [s for s in _df.set.unique() if s != "cf_cal"]
    _pre = {s: v_scores(_df[_df.set == s].k.to_numpy(), sigma)[:2] for s in _sets}
    for _al in V3["alpha_grid"]:
        for _be in V3["beta_grid"]:
            _rule = v_rule(_cal, _al, _be)
            for _er in V3["eps_rel"]:
                for _sn in _sets:
                    _m = v_eval_set(_df[_df.set == _sn], sigma, _rule, _er * sigma, pre=_pre[_sn])
                    _rows.append({"model": name, "sigma": sigma, "alpha": _al, "beta": _be, "eps_over_sigma": _er,
                                  "set": _sn, "tau_a": _rule["tau_a"], "tau_f": _rule["tau_f"], **_m})
    _t = pd.DataFrame(_rows)
    _te = _df[_df.set == "cf_test"]
    _vf = _te[_te.y == 1].k.to_numpy() / V3["N"]
    _vr = _te[_te.y == 0].k.to_numpy() / V3["N"]
    _votes = {"model": name, "sigma": sigma,
              "fake_real_vote_q99": float(np.quantile(_vf, 0.99)) if len(_vf) else np.nan,
              "share_fakes_le_10pct_real_votes": float(np.mean(_vf <= 0.10)) if len(_vf) else np.nan,
              "real_real_vote_median": float(np.median(_vr)) if len(_vr) else np.nan,
              "share_reals_ge_64pct_real_votes": float(np.mean(_vr >= 0.64)) if len(_vr) else np.nan,
              "share_reals_ge_90pct_real_votes": float(np.mean(_vr >= 0.90)) if len(_vr) else np.nan}
    return pd.DataFrame(_front), _t, _votes

def a3_run():
    _F, _T, _V = [], [], []
    for _n, _s in V_MODELS:
        _f, _t, _v = a3_frontier(_n, _s)
        _F.append(_f)
        _T.append(_t)
        _V.append(_v)
    if not _F:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    _F, _T, _V = pd.concat(_F), pd.concat(_T), pd.DataFrame(_V)
    _F.to_csv(v_path("tables", "A3_feasibility_frontier.csv"), index=False)
    _T.to_csv(v_path("tables", "A3_operating_grid_all.csv"), index=False)
    _V.to_csv(v_path("tables", "A3_vote_distributions_RQ3.csv"), index=False)
    # Figures: (1) alpha_min vs eps/sigma, (2) certified real acceptance vs alpha at eps = sigma (cf_test),
    # with the held-out guarantee event on the same axis.
    _fig, _ax = plt.subplots(1, 3, figsize=(15, 4))
    for _n, _g in _F.groupby("model"):
        _ax[0].plot(_g.eps_over_sigma, _g.alpha_min, marker="o", label=_n)
    _ax[0].set_yscale("log")
    _ax[0].set_xlabel("certified budget eps / sigma")
    _ax[0].set_ylabel("alpha_min (smallest feasible accepted-fake level)")
    _ax[0].set_title("Feasibility frontier")
    _ax[0].legend(fontsize=7)
    _m = _T[(_T.set == "cf_test") & (_T.beta == V3["main_beta"])]
    for _n, _g in _m[np.isclose(_m.eps_over_sigma, 1.0)].groupby("model"):
        _ax[1].plot(_g.alpha, _g.accept_real_rate, marker="o", label=_n)
        _ax[2].plot(_g.alpha, _g["guarantee_event_gen_avg P(S_hi>=tau_a)"], marker="o", label=_n)
    _ax[1].set_xscale("log")
    _ax[1].set_xlabel("alpha (expected accepted-fake level, new generator)")
    _ax[1].set_ylabel("certified real acceptance, eps = sigma")
    _ax[1].set_title("What the guarantee costs (held-out CF)")
    _ax[2].plot(V3["alpha_grid"], V3["alpha_grid"], "k--", label="y = alpha (target)")
    _ax[2].set_xscale("log")
    _ax[2].set_yscale("log")
    _ax[2].set_xlabel("alpha")
    _ax[2].set_ylabel("held-out generator-avg P(S_hi >= tau_a)")
    _ax[2].set_title("Guarantee event vs target")
    for _a in _ax[1:]:
        _a.legend(fontsize=7)
    v_savefig(_fig, "A3_frontier")
    v_push(v_path("tables"), "v3 A3 frontier")
    v_push(v_path("figs"), "v3 A3 figures")
    return _F, _T, _V

A3_FRONT, A3_GRID, A3_VOTES = a3_run()
_main = (A3_GRID[(A3_GRID.set == "cf_test") & A3_GRID.alpha.isin(V3["main_alphas"]) & (A3_GRID.beta == V3["main_beta"])
                 & A3_GRID.eps_over_sigma.isin([0.0, 1.0])]
         [["model", "alpha", "eps_over_sigma", "accept_real_rate", "AF", "FR", "cov_real",
           "guarantee_event_gen_avg P(S_hi>=tau_a)", "framing_event P(S_lo<tau_f)", "tau_a", "tau_f"]]
         if len(A3_GRID) else pd.DataFrame())
if len(_main):
    _main.to_csv(v_path("tables", "A3_main_operating_points_cf_test.csv"), index=False)
mo.vstack([mo.md("### A3. Feasibility frontier"),
           mo.ui.table(A3_FRONT.round(4), selection=None) if len(A3_FRONT) else mo.md("_no counts_"),
           mo.md("**Main operating points on held-out CommunityForensics generators**"),
           mo.ui.table(_main.round(4), selection=None, page_size=40) if len(_main) else mo.md("_no counts_"),
           mo.md("**RQ3 vote distributions (held-out CF test)**"),
           mo.ui.table(A3_VOTES.round(4), selection=None) if len(A3_VOTES) else mo.md("")])

In [ ]:
# ---- A4. GUARANTEE VALIDITY over random generator re-splits, and the ablations that show
#          WHY each design choice is needed (CPU) ----------------------------------------
# Calibration and test generators of CommunityForensics are both random draws (hash split),
# so pooling them and re-splitting at random many times measures E[guarantee event] directly.
#   GEN-CRC on S_hi     : our rule (Theorem 5' + Theorem 3). Target: mean event <= alpha.
#   GEN-CRC on S_lo     : critique row 4 (calibrating fakes on S_lo). Expected to break.
#   GEN-CRC on S-point  : plug-in score, no Monte-Carlo bound. Expected to break slightly.
#   IMAGE-conformal     : pooled image-level split conformal fitted on images of the calibration
#                         generators; evaluated on other images of THOSE generators (looks valid)
#                         and on NEW generators (the deployment question).
#   Framing, S_lo basis : corrected tau_f.   Framing, S_hi basis : the original tau_f.
def a4_validity(name, sigma):
    _df = v_counts(name)
    _p = _df[_df.set.isin(["cf_cal", "cf_test"])]
    _lo, _hi, _pt = v_scores(_p.k.to_numpy(), sigma)
    _y, _g = _p.y.to_numpy(), _p.model_name.to_numpy()
    _gens = [g for g in np.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= V3["min_per_gen"]]
    _idx = {g: np.where((_y == 1) & (_g == g))[0] for g in _gens}
    _ri = np.where(_y == 0)[0]
    _rng = np.random.default_rng(CFG["seed"] + 77)
    _rows = []
    for _r in range(V3["n_resplits"]):
        _perm = _rng.permutation(len(_gens))
        _Gc = [_gens[i] for i in _perm[: len(_gens) // 2]]
        _Gn = [_gens[i] for i in _perm[len(_gens) // 2:]]
        _rp = _rng.permutation(_ri)
        _Rc, _Rn = _rp[: len(_rp) // 2], _rp[len(_rp) // 2:]

        def _ev(arr, t, G):
            _v = [np.mean(arr[_idx[g]] >= t) for g in G]
            return float(np.mean(_v)), float(np.max(_v)), _v

        for _al in V3["main_alphas"]:
            for _meth, _arr in [("GEN-CRC on S_hi (ours)", _hi), ("GEN-CRC on S_lo", _lo), ("GEN-CRC on S-point", _pt)]:
                _t, _ = v_crc([_arr[_idx[g]] for g in _Gc], _al)
                _e_hi, _w_hi, _v_hi = _ev(_hi, _t, _Gn)
                _e_pt, _, _ = _ev(_pt, _t, _Gn)
                _rows.append({"model": name, "resplit": _r, "side": "accept", "level": _al, "method": _meth,
                              "event_S_hi_new_gens": _e_hi, "event_S_point_new_gens": _e_pt,
                              "share_new_gens_above_level": float(np.mean(np.array(_v_hi) > _al)),
                              "worst_new_gen": _w_hi, "event_same_gens_other_images": np.nan,
                              "real_accept_eps_sigma": float(np.mean(_lo[_Rn] >= _t + sigma))})
            # image-level pooled split conformal on S_hi, fitted on half of the images of Gc
            _imgs = np.concatenate([_idx[g] for g in _Gc])
            _h = _rng.random(len(_imgs)) < 0.5
            _I1, _I2 = _imgs[_h], _imgs[~_h]
            _s = np.sort(_hi[_I1])[::-1]
            _kmax = int(np.floor(_al * (len(_s) + 1))) - 1  # largest #exceedances allowed
            _t = np.inf if _kmax < 0 else (float(np.nextafter(_s[_kmax], np.inf)) if _kmax < len(_s) else -np.inf)
            _e_hi, _w_hi, _v_hi = _ev(_hi, _t, _Gn)
            _e_pt, _, _ = _ev(_pt, _t, _Gn)
            _rows.append({"model": name, "resplit": _r, "side": "accept", "level": _al, "method": "IMAGE-conformal on S_hi",
                          "event_S_hi_new_gens": _e_hi, "event_S_point_new_gens": _e_pt,
                          "share_new_gens_above_level": float(np.mean(np.array(_v_hi) > _al)), "worst_new_gen": _w_hi,
                          "event_same_gens_other_images": float(np.mean(_hi[_I2] >= _t)),
                          "real_accept_eps_sigma": float(np.mean(_lo[_Rn] >= _t + sigma))})
        for _be in V3["beta_grid"]:
            for _meth, _arr in [("conformal on S_lo (corrected)", _lo), ("conformal on S_hi (original)", _hi)]:
                _tf = v_flag_threshold(_arr[_Rc], _be)
                _rows.append({"model": name, "resplit": _r, "side": "flag", "level": _be, "method": _meth,
                              "event_S_lo_new_reals": float(np.mean(_lo[_Rn] < _tf)),
                              "event_S_point_new_reals": float(np.mean(_pt[_Rn] < _tf))})
    return pd.DataFrame(_rows)

def a4_run():
    _files = [v_path("tables", "A4_resplits_raw.parquet"), v_path("tables", "A4_validity_summary.csv")]
    if v_cached("A4", _files):
        log("[v3] A4: saved re-split results are current, loading them instead of re-running")
        return pd.read_parquet(_files[0]), pd.read_csv(_files[1])
    _all = [a4_validity(_n, _s) for _n, _s in V_MODELS]
    if not _all:
        return pd.DataFrame(), pd.DataFrame()
    _D = pd.concat(_all)
    _D.to_parquet(v_path("tables", "A4_resplits_raw.parquet"))
    _num = [c for c in _D.columns if c.startswith(("event", "share", "worst", "real_accept"))]
    _S = _D.groupby(["model", "side", "method", "level"])[_num].agg(["mean", "std"]).reset_index()
    _S.columns = [" ".join(c).strip() for c in _S.columns]
    _tgt = np.where(_S["side"] == "accept", _S["event_S_hi_new_gens mean"], _S["event_S_lo_new_reals mean"])
    _sd = np.where(_S["side"] == "accept", _S["event_S_hi_new_gens std"], _S["event_S_lo_new_reals std"])
    _S["mean_event_over_target"] = _tgt / _S["level"]
    # valid if the mean over re-splits is within 2 Monte-Carlo standard errors of the target
    _S["verdict"] = np.where(_tgt <= _S["level"] + 2.0 * _sd / np.sqrt(V3["n_resplits"]), "valid", "VIOLATED")
    _S.to_csv(v_path("tables", "A4_validity_summary.csv"), index=False)
    _fig, _ax = plt.subplots(1, 2, figsize=(12, 4))
    _a = _S[(_S.side == "accept") & (_S.model == (V_BEST or _S.model.iloc[0]))]
    for _m, _g in _a.groupby("method"):
        _ax[0].errorbar(_g.level, _g["event_S_hi_new_gens mean"], yerr=_g["event_S_hi_new_gens std"], marker="o", capsize=3, label=_m)
    _ax[0].plot(V3["main_alphas"], V3["main_alphas"], "k--", label="target")
    _ax[0].set_xscale("log")
    _ax[0].set_yscale("log")
    _ax[0].set_xlabel("alpha")
    _ax[0].set_ylabel("new-generator avg P(S_hi >= tau_a)")
    _ax[0].set_title(f"Acceptance guarantee over {V3['n_resplits']} re-splits ({V_BEST})")
    _ax[0].legend(fontsize=7)
    _f = _S[(_S.side == "flag") & (_S.model == (V_BEST or _S.model.iloc[0]))]
    for _m, _g in _f.groupby("method"):
        _ax[1].errorbar(_g.level, _g["event_S_lo_new_reals mean"], yerr=_g["event_S_lo_new_reals std"], marker="o", capsize=3, label=_m)
    _ax[1].plot(V3["beta_grid"], V3["beta_grid"], "k--", label="target")
    _ax[1].set_xscale("log")
    _ax[1].set_yscale("log")
    _ax[1].set_xlabel("beta")
    _ax[1].set_ylabel("new-real P(S_lo < tau_f)")
    _ax[1].set_title("Framing guarantee: corrected vs original basis")
    _ax[1].legend(fontsize=7)
    v_savefig(_fig, "A4_validity")
    v_push(v_path("tables"), "v3 A4 validity")
    v_push(v_path("figs"), "v3 A4 figures")
    v_stamp("A4")
    return _D, _S

A4_RAW, A4_SUM = a4_run()
mo.vstack([mo.md("### A4. Guarantee validity over random generator re-splits"),
           mo.ui.table(A4_SUM.round(4), selection=None, page_size=40) if len(A4_SUM) else mo.md("_no counts_")])

In [ ]:
# ---- A5. CERTIFICATE AUDIT of the saved adaptive attacks (CPU) --------------------------
# Theorem 3 per image: if an attacked fake is accepted at certified budget eps_c >= its l2 attack
# radius, then S(x_clean) >= tau_a, hence (w.p. >= 1 - alpha_mc/2) S_hi(x_clean) >= tau_a.
# A "violation" is an accepted attacked fake whose clean S_hi < tau_a. The theorem allows at most
# a Monte-Carlo-sized share of them. Same for framing with S_lo. l_inf attacks whose l2 radius
# exceeds eps_c are outside the certificate and are reported as empirical numbers only.
def a5_audit_one(name, sigma, att_counts, th, tag, att_N=None):
    _df = v_counts(name)
    _te = _df[_df.set == "cf_test"]
    _kf = _te[_te.y == 1].k.to_numpy()
    _kr = _te[_te.y == 0].k.to_numpy()
    _lof, _hif, _ = v_scores(_kf, sigma)
    _lor, _hir, _ = v_scores(_kr, sigma)
    _rows = []
    for (_key, _goal), _g in att_counts.groupby(["attack", "goal"], sort=False):
        _k = _g.k.to_numpy()
        _n = len(_k)
        _lo, _hi, _ = v_scores(_k, sigma, att_N)
        _nrm, _e, _r2 = v_parse_attack(_key)
        for _ec in sorted(set(CFG["cert"]["eps_cert"] + [sigma])):
            _d = decide(_lo, _hi, th["tau_a"], th["tau_f"], _ec)
            # Laundering (JPEG / blur / resize) is applied AFTER the l2 attack, so the laundered image
            # is no longer within radius _r2 of the clean one: those goals are outside the certificate.
            _cov = _ec >= _r2 - 1e-12 and "+" not in _goal
            if _goal.startswith("evasion"):
                _acc = _d == 2
                _miss = int(np.sum(_acc & (_hif[:_n] < th["tau_a"])))
                _rate, _clean = float(np.mean(_acc)), float(np.mean(_hif[:_n] >= th["tau_a"]))
            else:
                _acc = _d == 0
                _miss = int(np.sum(_acc & (_lor[:_n] >= th["tau_f"])))
                _rate, _clean = float(np.mean(_acc)), float(np.mean(_lor[:_n] < th["tau_f"]))
            _rows.append({"model": name, "source": tag, "attack": _key, "norm": _nrm, "attack_l2_radius": _r2,
                          "goal": _goal, "eps_c": _ec, "certified_regime": bool(_cov), "n": _n,
                          "n_accepted_or_flagged": int(np.sum(_acc)),
                          "attacked_success_rate": _rate, "clean_bound_event_rate": _clean,
                          "violations": _miss if _cov else None, "clean_outside_bound_but_succeeded": _miss,
                          "success_ci95": v_cp(int(np.sum(_acc)), _n)})
    return pd.DataFrame(_rows)

def a5_run():
    _out = []
    for _n, _s in V_MODELS:
        _cp = os.path.join(PATHS["res"], f"S9c_{_n}_counts.parquet")
        _jp = os.path.join(PATHS["res"], f"S9c_{_n}.json")
        if os.path.exists(_cp) and os.path.exists(_jp):
            _out.append(a5_audit_one(_n, _s, pd.read_parquet(_cp), load_json(_jp)["thresholds"], "S9c (20 steps, 16 draws)"))
        _sp = v_path("gpu", f"B3_{_n}_counts.parquet")
        _sj = v_path("gpu", f"B3_{_n}.json")
        if os.path.exists(_sp) and os.path.exists(_sj):
            _out.append(a5_audit_one(_n, _s, pd.read_parquet(_sp), load_json(_sj)["thresholds"], "B3 stress"))
    if not _out:
        return pd.DataFrame()
    _A = pd.concat(_out)
    _A["success_ci95"] = _A["success_ci95"].astype(str)
    _A.to_csv(v_path("tables", "A5_certificate_audit.csv"), index=False)
    v_push(v_path("tables", "A5_certificate_audit.csv"))
    return _A

A5 = a5_run()
_a5v = (A5[A5.certified_regime].groupby(["model", "source", "goal"]).agg(
    n_checks=("violations", "size"), total_violations=("violations", "sum"), max_n=("n", "max"),
    non_vacuous_checks=("n_accepted_or_flagged", lambda x: int((x > 0).sum()))).reset_index()
    if len(A5) else pd.DataFrame())
mo.vstack([mo.md("### A5. Certificate audit (violations must be ~0 in the certified regime)"),
           mo.ui.table(_a5v, selection=None) if len(_a5v) else mo.md("_no saved attack counts (S9c) yet_"),
           mo.ui.table(A5.round(4), selection=None, page_size=30) if len(A5) else mo.md("")])

In [ ]:
# ---- A6. CLUSTER-BOOTSTRAP confidence intervals for every headline number (CPU) ----------
# Fakes are resampled by GENERATOR (the exchangeability unit of the guarantee), reals by image.
# Thresholds stay fixed (they come from calibration data); the intervals describe test-set
# sampling error only. 95% percentile intervals.
# Two fake-side summaries, because they answer different questions:
#   pooled  = share of ALL fake images (big generators weigh more): what a deployment sees;
#   gen-avg = mean over generators (>= min_per_gen images) of each generator's own rate: what the
#             CRC guarantee bounds. Only the gen-avg guarantee event may be compared with alpha.
def _boot_fake_rate(acc, gens, rng, B, min_n=1):
    """Returns (pooled, pooled CI, gen-avg, gen-avg CI, number of generators used for gen-avg)."""
    _d = pd.DataFrame({"a": acc.astype(float), "g": gens}).groupby("g").a.agg(["sum", "size"])
    _s, _n = _d["sum"].to_numpy(), _d["size"].to_numpy()
    if len(_s) == 0:
        return np.nan, (np.nan, np.nan), np.nan, (np.nan, np.nan), 0
    _W = rng.multinomial(len(_s), np.full(len(_s), 1 / len(_s)), size=B)
    _r = (_W @ _s) / np.maximum(_W @ _n, 1)
    _k = _n >= min_n
    _ga, _gaci = np.nan, (np.nan, np.nan)
    if _k.any():
        _rate = _s[_k] / _n[_k]
        _Wg = rng.multinomial(int(_k.sum()), np.full(int(_k.sum()), 1 / _k.sum()), size=B)
        _rg = (_Wg @ _rate) / _k.sum()
        _ga, _gaci = float(_rate.mean()), (float(np.quantile(_rg, 0.025)), float(np.quantile(_rg, 0.975)))
    return (float(_s.sum() / _n.sum()), (float(np.quantile(_r, 0.025)), float(np.quantile(_r, 0.975))),
            _ga, _gaci, int(_k.sum()))

def _boot_real_rate(acc, rng, B):
    _n = len(acc)
    if _n == 0:
        return np.nan, (np.nan, np.nan)
    _p = float(np.mean(acc))
    _r = rng.binomial(_n, _p, size=B) / _n
    return _p, (float(np.quantile(_r, 0.025)), float(np.quantile(_r, 0.975)))

def a6_run():
    _rng = np.random.default_rng(CFG["seed"] + 6)
    _B = V3["n_boot"]
    _rows = []
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        for _sn in [s for s in ["cf_test", "openfake", "raid_original", "raid_epsilon16"] if s in set(_df.set)]:
            _t = _df[_df.set == _sn]
            _lo, _hi, _ = v_scores(_t.k.to_numpy(), _s)
            _y, _g = _t.y.to_numpy(), _t.model_name.to_numpy()
            for _al in V3["main_alphas"]:
                _rule = v_rule(_cal, _al, V3["main_beta"])
                for _er in [0.0, 1.0]:
                    _d = decide(_lo, _hi, _rule["tau_a"], _rule["tau_f"], _er * _s)
                    _af, _afci, _afg, _afgci, _G = _boot_fake_rate(_d[_y == 1] == 2, _g[_y == 1], _rng, _B, V3["min_per_gen"])
                    _gp, _gpci, _ge, _geci, _ = _boot_fake_rate(_hi[_y == 1] >= _rule["tau_a"], _g[_y == 1], _rng, _B, V3["min_per_gen"])
                    _ar, _arci = _boot_real_rate(_d[_y == 0] == 2, _rng, _B)
                    _fr, _frci = _boot_real_rate(_d[_y == 0] == 0, _rng, _B)
                    _rows.append({"method": f"{_n} [alpha={_al}, beta={V3['main_beta']}]", "family": "ECA/Cohen",
                                  "set": _sn, "point": f"eps_c={_er}*sigma", "accept_real": _ar, "accept_real_ci": _arci,
                                  "AF_pooled": _af, "AF_pooled_ci": _afci, "AF_gen_avg": _afg, "AF_gen_avg_ci": _afgci,
                                  "guarantee_event_gen_avg (compare with alpha)": _ge, "guarantee_event_gen_avg_ci": _geci,
                                  "guarantee_event_pooled": _gp, "guarantee_event_pooled_ci": _gpci,
                                  "FR": _fr, "FR_ci": _frci, "n_generators_gen_avg": _G})
    for _b in V_BASELINES:
        _sp = os.path.join(PATHS["res"], f"S6_{_b}_scores.parquet")
        if not os.path.exists(_sp):
            continue
        _thr = load_json(os.path.join(PATHS["res"], f"S6_{_b}.json"))["thresholds"]
        _sc = pd.read_parquet(_sp)
        for _sn in [s for s in ["cf_test", "openfake", "raid_original", "raid_epsilon16"] if s in set(_sc.set)]:
            _t = _sc[_sc.set == _sn]
            _y, _g = _t.y.to_numpy(), _t.model_name.to_numpy()
            for _v in VARIANTS["clip" if _b == "clip" else "cnn"]:
                if f"tau_a_{_v}" not in _thr:
                    continue
                _d = baseline_decide(_v, _t.s.to_numpy(), _t.maha.to_numpy(), _t.mcvar.to_numpy(), _thr)
                _af, _afci, _afg, _afgci, _G = _boot_fake_rate(_d[_y == 1] == 2, _g[_y == 1], _rng, _B, V3["min_per_gen"])
                _ar, _arci = _boot_real_rate(_d[_y == 0] == 2, _rng, _B)
                _fr, _frci = _boot_real_rate(_d[_y == 0] == 0, _rng, _B)
                _rows.append({"method": f"{_b}+{_v} [alpha={CFG['cal']['alpha']}]", "family": "baseline (uncertified)",
                              "set": _sn, "point": "clean", "accept_real": _ar, "accept_real_ci": _arci,
                              "AF_pooled": _af, "AF_pooled_ci": _afci, "AF_gen_avg": _afg, "AF_gen_avg_ci": _afgci,
                              "guarantee_event_gen_avg (compare with alpha)": np.nan, "guarantee_event_gen_avg_ci": (np.nan, np.nan),
                              "guarantee_event_pooled": np.nan, "guarantee_event_pooled_ci": (np.nan, np.nan),
                              "FR": _fr, "FR_ci": _frci, "n_generators_gen_avg": _G})
    _T = pd.DataFrame(_rows)
    if len(_T):
        for _c in [c for c in _T.columns if c.endswith("_ci")]:
            _T[_c] = [f"[{a:.4f}, {b:.4f}]" for a, b in _T[_c]]
        _T.to_csv(v_path("tables", "A6_bootstrap_ci.csv"), index=False)
        v_push(v_path("tables", "A6_bootstrap_ci.csv"))
    return _T

A6 = a6_run()
mo.vstack([mo.md("### A6. Cluster-bootstrap 95% intervals"),
           mo.ui.table(A6.round(4), selection=None, page_size=30) if len(A6) else mo.md("_no inputs_")])

In [ ]:
# ---- A7. THREE-WAY DEFENDED BAYES RISK (the paper's evaluation currency) (CPU) -----------
# R_eps = lam_AF * pi_F * AF_eps + lam_FR * pi_R * FR_eps + lam_D * (1 - Cov), pi_F = pi_R = 0.5.
# ECA rows use CERTIFIED upper bounds (held-out guarantee events); baseline rows use EMPIRICAL
# joint-attack success (S7), which is only a lower bound on their true defended AF, and their
# clean framing rate (no framing attack was run on baselines). Both choices favour the baselines.
def a7_run():
    _rows = []
    _budgets = [0.0, 0.06, 0.12, 0.25]
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        _t = _df[_df.set == "cf_test"]
        _smax = _s * v_zN(V3["N"], V3["alpha_mc"])
        for _al in V3["main_alphas"]:
            _rule = v_rule(_cal, _al, V3["main_beta"])
            for _e in _budgets:
                if _e >= _smax:
                    continue
                _m = v_eval_set(_t, _s, _rule, _e)
                _cov = (_m["cov_real"] * _m["n_real"] + _m["cov_fake"] * _m["n_fake"]) / max(1, _m["n_real"] + _m["n_fake"])
                _rows.append({"method": f"{_n} [alpha={_al}]", "family": "ECA/Cohen (certified bound)", "l2_budget": _e,
                              "AF_eps": _m.get("guarantee_event_gen_avg P(S_hi>=tau_a)", np.nan) if _e > 0 else _m["AF"],
                              "FR_eps": _m.get("framing_event P(S_lo<tau_f)", np.nan) if _e > 0 else _m["FR"],
                              "coverage": _cov, "accept_real": _m["accept_real_rate"]})
    for _b in V_BASELINES:
        _j6 = load_json(os.path.join(PATHS["res"], f"S6_{_b}.json"))
        _p7 = os.path.join(PATHS["res"], f"S7_{_b}.json")
        _j7 = load_json(_p7) if os.path.exists(_p7) else {"joint": {}}
        _ev = _j6["eval"].get("cf_test", {})
        for _v in VARIANTS["clip" if _b == "clip" else "cnn"]:
            if _v not in _ev:
                continue
            _m = _ev[_v]
            _cov = (_m["cov_real"] * _m["n_real"] + _m["cov_fake"] * _m["n_fake"]) / max(1, _m["n_real"] + _m["n_fake"])
            for _e in _budgets:
                _af = _m["AF"] if _e == 0 else _j7["joint"].get(f"l2_{_e}", {}).get(_v, np.nan)
                _rows.append({"method": f"{_b}+{_v} [alpha={CFG['cal']['alpha']}]", "family": "baseline (attack lower bound)",
                              "l2_budget": _e, "AF_eps": _af, "FR_eps": _m["FR"], "coverage": _cov,
                              "accept_real": _m["accept_real_rate"]})
    _T = pd.DataFrame(_rows)
    if not len(_T):
        return _T
    for _la, _lf, _ld in V3["losses"]:
        _T[f"risk(lAF={_la},lFR={_lf},lD={_ld})"] = _la * 0.5 * _T.AF_eps + _lf * 0.5 * _T.FR_eps + _ld * (1 - _T.coverage)
    _T.to_csv(v_path("tables", "A7_defended_risk.csv"), index=False)
    v_push(v_path("tables", "A7_defended_risk.csv"))
    return _T

A7 = a7_run()
mo.vstack([mo.md("### A7. Three-way defended Bayes risk (lower is better)"),
           mo.ui.table(A7.round(4), selection=None, page_size=40) if len(A7) else mo.md("_no inputs_")])

In [ ]:
# ---- A8. OUT-OF-POPULATION: OpenFake (2025-26 generators), drift by release date (CPU) ----
# Theorem 5' promises nothing here. We measure (i) the guarantee event per OpenFake generator at
# thresholds calibrated on CommunityForensics, (ii) its rank correlation with release date, and
# (iii) leave-one-generator-out recalibration: adding the other OpenFake generators to the
# calibration pool, does the left-out one come back under alpha? (the deployment recipe)
def a8_run():
    _rows, _summ = [], []
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        if "openfake" not in set(_df.set):
            continue
        _cal = v_cal_parts(_df, _s)
        _of = _df[_df.set == "openfake"]
        _lo, _hi, _ = v_scores(_of.k.to_numpy(), _s)
        _y, _g = _of.y.to_numpy(), _of.model_name.to_numpy()
        _rd = (pd.to_datetime(_of["release_date"].astype(str), errors="coerce").to_numpy()
               if "release_date" in _of.columns else np.full(len(_of), np.datetime64("NaT")))
        _ofg = [g for g in np.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= 10]
        _hi_of = {g: _hi[(_y == 1) & (_g == g)] for g in _ofg}
        for _al in V3["main_alphas"]:
            _rule = v_rule(_cal, _al, V3["main_beta"])
            _ev = {g: float(np.mean(_hi_of[g] >= _rule["tau_a"])) for g in _ofg}
            _logo, _logo_of, _logo_acc, _logo_of_acc = {}, {}, {}, {}
            _lo_r = _lo[_y == 0]
            for _j in _ofg:
                _pool = [_cal["hi"][g] for g in _cal["gens"]] + [_hi_of[g] for g in _ofg if g != _j]
                _t, _ = v_crc(_pool, _al)
                _logo[_j] = float(np.mean(_hi_of[_j] >= max(_t, _rule["tau_f"])))
                # the price of recalibrating: OpenFake real acceptance at eps = sigma under that threshold
                _logo_acc[_j] = float(np.mean(_lo_r >= max(_t, _rule["tau_f"]) + _s)) if len(_lo_r) else np.nan
                _t2, _ok2 = v_crc([_hi_of[g] for g in _ofg if g != _j], _al)
                _logo_of[_j] = float(np.mean(_hi_of[_j] >= max(_t2, _rule["tau_f"]))) if _ok2 else np.nan
                _logo_of_acc[_j] = float(np.mean(_lo_r >= max(_t2, _rule["tau_f"]) + _s)) if (_ok2 and len(_lo_r)) else np.nan
            for _gname in _ofg:
                _m = (_y == 1) & (_g == _gname)
                _d0 = _rd[_m][0] if _m.any() else np.datetime64("NaT")
                _rows.append({"model": _n, "alpha": _al, "of_generator": _gname, "n": int(_m.sum()),
                              "release_date": str(_d0)[:10], "event_CF_calibrated": _ev[_gname],
                              "event_LOGO_CF+OF": _logo[_gname], "event_LOGO_OF_only": _logo_of[_gname],
                              "real_accept_LOGO_CF+OF_eps_sigma": _logo_acc[_gname],
                              "real_accept_LOGO_OF_only_eps_sigma": _logo_of_acc[_gname],
                              "clean_AF_eps_sigma": float(np.mean(_lo[_m] >= _rule["tau_a"] + _s))})
            _tt = pd.DataFrame([r for r in _rows if r["model"] == _n and r["alpha"] == _al])
            _dates = pd.to_datetime(_tt.release_date, errors="coerce")
            _ok = _dates.notna()
            _rho = (v_spearman(_dates[_ok].map(pd.Timestamp.toordinal), _tt.event_CF_calibrated[_ok]) if _ok.sum() >= 4 else (np.nan, np.nan))
            _summ.append({"model": _n, "alpha": _al, "K_openfake_generators": len(_ofg),
                          "mean_event_CF_calibrated": float(_tt.event_CF_calibrated.mean()) if len(_tt) else np.nan,
                          "share_gens_above_alpha": float(np.mean(_tt.event_CF_calibrated > _al)) if len(_tt) else np.nan,
                          "mean_event_LOGO_CF+OF": float(_tt["event_LOGO_CF+OF"].mean()) if len(_tt) else np.nan,
                          "mean_event_LOGO_OF_only": float(np.nanmean(_tt["event_LOGO_OF_only"])) if len(_tt) and _tt["event_LOGO_OF_only"].notna().any() else np.nan,
                          "real_accept_LOGO_CF+OF_eps_sigma": float(np.nanmean(_tt["real_accept_LOGO_CF+OF_eps_sigma"])) if len(_tt) and _tt["real_accept_LOGO_CF+OF_eps_sigma"].notna().any() else np.nan,
                          "real_accept_LOGO_OF_only_eps_sigma": float(np.nanmean(_tt["real_accept_LOGO_OF_only_eps_sigma"])) if len(_tt) and _tt["real_accept_LOGO_OF_only_eps_sigma"].notna().any() else np.nan,
                          # True: the CRC threshold sits below tau_f, so tau_a = tau_f and the accept side is
                          # set by the framing quantile, not by alpha (raising alpha changes nothing)
                          "tau_f_binds": bool(v_crc([_cal["hi"][g] for g in _cal["gens"]], _al)[0] <= _rule["tau_f"]),
                          "spearman_rho_event_vs_release_date": float(_rho[0]), "spearman_p": float(_rho[1]),
                          "openfake_real_accept_eps_sigma": float(np.mean(_lo[_y == 0] >= _rule["tau_a"] + _s)) if (_y == 0).any() else np.nan})
    _R, _S = pd.DataFrame(_rows), pd.DataFrame(_summ)
    if len(_R):
        _R.to_csv(v_path("tables", "A8_openfake_per_generator.csv"), index=False)
        _S.to_csv(v_path("tables", "A8_openfake_summary.csv"), index=False)
        _b = _R[(_R.model == (V_BEST or _R.model.iloc[0])) & (_R.alpha == V3["select_alpha"])].copy()
        _b["date"] = pd.to_datetime(_b.release_date, errors="coerce")
        _b = _b.sort_values("date")
        _fig, _ax = plt.subplots(figsize=(9, 4))
        _ax.scatter(_b.date, _b.event_CF_calibrated, label="CF-calibrated")
        _ax.scatter(_b.date, _b["event_LOGO_CF+OF"], marker="x", label="recalibrated with other OpenFake generators")
        _ax.axhline(V3["select_alpha"], color="k", ls="--", label="alpha")
        for _r in _b.itertuples():
            _ax.annotate(str(_r.of_generator)[:18], (_r.date, _r.event_CF_calibrated), fontsize=6, rotation=20)
        _ax.set_ylabel("P(S_hi >= tau_a) per generator")
        _ax.set_title(f"OpenFake: guarantee event vs release date ({V_BEST}, alpha={V3['select_alpha']})")
        _ax.legend(fontsize=7)
        v_savefig(_fig, "A8_openfake_drift")
        v_push(v_path("tables"), "v3 A8 openfake")
        v_push(v_path("figs"), "v3 A8 figure")
    return _R, _S

A8_PER, A8_SUM = a8_run()
mo.vstack([mo.md("### A8. OpenFake drift"),
           mo.ui.table(A8_SUM.round(4), selection=None) if len(A8_SUM) else mo.md("_no OpenFake counts_")])

In [ ]:
# ---- A11. GENERATOR HETEROGENEITY: pooled vs generator-averaged error (CPU) ---------------
# The CRC guarantee bounds the generator-AVERAGED event; a deployment counts IMAGES. When a few
# held-out generators are large and hard, the pooled accepted-fake rate can sit far above alpha
# while the guarantee holds. (i) shows where the gap comes from on the official split. (ii) asks,
# over random generator re-splits, which calibration controls which quantity:
#   GEN-CRC (ours)       loss r_g(t), bound 1                  -> generator average
#   SIZE-WEIGHTED CRC    loss (n_g / n_bar) r_g(t), bound B    -> E[n_g r_g] / n_bar, i.e. the pooled rate,
#                        B = largest calibration n_g / n_bar (assumes no new generator is larger)
#   CAPPED SIZE-WEIGHTED same with n_g -> min(n_g, cap)        -> pooled rate when each generator's images
#                        count at most `cap` times; B = cap / n_bar is known in advance
# n_bar comes from the calibration generators, so the two weighted rules are approximate CRC (the
# weights are not a symmetric function of all K+1 generators). The B / (K+1) term makes them
# infeasible when sizes are heavy-tailed; that limit is itself a result.
A11_CAP = 100  # pre-specified: 2.5x the median CF generator size (40 images)

def a11_crc(per_gen, alpha, weighting="equal", cap=A11_CAP):
    _K = len(per_gen)
    if _K == 0:
        return np.inf, False
    _n = np.array([len(p) for p in per_gen], float)
    if weighting == "equal":
        _w = np.ones(_K)
    else:
        _m = np.minimum(_n, cap) if weighting == "capped" else _n
        _w = _m / _m.mean()
    _B = float(cap / np.minimum(_n, cap).mean()) if weighting == "capped" else float(_w.max())
    _c = (alpha - _B / (_K + 1)) * (_K + 1) / _K
    if _c < 0:
        return np.inf, False
    _v = np.concatenate([np.asarray(p, float) for p in per_gen])
    _wi = np.concatenate([np.full(len(p), _w[i] / (len(p) * _K)) for i, p in enumerate(per_gen)])
    _o = np.argsort(_v, kind="stable")
    _v, _wi = _v[_o], _wi[_o]
    _u, _first = np.unique(_v, return_index=True)
    _tail = np.cumsum(_wi[::-1])[::-1][_first]
    _ok = np.where(_tail <= _c + 1e-12)[0]
    return (float(_u[_ok[0]]) if len(_ok) else np.inf), True

def a11_rates(hits, n, G, cap=A11_CAP):
    """gen-avg, pooled and capped-pooled rate over generator indices G."""
    _r = hits[G] / n[G]
    _m = np.minimum(n[G], cap)
    return float(_r.mean()), float(hits[G].sum() / n[G].sum()), float((_m * _r).sum() / _m.sum())

def a11_official(name, sigma):
    _df = v_counts(name)
    _cal = v_cal_parts(_df, sigma)
    _t = _df[_df.set == "cf_test"]
    _lo, _hi, _ = v_scores(_t.k.to_numpy(), sigma)
    _y, _g = _t.y.to_numpy(), _t.model_name.to_numpy()
    _ncal = np.array([len(_cal["hi"][g]) for g in _cal["gens"]])
    _per, _sum = [], []
    for _al in V3["main_alphas"]:
        _rule = v_rule(_cal, _al, V3["main_beta"])
        for _er in [0.0, 1.0]:
            _d = decide(_lo, _hi, _rule["tau_a"], _rule["tau_f"], _er * sigma)
            _f = pd.DataFrame({"g": _g[_y == 1], "acc": _d[_y == 1] == 2, "ev": _hi[_y == 1] >= _rule["tau_a"]})
            _pg = _f.groupby("g").agg(n=("acc", "size"), AF=("acc", "mean"), event=("ev", "mean"), n_acc=("acc", "sum")).reset_index()
            _pg = _pg[_pg.n >= V3["min_per_gen"]]
            for _r in _pg.itertuples():
                _per.append({"model": name, "alpha": _al, "eps_over_sigma": _er, "generator": _r.g, "n": int(_r.n),
                             "AF": float(_r.AF), "guarantee_event": float(_r.event), "n_accepted": int(_r.n_acc)})
            _big = _pg.n > A11_CAP
            _top3 = _pg.sort_values("n_acc", ascending=False).head(3)
            _m = np.minimum(_pg.n.to_numpy(), A11_CAP)
            _sum.append({"model": name, "alpha": _al, "eps_over_sigma": _er, "K_test": int(len(_pg)),
                         "AF_gen_avg": float(_pg.AF.mean()), "AF_pooled": float(_pg.n_acc.sum() / _pg.n.sum()),
                         "AF_capped_pooled": float((_m * _pg.AF.to_numpy()).sum() / _m.sum()),
                         "event_gen_avg (bounded by alpha)": float(_pg.event.mean()),
                         "event_pooled": float((_pg.event * _pg.n).sum() / _pg.n.sum()),
                         f"test_gens_n>{A11_CAP}": int(_big.sum()), f"test_images_in_gens_n>{A11_CAP}": int(_pg.n[_big].sum()),
                         f"AF_pooled_gens_n>{A11_CAP}": float(_pg.n_acc[_big].sum() / max(_pg.n[_big].sum(), 1)) if _big.any() else np.nan,
                         f"AF_pooled_gens_n<={A11_CAP}": float(_pg.n_acc[~_big].sum() / max(_pg.n[~_big].sum(), 1)),
                         "top3_generators_by_accepted": "; ".join(f"{r.g} ({int(r.n_acc)}/{int(r.n)})" for r in _top3.itertuples()),
                         "share_accepted_fakes_from_top3": float(_top3.n_acc.sum() / max(_pg.n_acc.sum(), 1)),
                         "cal_K": int(len(_ncal)), "cal_max_n": int(_ncal.max()) if len(_ncal) else 0,
                         f"cal_gens_n>{A11_CAP}": int((_ncal > A11_CAP).sum())})
    return pd.DataFrame(_per), pd.DataFrame(_sum)

def a11_resplits(name, sigma):
    _df = v_counts(name)
    _p = _df[_df.set.isin(["cf_cal", "cf_test"])]
    _lo, _hi, _ = v_scores(_p.k.to_numpy(), sigma)
    _y, _g = _p.y.to_numpy(), _p.model_name.to_numpy()
    _gens = [g for g in np.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= V3["min_per_gen"]]
    _code = {g: i for i, g in enumerate(_gens)}
    _fm = (_y == 1) & np.isin(_g, _gens)
    _hf = _hi[_fm]
    _cf = np.array([_code[g] for g in _g[_fm]])
    _n = np.bincount(_cf, minlength=len(_gens)).astype(float)
    _per = [_hf[_cf == i] for i in range(len(_gens))]
    _lr = _lo[_y == 0]
    _rng = np.random.default_rng(CFG["seed"] + 111)
    _rows = []
    for _rs in range(V3["n_resplits"]):
        _perm = _rng.permutation(len(_gens))
        _Gc, _Gn = _perm[: len(_gens) // 2], _perm[len(_gens) // 2:]
        _rp = _rng.permutation(len(_lr))
        _Rc, _Rn = _rp[: len(_rp) // 2], _rp[len(_rp) // 2:]
        _tf = v_flag_threshold(_lr[_Rc], V3["main_beta"])
        for _al in V3["main_alphas"]:
            for _meth in ["equal", "size", "capped"]:
                _t, _feas = a11_crc([_per[i] for i in _Gc], _al, _meth)
                _ta = max(_t, _tf)  # the deployed rule: tau_a = max(CRC threshold, tau_f)
                _hits = np.bincount(_cf, weights=(_hf >= _ta).astype(float), minlength=len(_gens))
                _ga, _po, _cp = a11_rates(_hits, _n, _Gn)
                _rows.append({"model": name, "resplit": _rs, "alpha": _al,
                              "method": {"equal": "GEN-CRC (ours)", "size": "SIZE-WEIGHTED CRC", "capped": f"CAPPED SIZE-WEIGHTED CRC (cap {A11_CAP})"}[_meth],
                              "feasible": bool(_feas and np.isfinite(_t)), "event_gen_avg": _ga, "event_pooled": _po,
                              "event_capped_pooled": _cp, "real_accept_eps_sigma": float(np.mean(_lr[_Rn] >= _ta + sigma)),
                              "largest_new_gen_n": int(_n[_Gn].max()), "largest_cal_gen_n": int(_n[_Gc].max())})
    return pd.DataFrame(_rows)

def a11_run():
    if not V_MODELS:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    _files = [v_path("tables", f) for f in ["A11_per_generator_cf_test.csv", "A11_official_split_pooled_vs_gen_avg.csv",
                                            "A11_resplits_summary.csv"]]
    if v_cached("A11", _files):
        log("[v3] A11: saved results are current, loading them instead of re-running")
        return tuple(pd.read_csv(f) for f in _files)
    _o = [a11_official(_n, _s) for _n, _s in V_MODELS]
    _PER = pd.concat([o[0] for o in _o])
    _OFF = pd.concat([o[1] for o in _o])
    _RAW = pd.concat([a11_resplits(_n, _s) for _n, _s in V_MODELS])
    _num = ["feasible", "event_gen_avg", "event_pooled", "event_capped_pooled", "real_accept_eps_sigma"]
    _RS = _RAW.groupby(["model", "method", "alpha"])[_num].agg(["mean", "std"]).reset_index()
    _RS.columns = [" ".join(c).strip() for c in _RS.columns]
    _se = 2.0 / np.sqrt(V3["n_resplits"])
    _RS["gen_avg_valid"] = _RS["event_gen_avg mean"] <= _RS["alpha"] + _se * _RS["event_gen_avg std"]
    _RS["pooled_valid"] = _RS["event_pooled mean"] <= _RS["alpha"] + _se * _RS["event_pooled std"]
    _RS["capped_pooled_valid"] = _RS["event_capped_pooled mean"] <= _RS["alpha"] + _se * _RS["event_capped_pooled std"]
    _PER.to_csv(v_path("tables", "A11_per_generator_cf_test.csv"), index=False)
    _OFF.to_csv(v_path("tables", "A11_official_split_pooled_vs_gen_avg.csv"), index=False)
    _RAW.to_parquet(v_path("tables", "A11_resplits_raw.parquet"))
    _RS.to_csv(v_path("tables", "A11_resplits_summary.csv"), index=False)
    _b = _PER[(_PER.model == (V_BEST or _PER.model.iloc[0])) & (_PER.alpha == V3["select_alpha"]) & (_PER.eps_over_sigma == 1.0)]
    if len(_b):
        _fig, _ax = plt.subplots(figsize=(7, 4))
        _ax.scatter(_b.n, _b.AF, s=10, alpha=0.6)
        for _r in _b.sort_values("n_accepted", ascending=False).head(5).itertuples():
            _ax.annotate(str(_r.generator)[:24], (_r.n, _r.AF), fontsize=6)
        _ax.axhline(V3["select_alpha"], color="k", ls="--", label="alpha")
        _ax.set_xscale("log")
        _ax.set_xlabel("images of the held-out generator (log)")
        _ax.set_ylabel("certified accepted-fake rate at eps = sigma")
        _ax.set_title(f"Per-generator AF on held-out CF ({V_BEST}, alpha={V3['select_alpha']})")
        _ax.legend(fontsize=7)
        v_savefig(_fig, "A11_per_generator_AF_vs_size")
    v_push(v_path("tables"), "v3 A11 heterogeneity")
    v_push(v_path("figs"), "v3 A11 figure")
    v_stamp("A11")
    return _PER, _OFF, _RS

A11_PER, A11_OFF, A11_RS = a11_run()
mo.vstack([mo.md("### A11. Pooled vs generator-averaged error"),
           mo.ui.table(A11_OFF.round(4), selection=None, page_size=40) if len(A11_OFF) else mo.md("_no counts_"),
           mo.ui.table(A11_RS.round(4), selection=None, page_size=40) if len(A11_RS) else mo.md("")])

In [ ]:
# ---- A9. RESEARCH-QUESTION SCOREBOARD: each pre-registered hypothesis against the data ----
def _get(path, keys, default=None):
    if not os.path.exists(path):
        return default
    _o = load_json(path)
    for _k in keys:
        if not isinstance(_o, dict) or _k not in _o:
            return default
        _o = _o[_k]
    return _o

def a9_run():
    _R = []
    _res = PATHS["res"]
    _add = lambda rq, hyp, crit, val, verdict, src: _R.append(
        {"RQ": rq, "hypothesis": hyp, "criterion": crit, "measured": val, "verdict": verdict, "source": src})
    # S0
    _s0 = os.path.join(_res, "S0_theory.json")
    if os.path.exists(_s0):
        _j = load_json(_s0)
        _v = {"A1 general": _j["A1_prop2"]["max cost / (r_max*sqrt(k/lambda_min)) [must be <= 1]"],
              "A1 nonneg": _j["A1_prop2"]["max cost / (r_max*sqrt(k)) when cosines >= 0 [must be <= 1]"],
              "A2 Lipschitz": _j["A2_lipschitz"]["max |dS|/||dx|| over random nets [must be <= 1 up to MC noise]"],
              "A3 framing": _j["A3_conformal_ties"]["P(flag real), strict '<' [must be <= beta]"],
              "A4 CRC": _j["A4_crc_generators"]["E[AF of new generator] [must be <= alpha]"]}
        _ok = _v["A1 general"] <= 1 + 1e-6 and _v["A1 nonneg"] <= 1 + 1e-6 and _v["A2 Lipschitz"] <= 1.1
        _add("S0", "proved claims hold numerically", "all ratios <= 1, rates <= targets", json.dumps({k: round(float(v), 4) for k, v in _v.items()}),
             "PASS" if _ok else "FAIL", "S0_theory.json")
    else:
        _add("S0", "proved claims hold numerically", "", "missing", "MISSING", "S0_theory.json")
    # RQ1
    _vals = {}
    for _b in V_BASELINES:
        _j = _get(os.path.join(_res, f"S7_{_b}.json"), ["joint"], {})
        _jb = _get(os.path.join(_res, f"S7_{_b}.json"), ["blind"], {})
        for _v, _x in (_j.get("l2_0.25") or {}).items():
            _vals[f"{_b}+{_v}"] = (_x, (_jb.get("l2_0.25") or {}).get(_v))
    if _vals:
        _mn = min(v[0] for v in _vals.values())
        _verd = "SUPPORTED" if _mn >= 0.5 else ("FALSIFIED" if _mn < 0.2 else "PARTIAL")
        _add("RQ1", "joint attack drives every gated baseline's AF above 50% at l2 0.25",
             "falsified if any gate keeps AF < 20%", json.dumps({k: [round(float(a), 3), None if b is None else round(float(b), 3)] for k, (a, b) in _vals.items()}) + " [joint, blind]",
             _verd, "S7_*.json")
    else:
        _add("RQ1", "joint attack beats gates", "", "missing", "MISSING", "S7_*.json")
    # RQ2
    _p2 = {b: _get(os.path.join(_res, f"S7_{b}.json"), ["prop2", "share_emp_le_bound"]) for b in V_BASELINES if b != "clip"}
    _p2 = {k: v for k, v in _p2.items() if v is not None}
    if _p2:
        _add("RQ2", "measured joint cost stays within the sqrt(k / lambda_min) bound", "falsified if > 1.2x bound for most images",
             json.dumps({k: round(float(v), 3) for k, v in _p2.items()}) + " share within 1.2x", "SUPPORTED" if min(_p2.values()) >= 0.5 else "FALSIFIED",
             "S7_*.json prop2 (quantised l2 grid; first-order)")
    # RQ3
    for _r in A3_VOTES.itertuples() if len(A3_VOTES) else []:
        _ok = _r.share_fakes_le_10pct_real_votes >= 0.99 and _r.share_reals_ge_90pct_real_votes >= 0.5
        _add("RQ3", f"{_r.model}: 99% of fakes get <= 10% real votes and the median real >= 90%",
             "both conditions on held-out CF", f"fakes<=10%: {_r.share_fakes_le_10pct_real_votes:.3f}, median real vote: {_r.real_real_vote_median:.3f}",
             "SUPPORTED" if _ok else "NOT MET", "S8 counts")
    _base_auc = _get(os.path.join(_res, "S6_base.json"), ["eval", "cf_test", "auroc"])
    for _n, _s in V_MODELS:
        _auc = _get(os.path.join(_res, f"S8_{_n}.json"), ["eval", "cf_test", "auroc"])
        if _auc is not None and _base_auc is not None:
            _add("Risk 1", f"{_n}: smoothing costs <= 5 AUROC points vs BASE", "drop > 0.05 = negative result we report",
                 f"BASE {_base_auc:.4f} vs {_auc:.4f} (drop {_base_auc - _auc:+.4f})", "OK" if _base_auc - _auc <= 0.05 else "MATERIALISED", "S6/S8 json")
    # RQ4
    if len(A3_GRID):
        _g = A3_GRID[(A3_GRID.set == "cf_test") & np.isclose(A3_GRID.eps_over_sigma, 1.0)]
        for _al, _be in [(0.01, 0.01), (0.05, V3["main_beta"])]:
            _x = _g[(_g.alpha == _al) & (_g.beta == _be)].set_index("model").accept_real_rate.to_dict()
            if _x:
                _best = max(_x.values())
                _add("RQ4", f"certified real acceptance >= 50% at eps = sigma (alpha={_al}, beta={_be})",
                     "falsified if < 20% at every sigma", json.dumps({k: round(float(v), 4) for k, v in _x.items()}),
                     "SUPPORTED" if _best >= 0.5 else ("FALSIFIED (safe but not useful)" if _best < 0.2 else "PARTIAL"),
                     "A3 grid" + (" (pre-registered point)" if _al == 0.01 else " (relaxed point)"))
    # RQ5
    if len(A4_SUM):
        _o = A4_SUM[(A4_SUM.side == "accept") & (A4_SUM.method == "GEN-CRC on S_hi (ours)")]
        _add("RQ5a", "held-out CF generators meet E[event] <= alpha", "violation beyond MC error = bug",
             json.dumps({f"{r.model}@{r.level}": round(float(r["mean_event_over_target"]), 3) for _, r in _o.iterrows()}) + " (mean/alpha)",
             "SUPPORTED" if (_o.verdict == "valid").all() else "VIOLATED (check for a bug)", "A4 re-splits")
    if len(A8_SUM):
        _o = A8_SUM[A8_SUM.alpha == V3["select_alpha"]]
        _viol = bool((_o.mean_event_CF_calibrated > _o.alpha).any())
        _trend = bool(((_o.spearman_rho_event_vs_release_date > 0) & (_o.spearman_p < 0.05)).any())
        _add("RQ5b", "OpenFake: does the CF-calibrated guarantee transfer, and does the event grow with release date?",
             "measured, not guaranteed",
             json.dumps({r.model: [round(float(r.mean_event_CF_calibrated), 3), round(float(r.spearman_rho_event_vs_release_date), 3),
                                   round(float(r.spearman_p), 3)] for r in _o.itertuples()}) + f" [mean event (alpha={V3['select_alpha']}), rho vs date, p]",
             ("DOES NOT TRANSFER" if _viol else "TRANSFERS") + ("; newer is worse" if _trend else "; no significant date trend"), "A8")
    # RQ6
    _fl = os.path.join(_res, "S10c_floor.json") if os.path.exists(os.path.join(_res, "S10c_floor.json")) else os.path.join(_res, "S10_floor.json")
    if os.path.exists(_fl):
        _j = load_json(_fl)
        _ok = all(all(rp <= fl + 1e-9 for rp, fl in zip(m.get("robust_pairs", []), _j.get("floor_three_way", [])))
                  for m in _j.get("models", {}).values())
        _add("RQ6", "information floor P(||a|| > eps) never binds; estimation does", "robust-pair rate above floor = bug",
             f"median ||a||_2 = {_j.get('a_l2', {}).get('median')}; floor at eps grid {[round(float(x), 3) for x in _j.get('floor_three_way', [])]}",
             "CONSISTENT" if _ok else "BUG", os.path.basename(_fl))
    # Secondary
    _sq = os.path.join(_res, "S11c_seq.json") if os.path.exists(os.path.join(_res, "S11c_seq.json")) else os.path.join(_res, "S11_seq.json")
    if os.path.exists(_sq):
        _j = load_json(_sq)
        _ms = {k: v.get("mean_samples") for k, v in _j.items() if isinstance(v, dict) and "mean_samples" in v}
        _eca = {k: v for k, v in _ms.items() if not k.startswith("cohen")}  # the claim is about ECA models
        if _eca:
            _add("KBS-2", "ECA models: mean sequential samples per decision < 128", "every ECA model below 128 (Cohen shown for reference)",
                 json.dumps({k: round(float(v), 1) for k, v in _ms.items()}),
                 "SUPPORTED" if max(_eca.values()) < 128 else "NOT MET", os.path.basename(_sq))
    if len(A5):
        _c = A5[A5.certified_regime]
        _nv = int((_c.n_accepted_or_flagged > 0).sum()) if "n_accepted_or_flagged" in _c else -1
        _add("Thm 3", "no certified-regime violations under adaptive EOT attacks (pure l2 attacks; laundering is outside the certificate)",
             "violations ~ 0", f"{int(_c.violations.sum())} violations in {int(len(_c))} checks ({_nv} non-vacuous: at least one success)",
             "SUPPORTED" if _c.violations.sum() == 0 else "CHECK", "A5")
    # Pooled vs generator-averaged error (A11)
    if len(A11_OFF):
        _o = A11_OFF[(A11_OFF.model == (V_BEST or A11_OFF.model.iloc[0])) & (A11_OFF.alpha == V3["select_alpha"]) & (A11_OFF.eps_over_sigma == 1.0)]
        if len(_o):
            _r = _o.iloc[0]
            _add("Pooled AF", f"{_r.model}: image-level accepted-fake rate on held-out CF stays near alpha={V3['select_alpha']}",
                 "pooled AF <= 2 alpha", f"pooled {_r.AF_pooled:.3f}, gen-avg {_r.AF_gen_avg:.3f}, top-3 generators carry "
                 f"{_r.share_accepted_fakes_from_top3:.0%} of accepted fakes ({_r.top3_generators_by_accepted})",
                 "OK" if _r.AF_pooled <= 2 * V3["select_alpha"] else "GAP (report both; guarantee is per generator)", "A11")
    if len(A11_RS):
        _o = A11_RS[(A11_RS.model == (V_BEST or A11_RS.model.iloc[0])) & (A11_RS.alpha == V3["select_alpha"])]
        _add("Pooled guarantee", "which calibration controls the POOLED rate on new generators (re-splits)",
             "pooled_valid per method",
             json.dumps({r.method: {"feasible": round(float(r["feasible mean"]), 2), "pooled": round(float(r["event_pooled mean"]), 3),
                                    "capped": round(float(r["event_capped_pooled mean"]), 3), "accept": round(float(r["real_accept_eps_sigma mean"]), 3)}
                         for _, r in _o.iterrows()}), "MEASURED", "A11")
    _T = pd.DataFrame(_R)
    _T.to_csv(v_path("tables", "A9_scoreboard.csv"), index=False)
    with open(v_path("A9_SCOREBOARD.md"), "w") as _fh:
        _fh.write("# Research-question scoreboard\n\n")
        for _r in _T.itertuples():
            _fh.write(f"- **{_r.RQ}** [{_r.verdict}] {_r.hypothesis}. Criterion: {_r.criterion}. Measured: {_r.measured}. ({_r.source})\n")
    v_push(v_path("A9_SCOREBOARD.md"))
    v_push(v_path("tables", "A9_scoreboard.csv"))
    return _T

A9 = a9_run()
mo.vstack([mo.md("### A9. Research-question scoreboard"), mo.ui.table(A9, selection=None, page_size=30)])

In [ ]:
# ---- A10. BUNDLE: one zip with every analysis output and every saved result file ---------
import zipfile

def a10_bundle():
    _z = "eca3wd_analysis_bundle.zip"
    with zipfile.ZipFile(_z + ".tmp", "w", zipfile.ZIP_DEFLATED) as _zf:
        for _d, _, _fs in os.walk(V3["out"]):
            for _f in _fs:
                _p = os.path.join(_d, _f)
                if _f.endswith((".json", ".csv", ".md", ".png", ".pdf", ".tex")) or (_f.endswith(".parquet") and os.path.getsize(_p) < 50 * 2**20):
                    _zf.write(_p, os.path.join("v3", os.path.relpath(_p, V3["out"])))
        for _d, _, _fs in os.walk(PATHS["res"]):
            if "_superseded" in _d:
                continue
            for _f in _fs:
                _p = os.path.join(_d, _f)
                # per-image vote counts and baseline scores too, so every analysis can be redone offline
                _raw = _f.endswith(".parquet") and _f.startswith(("S8_", "S9c_", "S6_")) and os.path.getsize(_p) < 200 * 2**20
                if _f.endswith((".json", ".csv", ".md", ".png")) or _raw:
                    _zf.write(_p, os.path.join("results", os.path.relpath(_p, PATHS["res"])))
    os.replace(_z + ".tmp", _z)
    v_push(V3["out"], "v3 analysis bundle")
    return {"zip": os.path.abspath(_z), "MB": round(os.path.getsize(_z) / 2**20, 2)}

_ = (A2, A3_GRID, A4_SUM, A5, A6, A7, A8_SUM, A9, A11_RS)
BUNDLE_INFO = a10_bundle()
mo.md(f"**Part A done.** Bundle: `{BUNDLE_INFO['zip']}` ({BUNDLE_INFO['MB']} MB). Download it from the file browser and share it. "
      f"RAM now {rss_gb():.1f} GB.")

In [ ]:
# ---- B-LIBRARY: data rebuild, certification with any N, seeded training ----------------
B_LOG = []

def b_step(name, fn):
    """Run one part-B unit; one failure never blocks the next unit or the viewer."""
    if time_left_h() < 0.3:
        B_LOG.append({"step": name, "result": "not started: session clock nearly out; re-run to continue"})
        return None
    _t = time.time()
    try:
        _r = fn()
        _msg = str(_r)[:300]
    except Exception as _e:
        _r, _msg = None, f"FAILED: {type(_e).__name__}: {str(_e)[:250]}"
    B_LOG.append({"step": name, "minutes": round((time.time() - _t) / 60, 1), "result": _msg})
    log(f"[v3] {name}: {_msg}")
    gpu_free()
    trim_ram()
    return _r

def b_weight(name):
    """Weights path of a trained model, pulled from the Hub if needed."""
    _p = os.path.join(PATHS["models"], name, "head.pt" if name == "clip" else "final.pt")
    ensure(_p)
    return _p

def b_cf_store():
    """CommunityForensics crop cache: local, else rebuilt with S1/S2 (same
    selection file from the Hub, deterministic crops, so the same images and order)."""
    _d = os.path.join(PATHS["data"], "cf")
    _st = open_store(_d, "cf_")
    if len(_st.meta):
        return _st
    stage_S1_index()
    stage_S2_cache()
    return open_store(_d, "cf_")

B_STORE = {}

def b_store_lazy():
    """The crop cache, opened (or rebuilt) only the first time a unit really needs images, so a
    re-run whose outputs all exist never touches the 30-60 min rebuild."""
    if "cf" not in B_STORE:
        B_STORE["cf"] = b_cf_store()
    return B_STORE["cf"]

def b_test_idx(store):
    _y = store.meta.label.to_numpy()
    _sp = store.meta.split.to_numpy()
    _test = np.where(_sp == "test")[0]
    return _test[_y[_test] == 1], _test[_y[_test] == 0]

def b_subset(store, split, per_gen_cap, n_real):
    """Balanced certification subset: up to per_gen_cap fakes from every generator that has at
    least min_per_gen fakes in the split, and n_real reals; chosen by hash (deterministic)."""
    _m = store.meta
    _i = np.where(_m.split.to_numpy() == split)[0]
    _f = _i[_m.label.to_numpy()[_i] == 1]
    _r = _i[_m.label.to_numpy()[_i] == 0]
    _g = _m.model_name.to_numpy()
    _key = np.array([stable_u(f"sub:{split}:{j}") for j in range(len(_m))])
    _pick = []
    for _gn in np.unique(_g[_f]):
        _gi = _f[_g[_f] == _gn]
        if len(_gi) >= min(V3["min_per_gen"], per_gen_cap):
            _pick.append(_gi[np.argsort(_key[_gi])][:per_gen_cap])
    _rr = _r[np.argsort(_key[_r])][:n_real]
    return np.sort(np.concatenate(_pick + [_rr])) if _pick else np.sort(_rr)

def b_certify(model, store, idx, sigma, N, launder=None):
    _k = []
    for _i in range(0, len(idx), CFG["cert"]["img_batch"]):
        _b = store.batch(idx[_i:_i + CFG["cert"]["img_batch"]])
        _b = np.stack([launder_eval(a, launder) for a in _b]) if launder else center224(_b)
        _k.append(smooth_votes(model, to_tensor01(_b, DEVICE), sigma, N, CFG["cert"]["chunk"], DEVICE))
    return np.concatenate(_k) if _k else np.zeros(0)

def b_counts_frame(store, idx, k, setname, N):
    _m = store.meta.iloc[idx]
    return pd.DataFrame({"set": setname, "k": k, "y": _m.label.to_numpy(), "model_name": _m.model_name.to_numpy(),
                         "idx": idx, "N": N})

def b_frontier_from_counts(df, sigma, N, tag):
    """Feasibility and certified operating points from a counts table (any N)."""
    _cal = v_cal_parts(df, sigma, N)
    _zN = v_zN(N, V3["alpha_mc"])
    _per = [_cal["hi"][g] for g in _cal["gens"]]
    _te = df[df.set == "cf_test"]
    _rows = []
    for _al in V3["main_alphas"]:
        _rule = v_rule(_cal, _al, V3["main_beta"])
        _m = v_eval_set(_te, sigma, _rule, sigma, N)
        _rows.append({"tag": tag, "N": N, "z_N": _zN, "sigma": sigma, "alpha": _al,
                      "alpha_min(eps=sigma)": v_risk(_per, sigma * _zN - sigma), "K": len(_per),
                      "accept_real_eps_sigma": _m["accept_real_rate"], "AF_eps_sigma": _m["AF"],
                      "guarantee_event": _m.get("guarantee_event_gen_avg P(S_hi>=tau_a)"),
                      "auroc_point": auroc_real(v_scores(_te.k.to_numpy(), sigma, N)[2], _te.y.to_numpy())})
    return pd.DataFrame(_rows)

def b_train(name, mode, sigma, seed, max_steps=None, eca=None):
    """Same recipe as S5 (BASE or ECA-3WD/ADRT), with a different seed.
    Resumable per epoch; weights synced to the Hub under models_*/<name>/."""
    _dir = os.path.join(PATHS["models"], name)
    os.makedirs(_dir, exist_ok=True)
    _final = os.path.join(_dir, "final.pt")
    if ensure(_final):
        return _final
    _tc, _ec = CFG["train"], {**CFG["eca"], **(eca or {})}  # `eca` overrides loss weights (C6 ablation)
    _store = b_store_lazy()
    _tr = np.where(_store.meta.split.to_numpy() == "train")[0]
    torch.manual_seed(seed)
    _model = Detector(CFG["model"]["backbone"], CFG["model"]["drop_path"],
                      pretrained=os.environ.get("V3_SMOKE", "0") != "1").to(DEVICE)
    _opt = torch.optim.AdamW(_model.parameters(), lr=_tc["lr"], weight_decay=_tc["wd"])
    _steps_ep = len(_tr) // _tc["batch"]
    if max_steps:
        _steps_ep = min(_steps_ep, max_steps)
    _total = max(1, _steps_ep * _tc["epochs"])
    _warm = max(1, int(_total * _tc["warmup_frac"]))
    _sched = torch.optim.lr_scheduler.LambdaLR(
        _opt, lambda s: min(1.0, (s + 1) / _warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / _total))))
    _ep0, _hist = 0, []
    _last = os.path.join(_dir, "last.pt")
    if ensure(_last):
        _ck = torch.load(_last, map_location=DEVICE, weights_only=False)
        _model.load_state_dict(_ck["model"])
        _opt.load_state_dict(_ck["opt"])
        _sched.load_state_dict(_ck["sched"])
        _ep0, _hist = _ck["epoch"] + 1, _ck["hist"]
    _k = _tc["k_noise"]
    _epochs = 1 if max_steps else _tc["epochs"]
    for _ep in range(_ep0, _epochs):
        _model.train()
        _dl = make_loader(_store, _tr, _tc["batch"], _tc["workers"], seed, _ep)
        _t0, _agg = time.time(), {"loss": 0.0, "n": 0}
        for _it, (_xb, _yb) in enumerate(_dl):
            if _it >= _steps_ep:
                break
            _x = _xb.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            _t = 1.0 - _yb.to(DEVICE).float()
            if mode == "eca":
                _xr = _x.repeat_interleave(_k, 0)
                _xr = _xr + sigma * torch.randn_like(_xr)
                with amp_ctx(DEVICE):
                    _lg = _model(_xr).float()
                _loss = F.binary_cross_entropy_with_logits(_lg, _t.repeat_interleave(_k))
                _q = torch.sigmoid(_lg).view(-1, _k).mean(1).clamp(_ec["gamma"], 1 - _ec["gamma"])
                _S = sigma * torch.special.ndtri(_q)
                _T = _ec["T"] * sigma
                _isF, _isR = _t < 0.5, _t > 0.5
                _laf = (_T * F.softplus((_S + _ec["mF"] * sigma) / _T))[_isF].mean() if _isF.any() else _S.sum() * 0
                _lfr = (_T * F.softplus((_ec["mR"] * sigma - _S) / _T))[_isR].mean() if _isR.any() else _S.sum() * 0
                _loss = _loss + _ec["mu"] * (_ec["lam_af"] * _laf + _ec["lam_fr"] * _lfr) / sigma
            else:
                with amp_ctx(DEVICE):
                    _lg = _model(_x).float()
                _loss = F.binary_cross_entropy_with_logits(_lg, _t)
            _opt.zero_grad(set_to_none=True)
            _loss.backward()
            torch.nn.utils.clip_grad_norm_(_model.parameters(), 5.0)
            _opt.step()
            _sched.step()
            _agg["loss"] += float(_loss.item())
            _agg["n"] += 1
            if _it % 200 == 0:
                log(f"B5[{name}] ep {_ep} it {_it}/{_steps_ep} loss {_agg['loss'] / _agg['n']:.4f}")
            if time_left_h() < 0.3:
                raise RuntimeError(f"session clock nearly out during {name} epoch {_ep}; re-run to resume")
        _hist.append({"epoch": _ep, "loss": _agg["loss"] / max(1, _agg["n"]), "sec": time.time() - _t0})
        atomic_write(_last, lambda p: torch.save({"model": _model.state_dict(), "opt": _opt.state_dict(),
                                                  "sched": _sched.state_dict(), "epoch": _ep, "hist": _hist}, p))
        hf_push(_last, f"{name} epoch {_ep}")
    _model.eval()
    atomic_write(_final, lambda p: torch.save(_model.state_dict(), p))
    save_json(os.path.join(_dir, "train_log.json"), {"mode": mode, "sigma": sigma, "seed": seed, "eca_override": eca or {}, "hist": _hist})
    hf_push(_final, f"{name} final")
    hf_push(os.path.join(_dir, "train_log.json"))
    del _model, _opt
    gpu_free()
    return _final

In [ ]:
# ---- B1. DEFENDED ROC OF THE BASELINES (GPU) -----------------------------------------------
# Proposition 1 makes the confidence-gated attack threshold-free: maximising s once gives the
# accepted-fake rate at EVERY threshold, so one attack run yields a whole curve of
# (clean real acceptance, attacked accepted-fake rate). For the stacked gates (Mahalanobis +
# MC-dropout, k = 3) the attack maximises s subject to the gates' calibrated limits.
# Saved per image: s after attack, whether the gates still pass, and the generator name.
def b1_one(name, kind, ac=None, tag="B1", variants=None):
    """`ac`, `tag`, `variants` let C2 reuse this attack with a stronger configuration; the
    defaults are exactly B1's, so B1's saved files keep their names."""
    _pq = v_path("gpu", f"{tag}_{name}.parquet")
    if ensure(_pq):
        return "exists"
    _ac = ac or V3["roc_attack"]
    _grid = [("l2", e) for e in _ac["l2"]] + [("linf", e) for e in _ac["linf"]]
    _keyof = lambda nrm, eps: f"{nrm}_{eps * 255:.1f}/255" if nrm == "linf" else f"{nrm}_{eps}"
    # one part file per attack budget: a dead kernel resumes at the first missing budget
    _part = lambda key: v_path("gpu", f"{tag}_parts", f"{name}__{key.replace('/', '_')}.parquet")
    os.makedirs(v_path("gpu", f"{tag}_parts"), exist_ok=True)
    _todo = [(n, e) for n, e in _grid if not ensure(_part(_keyof(n, e)))]
    if not _todo:
        _df = pd.concat([pd.read_parquet(_part(_keyof(n, e))) for n, e in _grid])
        _df.to_parquet(_pq)
        v_push(_pq)
        return f"{len(_df)} rows (assembled from saved parts)"
    path = b_weight(name)
    _cf = b_store_lazy()
    _fk, _rl = b_test_idx(_cf)
    _fk, _rl = _fk[: _ac["n"]], _rl[: _ac["n"]]
    _thr = dict(load_json(os.path.join(PATHS["res"], f"S6_{name}.json"))["thresholds"])
    _mpz = np.load(os.path.join(PATHS["res"], f"S6_{name}_maha.npz"))
    _mp = {k: _mpz[k] for k in ["mu_r", "P"]}
    _mp_t = {k: torch.as_tensor(_mpz[k], device=DEVICE) for k in ["mu_r", "P"]}
    _sc = pd.read_parquet(os.path.join(PATHS["res"], f"S6_{name}_scores.parquet"))
    _ref = _sc[(_sc.set == "cf_test") & (_sc.y == 0)]
    _scl = {"s": float(_ref.s.std() + 1e-6), "maha": float(_ref.maha.std() + 1e-6), "mc": float(_ref.mcvar.std() + 1e-6)}
    _model = load_any(kind, path)
    for _p in _model.parameters():
        _p.requires_grad_(False)
    _vars = variants or (["conf", "all3"] if kind == "cnn" else ["conf", "maha"])
    _names = _cf.meta.model_name.to_numpy()

    def _s_of(x):
        with amp_ctx(DEVICE):
            return _model(x).float()

    def _joint(x, v):
        with amp_ctx(DEVICE):
            _s, _f = _model.forward_both(x)
        _s = _s.float()
        _pen = 0.0
        if v in ("maha", "all3"):
            _d = _f.float() - _mp_t["mu_r"]
            _c = -(_d @ _mp_t["P"] * _d).sum(1)
            _pen = _pen + torch.relu(_thr["maha"] + 0.05 * _scl["maha"] - _c) / _scl["maha"]
        if v == "all3" and kind == "cnn":
            set_mc(_model, True)
            with amp_ctx(DEVICE):
                _ss = torch.stack([_model.forward_both(x)[0].float() for _ in range(CFG["attack"]["mc_eot"])])
            set_mc(_model, False)
            _pen = _pen + torch.relu(_thr["mc"] + 0.05 * _scl["mc"] + _ss.var(0)) / _scl["mc"]
        return -_s / _scl["s"] + 10.0 * _pen

    for _nrm, _eps in _todo:
        _key = _keyof(_nrm, _eps)
        _rows = []
        for _goal, _ix in [("evasion", _fk), ("framing", _rl)]:
            for _v in (_vars if _goal == "evasion" else ["conf"]):
                for _i in range(0, len(_ix), _ac["batch"]):
                    _bi = _ix[_i:_i + _ac["batch"]]
                    _x0 = to_tensor01(center224(_cf.batch(_bi)), DEVICE)
                    _best_s = torch.full((len(_bi),), -np.inf if _goal == "evasion" else np.inf, device=DEVICE)
                    _best_ok = torch.zeros(len(_bi), dtype=torch.bool, device=DEVICE)
                    for _rs in range(_ac["restarts"]):
                        if _goal == "evasion" and _v == "conf":
                            _obj = lambda x: (-_s_of(x), torch.zeros(x.shape[0], dtype=torch.bool, device=x.device))
                        elif _goal == "evasion":
                            _obj = lambda x, _v=_v: (_joint(x, _v), torch.zeros(x.shape[0], dtype=torch.bool, device=x.device))
                        else:
                            _obj = lambda x: (_s_of(x), torch.zeros(x.shape[0], dtype=torch.bool, device=x.device))
                        _xa, _ = pgd(_obj, _x0, _eps, _nrm, _ac["steps"], rand_init=_rs > 0)
                        _s, _f, _var = base_outputs(_model, kind, _xa, CFG["attack"]["mc_eval"])
                        _cm = maha_gate(_f.cpu().numpy(), _mp)
                        _ok = torch.as_tensor(gate_mask(_v, _s.cpu().numpy(), _cm, _var.cpu().numpy(), _thr), device=DEVICE)
                        if _goal == "evasion":
                            _cand = torch.where(_ok, _s, torch.full_like(_s, -np.inf))
                            _upd = _cand > _best_s
                        else:
                            _cand = _s
                            _upd = _cand < _best_s
                        _best_s = torch.where(_upd, _cand, _best_s)
                        _best_ok = torch.where(_upd, _ok, _best_ok)
                    for _j, _gi in enumerate(_bi):
                        _rows.append({"attack": _key, "norm": _nrm, "eps": _eps, "goal": _goal, "variant": _v, "idx": int(_gi),
                                      "model_name": _names[_gi], "s_adv": float(_best_s[_j]), "gate_ok": bool(_best_ok[_j])})
            if time_left_h() < 0.3:
                raise RuntimeError("session clock nearly out; re-run to continue")
        pd.DataFrame(_rows).to_parquet(_part(_key))
        v_push(_part(_key))
        log(f"{tag}[{name}] {_key} done")
    _df = pd.concat([pd.read_parquet(_part(_keyof(n, e))) for n, e in _grid])
    _df.to_parquet(_pq)
    v_push(_pq)
    del _model
    gpu_free()
    return f"{len(_df)} rows"

if V3["run"]["B1"]:
    _ = BUNDLE_INFO
    B1_OUT = {n: b_step(f"B1 defended ROC {n}", lambda n=n: b1_one(n, "clip" if n == "clip" else "cnn"))
              for n in V_BASELINES}
else:
    B1_OUT = {"skipped": True}
B1_OUT

In [ ]:
# ---- B2. DOMINANCE: certified upper bound of ECA vs attack lower bound of baselines (CPU) --
# At a matched clean real-acceptance level r, ECA's CERTIFIED accepted-fake rate (an upper bound
# on its true defended rate, by Theorem 3) is compared with the baseline's EMPIRICAL attacked
# accepted-fake rate (a lower bound on its true defended rate). Where ECA's upper confidence
# limit is below the baseline's lower confidence limit, ECA is provably better at that level.
# The attacker may also submit the clean image, so the baseline's attacked score is
# max(clean, attacked).
def _cp_upper(k, n, a=0.025):
    return 1.0 if k >= n else float(v_beta.ppf(1 - a, k + 1, n - k))

def _cp_lower(k, n, a=0.025):
    return 0.0 if k <= 0 else float(v_beta.ppf(a, k, n - k + 1))

def b2_run():
    _levels = np.round(np.concatenate([np.arange(0.01, 0.1, 0.01), np.arange(0.1, 1.0, 0.05)]), 3)
    _curves, _dom = [], []
    _thrq = np.linspace(0, 1, 401)
    for _b in V_BASELINES:
        _pq = v_path("gpu", f"B1_{_b}.parquet")
        if not ensure(_pq):
            continue
        _A = pd.read_parquet(_pq)
        _sc = pd.read_parquet(os.path.join(PATHS["res"], f"S6_{_b}_scores.parquet"))
        _thr = load_json(os.path.join(PATHS["res"], f"S6_{_b}.json"))["thresholds"]
        _te = _sc[_sc.set == "cf_test"]
        _r = _te[_te.y == 0]
        _f = _te[_te.y == 1]
        for (_key, _v), _g in _A[_A.goal == "evasion"].groupby(["attack", "variant"]):
            _gate_r = gate_mask(_v, _r.s.to_numpy(), _r.maha.to_numpy(), _r.mcvar.to_numpy(), _thr)
            _n = len(_g)
            _fc = _f.iloc[:_n]  # the attacked fakes are the first n test fakes, same order
            _gate_fc = gate_mask(_v, _fc.s.to_numpy(), _fc.maha.to_numpy(), _fc.mcvar.to_numpy(), _thr)
            _s_att = np.maximum(np.where(_g.gate_ok.to_numpy(), _g.s_adv.to_numpy(), -np.inf),
                                np.where(_gate_fc, _fc.s.to_numpy(), -np.inf))
            _ts = np.quantile(_r.s.to_numpy(), _thrq)
            for _t in _ts:
                _acc_r = float(np.mean((_r.s.to_numpy() >= _t) & _gate_r))
                _kk = int(np.sum(_s_att >= _t))
                _curves.append({"family": "baseline", "method": f"{_b}+{_v}", "attack": _key, "threshold": _t,
                                "accept_real": _acc_r, "AF": _kk / _n, "AF_lo": _cp_lower(_kk, _n), "AF_hi": _cp_upper(_kk, _n), "n_fake": _n})
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _te = _df[_df.set == "cf_test"]
        _lo, _hi, _ = v_scores(_te.k.to_numpy(), _s)
        _y = _te.y.to_numpy()
        _smax = _s * v_zN(V3["N"], V3["alpha_mc"])
        for _e in V3["roc_attack"]["l2"]:
            if _e >= _smax:
                continue
            _ts = np.unique(np.quantile(np.concatenate([_lo[_y == 0] - _e, _hi[_y == 1]]), _thrq))
            _nf = int((_y == 1).sum())
            for _t in _ts:
                _kk = int(np.sum(_hi[_y == 1] >= _t))
                _curves.append({"family": "ECA (certified bound)", "method": _n, "attack": f"l2_{_e}", "threshold": _t,
                                "accept_real": float(np.mean(_lo[_y == 0] >= _t + _e)), "AF": _kk / _nf,
                                "AF_lo": _cp_lower(_kk, _nf), "AF_hi": _cp_upper(_kk, _nf), "n_fake": _nf})
    _C = pd.DataFrame(_curves)
    if not len(_C):
        return _C, pd.DataFrame()
    _C.to_csv(v_path("tables", "B2_defended_roc_curves.csv"), index=False)

    def _best_at(sub, r, col):
        _ok = sub[sub.accept_real >= r]
        return float(_ok[col].min()) if len(_ok) else np.nan

    for _key in sorted(set(_C[_C.family != "baseline"].attack)):
        _E = _C[(_C.family != "baseline") & (_C.attack == _key)]
        _Bs = _C[(_C.family == "baseline") & (_C.attack == _key)]
        for _em, _eg in _E.groupby("method"):
            for _bm, _bg in _Bs.groupby("method"):
                for _r in _levels:
                    _eu = _best_at(_eg, _r, "AF_hi")
                    _bl = _best_at(_bg, _r, "AF_lo")
                    _dom.append({"attack": _key, "eca": _em, "baseline": _bm, "real_accept_level": _r,
                                 "eca_certified_AF_upper95": _eu, "baseline_attacked_AF_lower95": _bl,
                                 "eca_provably_better": bool(np.isfinite(_eu) and np.isfinite(_bl) and _eu < _bl)})
    _D = pd.DataFrame(_dom)
    if len(_D):
        _D.to_csv(v_path("tables", "B2_dominance.csv"), index=False)
        _S = (_D[_D.eca_provably_better].groupby(["attack", "eca", "baseline"]).real_accept_level.max()
              .rename("max_real_accept_where_ECA_provably_better").reset_index())
        _S.to_csv(v_path("tables", "B2_dominance_summary.csv"), index=False)
    _keys = sorted(set(_C.attack) & set(_C[_C.family != "baseline"].attack))
    if _keys:
        _fig, _ax = plt.subplots(1, len(_keys), figsize=(5 * len(_keys), 4), squeeze=False)
        for _j, _key in enumerate(_keys):
            _a = _ax[0, _j]
            for (_fam, _m), _g in _C[_C.attack == _key].groupby(["family", "method"]):
                _g = _g.sort_values("accept_real")
                _a.plot(_g.accept_real, np.maximum(_g.AF, 1e-4), "-" if _fam == "baseline" else "-", lw=1 if _fam == "baseline" else 2.5,
                        label=f"{_m} ({'attacked' if _fam == 'baseline' else 'certified'})")
            _a.set_yscale("log")
            _a.set_xlabel("clean real acceptance")
            _a.set_ylabel("accepted-fake rate under attack")
            _a.set_title(f"Defended ROC, {_key}")
            _a.legend(fontsize=6)
        v_savefig(_fig, "B2_defended_roc")
    v_push(v_path("tables"), "v3 B2 dominance")
    v_push(v_path("figs"), "v3 B2 figure")
    return _C, _D

_ = B1_OUT
B2_CURVES, B2_DOM = b2_run()
mo.vstack([mo.md("### B2. Defended ROC dominance"),
           mo.ui.table(pd.read_csv(v_path("tables", "B2_dominance_summary.csv")), selection=None)
           if os.path.exists(v_path("tables", "B2_dominance_summary.csv")) else mo.md("_needs B1 outputs_")])

In [ ]:
# ---- B3. ATTACK STRESS TEST on the main ECA model (GPU) ----------------------------------
# 3x the steps and 2x the noise draws of S9c, with 2 restarts. Two checks: (a) certified-regime
# violations stay ~0 (A5 audits this file too); (b) outside the certified regime, how much
# more a stronger attack gets than S9c did (attack-strength convergence).
def b3_run():
    if not V_BEST:
        return "no smoothed model"
    _jp = v_path("gpu", f"B3_{V_BEST}.json")
    if ensure(_jp) and ensure(v_path("gpu", f"B3_{V_BEST}_counts.parquet")):
        return "exists"
    _sigma = dict(V_MODELS)[V_BEST]
    _s9 = os.path.join(PATHS["res"], f"S9c_{V_BEST}.json")
    if os.path.exists(_s9):
        _th = load_json(_s9)["thresholds"]
    else:
        _th = v_rule(v_cal_parts(v_counts(V_BEST), _sigma), V3["select_alpha"], V3["main_beta"])
    _sa = V3["stress"]
    _cf = b_store_lazy()
    _fk, _rl = b_test_idx(_cf)
    _fk, _rl = _fk[: _sa["n"]], _rl[: _sa["n"]]
    _model = load_detector(b_weight(V_BEST), DEVICE)
    for _p in _model.parameters():
        _p.requires_grad_(False)
    _rows = []
    for _rel in _sa["l2_rel"]:
        _eps = round(_rel * _sigma, 4)
        _key = f"l2_{_eps}"
        for _goal, _ix, _tag in [("real", _fk, "evasion"), ("fake", _rl, "framing")]:
            _ks = []
            for _i in range(0, len(_ix), _sa["batch"]):
                _x0 = to_tensor01(center224(_cf.batch(_ix[_i:_i + _sa["batch"]])), DEVICE)
                _best_x, _best_v = None, None
                for _r in range(_sa["restarts"]):
                    _xa = eot_smoothed_attack(_model, _x0, _sigma, _eps, "l2", _sa["steps"], _sa["m"], DEVICE, goal=_goal)
                    _v = torch.as_tensor(smooth_votes(_model, _xa, _sigma, 128, CFG["cert"]["chunk"], DEVICE), device=DEVICE)
                    _v = _v if _goal == "real" else -_v
                    if _best_x is None:
                        _best_x, _best_v = _xa, _v
                    else:
                        _u = _v > _best_v
                        _best_x = torch.where(_u.view(-1, 1, 1, 1), _xa, _best_x)
                        _best_v = torch.where(_u, _v, _best_v)
                _ks.append(smooth_votes(_model, _best_x, _sigma, V3["N"], CFG["cert"]["chunk"], DEVICE))
            _rows.append(pd.DataFrame({"attack": _key, "goal": _tag, "k": np.concatenate(_ks)}))
            if time_left_h() < 0.3:
                raise RuntimeError("session clock nearly out; re-run")
        log(f"B3[{V_BEST}] {_key} done")
    _C = pd.concat(_rows)
    _C.to_parquet(v_path("gpu", f"B3_{V_BEST}_counts.parquet"))
    _summ = {"model": V_BEST, "sigma": _sigma, "thresholds": _th, "config": _sa, "rates": {}}
    for (_key, _goal), _g in _C.groupby(["attack", "goal"]):
        _lo, _hi, _ = v_scores(_g.k.to_numpy(), _sigma)
        _summ["rates"][f"{_key}|{_goal}"] = {
            f"eps_c{_e}": float(np.mean(decide(_lo, _hi, _th["tau_a"], _th["tau_f"], _e) == (2 if _goal == "evasion" else 0)))
            for _e in sorted(set(CFG["cert"]["eps_cert"] + [_sigma]))}
    save_json(_jp, _summ)
    v_push(v_path("gpu", f"B3_{V_BEST}_counts.parquet"))
    v_push(_jp)
    del _model
    gpu_free()
    return _summ["rates"]

if V3["run"]["B3"]:
    _ = (BUNDLE_INFO, B2_CURVES)
    B3_OUT = b_step("B3 stress test", b3_run)
else:
    B3_OUT = {"skipped": True}
B3_OUT

In [ ]:
# ---- B4. MORE NOISE SAMPLES: does estimation, not information, bind? (GPU) ---------------
# Proposition 6: the largest certifiable score is sigma * z_N; N = 256 -> z_N = 1.89,
# N = 1024 -> 2.20 (alpha_mc = 1e-3). The same balanced calibration and test subsets are
# certified at every N, so the change in alpha_min and in certified real acceptance is
# attributable to N alone.
def b4_run():
    if not V_BEST:
        return "no smoothed model"
    _out = v_path("gpu", f"B4_{V_BEST}_counts.parquet")
    _sigma = dict(V_MODELS)[V_BEST]
    _bn = V3["bigN"]
    if not ensure(_out):
        _cf = b_store_lazy()
        _ic = b_subset(_cf, "cal", _bn["per_gen_cap"], _bn["n_real"])
        _it = b_subset(_cf, "test", _bn["per_gen_cap"], _bn["n_real"])
        _model = load_detector(b_weight(V_BEST), DEVICE)
        _fr = []
        for _N in _bn["Ns"]:
            for _sn, _ix in [("cf_cal", _ic), ("cf_test", _it)]:
                _fr.append(b_counts_frame(_cf, _ix, b_certify(_model, _cf, _ix, _sigma, _N), _sn, _N))
                log(f"B4[{V_BEST}] N={_N} {_sn}: {len(_ix)} images")
        pd.concat(_fr).to_parquet(_out)
        v_push(_out)
        del _model
        gpu_free()
    _C = pd.read_parquet(_out)
    _T = pd.concat([b_frontier_from_counts(_g, _sigma, int(_N), f"{V_BEST} N={int(_N)}") for _N, _g in _C.groupby("N")])
    _T.to_csv(v_path("tables", "B4_larger_N.csv"), index=False)
    v_push(v_path("tables", "B4_larger_N.csv"))
    return _T

if V3["run"]["B4"]:
    _ = (BUNDLE_INFO, B3_OUT)
    B4_OUT = b_step("B4 larger N", b4_run)
else:
    B4_OUT = None
B4_OUT if isinstance(B4_OUT, pd.DataFrame) else mo.md(f"B4: {B4_OUT}")

In [ ]:
# ---- B5. EXTRA TRAINING SEEDS (GPU): is the headline stable across training randomness? --
# Trains BASE and the main ECA model again with seeds 1 and 2 (same recipe as S5), then
# certifies the same balanced subsets for seed 0 (existing weights) and the new seeds.
def b5_run():
    if not V_BEST:
        return "no smoothed model"
    _sigma = dict(V_MODELS)[V_BEST]
    _mode = "eca" if V_BEST.startswith("eca") else "cohen"
    _sd = V3["seeds"]
    _sub = {}

    def _subsets():
        if not _sub:
            _c = b_store_lazy()
            _sub.update(cf=_c, ic=b_subset(_c, "cal", _sd["per_gen_cap"], _sd["n_real"]),
                        it=b_subset(_c, "test", _sd["per_gen_cap"], _sd["n_real"]))
        return _sub["cf"], _sub["ic"], _sub["it"]

    _rows = []
    for _seed in [0] + _sd["seeds"]:
        for _kind in _sd["models"]:
            _name = (V_BEST if _kind == "eca" else "base") + ("" if _seed == 0 else f"_seed{_seed}")
            if _kind == "eca" and _mode != "eca" and _seed != 0:
                continue
            _cp = v_path("gpu", f"B5_{_name}_counts.parquet" if _kind == "eca" else f"B5_{_name}_scores.parquet")
            _done = ensure(_cp)  # finished unit: no training, no weights download, no image cache
            if _done:
                _w = None
            elif _seed == 0:
                _w = b_weight(V_BEST if _kind == "eca" else "base")
                if not ensure(_w):
                    continue
            else:
                if _kind == "eca" and _mode != "eca":
                    continue
                _w = b_train(_name, "eca" if _kind == "eca" else "base", _sigma if _kind == "eca" else None, _seed, _sd["max_steps"])
            if _kind == "eca":
                if not _done:
                    _cf, _ic, _it = _subsets()
                    _m = load_detector(_w, DEVICE)
                    _fr = [b_counts_frame(_cf, _ix, b_certify(_m, _cf, _ix, _sigma, V3["N"]), _sn, V3["N"])
                           for _sn, _ix in [("cf_cal", _ic), ("cf_test", _it)]]
                    pd.concat(_fr).to_parquet(_cp)
                    v_push(_cp)
                    del _m
                    gpu_free()
                _t = b_frontier_from_counts(pd.read_parquet(_cp), _sigma, V3["N"], _name)
                for _r in _t.to_dict("records"):
                    _rows.append({"seed": _seed, "model": _name, "kind": "eca", **_r})
            else:
                _sp = _cp
                if not _done:
                    _cf, _ic, _it = _subsets()
                    _m = load_detector(_w, DEVICE)
                    _s, _, _ = run_store(_m, "cnn", _cf, _it, 1)
                    pd.DataFrame({"s": _s, "y": _cf.meta.label.to_numpy()[_it]}).to_parquet(_sp)
                    v_push(_sp)
                    del _m
                    gpu_free()
                _d = pd.read_parquet(_sp)
                _rows.append({"seed": _seed, "model": _name, "kind": "base", "auroc_point": auroc_real(_d.s, _d.y),
                              "tpr@1%fpr": tpr_at_fpr(_d.s, _d.y)})
    _T = pd.DataFrame(_rows)
    _T.to_csv(v_path("tables", "B5_seeds.csv"), index=False)
    if len(_T):
        _num = [c for c in _T.columns if c not in ("seed", "model", "kind", "tag")]
        _agg = _T.groupby([c for c in ["kind", "alpha"] if c in _T.columns], dropna=False)[_num].agg(["mean", "std", "count"])
        _agg.columns = [" ".join(c) for c in _agg.columns]
        _agg.reset_index().to_csv(v_path("tables", "B5_seeds_summary.csv"), index=False)
    v_push(v_path("tables"), "v3 B5 seeds")
    return _T

if V3["run"]["B5"]:
    _ = (BUNDLE_INFO, B4_OUT)
    B5_OUT = b_step("B5 extra seeds", b5_run)
else:
    B5_OUT = None
B5_OUT if isinstance(B5_OUT, pd.DataFrame) else mo.md(f"B5: {B5_OUT}")

## Part C: the questions a journal reviewer will ask, answered with data

| cell | reviewer question | needs |
|---|---|---|
| C1 | Is this really a *three-way decision* method (Yao's decision-theoretic rough sets), and what does a cost-sensitive user lose or gain under attack? | CPU |
| C2 | Why not simply use adversarial training? Does its robustness survive a stronger attacker? | GPU, about 2–3 h |
| C3 | Where does the 5–7 point AUROC cost of smoothing come from? | CPU |
| C4 | OpenFake breaks the guarantee: what should a deployer do, and how many new generators are needed? | CPU |
| C5 | The per-image guarantee depends on a cap of 100 images per generator: how sensitive is that? | CPU |
| C6 | Which part of the ADRT loss does the work? (ablation) | GPU, two trainings the size of one B5 seed |
| C7 | Is the α floor an estimation artefact or a property of the detector? (evidence-floor law) | CPU |
| C8 | When does the fake data stop mattering? (one-class collapse of the certified accept rule) | CPU |
| C9 | Reproducibility manifest and a one-page summary of Part C | CPU |

GPU cells skip any unit whose output file exists locally or on your HF repo; CPU cells take
seconds to a few minutes and always recompute from the saved files.

In [ ]:
# ---- C1. THREE-WAY DECISIONS IN YAO'S DECISION-THEORETIC ROUGH-SET FORM, UNDER ATTACK (CPU) ----
# States: real (R) / fake (F). Actions: accept (P), defer (B), flag (N). Losses: accept-real = flag-fake = 0,
# l_PN = accept a fake, l_BN = defer a fake, l_NP = flag a real, l_BP = defer a real. Yao's rule on the
# posterior p = P(R | x): accept iff p >= a, flag iff p <= b, with
#   a = (l_PN - l_BN) / ((l_PN - l_BN) + l_BP),   b = l_BN / (l_BN + (l_NP - l_BP)).
# BASELINE (uncertified): the Bayes three-way rule on a Platt-calibrated BASE / pgdat posterior. The
# posterior is fitted on half of the held-out generators (and half the reals) and everything is
# evaluated on the other half. Under attack we use the B1 attacks: a LOWER bound on its cost.
# ECA (certified): (alpha, beta) chosen on calibration data to minimise the certified cost; on the
# evaluation half we report an UPPER bound on its cost under any l2 attack <= eps:
#   fake cost <= l_PN * P(S_hi >= tau_a) + l_BN * (1 - P(S_hi >= tau_a))   (worst case: all others deferred)
#   real cost <= l_NP * P(S_lo <  tau_f) + l_BP * (1 - P(S_lo <  tau_f))
# so the comparison is tilted in the baseline's favour (its lower bound vs our upper bound).
C1_LOSSES = [(10.0, 0.5, 2.0, 0.2), (20.0, 1.0, 5.0, 0.5), (50.0, 1.0, 5.0, 0.2), (100.0, 2.0, 10.0, 0.5)]
C1_BUDGETS = [0.0, 0.06, 0.12, 0.25]

def c1_yao(l_pn, l_bn, l_np, l_bp):
    return (l_pn - l_bn) / ((l_pn - l_bn) + l_bp), l_bn / (l_bn + (l_np - l_bp))

from scipy.optimize import minimize as c1_minimize

def c1_platt(s, y_real):
    """p = sigmoid(a s + b), maximum likelihood (L-BFGS on the logistic loss; plain Newton from
    a = 1 can diverge on badly scaled scores)."""
    _s, _sg = np.asarray(s, float), 2.0 * np.asarray(y_real, float) - 1.0
    _sc = float(np.std(_s)) or 1.0
    _f = lambda th: float(np.sum(np.logaddexp(0.0, -_sg * (th[0] * _s / _sc + th[1]))))
    _th = c1_minimize(_f, np.zeros(2), method="L-BFGS-B").x
    return float(_th[0] / _sc), float(_th[1])

def c1_cost(l, af, df, fr, dr):
    return 0.5 * (l[0] * af + l[1] * df) + 0.5 * (l[2] * fr + l[3] * dr)

def c1_halves(gens_f, n_real):
    _ef = np.array([stable_u("c1:" + str(g)) >= 0.5 for g in gens_f])  # True = evaluation half
    _er = np.array([stable_u(f"c1r:{j}") >= 0.5 for j in range(n_real)])
    return _ef, _er

def c1_run():
    _rows = []
    for _b in [b for b in V_BASELINES if b != "clip"]:
        _sp = os.path.join(PATHS["res"], f"S6_{_b}_scores.parquet")
        _ap = v_path("gpu", f"B1_{_b}.parquet")
        if not (os.path.exists(_sp) and ensure(_ap)):
            continue
        _sc = pd.read_parquet(_sp)
        _te = _sc[_sc.set == "cf_test"]
        _f, _r = _te[_te.y == 1], _te[_te.y == 0]
        _ef, _er = c1_halves(_f.model_name.to_numpy(), len(_r))
        _sfit = np.concatenate([_f.s.to_numpy()[~_ef], _r.s.to_numpy()[~_er]])
        _yfit = np.concatenate([np.zeros((~_ef).sum()), np.ones((~_er).sum())])
        _pa, _pb = c1_platt(_sfit, _yfit)
        if not _pa > 0:  # score carries no usable signal: Yao thresholds would be inverted
            print(f"[v3] C1: Platt slope {_pa:.3g} <= 0 for {_b}, skipped")
            continue
        _A = pd.read_parquet(_ap)
        for _l in C1_LOSSES:
            _a, _bb = c1_yao(*_l)
            # thresholds on the raw score s (Platt is monotone, slope _pa > 0 expected)
            _ta = (np.log(_a / (1 - _a)) - _pb) / _pa
            _tf = (np.log(_bb / (1 - _bb)) - _pb) / _pa
            _sf, _sr = _f.s.to_numpy()[_ef], _r.s.to_numpy()[_er]
            _af0, _fr0 = float(np.mean(_sf >= _ta)), float(np.mean(_sr <= _tf))
            _df0, _dr0 = float(np.mean((_sf < _ta) & (_sf > _tf))), float(np.mean((_sr < _ta) & (_sr > _tf)))
            for _e in C1_BUDGETS:
                if _e == 0:
                    _af, _fr, _src = _af0, _fr0, "clean"
                else:
                    _ge = _A[(_A.attack == f"l2_{_e}") & (_A.goal == "evasion") & (_A.variant == "conf")]
                    _gf = _A[(_A.attack == f"l2_{_e}") & (_A.goal == "framing") & (_A.variant == "conf")]
                    if not len(_ge) or not len(_gf):
                        continue
                    # attacked images are the first n test fakes / reals (B1 protocol); keep the evaluation half
                    _n1, _n2 = len(_ge), len(_gf)
                    _m1, _m2 = _ef[:_n1], _er[:_n2]
                    _sa = np.maximum(_f.s.to_numpy()[:_n1], _ge.s_adv.to_numpy())[_m1]
                    _sb = np.minimum(_r.s.to_numpy()[:_n2], _gf.s_adv.to_numpy())[_m2]
                    _af, _fr, _src = float(np.mean(_sa >= _ta)), float(np.mean(_sb <= _tf)), "B1 attack (lower bound)"
                _rows.append({"method": f"{_b} + Yao three-way (uncertified)", "loss": str(_l), "a": _a, "b": _bb, "l2_budget": _e,
                              "AF": _af, "FR": _fr, "defer_fake_clean": _df0, "defer_real_clean": _dr0,
                              "cost_lower_bound": c1_cost(_l, _af, 0.0, _fr, 0.0) if _e > 0 else c1_cost(_l, _af, _df0, _fr, _dr0),
                              "cost_with_clean_deferral": c1_cost(_l, _af, _df0, _fr, _dr0), "kind": _src})
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        _t = _df[_df.set == "cf_test"]
        _lo, _hi, _ = v_scores(_t.k.to_numpy(), _s)
        _y, _g = _t.y.to_numpy(), _t.model_name.to_numpy()
        _ef, _er = c1_halves(_g[_y == 1], int((_y == 0).sum()))
        _lof, _hif = _lo[_y == 1][_ef], _hi[_y == 1][_ef]
        _lor, _hir = _lo[_y == 0][_er], _hi[_y == 0][_er]
        _smax = _s * v_zN(V3["N"], V3["alpha_mc"])
        _calf = np.concatenate([_cal["hi"][g] for g in _cal["gens"]])
        for _l in C1_LOSSES:
            for _e in C1_BUDGETS:
                if _e >= _smax:
                    continue
                _best = None
                for _al in V3["alpha_grid"]:
                    for _be in V3["beta_grid"]:
                        _ru = v_rule(_cal, _al, _be)
                        _afu = float(np.mean(_calf >= _ru["tau_a"])) if _ru["tau_a"] + _e <= _smax else 0.0
                        _fru = float(np.mean(_cal["lo_r"] < _ru["tau_f"])) if _ru["tau_f"] - _e > -_smax else 0.0
                        _c = c1_cost(_l, _afu, 1 - _afu, _fru, 1 - _fru)
                        if _best is None or _c < _best[0]:
                            _best = (_c, _al, _be, _ru)
                _, _al, _be, _ru = _best
                _afu = float(np.mean(_hif >= _ru["tau_a"])) if _ru["tau_a"] + _e <= _smax else 0.0
                _fru = float(np.mean(_lor < _ru["tau_f"])) if _ru["tau_f"] - _e > -_smax else 0.0
                _d = decide(np.concatenate([_lof, _lor]), np.concatenate([_hif, _hir]), _ru["tau_a"], _ru["tau_f"], _e)
                _nf = len(_lof)
                _dfc, _drc = float(np.mean(_d[:_nf] == 1)), float(np.mean(_d[_nf:] == 1))
                _rows.append({"method": f"{_n} (certified)", "loss": str(_l), "a": _al, "b": _be, "l2_budget": _e,
                              "AF": _afu, "FR": _fru, "defer_fake_clean": _dfc, "defer_real_clean": _drc,
                              "cost_upper_bound": c1_cost(_l, _afu, 1 - _afu, _fru, 1 - _fru), "calibration_cost": _best[0],
                              "cost_with_clean_deferral": c1_cost(_l, _afu, _dfc, _fru, _drc),
                              "kind": "certified bound (any l2 attack <= budget)"})
    _T = pd.DataFrame(_rows)
    if len(_T):
        _T.to_csv(v_path("tables", "C1_yao_three_way_cost.csv"), index=False)
        # head-to-head: best certified ECA upper bound vs best baseline lower bound, per loss and budget
        _h = []
        for (_l, _e), _g in _T.groupby(["loss", "l2_budget"]):
            _ec = _g[_g.method.str.endswith("(certified)")]
            _bc = _g[~_g.method.str.endswith("(certified)")]
            if len(_ec) and len(_bc):
                _eb = _ec.loc[_ec.calibration_cost.idxmin()]  # ECA model chosen on calibration data only
                _bb = _bc.loc[_bc.cost_lower_bound.idxmin()]
                _h.append({"loss": _l, "l2_budget": _e, "best_ECA": _eb.method, "ECA_cost_upper_bound": _eb.cost_upper_bound,
                           "ECA_cost_clean_deferral": _eb.cost_with_clean_deferral, "best_baseline": _bb.method,
                           "baseline_cost_lower_bound": _bb.cost_lower_bound,
                           "ECA_provably_cheaper": bool(_eb.cost_upper_bound < _bb.cost_lower_bound)})
        _H = pd.DataFrame(_h)
        if len(_H):
            _H.to_csv(v_path("tables", "C1_yao_head_to_head.csv"), index=False)
        v_push(v_path("tables"), "v3 C1 Yao three-way cost")
        return _T, _H
    return _T, pd.DataFrame()

C1, C1_H2H = c1_run()
mo.vstack([mo.md("### C1. Yao three-way decision cost under attack"),
           mo.ui.table(C1_H2H.round(4), selection=None) if len(C1_H2H) else mo.md("_needs S6 scores and B1 attacks_")])

In [ ]:
# ---- C2. IS ADVERSARIAL TRAINING'S ROBUSTNESS REAL? Stronger attack on pgdat (GPU) -----------
# B2 found no provable dominance of ECA over pgdat. That compares ECA's certificate with pgdat's
# robustness against ONE attack. Empirical robustness can shrink under a stronger attacker; a
# certificate cannot. Same images and gates as B1, 4x the steps and 2.5x the restarts. Output:
# pgdat's attacked accepted-fake rate at matched clean real acceptance under B1 vs C2, next to
# ECA's certified upper bound (from B2) at the same level.
def c2_curve_at(parquet, b, variant, key, levels):
    _A = pd.read_parquet(parquet)
    _A = _A[(_A.goal == "evasion") & (_A.variant == variant) & (_A.attack == key)]
    if not len(_A):
        return {}
    _sc = pd.read_parquet(os.path.join(PATHS["res"], f"S6_{b}_scores.parquet"))
    _thr = load_json(os.path.join(PATHS["res"], f"S6_{b}.json"))["thresholds"]
    _te = _sc[_sc.set == "cf_test"]
    _r, _f = _te[_te.y == 0], _te[_te.y == 1]
    _n = len(_A)
    _fc = _f.iloc[:_n]
    _gr = gate_mask(variant, _r.s.to_numpy(), _r.maha.to_numpy(), _r.mcvar.to_numpy(), _thr)
    _gf = gate_mask(variant, _fc.s.to_numpy(), _fc.maha.to_numpy(), _fc.mcvar.to_numpy(), _thr)
    _sa = np.maximum(np.where(_A.gate_ok.to_numpy(), _A.s_adv.to_numpy(), -np.inf), np.where(_gf, _fc.s.to_numpy(), -np.inf))
    _out = {}
    for _lv in levels:
        # threshold giving clean real acceptance >= level (gate included)
        _cand = np.sort(_r.s.to_numpy()[_gr])[::-1]
        _need = int(np.ceil(_lv * len(_r)))
        if _need > len(_cand) or _need == 0:
            _out[_lv] = (np.nan, np.nan, np.nan)
            continue
        _t = _cand[_need - 1]
        _k = int(np.sum(_sa >= _t))
        _out[_lv] = (_k / _n, (0.0 if _k == 0 else float(v_beta.ppf(0.025, _k, _n - _k + 1))), _n)
    return _out

def c2_run():
    if "pgdat" not in V_BASELINES:
        return "no pgdat baseline"
    _r = b1_one("pgdat", "cnn", ac=V3["c2"], tag="C2", variants=["conf", "all3"])
    return _r

def c2_table():
    _p1, _p2 = v_path("gpu", "B1_pgdat.parquet"), v_path("gpu", "C2_pgdat.parquet")
    if not (ensure(_p1) and ensure(_p2)):
        return pd.DataFrame()
    _levels = [0.5, 0.7, 0.8, 0.9, 0.95]
    _cur = v_path("tables", "B2_defended_roc_curves.csv")
    _E = pd.read_csv(_cur) if os.path.exists(_cur) else pd.DataFrame()
    _rows = []
    for _e in V3["c2"]["l2"]:
        _key = f"l2_{_e}"
        for _v in ["conf", "all3"]:
            _a1 = c2_curve_at(_p1, "pgdat", _v, _key, _levels)
            _a2 = c2_curve_at(_p2, "pgdat", _v, _key, _levels)
            for _lv in _levels:
                _ecah = np.nan
                if len(_E):
                    _x = _E[(_E.family != "baseline") & (_E.attack == _key) & (_E.accept_real >= _lv)]
                    _ecah = float(_x.AF_hi.min()) if len(_x) else np.nan
                _b1 = _a1.get(_lv, (np.nan,) * 3)
                _b2 = _a2.get(_lv, (np.nan,) * 3)
                _rows.append({"attack": _key, "pgdat_variant": _v, "clean_real_accept": _lv,
                              "pgdat_AF_B1_attack": _b1[0], "pgdat_AF_C2_stronger": _b2[0], "pgdat_AF_C2_lower95": _b2[1],
                              "increase_from_stronger_attack": (_b2[0] - _b1[0]) if np.isfinite(_b1[0]) and np.isfinite(_b2[0]) else np.nan,
                              "best_ECA_certified_AF_upper95": _ecah,
                              "ECA_provably_better_vs_stronger_attack": bool(np.isfinite(_ecah) and np.isfinite(_b2[1]) and _ecah < _b2[1])})
    _T = pd.DataFrame(_rows)
    _T.to_csv(v_path("tables", "C2_pgdat_stronger_attack.csv"), index=False)
    v_push(v_path("tables", "C2_pgdat_stronger_attack.csv"))
    return _T

if V3["run"]["C2"]:
    _ = (BUNDLE_INFO, B5_OUT)
    C2_OUT = b_step("C2 stronger attack on pgdat", c2_run)
else:
    C2_OUT = {"skipped": True}
C2 = c2_table()
mo.vstack([mo.md(f"### C2. Adversarial training under a stronger attacker: `{C2_OUT}`"),
           mo.ui.table(C2.round(4), selection=None, page_size=30) if len(C2) else mo.md("_needs B1 and C2 outputs_")])

In [ ]:
# ---- C3. WHERE DOES THE AUROC COST OF SMOOTHING COME FROM? (CPU) ---------------------------------
# AUROC (realness score, real = positive) of BASE and each smoothed model on strata of held-out CF:
# all images; generators with <= A11_CAP images; the large generators only; a size-balanced subset
# (<= 20 images per generator, as in B4/B5); and the mean of per-generator AUROCs (each generator's
# fakes against all reals). If the drop concentrates on the few large generators, the "cost of
# certification" is a cost on specific architectures, not a uniform loss of evidence.
def c3_auroc_strata(score, y, gens):
    _out = {}
    _f = y == 1
    _sz = pd.Series(gens[_f]).value_counts()
    _big = set(_sz[_sz > A11_CAP].index)
    _isbig = np.array([g in _big for g in gens])
    _out["all"] = auroc_real(score, y)
    _m = ~_f | ~_isbig
    _out[f"gens_n<={A11_CAP}"] = auroc_real(score[_m], y[_m])
    _m = ~_f | _isbig
    _out[f"gens_n>{A11_CAP}_only"] = auroc_real(score[_m], y[_m]) if _isbig.any() else np.nan
    _key = np.array([stable_u(f"c3:{i}") for i in range(len(y))])
    _keep = ~_f.copy()
    for _g in np.unique(gens[_f]):
        _ix = np.where(_f & (gens == _g))[0]
        _keep[_ix[np.argsort(_key[_ix])][:20]] = True
    _out["balanced_<=20_per_gen"] = auroc_real(score[_keep], y[_keep])
    _per = []
    for _g in np.unique(gens[_f]):
        _m = ~_f | (gens == _g)
        if ((gens == _g) & _f).sum() >= V3["min_per_gen"]:
            _per.append(auroc_real(score[_m], y[_m]))
    _out["mean_per_generator"] = float(np.mean(_per)) if _per else np.nan
    _out["worst_generator"] = float(np.min(_per)) if _per else np.nan
    return _out

def c3_run():
    _rows = []
    for _b in [b for b in V_BASELINES if b != "clip"] + (["clip"] if "clip" in V_BASELINES else []):
        _sp = os.path.join(PATHS["res"], f"S6_{_b}_scores.parquet")
        if os.path.exists(_sp):
            _t = pd.read_parquet(_sp)
            _t = _t[_t.set == "cf_test"]
            _rows.append({"model": _b, **c3_auroc_strata(_t.s.to_numpy(), _t.y.to_numpy(), _t.model_name.to_numpy())})
    for _n, _s in V_MODELS:
        _t = v_counts(_n)
        _t = _t[_t.set == "cf_test"]
        _pt = v_scores(_t.k.to_numpy(), _s)[2]
        _rows.append({"model": _n, **c3_auroc_strata(_pt, _t.y.to_numpy(), _t.model_name.to_numpy())})
    _T = pd.DataFrame(_rows)
    if len(_T) and "base" in set(_T.model):
        _base = _T[_T.model == "base"].iloc[0]
        for _c in [c for c in _T.columns if c != "model"]:
            _T[f"drop_vs_base:{_c}"] = _base[_c] - _T[_c]
    _T.to_csv(v_path("tables", "C3_auroc_decomposition.csv"), index=False)
    v_push(v_path("tables", "C3_auroc_decomposition.csv"))
    return _T

C3 = c3_run()
mo.vstack([mo.md("### C3. AUROC cost of smoothing, by stratum"), mo.ui.table(C3.round(4), selection=None)])

In [ ]:
# ---- C4. OUT-OF-POPULATION: what it takes to restore the guarantee on a new family (CPU) ---------
# m OpenFake generators are added to the calibration pool; the guarantee event is measured on the
# OpenFake generators left out (and OpenFake real acceptance on held-out OpenFake reals), averaged
# over random draws. Three recipes:
#   POOLED      CRC over CF + m OpenFake generators (tau_f from CF reals): the naive recipe
#   POOLED+TF   as POOLED, and tau_f re-fitted on half of the OpenFake reals
#   FAMILY-CRC  CRC over the m OpenFake generators alone (feasible only when 1/(m+1) <= alpha:
#               the generator-count law: a family-level guarantee needs >= 1/alpha - 1 generators)
def c4_run(repeats=100):
    _rows = []
    _rng = np.random.default_rng(CFG["seed"] + 404)
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        if "openfake" not in set(_df.set):
            continue
        _cal = v_cal_parts(_df, _s)
        _of = _df[_df.set == "openfake"]
        _lo, _hi, _ = v_scores(_of.k.to_numpy(), _s)
        _y, _g = _of.y.to_numpy(), _of.model_name.to_numpy()
        _gens = [g for g in np.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= 10]
        _hg = {g: _hi[(_y == 1) & (_g == g)] for g in _gens}
        _lr = _lo[_y == 0]
        _cfp = [_cal["hi"][g] for g in _cal["gens"]]
        _ms = [m for m in [0, 1, 2, 4, 8, 12, 16] if m <= len(_gens) - 3]
        for _al in V3["main_alphas"]:
            for _m in _ms:
                _acc = {k: [] for k in ["POOLED", "POOLED+TF", "FAMILY-CRC"]}
                for _rep in range(repeats if _m > 0 else 1):
                    _perm = _rng.permutation(len(_gens))
                    _G = [_gens[i] for i in _perm[:_m]]
                    _H = [_gens[i] for i in _perm[_m:]]
                    _rp = _rng.permutation(len(_lr))
                    _R1, _R2 = _lr[_rp[: len(_rp) // 2]], _lr[_rp[len(_rp) // 2:]]
                    _tf_cf = v_flag_threshold(_cal["lo_r"], V3["main_beta"])
                    _tf_of = v_flag_threshold(_R1, V3["main_beta"]) if len(_R1) else _tf_cf
                    _tp, _ = v_crc(_cfp + [_hg[g] for g in _G], _al)
                    _tfam, _okf = v_crc([_hg[g] for g in _G], _al) if _m > 0 else (np.inf, False)
                    for _rec, _ta, _tf in [("POOLED", max(_tp, _tf_cf), _tf_cf), ("POOLED+TF", max(_tp, _tf_of), _tf_of),
                                           ("FAMILY-CRC", max(_tfam, _tf_of), _tf_of)]:
                        _ev = float(np.mean([np.mean(_hg[h] >= _ta) for h in _H]))
                        _acc[_rec].append((_ev, float(np.mean(_R2 >= _ta + _s)) if len(_R2) else np.nan,
                                           float(np.mean(_R2 < _tf)) if len(_R2) else np.nan, bool(np.isfinite(_ta))))
                for _rec, _v in _acc.items():
                    if not _v:
                        continue
                    _a = np.array(_v, dtype=float)
                    _rows.append({"model": _n, "alpha": _al, "recipe": _rec, "m_openfake_gens_in_calibration": _m,
                                  "heldout_gen_avg_event": float(np.nanmean(_a[:, 0])), "event_sd": float(np.nanstd(_a[:, 0])),
                                  "openfake_real_accept_eps_sigma": float(np.nanmean(_a[:, 1])),
                                  "openfake_framing_event": float(np.nanmean(_a[:, 2])),
                                  "share_draws_with_finite_threshold": float(np.mean(_a[:, 3])),
                                  "generator_count_law_min_m": int(np.ceil(1 / _al - 1))})
    _T = pd.DataFrame(_rows)
    if len(_T):
        _T.to_csv(v_path("tables", "C4_openfake_adaptation.csv"), index=False)
        _b = _T[(_T.model == (V_BEST or _T.model.iloc[0])) & (_T.alpha == V3["select_alpha"])]
        if len(_b):
            _fig, _ax = plt.subplots(1, 2, figsize=(11, 4))
            for _rec, _g in _b.groupby("recipe"):
                _ax[0].plot(_g.m_openfake_gens_in_calibration, _g.heldout_gen_avg_event, marker="o", label=_rec)
                _ax[1].plot(_g.m_openfake_gens_in_calibration, _g.openfake_real_accept_eps_sigma, marker="o", label=_rec)
            _ax[0].axhline(V3["select_alpha"], color="k", ls="--", label="alpha")
            _ax[0].set_xlabel("OpenFake generators added to calibration")
            _ax[0].set_ylabel("held-out OpenFake guarantee event")
            _ax[1].set_xlabel("OpenFake generators added to calibration")
            _ax[1].set_ylabel("OpenFake real acceptance at eps = sigma")
            _ax[0].legend(fontsize=7)
            _ax[0].set_title(f"{V_BEST}, alpha={V3['select_alpha']}")
            v_savefig(_fig, "C4_openfake_adaptation")
        v_push(v_path("tables", "C4_openfake_adaptation.csv"))
        v_push(v_path("figs"), "v3 C4 figure")
    return _T

C4 = c4_run(20 if V3_SMOKE else 100)
mo.vstack([mo.md("### C4. Restoring the guarantee on a new generator family"),
           mo.ui.table(C4.round(4), selection=None, page_size=30) if len(C4) else mo.md("_no OpenFake counts_")])

In [ ]:
# ---- C5. THE SOURCE-MASS LAW: how the per-image guarantee depends on the per-generator cap (CPU) --
# With generator sizes n_g and a cap c, the image-level error with each generator counting at most c
# images is sum_g min(n_g, c) r_g / sum_g min(n_g, c). Size-weighted CRC controls it at level alpha
# only if B(c) / (K + 1) <= alpha, B(c) = c / mean_g min(n_g, c). As c grows the bound needs
# K >= B(c) / alpha - 1 generators: with heavy-tailed sizes no finite calibration set suffices for c
# -> infinity. This cell measures, over random re-splits, (i) the capped error of the plain
# generator rule (GEN-CRC) for each cap, (ii) feasibility and real acceptance of the capped
# size-weighted rule, and (iii) the analytic feasibility bound.
C5_CAPS = [10, 25, 50, 100, 200, 400, 1000000]

def c5_run(n_resplits=100):
    _rows = []
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _p = _df[_df.set.isin(["cf_cal", "cf_test"])]
        _lo, _hi, _ = v_scores(_p.k.to_numpy(), _s)
        _y, _g = _p.y.to_numpy(), _p.model_name.to_numpy()
        _gens = [g for g in np.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= V3["min_per_gen"]]
        _code = {g: i for i, g in enumerate(_gens)}
        _fm = (_y == 1) & np.isin(_g, _gens)
        _hf, _cf = _hi[_fm], np.array([_code[g] for g in _g[_fm]])
        _nn = np.bincount(_cf, minlength=len(_gens)).astype(float)
        _per = [_hf[_cf == i] for i in range(len(_gens))]
        _lr = _lo[_y == 0]
        _rng = np.random.default_rng(CFG["seed"] + 505)
        _acc = {}
        for _rs in range(n_resplits):
            _perm = _rng.permutation(len(_gens))
            _Gc, _Gn = _perm[: len(_gens) // 2], _perm[len(_gens) // 2:]
            _rp = _rng.permutation(len(_lr))
            _tf = v_flag_threshold(_lr[_rp[: len(_rp) // 2]], V3["main_beta"])
            _Rn = _lr[_rp[len(_rp) // 2:]]
            _t0, _ = a11_crc([_per[i] for i in _Gc], V3["select_alpha"], "equal")
            _h0 = np.bincount(_cf, weights=(_hf >= max(_t0, _tf)).astype(float), minlength=len(_gens))
            _r0 = _h0[_Gn] / _nn[_Gn]
            for _c in C5_CAPS:
                _m = np.minimum(_nn[_Gn], _c)
                # cap "none" = plain size weighting, with B = the largest calibration weight (as in A11)
                _wt = "size" if _c >= 1000000 else "capped"
                _tc, _feas = a11_crc([_per[i] for i in _Gc], V3["select_alpha"], _wt, cap=_c)
                _ta = max(_tc, _tf)
                _hc = np.bincount(_cf, weights=(_hf >= _ta).astype(float), minlength=len(_gens))
                _rc = _hc[_Gn] / _nn[_Gn]
                _Bc = float((_nn[_Gc].max() if _wt == "size" else _c) / np.minimum(_nn[_Gc], _c).mean())
                _acc.setdefault(_c, []).append((float((_m * _r0).sum() / _m.sum()), bool(_feas and np.isfinite(_tc)),
                                                float((_m * _rc).sum() / _m.sum()), float(np.mean(_Rn >= _ta + _s)),
                                                float(np.mean(_Rn >= max(_t0, _tf) + _s)), _Bc, len(_Gc)))
        for _c, _v in _acc.items():
            _a = np.array(_v, dtype=float)
            _K = _a[0, 6]
            _rows.append({"model": _n, "alpha": V3["select_alpha"], "cap": "none (B = largest calibration generator)" if _c >= 1000000 else _c,
                          "GEN-CRC capped error (mean)": float(_a[:, 0].mean()), "GEN-CRC capped error (sd)": float(_a[:, 0].std()),
                          "GEN-CRC real accept": float(_a[:, 4].mean()),
                          "capped-weighted CRC feasible share": float(_a[:, 1].mean()),
                          "capped-weighted CRC capped error": float(_a[:, 2].mean()),
                          "capped-weighted CRC real accept": float(_a[:, 3].mean()),
                          "B(cap) mean": float(_a[:, 5].mean()), "B/(K+1)": float(_a[:, 5].mean() / (_K + 1)),
                          "min generators needed K >= B/alpha - 1": float(np.ceil(_a[:, 5].mean() / V3["select_alpha"] - 1)),
                          "K calibration generators": int(_K)})
    _T = pd.DataFrame(_rows)
    if len(_T):
        _T.to_csv(v_path("tables", "C5_source_mass_law.csv"), index=False)
        v_push(v_path("tables", "C5_source_mass_law.csv"))
    return _T

C5 = c5_run(10 if V3_SMOKE else 100)
mo.vstack([mo.md("### C5. Source-mass law: per-image guarantee vs per-generator cap"),
           mo.ui.table(C5.round(4), selection=None, page_size=40) if len(C5) else mo.md("_no counts_")])

In [ ]:
# ---- C6. ADRT LOSS ABLATION (GPU, resumable) -----------------------------------------------------
# The main model, retrained with one hinge term removed: AF-only (lam_fr = 0) and FR-only
# (lam_af = 0); Cohen (both removed, mu = 0) is already trained. All are certified on the same
# balanced subsets as B5, so the rows compare directly with B5's seed-0 row of the main model.
def c6_run():
    if not V_BEST or not V_BEST.startswith("eca"):
        return "main model is not an ECA model"
    _sigma = dict(V_MODELS)[V_BEST]
    _sd = V3["seeds"]
    _sub = {}

    def _subsets():
        if not _sub:
            _c = b_store_lazy()
            _sub.update(cf=_c, ic=b_subset(_c, "cal", _sd["per_gen_cap"], _sd["n_real"]),
                        it=b_subset(_c, "test", _sd["per_gen_cap"], _sd["n_real"]))
        return _sub["cf"], _sub["ic"], _sub["it"]

    _units = [(f"{V_BEST}_abl_{k}", "train", v) for k, v in V3["c6"]["variants"].items()]
    _coh = [n for n, s in V_MODELS if n.startswith("cohen") and abs(s - _sigma) < 1e-9]
    _units += [(_coh[0], "certify", None)] if _coh else []
    _done = []
    for _name, _how, _over in _units:
        _cp = v_path("gpu", f"C6_{_name}_counts.parquet")
        if not ensure(_cp):
            if _how == "train":
                _w = b_train(_name, "eca", _sigma, 0, _sd["max_steps"], eca=_over)
            else:
                _w = b_weight(_name)
            _cf, _ic, _it = _subsets()
            _m = load_detector(_w, DEVICE)
            _fr = [b_counts_frame(_cf, _ix, b_certify(_m, _cf, _ix, _sigma, V3["N"]), _sn, V3["N"])
                   for _sn, _ix in [("cf_cal", _ic), ("cf_test", _it)]]
            pd.concat(_fr).to_parquet(_cp)
            v_push(_cp)
            del _m
            gpu_free()
        _done.append(_name)
    return _done

def c6_table():
    if not V_BEST:
        return pd.DataFrame()
    _sigma = dict(V_MODELS)[V_BEST]
    _src = [("ADRT full (B5 seed 0)", v_path("gpu", f"B5_{V_BEST}_counts.parquet"))]
    _src += [(f"ADRT {k}", v_path("gpu", f"C6_{V_BEST}_abl_{k}_counts.parquet")) for k in V3["c6"]["variants"]]
    _src += [(f"Cohen (no ADRT)", v_path("gpu", f"C6_{n}_counts.parquet")) for n, s in V_MODELS
             if n.startswith("cohen") and abs(s - _sigma) < 1e-9]
    _rows = []
    for _lab, _p in _src:
        if ensure(_p):
            _t = b_frontier_from_counts(pd.read_parquet(_p), _sigma, V3["N"], _lab)
            _rows.append(_t)
    _T = pd.concat(_rows) if _rows else pd.DataFrame()
    if len(_T):
        _T.to_csv(v_path("tables", "C6_adrt_ablation.csv"), index=False)
        v_push(v_path("tables", "C6_adrt_ablation.csv"))
    return _T

if V3["run"]["C6"]:
    _ = (BUNDLE_INFO, B5_OUT, C2_OUT)
    C6_OUT = b_step("C6 ADRT ablation", c6_run)
else:
    C6_OUT = {"skipped": True}
C6 = c6_table()
mo.vstack([mo.md(f"### C6. ADRT loss ablation: `{C6_OUT}`"),
           mo.ui.table(C6.round(4), selection=None, page_size=30) if len(C6) else mo.md("_needs C6 outputs_")])

In [ ]:
# ---- C7. THE EVIDENCE-FLOOR LAW: what limits alpha, estimation or the detector? (CPU) ---------
# With N noise draws the largest certifiable score is sigma*z_N, z_N = Phi^-1((alpha_mc/2)^(1/N)),
# which grows only like sqrt(2 ln N). Accepting anything at budget eps needs tau_a <= sigma*z_N - eps,
# so (Proposition, A3) alpha >= alpha_min(eps, N) = K/(K+1) * mean_g P_g(S_hi >= sigma z_N - eps) + 1/(K+1).
# As N -> infinity, S_hi -> S = sigma*Phi^-1(p) and z_N -> infinity, so for fixed eps the only fakes
# still inside the event are those with p = 1, i.e. fakes the base classifier calls real under EVERY
# noise draw. Hence the floor
#   alpha_inf = K/(K+1) * m_conf + 1/(K+1),  m_conf = mean_g P_g(p(x) = 1)  (confident-error mass),
# which no amount of sampling removes. This cell estimates m_conf from the N = 1024 counts of B4
# (plug-in p = k/N; an upper estimate of the p = 1 mass) and predicts alpha_min for larger N, next to
# the measured alpha_min at N = 256 and 1024.
def c7_run():
    _p = v_path("gpu", f"B4_{V_BEST}_counts.parquet") if V_BEST else None
    if not _p or not ensure(_p):
        return pd.DataFrame()
    _C = pd.read_parquet(_p)
    _sigma = dict(V_MODELS)[V_BEST]
    _Ns = sorted(_C.N.unique())
    _Nb = int(max(_Ns))
    _cb = _C[(_C.N == _Nb) & (_C.set == "cf_cal")]
    _f = _cb[_cb.y == 1]
    _gens = [g for g, n in _f.model_name.value_counts().items() if n >= min(V3["min_per_gen"], V3["bigN"]["per_gen_cap"])]
    _phat = {g: _f[_f.model_name == g].k.to_numpy() / _Nb for g in _gens}
    _K = len(_gens)
    _mconf = float(np.mean([np.mean(_phat[g] >= 1.0) for g in _gens])) if _K else np.nan
    _rows = []
    for _N in [256, 1024, 4096, 16384, 65536, 10**6, np.inf]:
        _zN = v_zN(_N, V3["alpha_mc"]) if np.isfinite(_N) else np.inf
        for _er in [0.0, 0.5, 1.0]:
            _e = _er * _sigma
            if np.isfinite(_zN):
                _thr = _sigma * _zN - _e
                # p-hat = 1 images: S = +inf in the plug-in, always inside the event
                _ev = float(np.mean([np.mean((_phat[g] >= 1.0) | (_sigma * v_norm.ppf(np.clip(_phat[g], 1e-15, 1 - 1e-15)) >= _thr))
                                     for g in _gens])) if _K else np.nan
            else:
                _ev = _mconf
            _meas = np.nan
            if np.isfinite(_N) and int(_N) in [int(n) for n in _Ns]:
                _cc = _C[(_C.N == _N) & (_C.set == "cf_cal")]
                _cal = v_cal_parts(_cc, _sigma, int(_N))
                _meas = v_risk([_cal["hi"][g] for g in _cal["gens"]], _sigma * _zN - _e) if _cal["gens"] else np.nan
            _rows.append({"N": "infinity" if not np.isfinite(_N) else int(_N), "z_N": _zN, "eps_over_sigma": _er,
                          "predicted_alpha_min (plug-in from N=%d)" % _Nb: _K / (_K + 1) * _ev + 1 / (_K + 1) if _K else np.nan,
                          "measured_alpha_min": _meas, "confident_error_mass m_conf": _mconf,
                          "floor alpha_inf": _K / (_K + 1) * _mconf + 1 / (_K + 1) if _K else np.nan, "K": _K})
    _T = pd.DataFrame(_rows)
    _T.to_csv(v_path("tables", "C7_evidence_floor_law.csv"), index=False)
    v_push(v_path("tables", "C7_evidence_floor_law.csv"))
    return _T

C7 = c7_run()
mo.vstack([mo.md("### C7. Evidence-floor law: alpha_min vs number of noise draws"),
           mo.ui.table(C7.round(5), selection=None, page_size=30) if len(C7) else mo.md("_needs B4 counts_")])

In [ ]:
# ---- C8. ONE-CLASS COLLAPSE: when does the fake data stop mattering? (CPU) ---------------------
# CERT-CAL sets tau_a = max(tau_CRC(alpha), tau_f(beta)). The CRC threshold falls as alpha rises; it
# reaches tau_f at
#   alpha*(beta) = K/(K+1) * mean_g P_g(S_hi >= tau_f(beta)) + 1/(K+1).
# For every alpha >= alpha*, tau_a = tau_f: the accept rule is "S_lo >= (beta-quantile of real S_lo)
# + eps", a ONE-CLASS certificate of typicality that never looks at a fake; the fake data then only
# certifies the risk bound alpha*. This cell computes alpha* per model and beta, checks it against
# the grid, and compares the one-class rule with ECA on held-out CF and on OpenFake.
# Validity of the one-class rule at alpha >= alpha*: the fake loss is non-increasing in the
# threshold and tau_f >= inf{t : risk_hat(t) <= alpha}, so the CRC bound carries over to tau_f.
# The implemented CRC returns an observed fake score, which can sit slightly above tau_f, so
# ECA and the one-class rule may still differ on a few images; the last column counts them.
def c8_run():
    _rows = []
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        _per = [_cal["hi"][g] for g in _cal["gens"]]
        for _be in V3["beta_grid"]:
            _tf = v_flag_threshold(_cal["lo_r"], _be)
            _astar = v_risk(_per, _tf)
            _one = {"tau_a": _tf, "tau_f": _tf}
            for _sn in [s for s in ["cf_test", "openfake"] if s in set(_df.set)]:
                _t = _df[_df.set == _sn]
                _m1 = v_eval_set(_t, _s, _one, _s)
                _ru = v_rule(_cal, max(V3["select_alpha"], _astar), _be)
                _m2 = v_eval_set(_t, _s, _ru, _s)
                _lo8, _hi8 = v_scores(_t.k.to_numpy(), _s)[:2]
                _d1 = np.asarray(decide(_lo8, _hi8, _tf, _tf, _s))
                _d2 = np.asarray(decide(_lo8, _hi8, _ru["tau_a"], _ru["tau_f"], _s))
                _rows.append({"model": _n, "beta": _be, "tau_f": _tf, "alpha_star (tau_f binds for alpha >= this)": _astar,
                              "set": _sn, "one_class_accept_real": _m1["accept_real_rate"], "one_class_AF": _m1["AF"],
                              "one_class_event_gen_avg": _m1.get("guarantee_event_gen_avg P(S_hi>=tau_a)"),
                              "ECA_accept_real (alpha=max(5%, alpha*))": _m2["accept_real_rate"], "ECA_AF": _m2["AF"],
                              "ECA_tau_a_minus_tau_f": float(_ru["tau_a"] - _tf),
                              "share_of_images_decided_differently": float(np.mean(_d1 != _d2))})
    _T = pd.DataFrame(_rows)
    _T.to_csv(v_path("tables", "C8_one_class_collapse.csv"), index=False)
    v_push(v_path("tables", "C8_one_class_collapse.csv"))
    return _T

C8 = c8_run()
mo.vstack([mo.md("### C8. One-class collapse of the certified accept rule"), mo.ui.table(C8.round(4), selection=None, page_size=40)])

In [ ]:
# ---- C9. REPRODUCIBILITY MANIFEST + PART C SUMMARY (CPU) -----------------------------------------
import hashlib as c9_hashlib
import platform as c9_platform
import importlib.metadata as c9_meta

def c9_run():
    _files = {}
    for _root in [PATHS["res"], V3["out"]]:
        for _d, _, _fs in os.walk(_root):
            for _f in sorted(_fs):
                _p = os.path.join(_d, _f)
                if os.path.getsize(_p) > 512 * 2**20 or _f.endswith(".tmp"):
                    continue
                _h = c9_hashlib.sha256()
                with open(_p, "rb") as _fh:
                    for _blk in iter(lambda: _fh.read(1 << 20), b""):
                        _h.update(_blk)
                _files[os.path.relpath(_p, CFG["work"])] = _h.hexdigest()
    _ver = {}
    for _pk in ["torch", "numpy", "pandas", "scipy", "timm", "transformers", "huggingface_hub", "marimo", "pillow"]:
        try:
            _ver[_pk] = c9_meta.version(_pk)
        except Exception:
            _ver[_pk] = None
    _man = {"python": c9_platform.python_version(), "packages": _ver, "config": CFG, "v3": V3,
            "folders": {k: os.path.basename(v) for k, v in PATHS.items()}, "sha256": _files}
    save_json(v_path("C9_MANIFEST.json"), _man)
    # one-page summary of Part C
    _L = ["# Part C summary (auto-generated; numbers only)\n"]
    def _sec(title, path, cols=None, n=40):
        if os.path.exists(path):
            _t = pd.read_csv(path)
            _L.append(f"## {title}\n\n{(_t[cols] if cols else _t).round(4).to_string(index=False, max_rows=n)}\n")
    _sec("C1 Yao three-way cost: ECA certified upper bound vs baseline attack lower bound", v_path("tables", "C1_yao_head_to_head.csv"))
    _sec("C2 pgdat under a stronger attack", v_path("tables", "C2_pgdat_stronger_attack.csv"))
    _sec("C3 AUROC by stratum", v_path("tables", "C3_auroc_decomposition.csv"))
    _sec("C4 OpenFake adaptation", v_path("tables", "C4_openfake_adaptation.csv"))
    _sec("C5 source-mass law", v_path("tables", "C5_source_mass_law.csv"))
    _sec("C6 ADRT ablation", v_path("tables", "C6_adrt_ablation.csv"))
    _sec("C7 evidence-floor law", v_path("tables", "C7_evidence_floor_law.csv"))
    _sec("C8 one-class collapse", v_path("tables", "C8_one_class_collapse.csv"))
    with open(v_path("C_SUMMARY.md"), "w") as _fh:
        _fh.write("\n".join(_L))
    v_push(v_path("C9_MANIFEST.json"))
    v_push(v_path("C_SUMMARY.md"))
    return {"files_hashed": len(_files)}

_ = (C1, C2, C3, C4, C5, C6, C7, C8)
C9 = c9_run()
mo.md(f"### C9. Manifest: {C9['files_hashed']} files hashed → `{v_path('C9_MANIFEST.json')}`; summary → `{v_path('C_SUMMARY.md')}`")

In [ ]:
# ---- B6. FINAL: re-run the audit with the new attack files, rebuild the bundle -------------
_ = (B1_OUT, B2_CURVES, B3_OUT, B4_OUT, B5_OUT, C9)
A5_FINAL = a5_run()
BUNDLE_FINAL = a10_bundle()
mo.vstack([mo.md(f"**All done.** Bundle: `{BUNDLE_FINAL['zip']}` ({BUNDLE_FINAL['MB']} MB) · RAM {rss_gb():.1f} GB"),
           mo.ui.table(pd.DataFrame(B_LOG), selection=None) if B_LOG else mo.md("_no part-B steps ran_")])

In [ ]:
# ---- VIEWER: every table and figure written by this notebook (reads files only) ----------
_ = BUNDLE_FINAL
_tabs = {}
for _f in sorted(os.listdir(v_path("tables"))):
    if _f.endswith(".csv"):
        _t = pd.read_csv(v_path("tables", _f))
        _tabs[_f[:-4]] = mo.ui.table(_t.round(4), selection=None, page_size=25) if len(_t) < 20000 else mo.md(f"{len(_t):,} rows (see file)")
for _f in sorted(os.listdir(v_path("figs"))):
    if _f.endswith(".png"):
        _tabs["fig " + _f[:-4]] = mo.image(v_path("figs", _f))
mo.ui.tabs(_tabs) if _tabs else mo.md("_Nothing written yet._")

In [ ]:
import os as _os, numpy as _np, pandas as _pd
from scipy.stats import binom as _binom
# C1b. Yao three-way cost with a TIGHT certified bound (replaces C1's ECA rows, which assumed every
# real could be pushed to "defer" and so always picked "defer everything").
# Real x, clean S_lo = L, attack ||d|| <= eps:  S(x+d) >= L - eps, so p' >= Phi((L - eps)/sigma) and
#   P(accept x+d) >= P(Binom(N, p') >= k*),  k* = smallest k with S_lo(k) >= tau_a + eps;
#   x+d can be flagged only if L < tau_f.
# Fake x, clean S_hi = H:  accepted only if H >= tau_a (Theorem 3); otherwise
#   P(flag x+d) >= P(Binom(N, p'') <= k_f),  p'' <= Phi((H + eps)/sigma),  k_f = largest k with S_hi(k) < tau_f - eps.
# Cost upper bound (per class, 0.5/0.5 prior, as in C1):
#   fake: l_PN 1[H >= tau_a] + l_BN (1 - P_flag_lb)      real: l_NP 1[L < tau_f] + l_BP (1 - P_acc_lb)
# All bounds hold with probability >= 1 - 2 alpha_mc per image. (alpha, beta) are chosen on calibration
# data with the same bound; ECA models only; Cohen reported as its own row.
def _c1b_bound(lo_r, hi_f, ta, tf, e, s, N, l):
    _k = _np.arange(N + 1)
    _L, _H, _ = v_scores(_k, s, N)
    _ka = _k[_L >= ta + e]
    _kf = _k[_H < tf - e]
    _pr = v_norm.cdf((_np.asarray(lo_r) - e) / s)
    _pacc = _binom.sf(_ka.min() - 1, N, _pr) if len(_ka) else _np.zeros(len(lo_r))
    _pacc = _np.where(_np.asarray(lo_r) >= tf, _pacc, 0.0)
    _pf = v_norm.cdf((_np.asarray(hi_f) + e) / s)
    _pflag = _binom.cdf(_kf.max(), N, _pf) if len(_kf) else _np.zeros(len(hi_f))
    _acc_f = _np.asarray(hi_f) >= ta
    _pflag = _np.where(_acc_f, 0.0, _pflag)
    _cf = l[0] * _acc_f.mean() + l[1] * float(_np.mean((~_acc_f) * (1 - _pflag)))
    _cr = l[2] * float(_np.mean(_np.asarray(lo_r) < tf)) + l[3] * float(_np.mean((_np.asarray(lo_r) >= tf) * (1 - _pacc)))
    return 0.5 * _cf + 0.5 * _cr, float(_acc_f.mean()), float(_np.mean(_pacc)), float(_np.mean(_pflag))

def _c1b_run():
    _N = V3["N"]
    _rows = []
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        _t = _df[_df.set == "cf_test"]
        _lo, _hi, _ = v_scores(_t.k.to_numpy(), _s)
        _y, _g = _t.y.to_numpy(), _t.model_name.to_numpy()
        _ef, _er = c1_halves(_g[_y == 1], int((_y == 0).sum()))
        _hif, _lor = _hi[_y == 1][_ef], _lo[_y == 0][_er]
        _calf = _np.concatenate([_cal["hi"][g] for g in _cal["gens"]])
        for _l in C1_LOSSES:
            for _e in C1_BUDGETS:
                _best = None
                for _al in V3["alpha_grid"]:
                    for _be in V3["beta_grid"]:
                        _ru = v_rule(_cal, _al, _be)
                        _c = _c1b_bound(_cal["lo_r"], _calf, _ru["tau_a"], _ru["tau_f"], _e, _s, _N, _l)[0]
                        if _best is None or _c < _best[0]:
                            _best = (_c, _al, _be, _ru)
                _cc, _al, _be, _ru = _best
                _c, _afu, _pacc, _pfl = _c1b_bound(_lor, _hif, _ru["tau_a"], _ru["tau_f"], _e, _s, _N, _l)
                _rows.append({"model": _n, "loss": str(_l), "l2_budget": _e, "alpha": _al, "beta": _be,
                              "cost_upper_bound": _c, "calibration_cost": _cc, "AF_upper": _afu,
                              "real_accept_lower (attacked)": _pacc, "fake_flag_lower (attacked)": _pfl,
                              "defer_all_cost": 0.5 * _l[1] + 0.5 * _l[3]})
    _T = _pd.DataFrame(_rows)
    _B = _pd.read_csv(v_path("tables", "C1_yao_three_way_cost.csv"))
    _B = _B[_B.method.str.contains("uncertified")]
    _h = []
    for (_l, _e), _g in _T.groupby(["loss", "l2_budget"]):
        _ec = _g[_g.model.str.startswith("eca")]
        _co = _g[_g.model.str.startswith("cohen")]
        _bb = _B[(_B.loss == _l) & (_np.isclose(_B.l2_budget, _e))]
        if not len(_ec):
            continue
        _eb = _ec.loc[_ec.calibration_cost.idxmin()]
        _row = {"loss": _l, "l2_budget": _e, "ECA_model (chosen on calibration)": _eb.model,
                "ECA_cost_upper_bound": _eb.cost_upper_bound, "ECA_real_accept_lower": _eb["real_accept_lower (attacked)"],
                "Cohen_cost_upper_bound": float(_co.cost_upper_bound.min()) if len(_co) else _np.nan,
                "defer_all_cost": _eb.defer_all_cost}
        if len(_bb):
            _bm = _bb.loc[_bb.cost_lower_bound.idxmin()]
            _row.update({"best_baseline": _bm.method, "baseline_cost_lower_bound": _bm.cost_lower_bound,
                         "ECA_provably_cheaper_than_baseline": bool(_eb.cost_upper_bound < _bm.cost_lower_bound)})
        _row["ECA_provably_cheaper_than_defer_all"] = bool(_eb.cost_upper_bound < _eb.defer_all_cost - 1e-9)
        _h.append(_row)
    _H = _pd.DataFrame(_h)
    _T.to_csv(v_path("tables", "C1b_yao_tight_bound.csv"), index=False)
    _H.to_csv(v_path("tables", "C1b_yao_head_to_head.csv"), index=False)
    return _T, _H

_T1b, _H1b = _c1b_run()

In [ ]:
import numpy as _np6, pandas as _pd6
from scipy.stats import binom as _binom6
# C6b. Does "accept-term-only" ADRT really beat full ADRT? (GPU: 2 trainings, resumable; then CPU)
# Trains the AF-only variant with seeds 1 and 2 (seed 0 exists from C6), certifies on the same B5
# subsets, and compares 3 seeds of AF-only with 3 seeds of full ADRT (B5). Also reports what the flag
# term buys: certified flagging of fakes and the C1b certified Yao cost, where C1b located all the value.
_SEEDS = [1, 2]

def _c6b_train():
    _sigma = dict(V_MODELS)[V_BEST]
    _sd = V3["seeds"]
    _sub = {}
    _done = []
    for _s in _SEEDS:
        _name = f"{V_BEST}_abl_afonly_seed{_s}"
        _cp = v_path("gpu", f"C6_{_name}_counts.parquet")
        if not ensure(_cp):
            if not _sub:
                _c = b_store_lazy()
                _sub.update(cf=_c, ic=b_subset(_c, "cal", _sd["per_gen_cap"], _sd["n_real"]),
                            it=b_subset(_c, "test", _sd["per_gen_cap"], _sd["n_real"]))
            _w = b_train(_name, "eca", _sigma, _s, _sd["max_steps"], eca=V3["c6"]["variants"]["afonly"])
            _m = load_detector(_w, DEVICE)
            _fr = [b_counts_frame(_sub["cf"], _ix, b_certify(_m, _sub["cf"], _ix, _sigma, V3["N"]), _sn, V3["N"])
                   for _sn, _ix in [("cf_cal", _sub["ic"]), ("cf_test", _sub["it"])]]
            _pd6.concat(_fr).to_parquet(_cp)
            v_push(_cp)
            del _m
            gpu_free()
        _done.append(_name)
    return _done

def _flag_cost(df, s, ta, tf, e, l=(10.0, 0.5, 2.0, 0.2)):
    """C1b certified Yao cost bound on cf_test (all generators), plus certified flag rates."""
    _N = V3["N"]
    _t = df[df.set == "cf_test"]
    _lo, _hi, _ = v_scores(_t.k.to_numpy(), s)
    _y = _t.y.to_numpy()
    _lr, _hf = _lo[_y == 0], _hi[_y == 1]
    _k = _np6.arange(_N + 1)
    _L, _H, _ = v_scores(_k, s, _N)
    _ka, _kf = _k[_L >= ta + e], _k[_H < tf - e]
    _pacc = _binom6.sf(_ka.min() - 1, _N, v_norm.cdf((_lr - e) / s)) if len(_ka) else _np6.zeros(len(_lr))
    _pacc = _np6.where(_lr >= tf, _pacc, 0.0)
    _acc = _hf >= ta
    _pfl = _binom6.cdf(_kf.max(), _N, v_norm.cdf((_hf + e) / s)) if len(_kf) else _np6.zeros(len(_hf))
    _pfl = _np6.where(_acc, 0.0, _pfl)
    _cost = 0.5 * (l[0] * _acc.mean() + l[1] * _np6.mean(~_acc * (1 - _pfl))) + \
            0.5 * (l[2] * _np6.mean(_lr < tf) + l[3] * _np6.mean((_lr >= tf) * (1 - _pacc)))
    return float(_cost), float(_np6.mean(_pfl)), float(_np6.mean(_pacc))

def _c6b_table():
    _sigma = dict(V_MODELS)[V_BEST]
    _src = [("full", 0, f"B5_{V_BEST}_counts.parquet"), ("full", 1, f"B5_{V_BEST}_seed1_counts.parquet"),
            ("full", 2, f"B5_{V_BEST}_seed2_counts.parquet"), ("afonly", 0, f"C6_{V_BEST}_abl_afonly_counts.parquet")]
    _src += [("afonly", s, f"C6_{V_BEST}_abl_afonly_seed{s}_counts.parquet") for s in _SEEDS]
    _src += [("fronly", 0, f"C6_{V_BEST}_abl_fronly_counts.parquet")]
    _src += [("cohen", 0, f"C6_{n}_counts.parquet") for n, s in V_MODELS if n.startswith("cohen") and abs(s - _sigma) < 1e-9]
    _rows = []
    for _v, _sd, _f in _src:
        _p = v_path("gpu", _f)
        if not ensure(_p):
            continue
        _df = _pd6.read_parquet(_p)
        _fr = b_frontier_from_counts(_df, _sigma, V3["N"], f"{_v} s{_sd}")
        _cal = v_cal_parts(_df, _sigma)
        _ru = v_rule(_cal, V3["select_alpha"], V3["main_beta"])
        _r = {"variant": _v, "seed": _sd, "alpha_min(eps=sigma)": float(_fr["alpha_min(eps=sigma)"].iloc[0]),
              "tau_f(beta=5%)/sigma": _ru["tau_f"] / _sigma, "cap_minus_1 = z_N - 1": v_zN(V3["N"], V3["alpha_mc"]) - 1,
              "auroc": float(_fr["auroc_point"].iloc[0])}
        for _a in [0.02, 0.05]:
            _q = _fr[_np6.isclose(_fr.alpha, _a)]
            if len(_q):
                _r[f"accept_real a={_a}"] = float(_q.accept_real_eps_sigma.iloc[0])
                _r[f"AF a={_a}"] = float(_q.AF_eps_sigma.iloc[0])
                _r[f"event a={_a}"] = float(_q.guarantee_event.iloc[0])
        for _e in [0.5, 1.0]:
            _c, _pf, _pa = _flag_cost(_df, _sigma, _ru["tau_a"], _ru["tau_f"], _e * _sigma)
            _r[f"certified_fake_flag eps={_e}sigma"] = _pf
            _r[f"C1b_cost(10,.5,2,.2) eps={_e}sigma"] = _c
        _rows.append(_r)
    _T = _pd6.DataFrame(_rows)
    _num = [c for c in _T.columns if c not in ("variant", "seed")]
    _S = _T.groupby("variant")[_num].agg(["mean", "std", "count"])
    _S.columns = [f"{a} [{b}]" for a, b in _S.columns]
    _T.to_csv(v_path("tables", "C6b_ablation_seeds.csv"), index=False)
    _S.to_csv(v_path("tables", "C6b_ablation_seed_summary.csv"))
    v_push(v_path("tables"), "v3 C6b ablation seeds")
    return _T, _S

_C6B_DONE = b_step("C6b AF-only extra seeds", _c6b_train) if V_BEST and V_BEST.startswith("eca") else "main model is not ECA"
_T6b, _S6b = _c6b_table()

In [ ]:
import numpy as _np7, pandas as _pd7
from scipy.stats import binom as _binom7
# C6c. The accept/flag trade-off of ADRT: sweep the flag weight lam_fr (GPU: 3 trainings, resumable).
# C6b showed the two hinge terms move tau_f in opposite directions: lam_fr = 0 puts tau_f far below
# (no certified flags), lam_af = 0 puts it above sigma*(z_N - 1) (no certified accepts). Full ADRT
# (lam_af = 4, lam_fr = 1) sits in between and is seed-unstable on flagging. This sweep (lam_af = 4,
# lam_fr in {2, 4, 8}, seed 0) maps whether one model can certify both sides at once.
_LAMS = [2.0, 4.0, 8.0]

def _c6c_train():
    _sigma = dict(V_MODELS)[V_BEST]
    _sd = V3["seeds"]
    _sub = {}
    _done = []
    for _lf in _LAMS:
        _s = 0
        _name = f"{V_BEST}_lamfr{_lf:g}"
        _cp = v_path("gpu", f"C6_{_name}_counts.parquet")
        if not ensure(_cp):
            if not _sub:
                _c = b_store_lazy()
                _sub.update(cf=_c, ic=b_subset(_c, "cal", _sd["per_gen_cap"], _sd["n_real"]),
                            it=b_subset(_c, "test", _sd["per_gen_cap"], _sd["n_real"]))
            _w = b_train(_name, "eca", _sigma, _s, _sd["max_steps"], eca={"lam_fr": _lf})
            _m = load_detector(_w, DEVICE)
            _fr = [b_counts_frame(_sub["cf"], _ix, b_certify(_m, _sub["cf"], _ix, _sigma, V3["N"]), _sn, V3["N"])
                   for _sn, _ix in [("cf_cal", _sub["ic"]), ("cf_test", _sub["it"])]]
            _pd7.concat(_fr).to_parquet(_cp)
            v_push(_cp)
            del _m
            gpu_free()
        _done.append(_name)
    return _done

def _flag_cost(df, s, ta, tf, e, l=(10.0, 0.5, 2.0, 0.2)):
    """C1b certified Yao cost bound on cf_test (all generators), plus certified flag rates."""
    _N = V3["N"]
    _t = df[df.set == "cf_test"]
    _lo, _hi, _ = v_scores(_t.k.to_numpy(), s)
    _y = _t.y.to_numpy()
    _lr, _hf = _lo[_y == 0], _hi[_y == 1]
    _k = _np7.arange(_N + 1)
    _L, _H, _ = v_scores(_k, s, _N)
    _ka, _kf = _k[_L >= ta + e], _k[_H < tf - e]
    _pacc = _binom7.sf(_ka.min() - 1, _N, v_norm.cdf((_lr - e) / s)) if len(_ka) else _np7.zeros(len(_lr))
    _pacc = _np7.where(_lr >= tf, _pacc, 0.0)
    _acc = _hf >= ta
    _pfl = _binom7.cdf(_kf.max(), _N, v_norm.cdf((_hf + e) / s)) if len(_kf) else _np7.zeros(len(_hf))
    _pfl = _np7.where(_acc, 0.0, _pfl)
    _cost = 0.5 * (l[0] * _acc.mean() + l[1] * _np7.mean(~_acc * (1 - _pfl))) + \
            0.5 * (l[2] * _np7.mean(_lr < tf) + l[3] * _np7.mean((_lr >= tf) * (1 - _pacc)))
    return float(_cost), float(_np7.mean(_pfl)), float(_np7.mean(_pacc))

def _c6c_table():
    _sigma = dict(V_MODELS)[V_BEST]
    _src = [("lam_fr=0 (afonly)", 0, f"C6_{V_BEST}_abl_afonly_counts.parquet"),
            ("lam_fr=1 (full)", 0, f"B5_{V_BEST}_counts.parquet")]
    _src += [(f"lam_fr={_lf:g}", 0, f"C6_{V_BEST}_lamfr{_lf:g}_counts.parquet") for _lf in _LAMS]
    _src += [("lam_af=0 (fronly)", 0, f"C6_{V_BEST}_abl_fronly_counts.parquet")]
    _src += [("cohen", 0, f"C6_{n}_counts.parquet") for n, s in V_MODELS if n.startswith("cohen") and abs(s - _sigma) < 1e-9]
    _rows = []
    for _v, _sd, _f in _src:
        _p = v_path("gpu", _f)
        if not ensure(_p):
            continue
        _df = _pd7.read_parquet(_p)
        _fr = b_frontier_from_counts(_df, _sigma, V3["N"], f"{_v} s{_sd}")
        _cal = v_cal_parts(_df, _sigma)
        _ru = v_rule(_cal, V3["select_alpha"], V3["main_beta"])
        _r = {"variant": _v, "seed": _sd, "alpha_min(eps=sigma)": float(_fr["alpha_min(eps=sigma)"].iloc[0]),
              "tau_f(beta=5%)/sigma": _ru["tau_f"] / _sigma, "cap_minus_1 = z_N - 1": v_zN(V3["N"], V3["alpha_mc"]) - 1,
              "auroc": float(_fr["auroc_point"].iloc[0])}
        for _a in [0.02, 0.05]:
            _q = _fr[_np7.isclose(_fr.alpha, _a)]
            if len(_q):
                _r[f"accept_real a={_a}"] = float(_q.accept_real_eps_sigma.iloc[0])
                _r[f"AF a={_a}"] = float(_q.AF_eps_sigma.iloc[0])
                _r[f"event a={_a}"] = float(_q.guarantee_event.iloc[0])
        for _e in [0.5, 1.0]:
            _c, _pf, _pa = _flag_cost(_df, _sigma, _ru["tau_a"], _ru["tau_f"], _e * _sigma)
            _r[f"certified_fake_flag eps={_e}sigma"] = _pf
            _r[f"C1b_cost(10,.5,2,.2) eps={_e}sigma"] = _c
        _rows.append(_r)
    _T = _pd7.DataFrame(_rows)
    _T.to_csv(v_path("tables", "C6c_lamfr_sweep.csv"), index=False)
    v_push(v_path("tables"), "v3 C6c lam_fr sweep")
    _S = None
    return _T, _S

_C6C_DONE = b_step("C6c lam_fr sweep", _c6c_train) if V_BEST and V_BEST.startswith("eca") else "main model is not ECA"
_T6c, _ = _c6c_table()

In [ ]:
import numpy as _np8, pandas as _pd8
from scipy.stats import binom as _binom8
# C6d: seeds 1 and 2 for the two operating points of the lam_fr sweep that matter (GPU: 4 trainings,
# resumable): lam_fr = 4 (best certified cost at eps = sigma/2, best AUROC) and lam_fr = 8 (certifies
# accept AND flag at eps = sigma). Seed 0 of each exists from C6c. Prints all rows and a mean/sd summary.
_LAMS = [4.0, 8.0]
_SEEDS8 = [1, 2]

def _c6d_train():
    _sigma = dict(V_MODELS)[V_BEST]
    _sd = V3["seeds"]
    _sub = {}
    _done = []
    for _lf, _s in [(a, b) for a in _LAMS for b in _SEEDS8]:
        _name = f"{V_BEST}_lamfr{_lf:g}_seed{_s}"
        _cp = v_path("gpu", f"C6_{_name}_counts.parquet")
        if not ensure(_cp):
            if not _sub:
                _c = b_store_lazy()
                _sub.update(cf=_c, ic=b_subset(_c, "cal", _sd["per_gen_cap"], _sd["n_real"]),
                            it=b_subset(_c, "test", _sd["per_gen_cap"], _sd["n_real"]))
            _w = b_train(_name, "eca", _sigma, _s, _sd["max_steps"], eca={"lam_fr": _lf})
            _m = load_detector(_w, DEVICE)
            _fr = [b_counts_frame(_sub["cf"], _ix, b_certify(_m, _sub["cf"], _ix, _sigma, V3["N"]), _sn, V3["N"])
                   for _sn, _ix in [("cf_cal", _sub["ic"]), ("cf_test", _sub["it"])]]
            _pd8.concat(_fr).to_parquet(_cp)
            v_push(_cp)
            del _m
            gpu_free()
        _done.append(_name)
    return _done

def _flag_cost(df, s, ta, tf, e, l=(10.0, 0.5, 2.0, 0.2)):
    """C1b certified Yao cost bound on cf_test (all generators), plus certified flag rates."""
    _N = V3["N"]
    _t = df[df.set == "cf_test"]
    _lo, _hi, _ = v_scores(_t.k.to_numpy(), s)
    _y = _t.y.to_numpy()
    _lr, _hf = _lo[_y == 0], _hi[_y == 1]
    _k = _np8.arange(_N + 1)
    _L, _H, _ = v_scores(_k, s, _N)
    _ka, _kf = _k[_L >= ta + e], _k[_H < tf - e]
    _pacc = _binom8.sf(_ka.min() - 1, _N, v_norm.cdf((_lr - e) / s)) if len(_ka) else _np8.zeros(len(_lr))
    _pacc = _np8.where(_lr >= tf, _pacc, 0.0)
    _acc = _hf >= ta
    _pfl = _binom8.cdf(_kf.max(), _N, v_norm.cdf((_hf + e) / s)) if len(_kf) else _np8.zeros(len(_hf))
    _pfl = _np8.where(_acc, 0.0, _pfl)
    _cost = 0.5 * (l[0] * _acc.mean() + l[1] * _np8.mean(~_acc * (1 - _pfl))) + \
            0.5 * (l[2] * _np8.mean(_lr < tf) + l[3] * _np8.mean((_lr >= tf) * (1 - _pacc)))
    return float(_cost), float(_np8.mean(_pfl)), float(_np8.mean(_pacc))

def _c6d_table():
    _sigma = dict(V_MODELS)[V_BEST]
    _src = [("lam_fr=1 (full)", 0, f"B5_{V_BEST}_counts.parquet"), ("lam_fr=1 (full)", 1, f"B5_{V_BEST}_seed1_counts.parquet"),
            ("lam_fr=1 (full)", 2, f"B5_{V_BEST}_seed2_counts.parquet")]
    for _lf in _LAMS:
        _src += [(f"lam_fr={_lf:g}", 0, f"C6_{V_BEST}_lamfr{_lf:g}_counts.parquet")]
        _src += [(f"lam_fr={_lf:g}", _s, f"C6_{V_BEST}_lamfr{_lf:g}_seed{_s}_counts.parquet") for _s in _SEEDS8]
    _src += [("cohen", 0, f"C6_{n}_counts.parquet") for n, s in V_MODELS if n.startswith("cohen") and abs(s - _sigma) < 1e-9]
    _rows = []
    for _v, _sd, _f in _src:
        _p = v_path("gpu", _f)
        if not ensure(_p):
            continue
        _df = _pd8.read_parquet(_p)
        _fr = b_frontier_from_counts(_df, _sigma, V3["N"], f"{_v} s{_sd}")
        _cal = v_cal_parts(_df, _sigma)
        _ru = v_rule(_cal, V3["select_alpha"], V3["main_beta"])
        _r = {"variant": _v, "seed": _sd, "alpha_min(eps=sigma)": float(_fr["alpha_min(eps=sigma)"].iloc[0]),
              "tau_f(beta=5%)/sigma": _ru["tau_f"] / _sigma, "cap_minus_1 = z_N - 1": v_zN(V3["N"], V3["alpha_mc"]) - 1,
              "auroc": float(_fr["auroc_point"].iloc[0])}
        for _a in [0.02, 0.05]:
            _q = _fr[_np8.isclose(_fr.alpha, _a)]
            if len(_q):
                _r[f"accept_real a={_a}"] = float(_q.accept_real_eps_sigma.iloc[0])
                _r[f"AF a={_a}"] = float(_q.AF_eps_sigma.iloc[0])
                _r[f"event a={_a}"] = float(_q.guarantee_event.iloc[0])
        for _e in [0.5, 1.0]:
            _c, _pf, _pa = _flag_cost(_df, _sigma, _ru["tau_a"], _ru["tau_f"], _e * _sigma)
            _r[f"certified_fake_flag eps={_e}sigma"] = _pf
            _r[f"C1b_cost(10,.5,2,.2) eps={_e}sigma"] = _c
        _rows.append(_r)
    _T = _pd8.DataFrame(_rows)
    _num = [c for c in _T.columns if c not in ("variant", "seed")]
    _S = _T.groupby("variant")[_num].agg(["mean", "std", "count"])
    _S.columns = [f"{a} [{b}]" for a, b in _S.columns]
    _T.to_csv(v_path("tables", "C6d_lamfr_seeds.csv"), index=False)
    _S.to_csv(v_path("tables", "C6d_lamfr_seed_summary.csv"))
    v_push(v_path("tables"), "v3 C6d lam_fr seeds")
    return _T, _S

_C6D_DONE = b_step("C6d lam_fr seeds", _c6d_train) if V_BEST and V_BEST.startswith("eca") else "main model is not ECA"
_T6d, _S6d = _c6d_table()

In [ ]:
import os as _os9
# PRINT EVERYTHING: all A11 and Part C tables as plain text.
print("===== A11 / C tables =====")
for _f in sorted(_os9.listdir(v_path("tables"))):
    if (_f.startswith("C") or _f.startswith("A11")) and _f.endswith(".csv"):
        print(f"\n===== {_f} =====")
        print(open(v_path("tables", _f)).read())

## Part IV: paper figures and revision analyses

In [ ]:
import os as _osF, io as _ioF, zipfile as _zipF, traceback as _tbF
import numpy as _npF, pandas as _pdF
import matplotlib as _mplF
import matplotlib.pyplot as _pltF
# PAPER FIGURES + ONE ZIP OF EVERYTHING (CPU, ~1 min). Draws every paper figure from the saved tables
# (PDF + 300-dpi PNG in v3/paper_figs/), then zips the whole v3 folder (tables, figures, reports,
# manifest, vote counts) into eca3wd_all_artifacts.zip, pushes it to the HF results repo and offers a download.
_FD = v_path("paper_figs")
_osF.makedirs(_FD, exist_ok=True)
_C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]  # validated categorical order
_MK = ["o", "s", "^", "D", "v", "P"]
_LS = ["-", "--", "-.", ":", "-", "--"]
_INK, _INK2 = "#0b0b0b", "#52514e"
_RC = {"font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8,
       "ytick.labelsize": 8, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": _INK2,
       "axes.labelcolor": _INK, "xtick.color": _INK2, "ytick.color": _INK2, "axes.grid": True,
       "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "lines.linewidth": 2, "lines.markersize": 6,
       "legend.frameon": False, "axes.axisbelow": True, "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight"}

def _tab(name):
    _p = v_path("tables", name)
    return _pdF.read_csv(_p) if _osF.path.exists(_p) else None

def _save(fig, name):
    for _ext in ("pdf", "png"):
        fig.savefig(_osF.path.join(_FD, f"{name}.{_ext}"), dpi=300)
    _pltF.close(fig)
    return name

def _f_tradeoff():
    # Accept/flag trade-off law: one point per trained model (C6b, C6c, C6d), x = tau_f / sigma
    _fs = [t for t in [_tab("C6b_ablation_seeds.csv"), _tab("C6c_lamfr_sweep.csv"), _tab("C6d_lamfr_seeds.csv")] if t is not None]
    if not _fs:
        return None
    _T = _pdF.concat(_fs).drop_duplicates(subset=["variant", "seed", "tau_f(beta=5%)/sigma"])
    _cap = float(_T["cap_minus_1 = z_N - 1"].iloc[0])
    _T = _T.sort_values("tau_f(beta=5%)/sigma")
    _x = _T["tau_f(beta=5%)/sigma"].to_numpy()
    _fig, _ax = _pltF.subplots(1, 1, figsize=(5.2, 3.0))
    _ax.axvspan(_cap, max(_x.max(), _cap) + 0.15, color="#f0efec", zorder=0, lw=0)
    _ax.axvline(_cap, color=_INK2, lw=1, ls="--")
    _ax.text(_cap - 0.03, 0.5, "accept impossible →\nτ_f/σ > z_N − 1", fontsize=7, color=_INK2, va="center", ha="right")
    for _i, (_col, _lab) in enumerate([("accept_real a=0.05", "certified real acceptance, ε = σ"),
                                       ("certified_fake_flag eps=0.5sigma", "certified fake flagging, ε = σ/2"),
                                       ("certified_fake_flag eps=1.0sigma", "certified fake flagging, ε = σ")]):
        _ax.plot(_x, _T[_col].to_numpy(), ls="none", marker=_MK[_i], color=_C[_i], label=_lab,
                 markeredgecolor="white", markeredgewidth=0.8, ms=7)
    _ax.set_xlabel("flag threshold τ_f / σ  (β = 5%; one point per trained model)")
    _ax.set_ylabel("rate")
    _ax.set_ylim(-0.03, 1.03)
    _ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=7)
    return _save(_fig, "fig_tradeoff_law")

def _f_lamfr():
    _T = _tab("C6d_lamfr_seeds.csv")
    _c = _tab("C6c_lamfr_sweep.csv")
    if _T is None and _c is None:
        return None
    _A = _pdF.concat([t for t in [_T, _c] if t is not None]).drop_duplicates(subset=["variant", "seed", "tau_f(beta=5%)/sigma"])
    _A = _A[_A.variant.str.startswith("lam_fr=")]
    _A["lam"] = _A.variant.str.extract(r"lam_fr=([0-9.]+)")[0].astype(float)
    _g = _A.groupby("lam")
    _fig, _axs = _pltF.subplots(1, 3, figsize=(7.2, 2.4))
    for _ax, (_col, _lab) in zip(_axs, [("tau_f(beta=5%)/sigma", "τ_f / σ"), ("alpha_min(eps=sigma)", "α_min (ε = σ)"),
                                         ('C1b_cost(10,.5,2,.2) eps=0.5sigma', "certified Yao cost, ε = σ/2")]):
        _m, _s = _g[_col].mean(), _g[_col].std().fillna(0)
        _ax.errorbar(_m.index, _m.values, yerr=_s.values, color=_C[0], marker="o", capsize=3)
        _ax.scatter(_A.lam, _A[_col], color=_C[0], alpha=0.35, s=14, zorder=3)
        _ax.set_xscale("symlog", linthresh=1)
        _ax.set_xticks([0, 1, 2, 4, 8])
        _ax.set_xticklabels(["0", "1", "2", "4", "8"])
        _ax.set_xlabel("flag weight λ_fr")
        _ax.set_title(_lab)
    _axs[0].axhline(float(_A["cap_minus_1 = z_N - 1"].iloc[0]), color=_INK2, ls="--", lw=1)
    _fig.subplots_adjust(wspace=0.45)
    return _save(_fig, "fig_lamfr_knob")

def _f_evidence():
    _T = _tab("C7_evidence_floor_law.csv")
    if _T is None or not len(_T):
        return None
    _pc = [c for c in _T.columns if c.startswith("predicted_alpha_min")][0]
    _T = _T[_T.N.astype(str) != "infinity"].copy()
    _T["Nn"] = _T.N.astype(float)
    _fig, _ax = _pltF.subplots(figsize=(4.2, 2.8))
    for _i, _e in enumerate(sorted(_T.eps_over_sigma.unique())):
        _g = _T[_T.eps_over_sigma == _e]
        _ax.plot(_g.Nn, _g[_pc], color=_C[_i], ls=_LS[_i], marker=_MK[_i], label=f"plug-in prediction, ε = {_e:g}σ" if _e else "plug-in prediction, ε = 0")
        _gm = _g.dropna(subset=["measured_alpha_min"])
        if len(_gm):
            _ax.plot(_gm.Nn, _gm.measured_alpha_min, ls="none", marker="*", ms=11, color=_C[_i],
                     markeredgecolor=_INK, markeredgewidth=0.6)
    _ax.axhline(float(_T["floor alpha_inf"].iloc[0]), color=_INK2, ls="--", lw=1)
    _ax.text(_T.Nn.min(), float(_T["floor alpha_inf"].iloc[0]) * 1.005, "floor α_∞", fontsize=7, color=_INK2, va="bottom")
    _ax.axhline(0.01, color=_C[5], lw=1, ls=":")
    _ax.text(_T.Nn.min(), 0.0099, "α = 1% target", fontsize=7, color=_C[5], va="top")
    _ax.set_xscale("log")
    _ax.set_xlabel("noise draws N  (stars: measured)")
    _ax.set_ylabel("smallest feasible α")
    _ax.legend()
    return _save(_fig, "fig_evidence_floor")

def _f_sourcemass():
    _T = _tab("C5_source_mass_law.csv")
    if _T is None or not len(_T):
        return None
    _fig, _axs = _pltF.subplots(1, 2, figsize=(7.0, 2.6))
    for _i, (_m, _g) in enumerate(_T.groupby("model", sort=False)):
        _x = _npF.arange(len(_g))
        _axs[0].errorbar(_x, _g["GEN-CRC capped error (mean)"], yerr=_g["GEN-CRC capped error (sd)"], color=_C[_i % 6],
                         marker=_MK[_i % 6], ls=_LS[_i % 6], capsize=2, label=_m)
        _axs[1].plot(_x, _g["capped-weighted CRC feasible share"], color=_C[_i % 6], marker=_MK[_i % 6], ls=_LS[_i % 6], label=_m)
    _labs = [str(c).split(" ")[0] for c in _T.groupby("model", sort=False).get_group(_T.model.iloc[0]).cap]
    for _ax in _axs:
        _ax.set_xticks(range(len(_labs)))
        _ax.set_xticklabels(_labs)
        _ax.set_xlabel("per-generator image cap c")
    _axs[0].axhline(float(_T.alpha.iloc[0]), color=_INK2, ls="--", lw=1)
    _axs[0].set_title("image-level error, capped (GEN-CRC)")
    _axs[1].set_title("size-weighted CRC: feasible share of splits")
    _axs[1].legend()
    return _save(_fig, "fig_source_mass_law")

def _f_pooled():
    _hit = None
    ensure(v_path("tables", "A11_official_split_pooled_vs_gen_avg.csv"))
    for _f in sorted(_osF.listdir(v_path("tables"))):
        if _f.startswith("A11") and _f.endswith(".csv"):
            _t = _tab(_f)
            if _t is not None and {"AF_gen_avg", "AF_pooled", "AF_capped_pooled", "alpha", "eps_over_sigma"} <= set(_t.columns):
                _hit = _t
                break
    if _hit is None:
        return None
    _t = _hit[(_npF.isclose(_hit.alpha, 0.05)) & (_npF.isclose(_hit.eps_over_sigma, 1.0))]
    _fig, _ax = _pltF.subplots(figsize=(4.6, 2.6))
    _x = _npF.arange(len(_t))
    _w = 0.26
    for _i, (_col, _lab) in enumerate([("AF_gen_avg", "generator average (guaranteed)"),
                                       ("AF_capped_pooled", "per image, ≤100 per generator"),
                                       ("AF_pooled", "per image, uncapped")]):
        _ax.bar(_x + (_i - 1) * _w, _t[_col], width=_w - 0.02, color=_C[_i], label=_lab)
    _ax.axhline(0.05, color=_INK2, ls="--", lw=1)
    _ax.set_xticks(_x)
    _ax.set_xticklabels(_t.model)
    _ax.set_ylabel("accepted fakes (α = 5%, ε = σ)")
    _ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=7)
    return _save(_fig, "fig_pooled_vs_generator")

def _f_oneclass():
    _T = _tab("C8_one_class_collapse.csv")
    if _T is None or not len(_T):
        return None
    _fig, _axs = _pltF.subplots(1, 2, figsize=(7.0, 2.6))
    for _i, (_m, _g) in enumerate(_T.groupby("model", sort=False)):
        _cf, _of = _g[_g.set == "cf_test"], _g[_g.set == "openfake"]
        _axs[0].plot(_cf.beta, _cf.one_class_accept_real, color=_C[_i % 6], marker=_MK[_i % 6], ls=_LS[_i % 6], label=_m)
        _axs[1].plot(_of.beta, _of.one_class_event_gen_avg, color=_C[_i % 6], marker=_MK[_i % 6], ls=_LS[_i % 6], label=_m)
    _axs[0].set_title("CF reals accepted (one-class rule, ε = σ)")
    _axs[1].set_title("OpenFake: generator-avg fake event")
    _axs[1].axhline(0.05, color=_INK2, ls="--", lw=1)
    for _ax in _axs:
        _ax.set_xlabel("framing level β")
        _ax.set_ylim(-0.02, 1.05)
    _axs[1].legend()
    return _save(_fig, "fig_one_class_openfake")

def _f_openfake_m():
    _T = _tab("C4_openfake_adaptation.csv")
    if _T is None or not len(_T):
        return None
    _mod = V_BEST if V_BEST in set(_T.model) else _T.model.iloc[0]
    _t = _T[(_T.model == _mod) & _npF.isclose(_T.alpha, 0.05)]
    _fig, _axs = _pltF.subplots(1, 2, figsize=(7.0, 2.6))
    for _i, (_r, _g) in enumerate(_t.groupby("recipe", sort=False)):
        _axs[0].errorbar(_g.m_openfake_gens_in_calibration, _g.heldout_gen_avg_event, yerr=_g.event_sd, color=_C[_i],
                         marker=_MK[_i], ls=_LS[_i], capsize=2, label=_r)
        _axs[1].plot(_g.m_openfake_gens_in_calibration, _g.openfake_real_accept_eps_sigma, color=_C[_i], marker=_MK[_i], ls=_LS[_i], label=_r)
    _axs[0].axhline(0.05, color=_INK2, ls="--", lw=1)
    _axs[0].set_title(f"held-out OpenFake event ({_mod}, α = 5%)")
    _axs[1].set_title("OpenFake reals accepted, ε = σ")
    for _ax in _axs:
        _ax.set_xlabel("OpenFake generators added to calibration, m")
    _axs[1].legend()
    return _save(_fig, "fig_openfake_adaptation")

def _f_auroc():
    _T = _tab("C3_auroc_decomposition.csv")
    if _T is None or not len(_T):
        return None
    _cols = [c for c in ["all", "gens_n<=100", "gens_n>100_only", "mean_per_generator"] if c in _T.columns]
    _labs = {"all": "all images", "gens_n<=100": "generators ≤100 images", "gens_n>100_only": "3 large generators",
             "mean_per_generator": "mean per generator"}
    _fig, _ax = _pltF.subplots(figsize=(6.4, 2.6))
    _x = _npF.arange(len(_T))
    _w = 0.8 / len(_cols)
    for _i, _c in enumerate(_cols):
        _ax.bar(_x + (_i - (len(_cols) - 1) / 2) * _w, _T[_c], width=_w - 0.02, color=_C[_i], label=_labs[_c])
    _ax.set_xticks(_x)
    _ax.set_xticklabels(_T.model)
    _ax.set_ylim(max(0.0, float(_npF.nanmin(_T[_cols].to_numpy())) - 0.05), 1.0)
    _ax.set_ylabel("AUROC")
    _ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=7)
    return _save(_fig, "fig_auroc_strata")

def _f_pgdat():
    _T = _tab("C2_pgdat_stronger_attack.csv")
    if _T is None or not len(_T):
        return None
    _t = _T[_T.pgdat_variant == "conf"]
    _at = list(dict.fromkeys(_t.attack))
    _fig, _axs = _pltF.subplots(1, len(_at), figsize=(2.5 * len(_at), 2.5), sharey=True)
    _axs = _npF.atleast_1d(_axs)
    for _ax, _a in zip(_axs, _at):
        _g = _t[_t.attack == _a]
        for _i, (_c, _l) in enumerate([("pgdat_AF_B1_attack", "pgdat, B1 attack"), ("pgdat_AF_C2_stronger", "pgdat, stronger attack"),
                                       ("best_ECA_certified_AF_upper95", "ECA certified upper bound")]):
            _ax.plot(_g.clean_real_accept, _g[_c].clip(upper=1.0), color=_C[_i], marker=_MK[_i], ls=_LS[_i], label=_l)
        _ax.set_title(_a.replace("l2_", "l2 budget "))
        _ax.set_xlabel("clean real acceptance")
    _axs[0].set_ylabel("accepted fakes under attack")
    _axs[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    return _save(_fig, "fig_pgdat_vs_certified")

def _f_yao():
    _T = _tab("C1b_yao_head_to_head.csv")
    if _T is None or not len(_T):
        return None
    _ls = sorted(dict.fromkeys(_T["loss"]), key=lambda v: float(str(v).strip("()").split(",")[0]))
    _fig, _axs = _pltF.subplots(1, len(_ls), figsize=(2.3 * len(_ls), 2.5))
    _axs = _npF.atleast_1d(_axs)
    for _ax, _l in zip(_axs, _ls):
        _g = _T[_T["loss"] == _l].sort_values("l2_budget")
        for _i, (_c, _lab) in enumerate([("ECA_cost_upper_bound", "ECA (certified upper bound)"),
                                         ("Cohen_cost_upper_bound", "Cohen (certified upper bound)"),
                                         ("baseline_cost_lower_bound", "best uncertified Yao rule (lower bound)")]):
            if _c in _g:
                _ax.plot(_g.l2_budget, _g[_c], color=_C[_i], marker=_MK[_i], ls=_LS[_i], label=_lab)
        _ax.plot(_g.l2_budget, _g.defer_all_cost, color=_INK2, ls=":", lw=1.2, label="defer everything")
        _ax.set_yscale("log")
        _ax.yaxis.set_major_locator(_mplF.ticker.LogLocator(subs=(1.0, 2.0, 5.0)))
        _ax.yaxis.set_major_formatter(_mplF.ticker.FormatStrFormatter("%g"))
        _ax.yaxis.set_minor_formatter(_mplF.ticker.NullFormatter())
        _ax.set_title(f"losses {_l}", fontsize=7)
        _ax.set_xlabel("l2 budget")
    _axs[0].set_ylabel("expected three-way cost")
    _axs[-1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
    return _save(_fig, "fig_yao_cost")

_made, _failed = [], []
with _mplF.rc_context(_RC):
    for _fn in [_f_tradeoff, _f_lamfr, _f_evidence, _f_sourcemass, _f_pooled, _f_oneclass, _f_openfake_m,
                _f_auroc, _f_pgdat, _f_yao]:
        try:
            _r = _fn()
            (_made if _r else _failed).append(_r or f"{_fn.__name__}: table missing")
        except Exception as _e:
            _failed.append(f"{_fn.__name__}: {_e!r}")
            _tbF.print_exc()

# zip the whole v3 folder (everything this project produced on your side)
_root = v_path()
_zp = v_path("eca3wd_all_artifacts.zip")
with _zipF.ZipFile(_zp, "w", _zipF.ZIP_DEFLATED) as _z:
    for _dp, _, _fs in _osF.walk(_root):
        for _f in _fs:
            _full = _osF.path.join(_dp, _f)
            if _osF.path.abspath(_full) == _osF.path.abspath(_zp):
                continue
            _z.write(_full, _osF.path.join("v3", _osF.path.relpath(_full, _root)))
v_push(_FD, "v3 paper figures")
v_push(_zp)
print("figures saved:", _made)
print("figures skipped:", _failed)
print("zip:", _zp, f"{_osF.path.getsize(_zp) / 1e6:.1f} MB")
mo.download(data=open(_zp, "rb").read(), filename="eca3wd_all_artifacts.zip", label="Download all figures and results (zip)")

In [ ]:
import os as _osR, numpy as _npR, pandas as _pdR
from scipy.stats import binom as _binomR
from scipy.special import gammaln as _gamR
# R1. REVISION ANALYSES (CPU only, no training, no attacks).
# It re-analyses vote counts and attack outputs that already exist, answers the pre-review, and
# skips itself if its outputs are already saved (delete v3/tables/R1_*.csv to force a rerun).
#   R1a  frozen procedure: model + (alpha = beta) chosen on one half of the calibration generators,
#        final thresholds fitted on the OTHER half, evaluated once on the test set; plus 200 re-splits of
#        the generator pool into selection / final-calibration / test thirds.
#   R1b  Table 2 rows on the official split: threshold, proxy event, clean certified AF, plug-in AF,
#        attacked certified acceptance (saved SmoothAdv outputs), real acceptance; flag side likewise.
#   R1c  Yao cost: certified upper bound WITH the Monte Carlo failure term, 95% generator-cluster
#        bootstrap intervals, uncertified Yao rule, and the new validation-tuned FLAG/DEFER baselines.
#   R1d  test generators by size (theorem covers generators with >= min_per_gen images).
#   R1e  finite-N confident-error bound from the N = 1024 counts (unbiased estimate of E[p^N]).
_RB = 1000                      # bootstrap resamples
_RSPLITS = 200                  # re-splits for the frozen procedure
_RGRID = [0.01, 0.02, 0.05, 0.10]
_RACC_MIN = 0.80                # selection rule, fixed before this run: smallest alpha whose best model accepts >= 80% on the selection hold-out
_ROUT = {k: v_path("tables", f"R1_{k}.csv") for k in ["a_official", "a_resplits", "a_resplit_summary", "b_table2",
                                                        "b_attacked", "c_cost", "d_sizes", "e_floor"]}
_ramc = V3["alpha_mc"]
_rmin = V3["min_per_gen"]


def _r_parts(df, s, sets):
    """Per-generator S_lo / S_hi / point score of fakes (generators with >= min_per_gen fakes), reals' bounds."""
    _c = df[df.set.isin(sets)]
    _lo, _hi, _pt = v_scores(_c.k.to_numpy(), s)
    _y, _g = _c.y.to_numpy(), _c.model_name.to_numpy().astype(str)
    _gens = sorted(g for g in _npR.unique(_g[_y == 1]) if ((_y == 1) & (_g == g)).sum() >= _rmin)
    _m = {g: (_y == 1) & (_g == g) for g in _gens}
    _ri = _npR.where(_y == 0)[0]
    return {"gens": _gens, "hi": {g: _hi[_m[g]] for g in _gens}, "lo": {g: _lo[_m[g]] for g in _gens},
            "pt": {g: _pt[_m[g]] for g in _gens}, "lo_r": _lo[_ri], "hi_r": _hi[_ri]}


def _r_thr(P, gens, rmask, alpha, beta):
    _tc, _ = v_crc([P["hi"][g] for g in gens], alpha)
    _tf = v_flag_threshold(P["lo_r"][rmask], beta)
    return float(max(_tc, _tf)), float(_tf)


def _r_eval(P, gens, rmask, ta, tf, s):
    _ev = float(_npR.mean([_npR.mean(P["hi"][g] >= ta) for g in gens]))
    _af = float(_npR.mean([_npR.mean(P["lo"][g] >= ta + s) for g in gens]))
    _pool = _npR.concatenate([P["lo"][g] for g in gens])
    return {"event_gen_avg": _ev, "AF_cert_gen_avg": _af, "AF_cert_pooled": float(_npR.mean(_pool >= ta + s)),
            "accept_real_eps_sigma": float(_npR.mean(P["lo_r"][rmask] >= ta + s)),
            "accept_real_eps0": float(_npR.mean(P["lo_r"][rmask] >= ta)),
            "framing_event": float(_npR.mean(P["lo_r"][rmask] < tf)),
            "flagged_real_eps_sigma": float(_npR.mean(P["hi_r"][rmask] < tf - s))}


def _r_select(PM, gens, rmask, u_gen, u_real):
    """Selection rule on the selection part only: thresholds on sub-half a, acceptance at eps = sigma on sub-half b."""
    _ga = [g for g in gens if u_gen[g] < 0.5]
    _gb = [g for g in gens if u_gen[g] >= 0.5]
    _ra, _rb = rmask & (u_real < 0.5), rmask & (u_real >= 0.5)
    _tab = []
    for _al in _RGRID:
        for _n, (_P, _s) in PM.items():
            _ta, _tf = _r_thr(_P, _ga, _ra, _al, _al)
            _tab.append({"alpha": _al, "model": _n, "sigma": _s, "acc_b": float(_npR.mean(_P["lo_r"][_rb] >= _ta + _s))})
    _T = _pdR.DataFrame(_tab)
    for _al in _RGRID:
        _t = _T[_T.alpha == _al].sort_values(["acc_b", "sigma"], ascending=[False, False])
        if _t.acc_b.iloc[0] >= _RACC_MIN or _al == _RGRID[-1]:
            return _al, _t.model.iloc[0], _T
    return None


def _r1a():
    PM = {}
    for _n, _s in V_MODELS:
        PM[_n] = (_r_parts(v_counts(_n), _s, ["cf_cal"]), _s)
    _gens = sorted(set.intersection(*[set(v[0]["gens"]) for v in PM.values()]))  # same images for every model
    _nr = min(len(v[0]["lo_r"]) for v in PM.values())
    _sel = {g: stable_u("r1sel:" + str(g)) < 0.5 for g in _gens}
    _rsel = _npR.array([stable_u(f"r1selr:{i}") < 0.5 for i in range(_nr)])
    _ug = {g: stable_u("r1sub:" + str(g)) for g in _gens}
    _ur = _npR.array([stable_u(f"r1subr:{i}") for i in range(_nr)])
    _gS = [g for g in _gens if _sel[g]]
    _gF = [g for g in _gens if not _sel[g]]
    _al, _m, _ST = _r_select(PM, _gS, _rsel, _ug, _ur)
    _P, _s = PM[_m]
    _ta, _tf = _r_thr(_P, _gF, ~_rsel, _al, _al)
    _TP = _r_parts(v_counts(_m), _s, ["cf_test"])
    _rows = [{"protocol": "frozen: select on half S, calibrate on half F", "model": _m, "alpha=beta": _al,
              "K_select": len(_gS), "K_final": len(_gF), "n_real_final": int((~_rsel).sum()), "tau_a": _ta, "tau_f": _tf,
              **_r_eval(_TP, _TP["gens"], _npR.ones(len(_TP["lo_r"]), bool), _ta, _tf, _s)}]
    # reference: the published protocol (thresholds on the full calibration set, alpha = beta = 5%)
    _Pb, _sb = PM[V_BEST]
    _ta0, _tf0 = _r_thr(_Pb, _gens, _npR.ones(_nr, bool), 0.05, 0.05)
    _TB = _r_parts(v_counts(V_BEST), _sb, ["cf_test"])
    _rows.append({"protocol": "published: full calibration set", "model": V_BEST, "alpha=beta": 0.05, "K_select": _npR.nan,
                  "K_final": len(_gens), "n_real_final": _nr, "tau_a": _ta0, "tau_f": _tf0,
                  **_r_eval(_TB, _TB["gens"], _npR.ones(len(_TB["lo_r"]), bool), _ta0, _tf0, _sb)})
    # generator-cluster bootstrap for the frozen protocol
    _rng = _npR.random.default_rng(CFG["seed"] + 911)
    _G = _TP["gens"]
    _evg = _npR.array([_npR.mean(_TP["hi"][g] >= _ta) for g in _G])
    _afg = _npR.array([_npR.mean(_TP["lo"][g] >= _ta + _s) for g in _G])
    _acc = _TP["lo_r"] >= _ta + _s
    _bs = []
    for _ in range(_RB):
        _i = _rng.integers(0, len(_G), len(_G))
        _j = _rng.integers(0, len(_acc), len(_acc))
        _bs.append((_evg[_i].mean(), _afg[_i].mean(), _acc[_j].mean()))
    _bs = _npR.array(_bs)
    for _c, _k in zip(range(3), ["event_gen_avg", "AF_cert_gen_avg", "accept_real_eps_sigma"]):
        _rows[0][_k + " CI95"] = f"[{_npR.quantile(_bs[:, _c], 0.025):.4f}, {_npR.quantile(_bs[:, _c], 0.975):.4f}]"
    _A = _pdR.DataFrame(_rows)
    _ST.to_csv(v_path("tables", "R1_a_selection_table.csv"), index=False)
    # ---- 200 re-splits of the generator pool into thirds (selection / final calibration / test) ----
    PP = {_n: (_r_parts(v_counts(_n), _s, ["cf_cal", "cf_test"]), _s) for _n, _s in V_MODELS}
    _pg = sorted(set.intersection(*[set(v[0]["gens"]) for v in PP.values()]))
    _pr = min(len(v[0]["lo_r"]) for v in PP.values())
    _rng = _npR.random.default_rng(CFG["seed"] + 912)
    _R = []
    for _r in range(_RSPLITS):
        _gp = _rng.permutation(len(_pg))
        _t3 = len(_pg) // 3
        _GS = [_pg[i] for i in _gp[:_t3]]
        _GF = [_pg[i] for i in _gp[_t3:2 * _t3]]
        _GT = [_pg[i] for i in _gp[2 * _t3:]]
        _rp = _rng.permutation(_pr)
        _mS, _mF, _mT = (_npR.zeros(_pr, bool) for _ in range(3))
        _mS[_rp[:_pr // 3]] = True
        _mF[_rp[_pr // 3:2 * (_pr // 3)]] = True
        _mT[_rp[2 * (_pr // 3):]] = True
        _ug = {g: _rng.random() for g in _GS}
        _ur = _rng.random(_pr)
        _al, _m, _ = _r_select(PP, _GS, _mS, _ug, _ur)
        _P, _s = PP[_m]
        _ta, _tf = _r_thr(_P, _GF, _mF, _al, _al)
        _R.append({"resplit": _r, "model": _m, "alpha=beta": _al, **_r_eval(_P, _GT, _mT, _ta, _tf, _s)})
    _R = _pdR.DataFrame(_R)
    _R["event/alpha"] = _R["event_gen_avg"] / _R["alpha=beta"]
    _R["framing/beta"] = _R["framing_event"] / _R["alpha=beta"]
    _S = _R.groupby("alpha=beta").agg(n_splits=("resplit", "size"), mean_event=("event_gen_avg", "mean"),
                                      mean_event_over_alpha=("event/alpha", "mean"),
                                      share_splits_event_above_alpha=("event/alpha", lambda x: float((x > 1).mean())),
                                      mean_AF_cert=("AF_cert_gen_avg", "mean"), mean_accept_eps_sigma=("accept_real_eps_sigma", "mean"),
                                      mean_framing_over_beta=("framing/beta", "mean")).reset_index()
    _S = _pdR.concat([_S, _pdR.DataFrame([{"alpha=beta": "ALL", "n_splits": len(_R), "mean_event": _R.event_gen_avg.mean(),
                                           "mean_event_over_alpha": _R["event/alpha"].mean(),
                                           "share_splits_event_above_alpha": float((_R["event/alpha"] > 1).mean()),
                                           "mean_AF_cert": _R.AF_cert_gen_avg.mean(), "mean_accept_eps_sigma": _R.accept_real_eps_sigma.mean(),
                                           "mean_framing_over_beta": _R["framing/beta"].mean()}])])
    _S["models_chosen"] = str(_R.model.value_counts().to_dict())
    return _A, _R, _S


def _r_att(name):
    """Saved adaptive-attack vote counts (SmoothAdv, S9c audit run and B3 stress run) for one model."""
    _out = []
    for _cp, _tag in [(_osR.path.join(PATHS["res"], f"S9c_{name}_counts.parquet"), "S9c"),
                      (v_path("gpu", f"B3_{name}_counts.parquet"), "B3 stress")]:
        if ensure(_cp):
            _d = _pdR.read_parquet(_cp)
            _d["source"] = _tag
            _out.append(_d)
    return _pdR.concat(_out) if _out else None


def _r1b():
    _s = dict(V_MODELS)[V_BEST]
    _df = v_counts(V_BEST)
    _C = _r_parts(_df, _s, ["cf_cal"])
    _T = _r_parts(_df, _s, ["cf_test"])
    _te = _df[_df.set == "cf_test"]
    _lof, _hif, _ = v_scores(_te[_te.y == 1].k.to_numpy(), _s)
    _lor, _hir, _ = v_scores(_te[_te.y == 0].k.to_numpy(), _s)
    _al = 0.05
    _tf_ours = v_flag_threshold(_C["lo_r"], _al)
    _thr = {"GEN-CRC on S_hi (ours)": v_crc([_C["hi"][g] for g in _C["gens"]], _al)[0],
            "GEN-CRC on plug-in S": v_crc([_C["pt"][g] for g in _C["gens"]], _al)[0],
            "GEN-CRC on S_lo": v_crc([_C["lo"][g] for g in _C["gens"]], _al)[0]}
    _all = _npR.sort(_npR.concatenate([_C["hi"][g] for g in _C["gens"]]))[::-1]
    _km = int(_npR.floor(_al * (len(_all) + 1))) - 1
    _thr["image-level conformal on S_hi"] = _npR.inf if _km < 0 else float(_npR.nextafter(_all[_km], _npR.inf))
    _A = _r_att(V_BEST)
    _rows, _att = [], []
    for _meth, _t in _thr.items():
        _ta = _t  # the threshold of the row itself, as in A4 / Table 2 (max with tau_f would hide the differences)
        _r = {"side": "accept", "calibration": _meth, "threshold t": _t, "max(t, tau_f) as deployed": max(_t, _tf_ours),
              "proxy event P(S_hi(x) >= tau_a) [certified upper bound on attacked acceptance]": float(_npR.mean([_npR.mean(_T["hi"][g] >= _ta) for g in _T["gens"]])),
              "clean certified AF eps=0  P(S_lo >= tau_a)": float(_npR.mean([_npR.mean(_T["lo"][g] >= _ta) for g in _T["gens"]])),
              "clean certified AF eps=sigma  P(S_lo >= tau_a+sigma)": float(_npR.mean([_npR.mean(_T["lo"][g] >= _ta + _s) for g in _T["gens"]])),
              "plug-in AF  P(S_point >= tau_a)": float(_npR.mean([_npR.mean(_T["pt"][g] >= _ta) for g in _T["gens"]])),
              "real accept eps=sigma": float(_npR.mean(_T["lo_r"] >= _ta + _s))}
        _rows.append(_r)
        if _A is not None:
            for (_src, _key, _goal), _g in _A.groupby(["source", "attack", "goal"]):
                if not _goal.startswith("evasion") or "+" in _goal:
                    continue
                _nrm, _e, _r2 = v_parse_attack(_key)
                if _nrm != "l2" or _r2 > _s + 1e-12:
                    continue
                _lo, _hi, _ = v_scores(_g.k.to_numpy(), _s)
                _n = len(_lo)
                _acc = _lo >= _ta + _s
                _att.append({"calibration": _meth, "source": _src, "attack": _key, "goal": _goal, "n": _n, "eps_c": _s,
                             "attacked certified acceptance": float(_acc.mean()), "CI95": str(v_cp(int(_acc.sum()), _n)),
                             "clean certified acceptance, same images": float(_npR.mean(_lof[:_n] >= _ta + _s)),
                             "proxy P(S_hi(clean) >= tau_a), same images": float(_npR.mean(_hif[:_n] >= _ta))})
    for _meth, _arr in [("conformal on real S_lo (ours)", _C["lo_r"]), ("conformal on real S_hi", _C["hi_r"])]:
        _tf = v_flag_threshold(_arr, _al)
        _rows.append({"side": "flag", "calibration": _meth, "threshold t": _tf,
                      "proxy event P(S_lo(x) < tau_f) [certified upper bound on attacked framing]": float(_npR.mean(_T["lo_r"] < _tf)),
                      "clean certified framing eps=sigma  P(S_hi < tau_f-sigma)": float(_npR.mean(_T["hi_r"] < _tf - _s)),
                      "clean certified framing eps=0  P(S_hi < tau_f)": float(_npR.mean(_T["hi_r"] < _tf))})
        if _A is not None:
            for (_src, _key, _goal), _g in _A.groupby(["source", "attack", "goal"]):
                if not _goal.startswith("framing") or "+" in _goal:
                    continue
                _nrm, _e, _r2 = v_parse_attack(_key)
                if _nrm != "l2" or _r2 > _s + 1e-12:
                    continue
                _lo, _hi, _ = v_scores(_g.k.to_numpy(), _s)
                _fl = _hi < _tf - _s
                _att.append({"calibration": _meth, "source": _src, "attack": _key, "goal": _goal, "n": len(_hi), "eps_c": _s,
                             "attacked certified framing": float(_fl.mean()), "CI95": str(v_cp(int(_fl.sum()), len(_hi))),
                             "proxy P(S_lo(clean) < tau_f), same images": float(_npR.mean(_lor[:len(_hi)] < _tf))})
    return _pdR.DataFrame(_rows), _pdR.DataFrame(_att)


def _r_kgrid(s, N):
    _k = _npR.arange(N + 1)
    _L, _H, _ = v_scores(_k, s, N)
    return _k, _L, _H


def _r_cert_cost(lo_r, hi_f, g_f, ta, tf, e, s, l):
    """Per-image certified cost upper bounds (C1b, Proposition 6) plus the MC failure term alpha_mc * lambda_max."""
    _N = V3["N"]
    _k, _L, _H = _r_kgrid(s, _N)
    _ka, _kf = _k[_L >= ta + e], _k[_H < tf - e]
    _pacc = _binomR.sf(_ka.min() - 1, _N, v_norm.cdf((lo_r - e) / s)) if len(_ka) else _npR.zeros(len(lo_r))
    _pacc = _npR.where(lo_r >= tf, _pacc, 0.0)
    _pfl = _binomR.cdf(_kf.max(), _N, v_norm.cdf((hi_f + e) / s)) if len(_kf) else _npR.zeros(len(hi_f))
    _accf = hi_f >= ta
    _pfl = _npR.where(_accf, 0.0, _pfl)
    _cf = l[0] * _accf + l[1] * (~_accf) * (1 - _pfl) + _ramc * l[0]
    _cr = l[2] * (lo_r < tf) + l[3] * (lo_r >= tf) * (1 - _pacc) + _ramc * l[2]
    return _cf, _cr


def _r_boot(cf, gf, cr, rng):
    """95% interval of 0.5 mean(cf) + 0.5 mean(cr), resampling fake generators (clusters) and reals."""
    _gs = _npR.unique(gf)
    _sum = _npR.array([cf[gf == g].sum() for g in _gs])
    _cnt = _npR.array([(gf == g).sum() for g in _gs])
    _o = []
    for _ in range(_RB):
        _i = rng.integers(0, len(_gs), len(_gs))
        _j = rng.integers(0, len(cr), len(cr))
        _o.append(0.5 * _sum[_i].sum() / _cnt[_i].sum() + 0.5 * cr[_j].mean())
    return float(_npR.quantile(_o, 0.025)), float(_npR.quantile(_o, 0.975))


def _r_platt(s, y_real):
    from scipy.optimize import minimize as _min
    _s, _sg = _npR.asarray(s, float), 2.0 * _npR.asarray(y_real, float) - 1.0
    _sc = float(_npR.std(_s)) or 1.0
    _f = lambda th: float(_npR.sum(_npR.logaddexp(0.0, -_sg * (th[0] * _s / _sc + th[1]))))
    _th = _min(_f, _npR.zeros(2), method="L-BFGS-B").x
    return float(_th[0] / _sc), float(_th[1])


def _r1c():
    _rng = _npR.random.default_rng(CFG["seed"] + 913)
    _N = V3["N"]
    _rows = []
    _budgets = [0.06, 0.12, 0.25]
    # ---- certified smoothed models: (alpha, beta) by certified cost on calibration, bound on the evaluation half ----
    _cert = {}
    for _n, _s in V_MODELS:
        _df = v_counts(_n)
        _cal = v_cal_parts(_df, _s)
        _t = _df[_df.set == "cf_test"]
        _lo, _hi, _ = v_scores(_t.k.to_numpy(), _s)
        _y, _g = _t.y.to_numpy(), _t.model_name.to_numpy().astype(str)
        _ef, _er = c1_halves(_g[_y == 1], int((_y == 0).sum()))
        _calf = _npR.concatenate([_cal["hi"][g] for g in _cal["gens"]])
        _calg = _npR.concatenate([[g] * len(_cal["hi"][g]) for g in _cal["gens"]])
        _cert[_n] = dict(s=_s, cal=_cal, calf=_calf, calg=_calg, lor=_lo[_y == 0][_er], hif=_hi[_y == 1][_ef], gf=_g[_y == 1][_ef])
    for _l in C1_LOSSES:
        for _e in _budgets:
            _best = {}
            for _n, _c in _cert.items():
                _cands = [(a, b) for a in V3["alpha_grid"] for b in V3["beta_grid"]]
                _bb = None
                for _a, _b in _cands:
                    _ru = v_rule(_c["cal"], _a, _b)
                    _cf, _cr = _r_cert_cost(_c["cal"]["lo_r"], _c["calf"], _c["calg"], _ru["tau_a"], _ru["tau_f"], _e, _c["s"], _l)
                    _v = 0.5 * _cf.mean() + 0.5 * _cr.mean()
                    if _bb is None or _v < _bb[0]:
                        _bb = (_v, _a, _b, _ru)
                # flag/defer variant of the same model: tau_a = +inf, beta chosen on calibration
                _bf = None
                for _b in V3["beta_grid"]:
                    _tf = v_flag_threshold(_c["cal"]["lo_r"], _b)
                    _cf, _cr = _r_cert_cost(_c["cal"]["lo_r"], _c["calf"], _c["calg"], _npR.inf, _tf, _e, _c["s"], _l)
                    _v = 0.5 * _cf.mean() + 0.5 * _cr.mean()
                    if _bf is None or _v < _bf[0]:
                        _bf = (_v, _b, _tf)
                _best[_n] = (_bb, _bf)
            for _fam in ["eca", "cohen"]:
                _ns = [n for n in _cert if n.startswith(_fam)]
                if not _ns:
                    continue
                _nb = min(_ns, key=lambda n: _best[n][0][0])
                _c = _cert[_nb]
                (_vc, _a, _b, _ru), (_vf, _bfl, _tff) = _best[_nb]
                for _kind, _ta, _tf, _desc in [("three-way (alpha, beta from calibration)", _ru["tau_a"], _ru["tau_f"], f"alpha={_a}, beta={_b}"),
                                                ("flag/defer only (tau_a = inf, beta from calibration)", _npR.inf, _tff, f"beta={_bfl}")]:
                    _cf, _cr = _r_cert_cost(_c["lor"], _c["hif"], _c["gf"], _ta, _tf, _e, _c["s"], _l)
                    _ci = _r_boot(_cf, _c["gf"], _cr, _rng)
                    _rows.append({"loss": str(_l), "l2_budget": _e, "method": f"{_nb} certified {_kind}", "setting": _desc,
                                  "bound": "UPPER (any l2 attack <= budget, incl. MC failure term)",
                                  "cost": 0.5 * _cf.mean() + 0.5 * _cr.mean(), "CI95_low": _ci[0], "CI95_high": _ci[1],
                                  "MC failure term": 0.5 * _ramc * (_l[0] + _l[2])})
            _rows.append({"loss": str(_l), "l2_budget": _e, "method": "defer everything", "setting": "", "bound": "exact",
                          "cost": 0.5 * _l[1] + 0.5 * _l[3], "CI95_low": _npR.nan, "CI95_high": _npR.nan, "MC failure term": 0.0})
    # ---- uncertified baselines: Yao three-way rule and the validation-tuned flag/defer rule, attacked lower bounds ----
    for _bn in [b for b in V_BASELINES if b != "clip"]:
        _sp = _osR.path.join(PATHS["res"], f"S6_{_bn}_scores.parquet")
        _ap = v_path("gpu", f"B1_{_bn}.parquet")
        if not (_osR.path.exists(_sp) and ensure(_ap)):
            print(f"[R1c] {_bn}: scores or B1 attacks missing, skipped")
            continue
        _sc = _pdR.read_parquet(_sp)
        _te = _sc[_sc.set == "cf_test"]
        _f, _r = _te[_te.y == 1], _te[_te.y == 0]
        _gf = _f.model_name.to_numpy().astype(str)
        _ef, _er = c1_halves(_gf, len(_r))
        _sf, _sr = _f.s.to_numpy(), _r.s.to_numpy()
        _pa, _pb = _r_platt(_npR.concatenate([_sf[~_ef], _sr[~_er]]), _npR.concatenate([_npR.zeros((~_ef).sum()), _npR.ones((~_er).sum())]))
        _A = _pdR.read_parquet(_ap)
        for _l in C1_LOSSES:
            _ya = (_l[0] - _l[1]) / ((_l[0] - _l[1]) + _l[3])
            _yb = _l[1] / (_l[1] + (_l[2] - _l[3]))
            _ta = (_npR.log(_ya / (1 - _ya)) - _pb) / _pa
            _tfy = (_npR.log(_yb / (1 - _yb)) - _pb) / _pa
            # flag/defer threshold tuned on the FIT half: clean cost, and cost under the fit half's own attacks
            _cand = _npR.unique(_npR.concatenate([_sf[~_ef], _sr[~_er]]))
            _cand = _npR.quantile(_cand, _npR.linspace(0, 1, 400)) if len(_cand) > 400 else _cand

            def _fd_cost(tf, sf, sr):
                return 0.5 * _l[1] * _npR.mean(sf > tf) + 0.5 * (_l[2] * _npR.mean(sr <= tf) + _l[3] * _npR.mean(sr > tf))
            _tf_clean = min(_cand, key=lambda t: _fd_cost(t, _sf[~_ef], _sr[~_er]))
            for _e in _budgets:
                _ge = _A[(_A.attack == f"l2_{_e}") & (_A.goal == "evasion") & (_A.variant == "conf")]
                _gfr = _A[(_A.attack == f"l2_{_e}") & (_A.goal == "framing") & (_A.variant == "conf")]
                if not len(_ge) or not len(_gfr):
                    continue
                _n1, _n2 = len(_ge), len(_gfr)
                _sa = _npR.maximum(_sf[:_n1], _ge.s_adv.to_numpy())
                _sb = _npR.minimum(_sr[:_n2], _gfr.s_adv.to_numpy())
                _m1, _m2 = _ef[:_n1], _er[:_n2]
                _tf_rob = min(_cand, key=lambda t: _fd_cost(t, _sa[~_m1], _sb[~_m2]))
                _g1 = _gf[:_n1][_m1]
                # Yao three-way (lower bound: deferral costs dropped under attack, as in C1)
                _cfy = _l[0] * (_sa[_m1] >= _ta)
                _cry = _l[2] * (_sb[_m2] <= _tfy)
                _ci = _r_boot(_cfy.astype(float), _g1, _cry.astype(float), _rng)
                _rows.append({"loss": str(_l), "l2_budget": _e, "method": f"{_bn} uncertified Yao three-way (Platt)", "setting": f"a={_ya:.3f}, b={_yb:.3f}",
                              "bound": "LOWER (observed adaptive attack)", "cost": 0.5 * _cfy.mean() + 0.5 * _cry.mean(),
                              "CI95_low": _ci[0], "CI95_high": _ci[1], "MC failure term": 0.0})
                for _tag, _tfd in [("tuned on fit half, clean", _tf_clean), ("tuned on fit half, under attack", _tf_rob)]:
                    _cff = _l[1] * (_sa[_m1] > _tfd)
                    _crf = _l[2] * (_sb[_m2] <= _tfd) + _l[3] * (_sb[_m2] > _tfd)
                    _ci = _r_boot(_cff.astype(float), _g1, _crf.astype(float), _rng)
                    _rows.append({"loss": str(_l), "l2_budget": _e, "method": f"{_bn} uncertified flag/defer", "setting": _tag,
                                  "bound": "LOWER (observed adaptive attack)", "cost": 0.5 * _cff.mean() + 0.5 * _crf.mean(),
                                  "CI95_low": _ci[0], "CI95_high": _ci[1], "MC failure term": 0.0})
    return _pdR.DataFrame(_rows)


def _r1d():
    _s = dict(V_MODELS)[V_BEST]
    _df = v_counts(V_BEST)
    _C = _r_parts(_df, _s, ["cf_cal"])
    _ta, _tf = _r_thr(_C, _C["gens"], _npR.ones(len(_C["lo_r"]), bool), 0.05, 0.05)
    _t = _df[(_df.set == "cf_test") & (_df.y == 1)]
    _lo, _hi, _ = v_scores(_t.k.to_numpy(), _s)
    _g = _t.model_name.to_numpy().astype(str)
    _n = _pdR.Series(_g).value_counts()
    _rows = []
    for _lab, _sel in [(f"test generators with >= {_rmin} images (theorem's population)", _n[_n >= _rmin].index),
                       (f"test generators with < {_rmin} images", _n[_n < _rmin].index),
                       ("all test generators", _n.index)]:
        _sel = list(_sel)
        if not _sel:
            _rows.append({"stratum": _lab, "n_generators": 0})
            continue
        _m = _npR.isin(_g, _sel)
        _rows.append({"stratum": _lab, "n_generators": len(_sel), "n_images": int(_m.sum()),
                      "AF_cert_gen_avg eps=sigma": float(_npR.mean([_npR.mean(_lo[_g == x] >= _ta + _s) for x in _sel])),
                      "event_gen_avg": float(_npR.mean([_npR.mean(_hi[_g == x] >= _ta) for x in _sel])),
                      "AF_cert_pooled eps=sigma": float(_npR.mean(_lo[_m] >= _ta + _s))})
    _sz = [len(_C["hi"][g]) for g in _C["gens"]]
    _rows.append({"stratum": "calibration generators (sizes: min %d, max %d)" % (min(_sz), max(_sz)), "n_generators": len(_sz),
                  "n_images": int(sum(_sz))})
    return _pdR.DataFrame(_rows)


def _r1e():
    _p = v_path("gpu", f"B4_{V_BEST}_counts.parquet")
    if not ensure(_p):
        return _pdR.DataFrame([{"note": "B4 counts missing"}])
    _C = _pdR.read_parquet(_p)
    _s = dict(V_MODELS)[V_BEST]
    _Nb = int(_C.N.max())
    _rows = []
    _cb = _C[(_C.N == _Nb) & (_C.set == "cf_cal") & (_C.y == 1)]
    _gens = [g for g, n in _cb.model_name.value_counts().items() if n >= min(_rmin, V3["bigN"]["per_gen_cap"])]
    _K = len(_gens)

    def _lc(a, b):
        return _gamR(a + 1) - _gamR(b + 1) - _gamR(a - b + 1)
    for _N in [64, 128, 256, 512, 1024]:
        # unbiased estimate of E[p^N] from N_b draws: C(k, N) / C(N_b, N)
        _mg = []
        for _g in _gens:
            _k = _cb[_cb.model_name == _g].k.to_numpy().astype(float)
            _v = _npR.where(_k >= _N, _npR.exp(_lc(_k, _N) - _lc(float(_Nb), _N)), 0.0)
            _mg.append(_v.mean())
        _m = float(_npR.mean(_mg))
        _row = {"N": _N, "estimated m_N = mean_g E[p^N] (unbiased, from N=%d counts)" % _Nb: _m,
                "finite-N lower bound on E[alpha_min] = K/(K+1) m_N + 1/(K+1)": _K / (_K + 1) * _m + 1 / (_K + 1), "K": _K}
        if _N in [int(n) for n in _C.N.unique()]:
            _cc = _C[(_C.N == _N) & (_C.set == "cf_cal")]
            _cal = v_cal_parts(_cc, _s, _N)
            _zN = v_zN(_N, _ramc)
            _ff = _cc[_cc.y == 1]
            _row["measured all-votes share (k = N)"] = float(_npR.mean([_npR.mean(_ff[_ff.model_name == g].k.to_numpy() == _N) for g in _gens]))
            _row["measured alpha_min(eps=sigma)"] = v_risk([_cal["hi"][g] for g in _cal["gens"]], _s * _zN - _s)
            _row["measured alpha_min(eps=0)"] = v_risk([_cal["hi"][g] for g in _cal["gens"]], _s * _zN)
        _rows.append(_row)
    return _pdR.DataFrame(_rows)


if all(_osR.path.exists(p) or ensure(p) for p in _ROUT.values()):
    print("[R1] outputs already saved: loading them (delete v3/tables/R1_*.csv to recompute)")
    _RES = {k: _pdR.read_csv(p) for k, p in _ROUT.items()}
else:
    _a1, _a2, _a3 = _r1a()
    _b1, _b2 = _r1b()
    _RES = {"a_official": _a1, "a_resplits": _a2, "a_resplit_summary": _a3, "b_table2": _b1, "b_attacked": _b2,
            "c_cost": _r1c(), "d_sizes": _r1d(), "e_floor": _r1e()}
    for _k, _v in _RES.items():
        _v.to_csv(_ROUT[_k], index=False)
    v_push(v_path("tables"), "v3 R1 revision analyses")
for _k in ["a_official", "a_resplit_summary", "b_table2", "b_attacked", "c_cost", "d_sizes", "e_floor"]:
    print(f"===== R1_{_k} =====")
    print(_RES[_k].to_csv(index=False))